# MEA Visual Stimuli v2.1.3

Scientific visual-stimulation software for MEA experiments, built around a PsychoPy presentation worker, reproducible stimulus protocols, timing logs, and optional TTL synchronization.

## Current version

**v2.1.3**

The v2.1.x branch keeps the validated stimulation/timing architecture of v2.0 while improving the bilingual Spanish/English operator interface and translation coverage. English is the default language for new installations; saved protocol identifiers and internal stimulus values remain unchanged for compatibility.

## Documentation

The offline technical/user manual distributed with the application is:

`docs/MEA_Visual_Stimuli_Manual_v2_0.pdf`

The v2.0 manual documents the validated experimental architecture used by the v2.1.x branch. See the repository `CHANGELOG.md` for later interface and maintenance changes.

## Source layout

This notebook contains the same application source published in the accompanying `.py` export. The notebook is kept for development continuity; the `.py` file is easier to inspect and diff on GitHub.

## Validation scope

Timing and TTL validation documented in the v2.0 manual applies to the specific hardware/software setup described there. Any different monitor, FTDI device, cabling, acquisition hardware, operating-system configuration, or display calibration should be validated locally before experimental use.

See `CHANGELOG.md` for version history.


In [ ]:

import json
import mmap
import os
import subprocess
import sys
import tempfile
import time
import math
import random
import atexit
from datetime import datetime
from pathlib import Path
import tkinter as tk
from tkinter import ttk, messagebox, colorchooser, simpledialog, filedialog

APP_VERSION = "2.1.3"
MANUAL_FILENAME = "MEA_Visual_Stimuli_Manual_v2_0.pdf"

TRIGGER_SOURCE_EMBEDDED = 'class MaxOneFTDI:\n    """Shared FTDI backend for the operator diagnostic and display worker.\n\n    Logical signals use the validated inverted VisExpMan polarity.\n    USB/driver latency is measured by the experiment, not assumed zero.\n    """\n\n    FRAME_PULSE_MIN_S = 0.001\n\n    def __init__(self, port="auto"):\n        self.requested_port = str(port or "auto").strip() or "auto"\n        self.resolved_port = ""\n        self.serial = None\n        self._frame_high_at = None\n\n    def open(self):\n        if self.serial is not None:\n            raise RuntimeError("El puerto FTDI ya está abierto.")\n        try:\n            import serial\n        except ImportError as exc:\n            raise RuntimeError(\n                "MaxOne Legacy FTDI requiere PySerial. Instálalo en el "\n                "entorno de este notebook: %pip install pyserial"\n            ) from exc\n\n        port = self.requested_port\n        if port.lower() in ("auto", "automatico", "automático"):\n            from serial.tools import list_ports\n            matches = [\n                p for p in list_ports.comports()\n                if getattr(p, "vid", None) == 0x0403\n                and getattr(p, "pid", None) == 0x6001\n            ]\n            if not matches:\n                raise RuntimeError("No se encuentra el FTDI MaxOne (0403:6001).")\n            if len(matches) != 1:\n                devices = ", ".join(str(p.device) for p in matches)\n                raise RuntimeError(\n                    f"Hay varios FTDI 0403:6001 ({devices}). "\n                    "Indica el puerto del montaje en el campo Puerto."\n                )\n            port = str(matches[0].device)\n\n        try:\n            # Configure the requested rest state before opening where the\n            # driver permits it; reapply both outputs immediately afterwards.\n            self.serial = serial.Serial(\n                port=None, baudrate=9600, timeout=0, write_timeout=0,\n                rtscts=False, dsrdtr=False, xonxoff=False,\n            )\n            self.serial.rts = True\n            self.serial.break_condition = True\n            self.serial.port = port\n            self.serial.open()\n            self.resolved_port = port\n            self.idle(strict=True)\n        except Exception as exc:\n            self.close()\n            raise RuntimeError(\n                f"No se pudo abrir/configurar {port}: {type(exc).__name__}: {exc}"\n            ) from exc\n        return self.resolved_port\n\n    def _set(self, signal, logical):\n        if self.serial is None:\n            raise RuntimeError("El backend MaxOne FTDI no está abierto.")\n        physical = not bool(logical)\n        if signal == "frame":\n            self.serial.rts = physical\n        else:\n            self.serial.break_condition = physical\n\n    def set_frame(self, logical):\n        self._set("frame", logical)\n        self._frame_high_at = _ftdi_time.perf_counter() if logical else None\n\n    def set_event(self, logical):\n        self._set("event", logical)\n\n    def frame_high(self):\n        self.set_frame(1)\n\n    def frame_low(self):\n        # Keep the requested RTS HIGH for at least 1 ms after the driver call.\n        # A bounded short wait avoids scheduling a Timer across later frames.\n        # Physical width can be longer; validate it in /bits on the real setup.\n        if self._frame_high_at is not None:\n            deadline = self._frame_high_at + self.FRAME_PULSE_MIN_S\n            while _ftdi_time.perf_counter() < deadline:\n                pass\n        self.set_frame(0)\n\n    def idle(self, strict=False):\n        errors = []\n        if self.serial is not None:\n            # Attempt BOTH lines even if one write fails.\n            for name, action in (("FRAME", self.set_frame), ("EVENT", self.set_event)):\n                try:\n                    action(0)\n                except Exception as exc:\n                    errors.append(f"{name}: {type(exc).__name__}: {exc}")\n        self._frame_high_at = None\n        if errors and strict:\n            raise RuntimeError("; ".join(errors))\n        return errors\n\n    def close(self):\n        errors = self.idle()\n        ser, self.serial = self.serial, None\n        if ser is not None:\n            try:\n                ser.close()\n            except Exception as exc:\n                errors.append(f"Cerrar FTDI: {type(exc).__name__}: {exc}")\n        return errors\n\n\nclass GenericTTLFTDI:\n    """Shared FTDI backend for the generic trigger box and display worker.\n\n    Validated hardware mapping (active-low FTDI control-line polarity):\n      UNIVERSAL / FRAME_SYNC -> RTS#  (ser.rts = False)\n      AUX EVENT              -> TX/BREAK (break_condition = False)\n\n    Encoding on UNIVERSAL is generated in software:\n      visual flip -> FRAME pulse on RTS\n      condition onset -> RTS EVENT pulse ~3 ms later + simultaneous BREAK EVENT\n    The EVENT replication onto RTS uses _set() directly so it never overwrites\n    the visual FRAME anchor used for timing and logging.\n    """\n\n    FRAME_PULSE_MIN_S = 0.001\n    EVENT_PULSE_MIN_S = 0.001\n    EVENT_OFFSET_S = 0.003\n\n    def __init__(self, port="auto"):\n        self.requested_port = str(port or "auto").strip() or "auto"\n        self.resolved_port = ""\n        self.serial = None\n        self._frame_high_at = None\n        self._last_frame_anchor = None\n        self._event_high_at = None\n\n    @staticmethod\n    def _wait_until(deadline):\n        """Hybrid sleep/spin wait, matching the validated UNIVERSAL tester."""\n        while True:\n            remaining = float(deadline) - _ftdi_time.perf_counter()\n            if remaining <= 0:\n                return\n            if remaining > 0.002:\n                _ftdi_time.sleep(max(0.0, remaining - 0.001))\n            # Final sub-millisecond section intentionally busy-waits.\n\n    def open(self):\n        if self.serial is not None:\n            raise RuntimeError("El puerto FTDI ya está abierto.")\n        try:\n            import serial\n        except ImportError as exc:\n            raise RuntimeError(\n                "Trigger Box TTL genérica requiere PySerial. Instálalo en el "\n                "entorno de este notebook: %pip install pyserial"\n            ) from exc\n\n        port = self.requested_port\n        if port.lower() in ("auto", "automatico", "automático"):\n            from serial.tools import list_ports\n            matches = [\n                p for p in list_ports.comports()\n                if getattr(p, "vid", None) == 0x0403\n                and getattr(p, "pid", None) == 0x6001\n            ]\n            if not matches:\n                raise RuntimeError("No se encuentra el FTDI de la Trigger Box (0403:6001).")\n            if len(matches) != 1:\n                devices = ", ".join(str(p.device) for p in matches)\n                raise RuntimeError(\n                    f"Hay varios FTDI 0403:6001 ({devices}). "\n                    "Indica manualmente el puerto de la Trigger Box."\n                )\n            port = str(matches[0].device)\n\n        try:\n            self.serial = serial.Serial(\n                port=None, baudrate=9600, timeout=0, write_timeout=0,\n                rtscts=False, dsrdtr=False, xonxoff=False,\n            )\n            # Validated idle polarity: both FTDI lines physically inactive.\n            self.serial.rts = True\n            self.serial.break_condition = True\n            self.serial.port = port\n            self.serial.open()\n            self.resolved_port = port\n            self.idle(strict=True)\n        except Exception as exc:\n            self.close()\n            raise RuntimeError(\n                f"No se pudo abrir/configurar {port}: {type(exc).__name__}: {exc}"\n            ) from exc\n        return self.resolved_port\n\n    def _set(self, signal, logical):\n        if self.serial is None:\n            raise RuntimeError("El backend FTDI genérico no está abierto.")\n        physical = not bool(logical)\n        if signal == "frame":\n            self.serial.rts = physical\n        elif signal == "event":\n            self.serial.break_condition = physical\n        else:\n            raise ValueError(f"Señal FTDI desconocida: {signal}")\n\n    def set_frame(self, logical):\n        self._set("frame", logical)\n        now = _ftdi_time.perf_counter()\n        if logical:\n            self._frame_high_at = now\n            self._last_frame_anchor = now\n        else:\n            self._frame_high_at = None\n        return now\n\n    def set_event(self, logical):\n        self._set("event", logical)\n        now = _ftdi_time.perf_counter()\n        self._event_high_at = now if logical else None\n        return now\n\n    def frame_high(self):\n        return self.set_frame(1)\n\n    def frame_low(self):\n        # Keep FRAME HIGH for at least 1 ms. The validated hardware produced\n        # ~1.1-1.35 ms physical pulses with this requested width.\n        if self._frame_high_at is not None:\n            self._wait_until(self._frame_high_at + self.FRAME_PULSE_MIN_S)\n        return self.set_frame(0)\n\n    def _event_pair_high(self):\n        """Assert EVENT on AUX EVENT and replicate it onto UNIVERSAL."""\n        if self.serial is None:\n            raise RuntimeError("El backend FTDI genérico no está abierto.")\n        errors = []\n        # RTS carries UNIVERSAL. BREAK carries the independent AUX EVENT.\n        # Use _set() directly for RTS so the visual FRAME anchor is NOT changed.\n        for signal in ("frame", "event"):\n            try:\n                self._set(signal, 1)\n            except Exception as exc:\n                errors.append(f"{signal}: {type(exc).__name__}: {exc}")\n        if errors:\n            # Best-effort fail-safe: release both outputs before propagating.\n            for signal in ("event", "frame"):\n                try:\n                    self._set(signal, 0)\n                except Exception:\n                    pass\n            raise RuntimeError("; ".join(errors))\n        now = _ftdi_time.perf_counter()\n        self._event_high_at = now\n        return now\n\n    def _event_pair_low(self):\n        """Release AUX EVENT and UNIVERSAL after an EVENT pulse."""\n        errors = []\n        # Release UNIVERSAL first, then AUX EVENT; both operations are attempted.\n        for signal in ("frame", "event"):\n            try:\n                self._set(signal, 0)\n            except Exception as exc:\n                errors.append(f"{signal}: {type(exc).__name__}: {exc}")\n        now = _ftdi_time.perf_counter()\n        self._event_high_at = None\n        if errors:\n            raise RuntimeError("; ".join(errors))\n        return now\n\n    def pulse_event_now(self, width_s=None):\n        """Generate EVENT simultaneously on UNIVERSAL (RTS) and AUX EVENT (BREAK)."""\n        width = self.EVENT_PULSE_MIN_S if width_s is None else max(\n            self.EVENT_PULSE_MIN_S, float(width_s)\n        )\n        high_at = self._event_pair_high()\n        try:\n            self._wait_until(high_at + width)\n        finally:\n            low_at = self._event_pair_low()\n        return {\n            "event_high_perf_s": high_at,\n            "event_low_perf_s": low_at,\n            "event_width_s": max(0.0, low_at - high_at),\n        }\n\n    def pulse_event_after_frame(self, offset_s=None, width_s=None):\n        """Emit EVENT as the second UNIVERSAL pulse after the last visual FRAME."""\n        if self._last_frame_anchor is None:\n            raise RuntimeError(\n                "No hay un FRAME_SYNC previo al que asociar el EVENT."\n            )\n        offset = self.EVENT_OFFSET_S if offset_s is None else max(0.0, float(offset_s))\n        width = self.EVENT_PULSE_MIN_S if width_s is None else max(\n            self.EVENT_PULSE_MIN_S, float(width_s)\n        )\n        frame_anchor = float(self._last_frame_anchor)\n        self._wait_until(frame_anchor + offset)\n\n        # IMPORTANT: this pulses RTS directly instead of set_frame(), so the\n        # original visual FRAME anchor remains unchanged for timing/logging.\n        high_at = self._event_pair_high()\n        try:\n            self._wait_until(high_at + width)\n        finally:\n            low_at = self._event_pair_low()\n\n        return {\n            "frame_anchor_perf_s": frame_anchor,\n            "event_high_perf_s": high_at,\n            "event_low_perf_s": low_at,\n            "event_offset_s": max(0.0, high_at - frame_anchor),\n            "event_width_s": max(0.0, low_at - high_at),\n        }\n\n    def pulse_universal_event_after_frame(self, aux_state, offset_s=None, width_s=None):\n        """Pulse only UNIVERSAL ~3 ms after FRAME while setting AUX EVENT state.\n\n        This is the v1.27 experimental encoding used by the display worker:\n        RTS/UNIVERSAL keeps the validated short EVENT pulse, whereas TX/BREAK\n        becomes a held condition-state channel. The visual FRAME anchor is never\n        overwritten.\n        """\n        if self._last_frame_anchor is None:\n            raise RuntimeError(\n                "No hay un FRAME_SYNC previo al que asociar el EVENT."\n            )\n        offset = self.EVENT_OFFSET_S if offset_s is None else max(0.0, float(offset_s))\n        width = self.EVENT_PULSE_MIN_S if width_s is None else max(\n            self.EVENT_PULSE_MIN_S, float(width_s)\n        )\n        frame_anchor = float(self._last_frame_anchor)\n        desired_aux = 1 if int(aux_state) else 0\n        self._wait_until(frame_anchor + offset)\n\n        universal_high_at = None\n        aux_transition_at = None\n        try:\n            # UNIVERSAL receives the precise second pulse; use _set() directly so\n            # the FRAME anchor remains the original visual flip.\n            self._set("frame", 1)\n            universal_high_at = _ftdi_time.perf_counter()\n            # AUX EVENT is a held state, changed at the same condition onset.\n            self._set("event", desired_aux)\n            aux_transition_at = _ftdi_time.perf_counter()\n            self._event_high_at = aux_transition_at if desired_aux else None\n            self._wait_until(universal_high_at + width)\n        finally:\n            # Release ONLY UNIVERSAL. AUX EVENT deliberately remains at its\n            # requested condition state until the next onset or explicit reset.\n            self._set("frame", 0)\n            universal_low_at = _ftdi_time.perf_counter()\n\n        return {\n            "frame_anchor_perf_s": frame_anchor,\n            "event_high_perf_s": universal_high_at,\n            "event_low_perf_s": universal_low_at,\n            "event_offset_s": max(0.0, universal_high_at - frame_anchor),\n            "event_width_s": max(0.0, universal_low_at - universal_high_at),\n            "aux_transition_perf_s": aux_transition_at,\n            "aux_state": desired_aux,\n        }\n\n    def pulse_aux_marker(self, width_s=None):\n        """Emit a short marker only on AUX EVENT (TX/BREAK).\n\n        UNIVERSAL/RTS is deliberately untouched. This is used for protocol\n        pause markers in v1.28. AUX always returns LOW when the pulse ends.\n        """\n        width = self.EVENT_PULSE_MIN_S if width_s is None else max(\n            self.EVENT_PULSE_MIN_S, float(width_s)\n        )\n        # Start from a defined LOW state, then generate a clean HIGH pulse.\n        self._set("event", 0)\n        self._event_high_at = None\n        self._set("event", 1)\n        high_at = _ftdi_time.perf_counter()\n        self._event_high_at = high_at\n        try:\n            self._wait_until(high_at + width)\n        finally:\n            self._set("event", 0)\n            low_at = _ftdi_time.perf_counter()\n            self._event_high_at = None\n        return {\n            "aux_high_perf_s": high_at,\n            "aux_low_perf_s": low_at,\n            "aux_width_s": max(0.0, low_at - high_at),\n        }\n\n    def idle(self, strict=False):\n        errors = []\n        if self.serial is not None:\n            # Attempt BOTH lines even if one write fails.\n            for name, action in (("FRAME", self.set_frame), ("EVENT", self.set_event)):\n                try:\n                    action(0)\n                except Exception as exc:\n                    errors.append(f"{name}: {type(exc).__name__}: {exc}")\n        self._frame_high_at = None\n        self._last_frame_anchor = None\n        self._event_high_at = None\n        if errors and strict:\n            raise RuntimeError("; ".join(errors))\n        return errors\n\n    def close(self):\n        errors = self.idle()\n        ser, self.serial = self.serial, None\n        if ser is not None:\n            try:\n                ser.close()\n            except Exception as exc:\n                errors.append(f"Cerrar FTDI: {type(exc).__name__}: {exc}")\n        return errors\n'
exec(TRIGGER_SOURCE_EMBEDDED, globals())

# SMN-Systems splash artwork embedded inside the notebook.
# It is a resized copy of the selected original image and requires
# no external image file at runtime.
SPLASH_IMAGE_B64 = (
    "..."
)
SPLASH_MIN_VISIBLE_MS = 1800


# Hidden SMN-Systems easter egg.
# Six generated animation poses are embedded directly in the notebook.
EASTER_EGG_FRAMES_B64 = [
    (
        "..."
        "rkJggg=="
    ),
    (
        "..."
        "TkSuQmCC"
    ),
    (
        "..."
        "u1QAAAAASUVORK5CYII="
    )
]
EASTER_EGG_FRAME_MS = 125
EASTER_EGG_LOOPS = 3

CONFIG_DIR = Path.home() / "MEA_Visual_Stimulus"
CONFIG_FILE = CONFIG_DIR / "visual_stimuli_config.json"
PROTOCOLS_FILE = CONFIG_DIR / "protocols.json"
PROTOCOL_EXPORT_FORMAT = "MEA Visual Stimuli Protocol"
PROTOCOL_EXPORT_SCHEMA_VERSION = 1
RUN_DIR = CONFIG_DIR / "persistent_display"
LOG_DIR = CONFIG_DIR / "logs"

for p in (CONFIG_DIR, RUN_DIR, LOG_DIR):
    p.mkdir(parents=True, exist_ok=True)

def write_json_atomic(path, data):
    path = Path(path)
    tmp = path.with_suffix(path.suffix + ".tmp")
    tmp.write_text(
        json.dumps(data, indent=2, ensure_ascii=False),
        encoding="utf-8"
    )
    os.replace(tmp, path)

def detect_screens():
    import pyglet

    if hasattr(pyglet, "display"):
        display = pyglet.display.get_display()
    elif hasattr(pyglet, "canvas"):
        display = pyglet.canvas.get_display()
    else:
        raise RuntimeError(
            "pyglet no expone ni display ni canvas"
        )

    result = []
    for i, s in enumerate(display.get_screens()):
        x = int(getattr(s, "x", 0))
        y = int(getattr(s, "y", 0))
        w = int(getattr(s, "width", 0))
        h = int(getattr(s, "height", 0))
        result.append({
            "index": i,
            "x": x,
            "y": y,
            "width": w,
            "height": h,
            "label":
                (f"Monitor {i+1} — {w}×{h} px — "
                 f"position ({x}, {y})" if _I18N_LANGUAGE == "en" else
                 f"Monitor {i+1} — {w}×{h} px — posición ({x}, {y})")
        })
    return result

def load_saved():
    if CONFIG_FILE.exists():
        try:
            return json.loads(
                CONFIG_FILE.read_text(encoding="utf-8")
            )
        except Exception:
            pass
    return {}

def load_protocol_library():
    default = {
        "schema_version": 1,
        "protocols": []
    }
    if PROTOCOLS_FILE.exists():
        try:
            data = json.loads(
                PROTOCOLS_FILE.read_text(
                    encoding="utf-8"
                )
            )
            if (
                isinstance(data, dict)
                and isinstance(
                    data.get("protocols"),
                    list
                )
            ):
                return data
        except Exception:
            pass
    return default

def save_protocol_library(data):
    write_json_atomic(
        PROTOCOLS_FILE,
        data
    )

def rgb255_to_hex(rgb):
    try:
        r, g, b = [
            max(0, min(255, int(round(v))))
            for v in rgb[:3]
        ]
        return f"#{r:02X}{g:02X}{b:02X}"
    except Exception:
        return "#808080"

def hex_to_rgb255(value):
    value = str(value).strip()
    if value.startswith("#"):
        value = value[1:]
    if len(value) != 6:
        raise ValueError(
            f"Color hexadecimal inválido: {value}"
        )
    return [
        int(value[0:2], 16),
        int(value[2:4], 16),
        int(value[4:6], 16),
    ]

def best_text_color(hex_color):
    r, g, b = hex_to_rgb255(hex_color)
    luminance = (
        0.299 * r + 0.587 * g + 0.114 * b
    )
    return "black" if luminance > 150 else "white"

# ------------------------------------------------------------------
# v2.1.0 - Bilingual operator interface (Spanish / English)
# ------------------------------------------------------------------
# Internal stimulus identifiers and saved protocol values remain unchanged.
# Only their presentation is translated, preserving v2.0.1 compatibility.
_I18N_LANGUAGE = "en"
_I18N_HOOKS_INSTALLED = False

_I18N_EN_EXACT = {
    "Estimulación visual para MEA · PsychoPy · pantalla experimental protegida": "Visual stimulation for MEA · PsychoPy · protected experimental display",
    "Sesión / metadatos…": "Session / metadata…",
    "? Manual / Ayuda": "? Manual / Help",
    "🔴 MODO ROJO": "🔴 RED MODE",
    "⚫ MODO NORMAL": "⚫ NORMAL MODE",
    "Configuración del montaje": "Setup configuration",
    "Pantalla de estimulación:": "Stimulation display:",
    "Detectar": "Detect",
    "Reiniciar pantalla": "Restart display",
    "Ancho físico (cm):": "Physical width (cm):",
    "Alto físico (cm):": "Physical height (cm):",
    "Distancia (cm):": "Distance (cm):",
    "▶  INICIAR ESTÍMULO": "▶  START STIMULUS",
    "■  ABORTAR": "■  ABORT",
    "Protocolos": "Protocols",
    "Sistema / TTL": "System / TTL",
    "Parámetros comunes": "Common parameters",
    "Estímulo:": "Stimulus:",
    "Modo:": "Mode:",
    "Perfil:": "Profile:",
    "Cuadrado": "Square",
    "Distribución:": "Distribution:",
    "Diámetros:": "Diameters:",
    "Diámetros exteriores:": "Outer diameters:",
    "Duración:": "Duration:",
    "Duración (s):": "Duration (s):",
    "Repeticiones:": "Repetitions:",
    "Direcciones (°):": "Directions (°):",
    "Orientaciones (°):": "Orientations (°):",
    "Frecuencia temporal (Hz):": "Temporal frequency (Hz):",
    "Frecuencia espacial (ciclos/°):": "Spatial frequency (cycles/°):",
    "Frecuencia de actualización (Hz):": "Update frequency (Hz):",
    "Contraste:": "Contrast:",
    "Contraste (%):": "Contrast (%):",
    "Tamaño (°):": "Size (°):",
    "Diámetro (°):": "Diameter (°):",
    "Posición X (°):": "X position (°):",
    "Posición Y (°):": "Y position (°):",
    "Color A:": "Color A:",
    "Color B:": "Color B:",
    "Color ON:": "ON color:",
    "Color OFF:": "OFF color:",
    "Fondo:": "Background:",
    "Fondo de adaptación": "Adaptation background",
    "Negro": "Black",
    "Gris medio": "Middle gray",
    "Simulación": "Simulation",
    "Trigger Box TTL genérica": "Generic TTL Trigger Box",
    "Desactivado": "Disabled",
    "Guardar": "Save",
    "Cancelar": "Cancel",
    "Cerrar": "Close",
    "Abrir": "Open",
    "Cargar": "Load",
    "Eliminar": "Delete",
    "Editar": "Edit",
    "Duplicar": "Duplicate",
    "Nuevo protocolo": "New protocol",
    "Guardar protocolo": "Save protocol",
    "Añadir pausa": "Add pause",
    "＋ Añadir pausa": "＋ Add pause",
    "Reanudar protocolo": "Resume protocol",
    "Nombre del protocolo:": "Protocol name:",
    "Repeticiones del protocolo:": "Protocol repetitions:",
    "Biblioteca de protocolos": "Protocol library",
    "Pasos del protocolo": "Protocol steps",
    "Configuración del paso": "Step configuration",
    "Todos los archivos": "All files",
    "Metadatos de la sesión experimental": "Experimental session metadata",
    "ID de sesión": "Session ID",
    "ID de muestra": "Sample ID",
    "Preparación": "Preparation",
    "Ojo / retina": "Eye / retina",
    "Grupo / tratamiento": "Group / treatment",
    "Operador": "Operator",
    "Notas:": "Notes:",
    "Aplicar": "Apply",
    "Limpiar": "Clear",
    "Sesión: sin metadatos": "Session: no metadata",
    "Preparando pantalla de seguridad...": "Preparing safety display...",
    "Refresco: todavía no medido": "Refresh: not measured yet",
    "Información temporal: —": "Timing information: —",
    "Modo rojo: inactivo": "Red mode: inactive",
    "Modo rojo: activo": "Red mode: active",
    "Preparando MEA Visual Stimuli...": "Preparing MEA Visual Stimuli...",
    "Construyendo interfaz...": "Building interface...",
    "Preparando controles del operador": "Preparing operator controls",
    "Detectando pantallas...": "Detecting displays...",
    "Identificando monitor del operador y monitor experimental": "Identifying operator and experimental displays",
    "Pantallas detectadas": "Displays detected",
    "Preparando la capa de seguridad experimental": "Preparing experimental safety layer",
    "Pantalla experimental:": "Experimental display:",
    "Versión": "Version",
    "Error": "Error",
    "Aviso": "Notice",
    "Advertencia": "Warning",
    "Configuración": "Configuration",
    "Manual": "Manual",
    "Protocolo": "Protocol",
    "Error en estímulo": "Stimulus error",
    "Noise · frecuencia limitada": "Noise · frequency limited",
    "Guardar diagnóstico:": "Save diagnostic:",
    "Polaridad espacial:": "Spatial polarity:",
    "Contraste absoluto (usar los dos colores al 100 %)": "Absolute contrast (use both colors at 100%)",
    "Esquema para el operador · no se presenta en el monitor experimental": "Operator schematic · not shown on the experimental display",
    "Guardar esta configuración como predeterminada": "Save this configuration as default",
    "Actualizar vista / guardar con «Guardar»": "Refresh view / save with ‘Save’",
    "PROTOCOLO EN PAUSA": "PROTOCOL PAUSED",
    "Protocolo reanudado · ejecución en curso.": "Protocol resumed · execution in progress.",
    "Protocolo abortado · espera segura estable.": "Protocol aborted · stable safe idle state.",
    "Protocolo completado ·": "Protocol completed ·",
    "Último estímulo ·": "Last stimulus ·",
    "Último protocolo ·": "Last protocol ·",
    "Error desconocido": "Unknown error",
    "Sin metadatos": "No metadata",
    "Idioma:": "Language:",
    "Editar pausa": "Edit pause",
    "Duración de la pausa (s):": "Pause duration (s):",
    "Importar protocolo": "Import protocol",
    "Exportar protocolo": "Export protocol",
    "Archivo JSON": "JSON file",
    "Actualizar paso": "Update step",
    "Estado experimental": "Experimental status",
    "Parámetros del modo": "Mode parameters",
    "Duración": "Duration",
    "Hardware de triggers": "Trigger hardware",
    "Modo rojo / sala oscura": "Red mode / dark room",
    "✓ Actualizar paso": "✓ Update step",
    "⚫ SALIR MODO ROJO": "⚫ EXIT RED MODE",
    "Guardar metadatos": "Save metadata",
    "adaptación → ON → adaptación → OFF → adaptación": "adaptation → ON → adaptation → OFF → adaptation",
    "Blanco y negro son los colores predeterminados. Puedes cambiarlos directamente para realizar el mismo paradigma con cualquier pareja de colores.": "Black and white are the default colors. You can change them directly to run the same paradigm with any color pair.",
    "↻ Actualizar": "↻ Refresh",
    "Paso seleccionado": "Selected step",
    "Puerto:": "Port:",
    "modulación sinusoidal continua": "continuous sinusoidal modulation",
    "Parámetros del estímulo": "Stimulus parameters",
    "Vista del estímulo": "Stimulus preview",
    "Protocolos guardados": "Saved protocols",
    "Editor del protocolo": "Protocol editor",
    "Nuevo": "New",
    "Exportar…": "Export…",
    "Importar…": "Import…",
    "Nombre:": "Name:",
    "Repetir:": "Repeat:",
    "veces": "times",
    "Estímulo": "Stimulus",
    "↑ Subir": "↑ Move up",
    "↓ Bajar": "↓ Move down",
    "⏸ Pausar protocolo": "⏸ Pause protocol",
    "▶ Ejecutar protocolo": "▶ Run protocol",
    "＋ Añadir al protocolo": "＋ Add to protocol",
    "ABORTAR → SEGURO": "ABORT → SAFE",
    "Abrir logs": "Open logs",
    "Seleccionar": "Select",
    "  Seleccionar  ": "  Select  ",
    "Parámetros: —": "Parameters: —",
    "▶ Vista previa": "▶ Preview",
    "Pantalla segura y fondo de adaptación": "Safe display and adaptation background",
    "Adaptación previa:": "Pre-adaptation:",
    "Duración noise:": "Noise duration:",
    "Adaptación entre repeticiones:": "Adaptation between repetitions:",
    "Frecuencia de actualización:": "Update frequency:",
    "Tamaño de check:": "Check size:",
    "Semilla base:": "Base seed:",
    "Color bajo:": "Low color:",
    "Color medio:": "Mid color:",
    "Color alto:": "High color:",
    "Adaptación inicial:": "Initial adaptation:",
    "Duración ON:": "ON duration:",
    "Adaptación intermedia:": "Intermediate adaptation:",
    "Duración OFF:": "OFF duration:",
    "Adaptación final:": "Final adaptation:",
    "Anchura pantalla:": "Display width:",
    "Altura pantalla:": "Display height:",
    "Distancia a retina:": "Retina distance:",
    "Anchura de barra:": "Bar width:",
    "Longitud de barra:": "Bar length:",
    "Diámetro del spot:": "Spot diameter:",
    "Diámetro pequeño:": "Small diameter:",
    "Diámetro grande:": "Large diameter:",
    "Centro X:": "Center X:",
    "Centro Y:": "Center Y:",
    "Diámetro exterior pequeño:": "Small outer diameter:",
    "Diámetro exterior grande:": "Large outer diameter:",
    "Grosor radial del anillo:": "Annulus radial thickness:",
    "Frecuencia espacial:": "Spatial frequency:",
    "Contraste entre barras:": "Bar contrast:",
    "Frecuencia temporal:": "Temporal frequency:",
    "Frecuencia inversión:": "Reversal frequency:",
    "Tamaño Gabor:": "Gabor size:",
    "Duración / condición:": "Duration / condition:",
    "Adaptación entre condiciones:": "Adaptation between conditions:",
    "Posición X:": "X position:",
    "Posición Y:": "Y position:",
    "Diámetro interno:": "Inner diameter:",
    "Diámetro externo:": "Outer diameter:",
    "Tamaño del cuadrado:": "Square size:",
    "Paso de rejilla:": "Grid spacing:",
    "Adaptación basal:": "Baseline adaptation:",
    "Color alto / ON:": "High color / ON:",
    "Color bajo / OFF:": "Low color / OFF:",
    "Frecuencia mínima:": "Minimum frequency:",
    "Frecuencia máxima:": "Maximum frequency:",
    "Duración chirp:": "Chirp duration:",
    "Frecuencia ON/OFF:": "ON/OFF frequency:",
    "Frecuencia:": "Frequency:",
    "Frecuencia actualización:": "Update frequency:",
    "Frecuencia flicker:": "Flicker frequency:",
    "Adaptación entre contrastes:": "Adaptation between contrasts:",
    "Adaptación entre ensayos:": "Adaptation between trials:",
    "Contrastes:": "Contrasts:",
    "Direcciones:": "Directions:",
    "Direcciones movimiento:": "Motion directions:",
    "Duración del estímulo:": "Stimulus duration:",
    "Duración estímulo:": "Stimulus duration:",
    "Duración por contraste:": "Duration per contrast:",
    "Fase:": "Phase:",
    "Fase inicial:": "Initial phase:",
    "Fondo Gabor:": "Gabor background:",
    "Fondo estímulo:": "Stimulus background:",
    "Grosor radial:": "Radial thickness:",
    "Intensidad ON inicial:": "Initial ON intensity:",
    "Orientaciones:": "Orientations:",
    "Pausa B → C:": "B → C pause:",
    "Pausa C → D:": "C → D pause:",
    "Presentaciones / rep:": "Presentations / rep:",
    "Semilla:": "Seed:",
    "Sigma gaussiana:": "Gaussian sigma:",
    "Color del estímulo:": "Stimulus color:",
    "Color del fondo:": "Background color:",
    "Color estímulo:": "Stimulus color:",
    "Fondo de adaptación:": "Adaptation background:",
    "Duración estimada:": "Estimated duration:",
    "Pasos:": "Steps:",
    "¿Iniciar ahora?": "Start now?",
    "Prueba de triggers activa": "Trigger test active",
    "Prueba de triggers": "Trigger test",
}

_I18N_EN_PHRASES = [
    ("Todos los campos son opcionales.", "All fields are optional."),
    ("Se aplicarán a las ejecuciones realizadas mientras esta sesión permanezca activa", "They will apply to runs performed while this session remains active"),
    ("y quedarán guardados en el JSON de configuración.", "and will be stored in the configuration JSON."),
    ("La pantalla experimental permanece negra mientras no se ejecuta un estímulo.", "The experimental display remains black while no stimulus is running."),
    ("El fondo de adaptación se usa en baselines, intervalos y pausas experimentales.", "The adaptation background is used during baselines, intervals and experimental pauses."),
    ("Modo de sala oscura de la interfaz", "Dark-room interface mode"),
    ("No modifica Windows, la gamma del monitor ni la pantalla experimental", "It does not modify Windows, monitor gamma or the experimental display"),
    ("La paleta se mantiene estable mientras se presenta un estímulo.", "The palette remains stable while a stimulus is presented."),
    ("Pulsa 🔴 MODO ROJO en la cabecera para activarla o desactivarla.", "Press 🔴 RED MODE in the header to enable or disable it."),
    ("Familia de estímulos de movimiento.", "Motion stimulus family."),
    ("La barra atraviesa completamente la pantalla en cada dirección.", "The bar crosses the full display in each direction."),
    ("El spot circular entra desde fuera del campo visual y cruza completamente la pantalla.", "The circular spot enters from outside the visual field and crosses the full display."),
    ("Familia de gratings para caracterización espacial y temporal", "Grating family for spatial and temporal characterization"),
    ("Familia de ruido espaciotemporal de pantalla completa.", "Full-field spatiotemporal noise family."),
    ("Una sola familia para caracterizar posición, tamaño y organización centro-periferia del campo receptivo.", "A single family to characterize receptive-field position, size and center-surround organization."),
    ("Familia de estimulación temporal de campo completo", "Full-field temporal stimulation family"),
    ("0° = movimiento de izquierda a derecha.", "0° = motion from left to right."),
    ("0° = barras verticales.", "0° = vertical bars."),
    ("Los ángulos positivos avanzan en sentido antihorario.", "Positive angles advance counterclockwise."),
    ("Las orientaciones positivas giran en sentido antihorario.", "Positive orientations rotate counterclockwise."),
    ("La fase alterna 180°", "Phase alternates by 180°"),
    ("TTL sólo marca el inicio de cada condición.", "TTL only marks the onset of each condition."),
    ("patrón localizado e inmóvil mediante envolvente gaussiana", "localized stationary pattern using a Gaussian envelope"),
    ("el parche permanece fijo, pero las barras se desplazan dentro de la envolvente gaussiana", "the patch remains fixed while the bars drift within the Gaussian envelope"),
    ("Spot circular estático", "Static circular spot"),
    ("Anillo centrado", "Annulus centered"),
    ("Cada diámetro constituye una condición principal.", "Each diameter defines a main condition."),
    ("Sparse Noise usa el fondo de adaptación", "Sparse Noise uses the adaptation background"),
    ("Secuencia histórica", "Historical sequence"),
    ("adaptación basal", "baseline adaptation"),
    ("barrido lineal de frecuencia", "linear frequency sweep"),
    ("rampa de intensidad", "intensity ramp"),
    ("Modulación sinusoidal continua", "Continuous sinusoidal modulation"),
    ("Ruido temporal gaussiano de campo completo.", "Full-field Gaussian temporal noise."),
    ("Serie de flicker sinusoidal a frecuencia fija.", "Sinusoidal flicker series at a fixed frequency."),
    ("No se ha podido guardar el archivo.", "The file could not be saved."),
    ("No se puede importar este protocolo.", "This protocol cannot be imported."),
    ("No se pudo cargar la configuración del paso.", "The step configuration could not be loaded."),
    ("No se ha podido solicitar la pausa.", "The pause request could not be sent."),
    ("No se ha podido enviar la orden de reanudación.", "The resume command could not be sent."),
    ("No se pudo abrir PsychoPy:", "PsychoPy could not be opened:"),
    ("Introduce al menos una dirección.", "Enter at least one direction."),
    ("introduce al menos una dirección/orientación.", "enter at least one direction/orientation."),
    ("debe ser > 0", "must be > 0"),
    ("deben ser > 0", "must be > 0"),
    ("debe ser >= 1", "must be >= 1"),
    ("debe estar entre 0 y 1", "must be between 0 and 1"),
    ("debe estar entre 0 y 100 %", "must be between 0 and 100%"),
    ("deben estar entre 0 y 100 %", "must be between 0 and 100%"),
    ("No es posible presentar más de un patrón nuevo por frame.", "It is not possible to present more than one new pattern per frame."),
    ("la frecuencia real quedará registrada en los archivos de la sesión.", "the actual frequency will be recorded in the session files."),
    ("Limitado por refresco", "Limited by refresh rate"),
    ("frecuencia limitada", "frequency limited"),
    ("modo Motion no compatible", "unsupported Motion mode"),
    ("modo Grating no compatible", "unsupported Grating mode"),
    ("modo RF Mapping no compatible", "unsupported RF Mapping mode"),
    ("modo Temporal no compatible", "unsupported Temporal mode"),
    ("Modo Motion desconocido", "Unknown Motion mode"),
    ("Modo Temporal desconocido", "Unknown Temporal mode"),
    ("Tipo de estímulo no reconocido", "Unrecognized stimulus type"),
    ("configuración ausente o inválida", "missing or invalid configuration"),
    ("tipo de paso desconocido", "unknown step type"),
    ("estímulo no compatible", "unsupported stimulus"),
    ("estructura no válida", "invalid structure"),
    ("Guardar diagnóstico", "Save diagnostic"),
    ("Configuración guardada en", "Configuration saved to"),
    ("Pantalla", "Display"),
    ("pantalla", "display"),
    ("Refresco", "Refresh"),
    ("refresco", "refresh"),
    ("Sesión", "Session"),
    ("sesión", "session"),
    ("Configuración", "Configuration"),
    ("configuración", "configuration"),
    ("Protocolo", "Protocol"),
    ("protocolo", "protocol"),
    ("Estímulo", "Stimulus"),
    ("estímulo", "stimulus"),
    ("Pausa", "Pause"),
    ("pausa", "pause"),
    ("Activo", "Active"),
    ("activo", "active"),
    ("Inactivo", "Inactive"),
    ("inactivo", "inactive"),
    ("Preparando", "Preparing"),
    ("Detectando", "Detecting"),
    ("Guardado", "Saved"),
    ("guardado", "saved"),
    ("Error en", "Error in"),
    ("diámetros", "diameters"),
    ("diámetro", "diameter"),
    ("grosor", "thickness"),
    ("frecuencia", "frequency"),
    ("contraste", "contrast"),
    ("duración", "duration"),
    ("repetición", "repetition"),
    ("repeticiones", "repetitions"),
    ("dirección", "direction"),
    ("orientación", "orientation"),
    ("color", "color"),
    ("Sparse Noise permanece en RF Mapping.", "Sparse Noise remains in RF Mapping."),
    ("Crea un protocolo nuevo o carga uno guardado. Configura un estímulo en su pestaña y pulsa «Añadir al protocolo».", "Create a new protocol or load a saved one. Configure a stimulus in its tab and click ‘Add to protocol’."),
    ("pasos ·", "steps ·"),
    ("estímulos ·", "stimuli ·"),
    ("pausas", "pauses"),
    ("pasos", "steps"),
    ("estímulos", "stimuli"),
    ("vuelta", "run"),
    ("repetición", "repetition"),
    ("fondo", "background"),
]



# ------------------------------------------------------------------
# v2.1.2 - exhaustive bilingual UI audit
# ------------------------------------------------------------------
# These additions cover static labels, runtime status strings, previews,
# protocol-builder text and validation/dialog messages that are assembled
# dynamically after widget creation. Translation remains presentation-only.
_I18N_EN_EXACT.update({
    "Preparando pantalla de seguridad...": "Preparing safety display...",
    "Refresco: todavía no medido": "Refresh rate: not measured yet",
    "Información temporal: —": "Timing information: —",
    "Sesión: sin metadatos": "Session: no metadata",
    "Preparando MEA Visual Stimuli...": "Preparing MEA Visual Stimuli...",
    "Preparando controles del operador": "Preparing operator controls",
    "Preparando la capa de seguridad experimental": "Preparing experimental safety layer",
    "Pantalla experimental estable · espera segura": "Experimental display stable · safe idle",
    "Pantalla experimental estable · ": "Experimental display stable · ",
    " Hz · espera segura": " Hz · safe idle",
    "Abrir logs": "Open logs",
    "Abrir interfaz": "Open interface",
    "No se pudo completar la inicialización": "Initialization could not be completed",
    "Anchura pantalla": "Display width",
    "Altura pantalla": "Display height",
    "Distancia a retina": "Retina distance",
    "Parámetros del estímulo": "Stimulus parameters",
    "Vista del estímulo": "Stimulus preview",
    "Parámetros: —": "Parameters: —",
    "＋ Añadir al protocolo": "＋ Add to protocol",
    "Seleccionar": "Select",
    "  Seleccionar  ": "  Select  ",
    "Adaptación inicial": "Initial adaptation",
    "Adaptación intermedia": "Intermediate adaptation",
    "Adaptación final": "Final adaptation",
    "Duración ON": "ON duration",
    "Duración OFF": "OFF duration",
    "Color ON": "ON color",
    "Color OFF": "OFF color",
    "Adaptación previa": "Pre-adaptation",
    "Duración del estímulo": "Stimulus duration",
    "Adaptación entre ensayos": "Inter-trial adaptation",
    "Color del fondo": "Background color",
    "Color del estímulo": "Stimulus color",
    "Anchura de barra": "Bar width",
    "Longitud de barra": "Bar length",
    "Diámetro del spot": "Spot diameter",
    "Diámetro pequeño": "Small diameter",
    "Diámetro grande": "Large diameter",
    "Diámetro exterior pequeño": "Small outer diameter",
    "Diámetro exterior grande": "Large outer diameter",
    "Duración estímulo": "Stimulus duration",
    "Frecuencia espacial": "Spatial frequency",
    "Contraste entre barras": "Bar contrast",
    "Contraste absoluto (usar los dos colores al 100 %)": "Absolute contrast (use both colors at 100%)",
    "Frecuencia temporal": "Temporal frequency",
    "Polaridad espacial:": "Spatial polarity:",
    "Frecuencia inversión": "Reversal frequency",
    "Fase inicial": "Initial phase",
    "Tamaño Gabor": "Gabor size",
    "Fondo Gabor": "Gabor background",
    "Duración noise": "Noise duration",
    "Adaptación entre repeticiones": "Inter-repetition adaptation",
    "Frecuencia de actualización": "Update frequency",
    "Tamaño de check": "Check size",
    "Semilla base": "Base seed",
    "Color bajo": "Low color",
    "Color medio": "Middle color",
    "Color alto": "High color",
    "Duración / condición": "Duration / condition",
    "Adaptación entre condiciones": "Inter-condition adaptation",
    "Posición X": "X position",
    "Posición Y": "Y position",
    "Color estímulo": "Stimulus color",
    "Fondo estímulo": "Stimulus background",
    "Parámetros del modo": "Mode parameters",
    "Diámetro interno": "Inner diameter",
    "Diámetro externo": "Outer diameter",
    "Diámetros:": "Diameters:",
    "Diámetros exteriores:": "Outer diameters:",
    "Tamaño del cuadrado": "Square size",
    "Paso de rejilla": "Grid spacing",
    "Semilla": "Seed",
    "Adaptación basal": "Baseline adaptation",
    "Color alto / ON": "High color / ON",
    "Color bajo / OFF": "Low color / OFF",
    "Pausa B → C": "B → C pause",
    "Frecuencia mínima": "Minimum frequency",
    "Frecuencia máxima": "Maximum frequency",
    "Duración chirp": "Chirp duration",
    "Pausa C → D": "C → D pause",
    "Frecuencia ON/OFF": "ON/OFF frequency",
    "Intensidad ON inicial": "Initial ON intensity",
    "Frecuencia": "Frequency",
    "Frecuencia actualización": "Update frequency",
    "Duración por contraste": "Duration per contrast",
    "Frecuencia flicker": "Flicker frequency",
    "Adaptación entre contrastes": "Inter-contrast adaptation",
    "Protocolos guardados": "Saved protocols",
    "Editor del protocolo": "Protocol editor",
    "Nombre:": "Name:",
    "Repetir:": "Repeat:",
    "veces": "times",
    "Paso seleccionado": "Selected step",
    "↑ Subir": "↑ Move up",
    "↓ Bajar": "↓ Move down",
    "⏸ Pausar protocolo": "⏸ Pause protocol",
    "▶ Ejecutar protocolo": "▶ Run protocol",
    "Sin nombre": "Untitled",
    "Cambios sin guardar": "Unsaved changes",
    "Pausa": "Pause",
    "Editar pausa": "Edit pause",
    "Duración de la pausa (s):": "Pause duration (s):",
    "Duración estimada: ": "Estimated duration: ",
    "Refresco medido: ": "Measured refresh rate: ",
    "Información temporal": "Timing information",
    "Capa de seguridad activa.": "Safety layer active.",
    "Activando pantalla de seguridad...": "Activating safety display...",
    "Pantalla de seguridad activa": "Safety display active",
    "Orden de estímulo enviada...": "Stimulus command sent...",
    "Abortando estímulo → retorno a estado seguro...": "Aborting stimulus → returning to safe state...",
    "Midiendo frecuencia de refresco...": "Measuring refresh rate...",
    "PsychoPy está comprobando la temporización real del monitor": "PsychoPy is checking the monitor's actual timing",
    "Configuración guardada": "Configuration saved",
    "Desactivado": "Disabled",
    "Activado": "Enabled",
    "Español": "Spanish",
    "Idioma:": "Language:",
    "Temporal": "Temporal",
})

_I18N_EN_PHRASES_EXTRA = [
    ("Familia de estimulación temporal de campo completo: Chirp + Intensity, flicker sinusoidal, flicker gaussiano y serie de contraste.", "Full-field temporal stimulation family: Chirp + Intensity, sinusoidal flicker, Gaussian flicker and contrast series."),
    ("Familia de ruido espaciotemporal de pantalla completa. Sparse Noise permanece en RF Mapping.", "Full-field spatiotemporal noise family. Sparse Noise remains in RF Mapping."),
    ("En los modos continuos, los dos colores definen los extremos de modulación. El basal/final usa el fondo de adaptación configurado en Sistema / TTL.", "In continuous modes, the two colors define the modulation extremes. Baseline/final periods use the adaptation background configured in System / TTL."),
    ("Secuencia histórica A → B → C → D: adaptación basal, ON→OFF, barrido lineal de frecuencia y rampa de intensidad.", "Historical sequence A → B → C → D: baseline adaptation, ON→OFF, linear frequency sweep and intensity ramp."),
    ("ON/OFF inicial → chirp → rampa de intensidad", "initial ON/OFF → chirp → intensity ramp"),
    ("Duración estimada", "Estimated duration"),
    ("Refresco medido", "Measured refresh rate"),
    ("Información temporal", "Timing information"),
    ("Display experimental estable", "Experimental display stable"),
    ("Pantalla experimental estable", "Experimental display stable"),
    ("espera segura", "safe idle"),
    ("Crea un protocolo nuevo o carga uno guardado. Configura un estímulo en su pestaña y pulsa «Añadir al protocolo».", "Create a new protocol or load a saved one. Configure a stimulus in its tab and click ‘Add to protocol’."),
    ("Nuevo protocolo preparado. Configura cualquier estímulo y pulsa «Añadir al protocolo».", "New protocol ready. Configure any stimulus and click ‘Add to protocol’."),
    ("Abre primero una pestaña de estímulo, configúrala y pulsa «Añadir al protocolo».", "Open a stimulus tab first, configure it and click ‘Add to protocol’."),
    ("La duración de la pausa debe ser > 0 s.", "Pause duration must be > 0 s."),
    ("del protocolo. Modifica los parámetros y pulsa «Actualizar paso».", "of the protocol. Modify the parameters and click ‘Update step’."),
    ("adaptación → ON → adaptación → OFF → adaptación", "adaptation → ON → adaptation → OFF → adaptation"),
    ("Estado basal y periodos intermedios: fondo de adaptación.", "Baseline and intermediate periods: adaptation background."),
    ("TTL: un pulso al inicio de ON y otro al inicio de OFF.", "TTL: one pulse at ON onset and another at OFF onset."),
    ("0° = derecha, 90° = arriba; ángulos positivos en sentido antihorario.", "0° = right, 90° = up; positive angles run counterclockwise."),
    ("Expansión: diámetro pequeño → grande. Simula un objeto que se aproxima.", "Expansion: small → large diameter. Simulates an approaching object."),
    ("Contracción: diámetro grande → pequeño. Simula un objeto que se aleja.", "Contraction: large → small diameter. Simulates a receding object."),
    ("Frecuencia de actualización", "Update frequency"),
    ("Adaptación entre repeticiones", "Inter-repetition adaptation"),
    ("Adaptación entre condiciones", "Inter-condition adaptation"),
    ("Adaptación entre contrastes", "Inter-contrast adaptation"),
    ("Adaptación previa", "Pre-adaptation"),
    ("Adaptación basal", "Baseline adaptation"),
    ("Adaptación inicial", "Initial adaptation"),
    ("Adaptación intermedia", "Intermediate adaptation"),
    ("Adaptación final", "Final adaptation"),
    ("Parámetros comunes", "Common parameters"),
    ("Parámetros del estímulo", "Stimulus parameters"),
    ("Parámetros del modo", "Mode parameters"),
    ("Vista del estímulo", "Stimulus preview"),
    ("Seleccionar", "Select"),
    ("Añadir al protocolo", "Add to protocol"),
    ("Añadir pausa", "Add pause"),
    ("Guardar protocolo", "Save protocol"),
    ("Ejecutar protocolo", "Run protocol"),
    ("Pausar protocolo", "Pause protocol"),
    ("Reanudar protocolo", "Resume protocol"),
    ("Abrir logs", "Open logs"),
    ("Duración por contraste", "Duration per contrast"),
    ("Frecuencia flicker", "Flicker frequency"),
    ("Frecuencia mínima", "Minimum frequency"),
    ("Frecuencia máxima", "Maximum frequency"),
    ("Duración chirp", "Chirp duration"),
    ("Intensidad ON inicial", "Initial ON intensity"),
    ("Color alto", "High color"),
    ("Color medio", "Middle color"),
    ("Color bajo", "Low color"),
    ("Semilla base", "Base seed"),
    ("Semilla", "Seed"),
    ("Tamaño de check", "Check size"),
    ("Tamaño del cuadrado", "Square size"),
    ("Paso de rejilla", "Grid spacing"),
    ("Posición X", "X position"),
    ("Posición Y", "Y position"),
    ("Diámetro interno", "Inner diameter"),
    ("Diámetro externo", "Outer diameter"),
    ("Diámetros exteriores", "Outer diameters"),
    ("Diámetros", "Diameters"),
    ("Duración del estímulo", "Stimulus duration"),
    ("Duración estímulo", "Stimulus duration"),
    ("Duración noise", "Noise duration"),
    ("Frecuencia espacial", "Spatial frequency"),
    ("Frecuencia temporal", "Temporal frequency"),
    ("Frecuencia inversión", "Reversal frequency"),
    ("Fase inicial", "Initial phase"),
    ("Tamaño Gabor", "Gabor size"),
    ("Fondo Gabor", "Gabor background"),
    ("Protocolos guardados", "Saved protocols"),
    ("Editor del protocolo", "Protocol editor"),
    ("Paso seleccionado", "Selected step"),
    ("Nombre", "Name"),
    ("Repetir", "Repeat"),
    ("veces", "times"),
    ("Subir", "Move up"),
    ("Bajar", "Move down"),
    ("Sin nombre", "Untitled"),
    ("Cambios sin guardar", "Unsaved changes"),
    ("sin metadatos", "no metadata"),
    ("metadatos", "metadata"),
    ("Configuración del montaje", "Setup configuration"),
    ("Pantalla de estimulación", "Stimulation display"),
    ("Reiniciar pantalla", "Restart display"),
    ("Anchura pantalla", "Display width"),
    ("Altura pantalla", "Display height"),
    ("Distancia a retina", "Retina distance"),
    ("MODO ROJO", "RED MODE"),
    ("MODO NORMAL", "NORMAL MODE"),
    ("ABORTAR", "ABORT"),
    ("INICIAR ESTÍMULO", "START STIMULUS"),
    ("Guardar", "Save"),
    ("Cargar", "Load"),
    ("Eliminar", "Delete"),
    ("Duplicar", "Duplicate"),
    ("Nuevo", "New"),
    ("Editar", "Edit"),
    ("Exportar", "Export"),
    ("Importar", "Import"),
    ("Duración", "Duration"),
    ("Contraste", "Contrast"),
    ("Repeticiones", "Repetitions"),
]
# Put specific phrases before the generic list so complete sentences are handled first.
_I18N_EN_PHRASES = _I18N_EN_PHRASES_EXTRA + _I18N_EN_PHRASES

# Final exact-string audit for runtime/help/validation text that is not safely
# expressible as short phrase replacements.
_I18N_EN_EXACT.update({
    "● TTL GENÉRICO · UNIVERSAL + AUX ESTADO/PAUSA": "● GENERIC TTL · UNIVERSAL + AUX STATUS/PAUSE",
    "TTL: Genérico · auto": "TTL: Generic · auto",
    "Metadatos de sesión limpiados.": "Session metadata cleared.",
    "Metadatos de sesión actualizados.": "Session metadata updated.",
    "Versión ": "Version ",
    "Familia de gratings para caracterización espacial y temporal: drifting, rampa asimétrica, estático, inversión de fase, Static Gabor y Drifting Gabor.": "Grating family for spatial and temporal characterization: drifting, asymmetric ramp, static, phase reversal, Static Gabor and Drifting Gabor.",
    "A_TO_B: Color A → Color B con transición de medio coseno y salto brusco de vuelta a A. B_TO_A invierte la polaridad. 0° = movimiento de izquierda a derecha.": "A_TO_B: Color A → Color B with a half-cosine transition and an abrupt jump back to A. B_TO_A reverses the polarity. 0° = motion from left to right.",
    "Static Gabor: patrón localizado e inmóvil mediante envolvente gaussiana. El fondo se reaplica explícitamente en cada frame.": "Static Gabor: localized stationary pattern using a Gaussian envelope. The background is explicitly reapplied on every frame.",
    "Dense White Noise: muchos checks cambian de forma independiente en cada actualización. Binary usa Color bajo/alto; Ternary usa bajo/medio/alto. Cada repetición genera una secuencia nueva y reproducible (semilla + rep).": "Dense White Noise: many checks change independently at every update. Binary uses low/high color; Ternary uses low/middle/high. Each repetition generates a new reproducible sequence (seed + rep).",
    "Gaussian White Noise: cada check toma un valor continuo e independiente de una distribución gaussiana centrada en Color medio. Con contraste 1, bajo/alto corresponden aproximadamente a ±3σ; los extremos se recortan.": "Gaussian White Noise: each check takes an independent continuous value from a Gaussian distribution centered on the middle color. At contrast 1, low/high correspond approximately to ±3σ; extremes are clipped.",
    "Frozen Noise: una única secuencia aleatoria se genera con la semilla base y se reproduce exactamente igual en todas las repeticiones. Útil para medir fiabilidad trial-to-trial. Puede ser Binary, Ternary o Gaussian.": "Frozen Noise: a single random sequence is generated from the base seed and replayed identically in every repetition. Useful for measuring trial-to-trial reliability. It can be Binary, Ternary or Gaussian.",
    "Spot circular estático en la posición X/Y. Útil para polaridad, latencia y respuesta local.": "Static circular spot at X/Y position. Useful for polarity, latency and local response.",
    "Funciona como Size series, pero cada condición es un anillo. El grosor radial permanece constante para todos los diámetros exteriores.": "Works like Size series, but each condition is an annulus. Radial thickness remains constant for all outer diameters.",
    "Ruido temporal gaussiano de campo completo. Cada actualización toma un nivel independiente, centrado en el punto medio de los colores. A 100 %, bajo/alto corresponden aproximadamente a ±3σ. Si se pide más Hz que el refresco, se registra la frecuencia real alcanzable.": "Full-field Gaussian temporal noise. Each update takes an independent level centered on the midpoint of the two colors. At 100%, low/high correspond approximately to ±3σ. If the requested rate exceeds refresh, the actual achievable frequency is recorded.",
    "El protocolo actual contiene cambios sin guardar.\n\n¿Quieres descartarlos?": "The current protocol contains unsaved changes.\n\nDo you want to discard them?",
    ". Puedes modificarlo y guardarlo como una nueva revisión.": ". You can modify it and save it as a new revision.",
    "Copia creada como borrador. Guárdala para añadirla a la biblioteca.": "Copy created as a draft. Save it to add it to the library.",
    "»?\n\nEsta acción elimina la definición guardada, no los CSV de experimentos anteriores.": "»?\n\nThis deletes the saved definition, not CSV files from previous experiments.",
    "¿Eliminar «": "Delete «",
    "Escribe un nombre para el protocolo.": "Enter a name for the protocol.",
    "Añade al menos un paso antes de guardar.": "Add at least one step before saving.",
    " · versión ": " · version ",
    "El archivo no contiene un objeto JSON válido.": "The file does not contain a valid JSON object.",
    "Versión de formato no compatible: ": "Unsupported format version: ",
    ". Esta versión del programa admite schema ": ". This program version supports schema ",
    "Falta la definición del protocolo.": "Protocol definition is missing.",
    "El protocolo importado no tiene nombre.": "The imported protocol has no name.",
    "Versión o número de repeticiones no válido.": "Invalid version or repetition count.",
    "La versión del protocolo debe ser >= 1.": "Protocol version must be >= 1.",
    "El protocolo es válido, pero no se ha podido guardar en la biblioteca local.\n\n": "The protocol is valid, but it could not be saved to the local library.\n\n",
    " Ya existía ese nombre; se ha guardado como «": " That name already existed; it was saved as «",
    "No se puede añadir": "Cannot add",
    " actualizado. Vuelve a guardarlo para crear una nueva revisión.": " updated. Save it again to create a new revision.",
    "Añadido a «": "Added to «",
    "adaptación · ": "adaptation · ",
    "Reanudando protocolo… se mantiene el fondo de adaptación hasta que comience el siguiente paso.": "Resuming protocol… the adaptation background is maintained until the next step begins.",
    "Pausa solicitada · el paso actual terminará normalmente. El protocolo se detendrá sobre el fondo de adaptación antes de iniciar el siguiente paso.": "Pause requested · the current step will finish normally. The protocol will stop on the adaptation background before the next step begins.",
    "Termina o detén la prueba de triggers antes de ejecutar el estímulo.": "Finish or stop the trigger test before running the stimulus.",
    "Guarda el protocolo antes de ejecutarlo. Así el registro experimental queda asociado a una versión concreta y reproducible.": "Save the protocol before running it so the experimental record is associated with a specific reproducible version.",
    "PsychoPy todavía está preparando la pantalla o midiendo el refresco. Espera unos segundos; el botón se habilitará automáticamente cuando el estado sea estable.": "PsychoPy is still preparing the display or measuring refresh. Wait a few seconds; the button will be enabled automatically when the state is stable.",
    "La ventana PsychoPy no está preparada en el monitor seleccionado. La capa de seguridad permanece activa. Usa «Reiniciar pantalla» sólo si el estado no vuelve a preparado.": "The PsychoPy window is not ready on the selected monitor. The safety layer remains active. Use ‘Restart display’ only if the state does not return to ready.",
    "). Se permitirá y se registrará la frecuencia realmente presentada.": "). This will be allowed and the actual presented frequency will be recorded.",
    "\n\nPasos: ": "\n\nSteps: ",
    "\n\nTodo el protocolo se enviará al worker como una única ejecución. Antes del primer frame registrado, PsychoPy preparará los recursos gráficos manteniendo la pantalla en el fondo de adaptación. Después generará un único CSV visual y un único CSV TTL.": "\n\nThe entire protocol will be sent to the worker as a single run. Before the first recorded frame, PsychoPy will prepare graphical resources while keeping the display on the adaptation background. It will then generate one visual CSV and one TTL CSV.",
    "\n\n¿Iniciar ahora?": "\n\nStart now?",
    "expansión": "expansion",
    "contracción": "contraction",
    " Hz · tamaño ": " Hz · size ",
    "° · tamaño ": "° · size ",
    "\n⚠ Limitado por refresco: máx. ": "\n⚠ Limited by refresh rate: max. ",
    "Genérico: UNIVERSAL=FRAME + EVENT ~3 ms después; AUX EVENT muestra estados y marca cada pausa con un pulso corto.  MaxOne Legacy conserva exactamente la ruta v1.17.": "Generic: UNIVERSAL=FRAME + EVENT ~3 ms later; AUX EVENT shows states and marks each pause with a short pulse. MaxOne Legacy preserves the v1.17 path exactly.",
    "Espera segura: NEGRO": "Safe idle: BLACK",
    "Modo de sala oscura de la interfaz: convierte únicamente la ventana del operador a una paleta roja tenue y muy oscura. No modifica Windows, la gamma del monitor ni la pantalla experimental, que permanece negra en espera.": "Dark-room interface mode: converts only the operator window to a dim, very dark red palette. It does not modify Windows, monitor gamma or the experimental display, which remains black while idle.",
    "● SIMULACIÓN · EVENT por condición": "● SIMULATION · EVENT per condition",
    "Genérico": "Generic",
    "Guardar diagnóstico: ": "Save diagnostic: ",
    "● TRIGGERS · error de diagnóstico": "● TRIGGERS · diagnostic error",
    "Prueba TTL genérica enviada · contrasta FRAME/EVENT/UNIVERSAL. Registro: ": "Generic TTL test sent · compare FRAME/EVENT/UNIVERSAL. Log: ",
    "Selecciona la pantalla de estimulación.": "Select the stimulation display.",
    "Tamaño de pantalla y distancia deben ser > 0.": "Display size and distance must be > 0.",
    "Motion: duración > 0 y tiempos de adaptación >= 0.": "Motion: duration > 0 and adaptation times >= 0.",
    "Motion: diámetro grande > diámetro pequeño > 0.": "Motion: large diameter > small diameter > 0.",
    "Annulus: diámetro exterior grande > diámetro exterior pequeño > 0.": "Annulus: large outer diameter > small outer diameter > 0.",
    "Annulus: el grosor radial debe ser menor que la mitad del diámetro exterior pequeño.": "Annulus: radial thickness must be less than half the small outer diameter.",
    "Grating: duración > 0 y tiempos de adaptación >= 0.": "Grating: duration > 0 and adaptation times >= 0.",
    "Phase Reversal: la frecuencia de inversión debe ser > 0.": "Phase Reversal: reversal frequency must be > 0.",
    ": tamaño > 0 y 0 < sigma <= tamaño/2.": ": size > 0 and 0 < sigma <= size/2.",
    "Frozen Noise: distribución no válida.": "Frozen Noise: invalid distribution.",
    "Noise: duración > 0 y tiempos de adaptación >= 0.": "Noise: duration > 0 and adaptation times >= 0.",
    "Noise: la frecuencia de actualización debe ser > 0.": "Noise: update frequency must be > 0.",
    "Noise: el tamaño de check debe ser > 0.": "Noise: check size must be > 0.",
    "RF Mapping: duración > 0 y tiempos de adaptación >= 0.": "RF Mapping: duration > 0 and adaptation times >= 0.",
    "Los diámetros deben ser números separados por comas.": "Diameters must be comma-separated numbers.",
    "Annulus Size series: los diámetros exteriores deben ser números separados por comas.": "Annulus Size series: outer diameters must be comma-separated numbers.",
    "Sparse Noise: tamaño y paso deben ser > 0.": "Sparse Noise: size and spacing must be > 0.",
    "Temporal: los tiempos de adaptación deben ser >= 0.": "Temporal: adaptation times must be >= 0.",
    "La frecuencia máxima debe ser >= a la mínima.": "Maximum frequency must be >= minimum frequency.",
    "La intensidad ON inicial debe estar entre 0 y 100 %.": "Initial ON intensity must be between 0 and 100%.",
    "Gaussian Flicker: frecuencia de actualización debe ser > 0.": "Gaussian Flicker: update frequency must be > 0.",
    "Contrast Series: la adaptación entre contrastes debe ser >= 0.": "Contrast Series: inter-contrast adaptation must be >= 0.",
    "La pestaña Protocolos es un editor. La ejecución automática se activará en la siguiente fase.": "The Protocols tab is an editor. Automatic execution will be enabled in the next phase.",
    " Hz de actualización, pero la pantalla refresca a ": " Hz update rate, but the display refreshes at ",
    " Hz.\n\nNo es posible presentar más de un patrón nuevo por frame. El estímulo continuará aproximadamente a ": " Hz.\n\nIt is not possible to present more than one new pattern per frame. The stimulus will continue at approximately ",
    "La ventana PsychoPy no está preparada. La capa de seguridad permanece activa. Usa «Reiniciar pantalla» antes de presentar un estímulo.": "The PsychoPy window is not ready. The safety layer remains active. Use ‘Restart display’ before presenting a stimulus.",
    "Motor PsychoPy detenido. La capa de seguridad mantiene el monitor en estado seguro. Pulsa «Reiniciar pantalla».": "PsychoPy engine stopped. The safety layer keeps the monitor in a safe state. Press ‘Restart display’.",
    "Motor PsychoPy detenido. La capa segura de seguridad permanece activa. Puedes abrir la interfaz y usar «Reiniciar pantalla».": "PsychoPy engine stopped. The safety layer remains active. You can open the interface and use ‘Restart display’.",
    "Último protocolo · ": "Last protocol · ",
    "Último estímulo · ": "Last stimulus · ",
    "Último protocolo abortado con retorno al estado seguro.": "Last protocol aborted with return to safe state.",
    "Último estímulo abortado con retorno al estado seguro.": "Last stimulus aborted with return to safe state.",
    "PROTOCOLO EN PAUSA · fondo de adaptación estable · siguiente: paso ": "PROTOCOL PAUSED · stable adaptation background · next: step ",
    "Preparando la ventana de estimulación en estado seguro": "Preparing the stimulation window in a safe state",
    "Refresh medido: ": "Measured refresh rate: ",
    "Display experimental estable · espera segura": "Experimental display stable · safe idle",
})

# v2.1.3: deep i18n audit of secondary windows, runtime status strings,
# validation paths and labels that are assembled dynamically (for example,
# "ID de sesión" + ":").  Keep these presentation-only; internal protocol
# identifiers and saved values remain unchanged.
_I18N_EN_EXACT.update({
    "Sesión experimental / metadatos": "Experimental session / metadata",
    "Todos los campos son opcionales. Se aplicarán a las ejecuciones realizadas mientras esta sesión permanezca activa y quedarán guardados en el JSON de configuración.": "All fields are optional. They apply to runs performed while this session remains active and are stored in the configuration JSON.",
    "ID de sesión:": "Session ID:",
    "ID de muestra:": "Sample ID:",
    "Preparación:": "Preparation:",
    "Ojo / retina:": "Eye / retina:",
    "Grupo / tratamiento:": "Group / treatment:",
    "Operador:": "Operator:",
    "Limpiar campos": "Clear fields",
    "Cancelar": "Cancel",
    "Guardar metadatos": "Save metadata",
    "Sesión sin metadatos activos.": "Session has no active metadata.",
    "No se ha podido preparar la carpeta de logs.\n\n": "The logs folder could not be prepared.\n\n",
    "No se ha podido abrir la carpeta de logs.\n\n": "The logs folder could not be opened.\n\n",
    "No se encuentra el manual offline.\n\nArchivo esperado: ": "The offline manual could not be found.\n\nExpected file: ",
    "\n\nColoca el PDF junto al programa (o en una carpeta «docs» junto al programa) y vuelve a intentarlo.\n\nUbicaciones comprobadas:\n": "\n\nPlace the PDF next to the program (or in a 'docs' folder next to it) and try again.\n\nLocations checked:\n",
    "No se ha podido abrir el manual.\n\n": "The manual could not be opened.\n\n",
    "Abrir interfaz": "Open interface",
    "No se pudo completar la inicialización": "Initialization could not be completed",
    " Hz · espera segura": " Hz · safe idle",
    "Abriendo interfaz del operador": "Opening operator interface",
    "Distancia a retina": "Retina distance",
    "Adaptación inicial": "Initial adaptation",
    "Duración ON": "ON duration",
    "Adaptación intermedia": "Intermediate adaptation",
    "Duración OFF": "OFF duration",
    "Adaptación final": "Final adaptation",
    "Color ON": "ON color",
    "Color OFF": "OFF color",
    "Adaptación previa": "Pre-adaptation",
    "Adaptación entre ensayos": "Inter-trial adaptation",
    "Anchura de barra": "Bar width",
    "Longitud de barra": "Bar length",
    "Diámetro del spot": "Spot diameter",
    "Diámetro pequeño": "Small diameter",
    "Diámetro grande": "Large diameter",
    "Centro Y": "Y center",
    "Diámetro exterior pequeño": "Small outer diameter",
    "Diámetro exterior grande": "Large outer diameter",
    "Grosor radial del anillo": "Annulus radial thickness",
    "Frecuencia espacial": "Spatial frequency",
    "Color A": "Color A",
    "Color B": "Color B",
    "Frecuencia temporal": "Temporal frequency",
    "Frecuencia inversión": "Reversal frequency",
    "Tamaño Gabor": "Gabor size",
    "Duración noise": "Noise duration",
    "Frecuencia de actualización": "Update frequency",
    "Tamaño de check": "Check size",
    "Color bajo": "Low color",
    "Color medio": "Middle color",
    "Color alto": "High color",
    "Duración / condición": "Duration / condition",
    "Adaptación entre condiciones": "Inter-condition adaptation",
    "Posición X": "X position",
    "Posición Y": "Y position",
    "Diámetro interno": "Inner diameter",
    "Diámetro externo": "Outer diameter",
    "Tamaño del cuadrado": "Square size",
    "Paso de rejilla": "Grid spacing",
    "Adaptación basal": "Baseline adaptation",
    "Color alto / ON": "High color / ON",
    "Color bajo / OFF": "Low color / OFF",
    "Frecuencia mínima": "Minimum frequency",
    "Frecuencia máxima": "Maximum frequency",
    "Duración chirp": "Chirp duration",
    "Frecuencia ON/OFF": "ON/OFF frequency",
    "Frecuencia": "Frequency",
    "Frecuencia actualización": "Update frequency",
    "Frecuencia flicker": "Flicker frequency",
    "Sin nombre": "Untitled",
    "Cambios sin guardar": "Unsaved changes",
    "Añade al menos un paso antes de guardar.": "Add at least one step before saving.",
    "El archivo no contiene un objeto JSON válido.": "The file does not contain a valid JSON object.",
    "No se puede añadir": "Cannot add",
    "No se puede ejecutar": "Cannot run",
    "Duración estimada: ": "Estimated duration: ",
    "monitor experimental": "experimental display",
    "Espera segura: NEGRO": "Safe idle: BLACK",
    "Modo de trigger no reconocido": "Unrecognized trigger mode",
    "Detener prueba": "Stop test",
    "● TRIGGERS · error de diagnóstico": "● TRIGGERS · diagnostic error",
    "Prueba MaxOne Legacy enviada · comportamiento v1.17 conservado. Registro: ": "MaxOne Legacy test sent · v1.17 behavior preserved. Log: ",
    "Prueba TTL genérica enviada · contrasta FRAME/EVENT/UNIVERSAL. Registro: ": "Generic TTL test sent · compare FRAME/EVENT/UNIVERSAL. Log: ",
    "Prueba simulada completada · Registro: ": "Simulation test completed · Log: ",
    "● PRUEBA · ": "● TEST · ",
    "Modo de trigger desconocido: ": "Unknown trigger mode: ",
    "El pin simulado debe ser un entero >= 0.": "The simulated pin must be an integer >= 0.",
    "Asymmetric Drift: selecciona A_TO_B o B_TO_A.": "Asymmetric Drift: select A_TO_B or B_TO_A.",
    ": tamaño > 0 y 0 < sigma <= tamaño/2.": ": size > 0 and 0 < sigma <= size/2.",
    "Dense White Noise: usa Binary o Ternary.": "Dense White Noise: use Binary or Ternary.",
    "Frozen Noise: distribución no válida.": "Frozen Noise: invalid distribution.",
    "Temporal: los tiempos de adaptación deben ser >= 0.": "Temporal: adaptation times must be >= 0.",
    "Contrast Series: introduce porcentajes separados por comas.": "Contrast Series: enter comma-separated percentages.",
    "Capa de seguridad activa.": "Safety layer active.",
    "Capa segura activa · abriendo PsychoPy...": "Safety layer active · opening PsychoPy...",
    "Abriendo la ventana experimental sobre la capa segura": "Opening the experimental window over the safety layer",
    "El monitor experimental se mantiene en estado seguro": "The experimental display remains in a safe state",
    "Iniciando el motor PsychoPy": "Starting PsychoPy engine",
    "No se puede iniciar": "Cannot start",
    "PsychoPy está comprobando la temporización real del monitor": "PsychoPy is checking the display's actual timing",
    "TTL predeterminado": "Default TTL",
    "TTL predeterminado: ": "Default TTL: ",
    "Prueba de triggers": "Trigger test",
    "Capa segura activa": "Safety layer active",
    "Capa de seguridad": "Safety layer",
})

# Source code predates internationalisation and legitimately contains both
# Spanish and English UI strings. Build the Spanish presentation map from the
# English translations as well, then add native-English UI terms that were in
# the original interface.
_I18N_ES_EXACT = {v: k for k, v in _I18N_EN_EXACT.items()}
_I18N_ES_EXACT.update({
    "Stimulus parameters": "Parámetros del estímulo",
    "Stimulus preview": "Vista del estímulo",
    "Common parameters": "Parámetros comunes",
    "Selected step": "Paso seleccionado",
    "Saved protocols": "Protocolos guardados",
    "Protocol editor": "Editor del protocolo",
    "Add pause": "Añadir pausa",
    "Adaptation background": "Fondo de adaptación",
    "Save protocol": "Guardar protocolo",
    "Run protocol": "Ejecutar protocolo",
    "Open logs": "Abrir logs",
    "Refresh": "Actualizar",
    "Preview": "Vista previa",
    "Save": "Guardar",
    "Load": "Cargar",
    "Delete": "Eliminar",
    "Duplicate": "Duplicar",
    "New": "Nuevo",
    "Edit": "Editar",
    "Export...": "Exportar...",
    "Import...": "Importar...",
    "Motion": "Movimiento",
    "Grating": "Rejilla",
    "Noise": "Ruido",
    "RF Mapping": "Mapeo RF",
    "Protocols": "Protocolos",
    "System / TTL": "Sistema / TTL",
    "Language:": "Idioma:",
    "Spanish": "Español",
    "English": "English",
    "Experimental status": "Estado experimental",
    "Timing information: —": "Información temporal: —",
    "Setup configuration": "Configuración del montaje",
    "Stimulation display:": "Pantalla de estimulación:",
    "Display width:": "Anchura pantalla:",
    "Display height:": "Altura pantalla:",
    "Retina distance:": "Distancia a retina:",
})
_I18N_ES_PHRASES = [(en, es) for es, en in _I18N_EN_PHRASES]

def _tr_text(value, language=None):
    """Translate operator-facing text without changing internal identifiers."""
    if value is None:
        return value
    text = str(value)
    lang = language or _I18N_LANGUAGE
    if not text:
        return text
    if lang == "en":
        if text in _I18N_EN_EXACT:
            return _I18N_EN_EXACT[text]
        out = text
        for es, en in _I18N_EN_PHRASES:
            if es in out:
                out = out.replace(es, en)
        return out
    # Spanish mode must also normalize the English strings inherited from the
    # pre-i18n interface, so switching language never leaves a mixed UI.
    if text in _I18N_ES_EXACT:
        return _I18N_ES_EXACT[text]
    out = text
    for en, es in _I18N_ES_PHRASES:
        if en in out:
            out = out.replace(en, es)
    return out

def _set_i18n_language(language):
    global _I18N_LANGUAGE
    _I18N_LANGUAGE = "en" if str(language).lower().startswith("en") else "es"


def _install_i18n_hooks():
    """Install presentation-only Tk/ttk hooks once for this application."""
    global _I18N_HOOKS_INSTALLED
    if _I18N_HOOKS_INSTALLED:
        return
    _I18N_HOOKS_INSTALLED = True

    def hook_text_widget(namespace, name):
        original = getattr(namespace, name)
        def factory(master=None, *args, **kwargs):
            source_text = kwargs.get("text", None)
            source_var = kwargs.get("textvariable", None)
            mirror_var = None
            if source_text is not None:
                kwargs["text"] = _tr_text(source_text)
            if source_var is not None and name in ("Label",):
                try:
                    mirror_var = tk.StringVar(master=master, value=_tr_text(source_var.get()))
                    kwargs["textvariable"] = mirror_var
                except Exception:
                    mirror_var = None
            widget = original(master, *args, **kwargs)
            if source_text is not None:
                widget._i18n_source_text = source_text
            if source_var is not None and mirror_var is not None:
                widget._i18n_source_var = source_var
                widget._i18n_mirror_var = mirror_var
                def sync(*_):
                    try:
                        mirror_var.set(_tr_text(source_var.get()))
                    except Exception:
                        pass
                try:
                    source_var.trace_add("write", sync)
                except Exception:
                    pass
            # Translate later configure(text=...) calls while retaining source text.
            try:
                original_configure = widget.configure
                def configure_i18n(cnf=None, **kw):
                    if isinstance(cnf, dict):
                        kw = {**cnf, **kw}
                        cnf = None
                    if "text" in kw:
                        widget._i18n_source_text = kw["text"]
                        kw["text"] = _tr_text(kw["text"])
                    return original_configure(cnf, **kw) if cnf is not None else original_configure(**kw)
                widget.configure = configure_i18n
                widget.config = configure_i18n
            except Exception:
                pass
            return widget
        setattr(namespace, name, factory)

    for ns, names in ((ttk, ("Label", "Button", "Checkbutton", "Radiobutton", "LabelFrame")),
                      (tk, ("Label", "Button", "Checkbutton", "Radiobutton", "LabelFrame"))):
        for name in names:
            if hasattr(ns, name):
                hook_text_widget(ns, name)

    # Secondary-window titles (tk.Toplevel) are not widget text options.
    # Translate them globally so every submenu/dialog uses the active language.
    original_toplevel_title = tk.Toplevel.title
    def toplevel_title_i18n(self, string=None):
        if string is None:
            return original_toplevel_title(self)
        try:
            self._i18n_title_source = string
        except Exception:
            pass
        return original_toplevel_title(self, _tr_text(string))
    tk.Toplevel.title = toplevel_title_i18n

    # Notebook tabs.
    original_add = ttk.Notebook.add
    def add_i18n(self, child, **kw):
        source = kw.get("text")
        if source is not None:
            try:
                child._i18n_tab_source_text = source
            except Exception:
                pass
            kw["text"] = _tr_text(source)
        return original_add(self, child, **kw)
    ttk.Notebook.add = add_i18n

    # Read-only comboboxes get a translated display variable while preserving
    # the original source StringVar used by stimulus logic and saved configs.
    original_combo = ttk.Combobox
    def combo_factory(master=None, *args, **kwargs):
        source_values = list(kwargs.get("values", ()) or ())
        source_var = kwargs.get("textvariable", None)
        needs_layer = source_var is not None and source_values
        if not needs_layer:
            return original_combo(master, *args, **kwargs)
        display_values = [_tr_text(v) for v in source_values]
        try:
            current = source_var.get()
        except Exception:
            current = ""
        try:
            idx = source_values.index(current)
            display_current = display_values[idx]
        except Exception:
            display_current = _tr_text(current)
        display_var = tk.StringVar(master=master, value=display_current)
        kwargs["textvariable"] = display_var
        kwargs["values"] = display_values
        widget = original_combo(master, *args, **kwargs)
        widget._i18n_combo_source_values = source_values
        widget._i18n_combo_source_var = source_var
        widget._i18n_combo_display_var = display_var
        guard = {"busy": False}
        def from_display(*_):
            if guard["busy"]:
                return
            guard["busy"] = True
            try:
                shown = display_var.get()
                translated = [_tr_text(v) for v in source_values]
                if shown in translated:
                    source_var.set(source_values[translated.index(shown)])
                else:
                    source_var.set(shown)
            finally:
                guard["busy"] = False
        def from_source(*_):
            if guard["busy"]:
                return
            guard["busy"] = True
            try:
                value = source_var.get()
                translated = [_tr_text(v) for v in source_values]
                if value in source_values:
                    display_var.set(translated[source_values.index(value)])
                else:
                    display_var.set(_tr_text(value))
            finally:
                guard["busy"] = False
        try:
            display_var.trace_add("write", from_display)
            source_var.trace_add("write", from_source)
        except Exception:
            pass
        return widget
    ttk.Combobox = combo_factory

    # Standard dialogs are translated at call time.
    for fname in ("showinfo", "showwarning", "showerror", "askyesno", "askokcancel", "askretrycancel"):
        if hasattr(messagebox, fname):
            original = getattr(messagebox, fname)
            def make_dialog_wrapper(fn):
                def wrapped(title, message=None, *args, **kwargs):
                    return fn(_tr_text(title), _tr_text(message), *args, **kwargs)
                return wrapped
            setattr(messagebox, fname, make_dialog_wrapper(original))


    # Numeric prompt used by the Protocol Builder.
    if hasattr(simpledialog, "askfloat"):
        _askfloat_original = simpledialog.askfloat
        def askfloat_i18n(title, prompt, *args, **kwargs):
            return _askfloat_original(_tr_text(title), _tr_text(prompt), *args, **kwargs)
        simpledialog.askfloat = askfloat_i18n

    # Native open/save dialogs: translate title and file-type descriptions only.
    for _fd_name in ("askopenfilename", "asksaveasfilename"):
        if hasattr(filedialog, _fd_name):
            _fd_original = getattr(filedialog, _fd_name)
            def _make_fd_wrapper(fn):
                def wrapped(*args, **kwargs):
                    if "title" in kwargs:
                        kwargs["title"] = _tr_text(kwargs["title"])
                    if "filetypes" in kwargs:
                        kwargs["filetypes"] = [
                            (_tr_text(desc), pattern)
                            for desc, pattern in kwargs["filetypes"]
                        ]
                    return fn(*args, **kwargs)
                return wrapped
            setattr(filedialog, _fd_name, _make_fd_wrapper(_fd_original))

class App(tk.Tk):
    def __init__(self):
        super().__init__()

        # Keep the normal operator interface hidden until PsychoPy is ready.
        # The splash window is shown instead, only on the operator monitor.
        self.withdraw()

        self.title(
            f"MEA Visual Stimuli v{APP_VERSION}"
        )
        # v1.22: never create the operator window larger than the available
        # desktop. This is deliberately computed before building widgets so
        # 1366×768 laptops and Windows 125/150 % scaling remain usable.
        screen_w = max(800, int(self.winfo_screenwidth()))
        screen_h = max(600, int(self.winfo_screenheight()))
        win_w = max(760, min(1320, screen_w - 40))
        win_h = max(580, min(920, screen_h - 90))
        pos_x = max(0, (screen_w - win_w) // 2)
        pos_y = max(0, (screen_h - win_h) // 3)
        self.geometry(f"{win_w}x{win_h}+{pos_x}+{pos_y}")
        self.minsize(min(760, win_w), min(580, win_h))

        self.saved = load_saved()

        # v2.1.1 bilingual UI. English is the default for new/public installations.
        self.language = str(self.saved.get("language", "en")).lower()
        if self.language not in ("es", "en"):
            self.language = "en"
        _set_i18n_language(self.language)
        _install_i18n_hooks()
        self.language_display_var = tk.StringVar(
            value="English" if self.language == "en" else "Español"
        )

        # Persistent Protocol Builder library. Protocol definitions are
        # user data and therefore live outside the notebook in protocols.json.
        self.protocol_library = load_protocol_library()
        self.protocol_draft = {
            "id": None,
            "name": self._tr("Nuevo protocolo"),
            "version": 0,
            "repetitions": 1,
            "steps": [],
        }
        self.protocol_dirty = False
        self.protocol_edit_step_index = None
        self.protocol_add_buttons = {}

        self.screens = []

        self.safety_process = None
        self.safety_file = None

        self.worker_process = None
        self.worker_file = None
        self.worker_console = None

        self.abort_file_handle = None
        self.abort_map = None
        self.abort_path = RUN_DIR / "abort.bin"

        # Safe protocol pause control. It is never read from a timed frame
        # loop; the worker consults it only between protocol steps.
        self.protocol_pause_path = (
            RUN_DIR / "protocol_pause.json"
        )
        self.protocol_pause_state = "idle"
        self.active_run_type = None

        self.status_path = RUN_DIR / "status.json"
        self.command_path = RUN_DIR / "command.json"
        self.result_path = RUN_DIR / "result.json"
        self.startup_config_path = (
            RUN_DIR / "startup_config.json"
        )

        self.last_status_signature = None
        self.active_command_id = None
        self.pending_config = None
        self.current_worker_screen = None
        self.intentional_worker_stop = False

        self.screen_var = tk.StringVar()
        self.status = tk.StringVar(
            value="Preparando pantalla de seguridad..."
        )
        self.refresh_info = tk.StringVar(
            value="Refresco: todavía no medido"
        )
        self.timing_info = tk.StringVar(
            value="Información temporal: —"
        )
        self.current_measured_hz = None

        # Session metadata are intentionally session-local: they persist
        # while the app is open, but are not silently restored on a future
        # day/run, reducing the risk of carrying sample metadata forward.
        self.session_metadata = {
            "session_id": "",
            "sample_id": "",
            "preparation": "",
            "eye_retina": "",
            "group_treatment": "",
            "operator": "",
            "notes": "",
            "defined_at": "",
        }
        self.session_summary_var = tk.StringVar(
            value="Sesión: sin metadatos"
        )
        self.session_dialog = None

        self.last_run_folder = None

        self.width_cm = tk.StringVar(
            value=str(
                self.saved.get(
                    "physical_width_cm", 31.0
                )
            )
        )
        self.height_cm = tk.StringVar(
            value=str(
                self.saved.get(
                    "physical_height_cm", 19.4
                )
            )
        )
        self.distance_cm = tk.StringVar(
            value=str(
                self.saved.get(
                    "distance_cm", 20.0
                )
            )
        )

        ttl_saved = self.saved.get("ttl", {})
        # Fresh installations start with the validated generic TTL box.
        # Existing v1.19 users retain their last mode unless they explicitly
        # saved a different startup default in the new System / TTL tab.
        ttl_start_mode = ttl_saved.get(
            "default_mode",
            ttl_saved.get("mode", "Trigger Box TTL genérica")
        )
        ttl_start_port = ttl_saved.get(
            "default_port",
            ttl_saved.get("port", "auto")
        )
        ttl_start_pin = ttl_saved.get(
            "default_pin",
            ttl_saved.get("pin", 8)
        )
        self.ttl_mode = tk.StringVar(value=str(ttl_start_mode))
        self.ttl_pin = tk.StringVar(value=str(ttl_start_pin))
        self.ttl_port = tk.StringVar(value=str(ttl_start_port))
        self.ttl_status_text = tk.StringVar(
            value="● TTL GENÉRICO · UNIVERSAL + AUX ESTADO/PAUSA"
        )
        self.ttl_header_text = tk.StringVar(value="TTL: Genérico · auto")

        display_saved = self.saved.get("display_safety", {})
        self.adaptation_hex = tk.StringVar(
            value=str(display_saved.get("adaptation_hex", "#808080")).upper()
        )
        self.red_mode_intensity = tk.StringVar(
            value=str(display_saved.get("red_intensity_pct", 10))
        )
        # Kept only for backwards compatibility with v1.20/v1.21 settings.
        # v1.22 never alters the Windows desktop, gamma ramp or experimental monitor.
        self.red_global_filter = tk.BooleanVar(value=False)
        self.red_mode_active = False
        self.red_mode_status = tk.StringVar(value="Modo rojo: inactivo")
        # Dark-room state. Since v1.22 this is deliberately operator-GUI only.
        # Legacy filter state variables are retained solely so an old session can
        # always be restored safely; no new OS-wide filter is ever applied.
        self._gamma_original = None
        self._gamma_dc = None
        self._gamma_device_name = None
        self._gamma_filter_active = False
        self._mag_initialized = False
        self._mag_original_effect = None
        self._mag_filter_active = False
        self._global_red_backend = ""
        self._red_filter_suspended_for_run = False
        self._color_selector_widgets = []
        atexit.register(self._restore_global_gamma_safely)
        self.ttl_test_after_ids = []
        self._trigger_test_active = False
        self._trigger_test_backend = None
        self._trigger_test_path = None
        self._trigger_test_events = []
        self._trigger_test_mode = ""

        # Operator-side preview state. This never communicates with PsychoPy.
        self.preview_canvases = {}
        self.preview_summary_vars = {}
        self.preview_after_id = None
        self.preview_running_key = None
        self.preview_step = 0

        # Hidden operator-only easter egg animation.
        self.easter_after_id = None
        self.easter_frame_images = []
        self.easter_active_key = None
        self.easter_sequence = []
        self.easter_position = 0

        # Splash / startup state.
        self.splash = None
        self.splash_image = None
        self.splash_status_var = tk.StringVar(
            value="Preparando MEA Visual Stimuli..."
        )
        self.splash_detail_var = tk.StringVar(
            value="SMN-Systems"
        )
        self.splash_started = time.perf_counter()
        self.splash_ready_requested = False
        self.splash_error = False

        self._configure_styles()
        self._create_splash()
        self._splash_update(
            "Construyendo interfaz...",
            "Preparando controles del operador"
        )

        self._build()
        self._refresh_i18n_widgets()

        self._splash_update(
            "Detectando pantallas...",
            "Identificando monitor del operador y monitor experimental"
        )
        self.refresh_screens()

        self._splash_update(
            "Pantallas detectadas",
            "Preparando la capa de seguridad experimental"
        )

        self.combo.bind(
            "<<ComboboxSelected>>",
            self.on_screen_changed
        )

        self.protocol(
            "WM_DELETE_WINDOW",
            self.on_close
        )

        # Open safe backing layer first, then PsychoPy over it.
        self.after(200, self.initialize_display_stack)
        self.after(120, self.poll_worker)

    def _tr(self, text):
        return _tr_text(text, self.language)

    def _refresh_i18n_widgets(self):
        """Refresh already-created widgets after an in-session language change."""
        def walk(widget):
            try:
                source = getattr(widget, "_i18n_source_text", None)
                if source is not None:
                    widget.configure(text=_tr_text(source, self.language))
                    # configure() is itself i18n-aware; restore the canonical
                    # Spanish source so switching back remains lossless.
                    widget._i18n_source_text = source
            except Exception:
                pass
            try:
                source_var = getattr(widget, "_i18n_source_var", None)
                mirror_var = getattr(widget, "_i18n_mirror_var", None)
                if source_var is not None and mirror_var is not None:
                    mirror_var.set(_tr_text(source_var.get(), self.language))
            except Exception:
                pass
            try:
                source_values = getattr(widget, "_i18n_combo_source_values", None)
                source_var = getattr(widget, "_i18n_combo_source_var", None)
                display_var = getattr(widget, "_i18n_combo_display_var", None)
                if source_values is not None:
                    translated = [_tr_text(v, self.language) for v in source_values]
                    widget.configure(values=translated)
                    if source_var is not None and display_var is not None:
                        value = source_var.get()
                        if value in source_values:
                            display_var.set(translated[source_values.index(value)])
                        else:
                            display_var.set(_tr_text(value, self.language))
            except Exception:
                pass
            try:
                if isinstance(widget, ttk.Notebook):
                    for tab_id in widget.tabs():
                        child = self.nametowidget(tab_id)
                        source = getattr(child, "_i18n_tab_source_text", None)
                        if source is not None:
                            widget.tab(tab_id, text=_tr_text(source, self.language))
            except Exception:
                pass
            try:
                for child in widget.winfo_children():
                    walk(child)
            except Exception:
                pass
        walk(self)
        # Secondary windows have native titles, outside the widget tree text options.
        try:
            for child in self.winfo_children():
                if isinstance(child, tk.Toplevel):
                    source_title = getattr(child, "_i18n_title_source", None)
                    if source_title is not None:
                        child.title(_tr_text(source_title, self.language))
                        child._i18n_title_source = source_title
        except Exception:
            pass
        # Treeview headings are not normal child widgets, so refresh them explicitly.
        try:
            if hasattr(self, "protocol_tree"):
                self.protocol_tree.heading("number", text="#")
                self.protocol_tree.heading("type", text=self._tr("Estímulo"))
                self.protocol_tree.heading("summary", text=self._tr("Configuración"))
                self.protocol_tree.heading("duration", text=self._tr("Duración"))
        except Exception:
            pass
        # Rebuild dynamic protocol strings/list entries in the selected language.
        try:
            self._refresh_protocol_library()
            self._refresh_protocol_editor()
        except Exception:
            pass

    def _on_language_selected(self, _event=None):
        selected = self.language_display_var.get().strip().lower()
        self.language = "en" if selected.startswith("english") else "es"
        _set_i18n_language(self.language)
        # Persist only the language preference, preserving all other settings.
        try:
            saved = load_saved()
            saved["language"] = self.language
            write_json_atomic(CONFIG_FILE, saved)
            self.saved = saved
        except Exception:
            pass
        self._refresh_i18n_widgets()
        # Re-render dynamic header/status text using the presentation layer.
        try:
            self.update_ttl_ui()
        except Exception:
            pass

    def _session_metadata_snapshot(self):
        meta = {
            key: str(
                self.session_metadata.get(
                    key,
                    ""
                )
            ).strip()
            for key in (
                "session_id",
                "sample_id",
                "preparation",
                "eye_retina",
                "group_treatment",
                "operator",
                "notes",
                "defined_at",
            )
        }

        meta["captured_at"] = (
            datetime.now().isoformat(
                timespec="seconds"
            )
        )

        return meta

    def _update_session_summary(self):
        session_id = str(
            self.session_metadata.get(
                "session_id",
                ""
            )
        ).strip()
        sample_id = str(
            self.session_metadata.get(
                "sample_id",
                ""
            )
        ).strip()
        eye_retina = str(
            self.session_metadata.get(
                "eye_retina",
                ""
            )
        ).strip()

        primary = (
            session_id
            or sample_id
        )

        if not primary:
            self.session_summary_var.set(
                "Sesión: sin metadatos"
            )
            return

        text = (
            f"Sesión: {primary}"
        )
        if (
            eye_retina
            and eye_retina.casefold()
            != primary.casefold()
        ):
            text += (
                f" · {eye_retina}"
            )

        self.session_summary_var.set(
            text
        )

    def _clear_session_metadata(
        self,
        close_dialog=False
    ):
        self.session_metadata = {
            "session_id": "",
            "sample_id": "",
            "preparation": "",
            "eye_retina": "",
            "group_treatment": "",
            "operator": "",
            "notes": "",
            "defined_at": "",
        }
        self._update_session_summary()

        if (
            close_dialog
            and self.session_dialog is not None
        ):
            try:
                self.session_dialog.destroy()
            except Exception:
                pass
            self.session_dialog = None

        self.status.set(
            "Metadatos de sesión limpiados."
        )

    def _open_session_metadata_dialog(self):
        if (
            self.session_dialog is not None
        ):
            try:
                if self.session_dialog.winfo_exists():
                    self.session_dialog.lift()
                    self.session_dialog.focus_force()
                    return
            except Exception:
                pass

        dialog = tk.Toplevel(
            self
        )
        self.session_dialog = dialog

        dialog.title(
            self._tr("Sesión experimental / metadatos")
        )
        dialog.geometry(
            "760x470"
        )
        dialog.minsize(
            680,
            430
        )
        dialog.transient(
            self
        )

        outer = ttk.Frame(
            dialog,
            padding=16
        )
        outer.pack(
            fill="both",
            expand=True
        )

        ttk.Label(
            outer,
            text="Metadatos de la sesión experimental",
            style="Section.TLabel"
        ).grid(
            row=0,
            column=0,
            columnspan=4,
            sticky="w"
        )

        ttk.Label(
            outer,
            text=(
                "Todos los campos son opcionales. Se aplicarán a las "
                "ejecuciones realizadas mientras esta sesión permanezca "
                "activa y quedarán guardados en el JSON de configuración."
            ),
            style="Muted.TLabel",
            wraplength=700
        ).grid(
            row=1,
            column=0,
            columnspan=4,
            sticky="w",
            pady=(3, 14)
        )

        vars_local = {
            key: tk.StringVar(
                value=str(
                    self.session_metadata.get(
                        key,
                        ""
                    )
                )
            )
            for key in (
                "session_id",
                "sample_id",
                "preparation",
                "eye_retina",
                "group_treatment",
                "operator",
            )
        }

        labels = (
            (
                "ID de sesión",
                "session_id",
                2,
                0
            ),
            (
                "ID de muestra",
                "sample_id",
                2,
                2
            ),
            (
                "Preparación",
                "preparation",
                3,
                0
            ),
            (
                "Ojo / retina",
                "eye_retina",
                3,
                2
            ),
            (
                "Grupo / tratamiento",
                "group_treatment",
                4,
                0
            ),
            (
                "Operador",
                "operator",
                4,
                2
            ),
        )

        for (
            label_text,
            key,
            row,
            col,
        ) in labels:
            ttk.Label(
                outer,
                text=self._tr(label_text + ":")
            ).grid(
                row=row,
                column=col,
                sticky="w",
                padx=(
                    0,
                    8
                ),
                pady=6
            )

            ttk.Entry(
                outer,
                textvariable=vars_local[
                    key
                ],
                width=24
            ).grid(
                row=row,
                column=col + 1,
                sticky="ew",
                padx=(
                    0,
                    18
                ),
                pady=6
            )

        ttk.Label(
            outer,
            text="Notas:"
        ).grid(
            row=5,
            column=0,
            sticky="nw",
            pady=(10, 4)
        )

        notes = tk.Text(
            outer,
            height=7,
            wrap="word",
            bd=1,
            relief="solid"
        )
        notes.grid(
            row=6,
            column=0,
            columnspan=4,
            sticky="nsew",
            pady=(0, 12)
        )
        notes.insert(
            "1.0",
            str(
                self.session_metadata.get(
                    "notes",
                    ""
                )
            )
        )

        button_row = ttk.Frame(
            outer
        )
        button_row.grid(
            row=7,
            column=0,
            columnspan=4,
            sticky="e"
        )

        def close_dialog():
            try:
                dialog.destroy()
            finally:
                self.session_dialog = None

        def clear_dialog():
            for variable in (
                vars_local.values()
            ):
                variable.set("")
            notes.delete(
                "1.0",
                "end"
            )

        def save_dialog():
            defined_at = str(
                self.session_metadata.get(
                    "defined_at",
                    ""
                )
            ).strip()

            values = {
                key:
                    variable.get().strip()
                for (
                    key,
                    variable
                ) in vars_local.items()
            }
            values[
                "notes"
            ] = notes.get(
                "1.0",
                "end-1c"
            ).strip()

            any_content = any(
                values.values()
            )

            if (
                any_content
                and not defined_at
            ):
                defined_at = (
                    datetime.now().isoformat(
                        timespec="seconds"
                    )
                )

            if not any_content:
                defined_at = ""

            values[
                "defined_at"
            ] = defined_at

            self.session_metadata = values
            self._update_session_summary()

            if any_content:
                self.status.set(
                    "Metadatos de sesión actualizados."
                )
            else:
                self.status.set(
                    "Sesión sin metadatos activos."
                )

            close_dialog()

        ttk.Button(
            button_row,
            text="Limpiar campos",
            command=clear_dialog,
            style="Secondary.TButton"
        ).pack(
            side="left",
            padx=(0, 8)
        )

        ttk.Button(
            button_row,
            text="Cancelar",
            command=close_dialog,
            style="Secondary.TButton"
        ).pack(
            side="left",
            padx=(0, 8)
        )

        ttk.Button(
            button_row,
            text="Guardar metadatos",
            command=save_dialog,
            style="Accent.TButton"
        ).pack(
            side="left"
        )

        for col in (
            1,
            3
        ):
            outer.columnconfigure(
                col,
                weight=1
            )
        outer.rowconfigure(
            6,
            weight=1
        )

        dialog.protocol(
            "WM_DELETE_WINDOW",
            close_dialog
        )

        dialog.grab_set()
        dialog.focus_force()

    def _set_last_run_folder(
        self,
        folder
    ):
        folder = str(
            folder
            or ""
        ).strip()

        if not folder:
            return

        self.last_run_folder = folder


    def _open_logs_folder(self):
        path = Path(
            LOG_DIR
        )

        try:
            path.mkdir(
                parents=True,
                exist_ok=True
            )
        except Exception as exc:
            messagebox.showerror(
                "Abrir logs",
                (
                    "No se ha podido preparar la carpeta de logs.\n\n"
                    f"{exc}"
                ),
                parent=self
            )
            return

        try:
            if os.name == "nt":
                os.startfile(
                    str(path)
                )
            elif sys.platform == "darwin":
                subprocess.Popen(
                    [
                        "open",
                        str(path)
                    ]
                )
            else:
                subprocess.Popen(
                    [
                        "xdg-open",
                        str(path)
                    ]
                )
        except Exception as exc:
            messagebox.showerror(
                "Abrir logs",
                (
                    "No se ha podido abrir la carpeta de logs.\n\n"
                    f"{exc}"
                ),
                parent=self
            )


    def _open_manual(self):
        """Open the offline v2.0 user manual without requiring Internet."""
        candidates = []

        try:
            if getattr(sys, "frozen", False):
                candidates.append(
                    Path(sys.executable).resolve().parent / MANUAL_FILENAME
                )
        except Exception:
            pass

        try:
            if "__file__" in globals():
                candidates.append(
                    Path(__file__).resolve().parent / MANUAL_FILENAME
                )
        except Exception:
            pass

        try:
            candidates.append(Path.cwd() / MANUAL_FILENAME)
        except Exception:
            pass

        # Future installer builds may place documentation in a docs subfolder.
        expanded = []
        for candidate in candidates:
            expanded.append(candidate)
            expanded.append(candidate.parent / "docs" / MANUAL_FILENAME)

        manual_path = next(
            (p for p in expanded if p.exists() and p.is_file()),
            None
        )

        if manual_path is None:
            locations = "\n".join(
                f"• {p}" for p in expanded[:6]
            )
            messagebox.showwarning(
                "Manual / Ayuda",
                (
                    "No se encuentra el manual offline.\n\n"
                    f"Archivo esperado: {MANUAL_FILENAME}\n\n"
                    "Coloca el PDF junto al programa (o en una carpeta "
                    "«docs» junto al programa) y vuelve a intentarlo.\n\n"
                    f"Ubicaciones comprobadas:\n{locations}"
                ),
                parent=self
            )
            return

        try:
            if os.name == "nt":
                os.startfile(str(manual_path))
            elif sys.platform == "darwin":
                subprocess.Popen(["open", str(manual_path)])
            else:
                subprocess.Popen(["xdg-open", str(manual_path)])
            self.status.set(
                f"Manual abierto: {manual_path.name}"
            )
        except Exception as exc:
            messagebox.showerror(
                "Manual / Ayuda",
                (
                    "No se ha podido abrir el manual.\n\n"
                    f"{manual_path}\n\n{exc}"
                ),
                parent=self
            )


    def _get_operator_screen_for_splash(self):
        """
        Choose a monitor different from the saved experimental monitor.
        With one screen, fall back to that screen.
        """
        try:
            screens = detect_screens()
            if not screens:
                return None

            default_exp = (
                1 if len(screens) > 1 else 0
            )
            exp_idx = int(
                self.saved.get(
                    "screen_index",
                    default_exp
                )
            )
            if not (
                0 <= exp_idx < len(screens)
            ):
                exp_idx = default_exp

            for screen in screens:
                if screen["index"] != exp_idx:
                    return screen

            return screens[0]
        except Exception:
            return None

    def _create_splash(self):
        try:
            splash = tk.Toplevel(self)
            self.splash = splash

            splash.overrideredirect(True)
            splash.configure(bg="#FFFFFF")

            # Keep it above normal desktop windows, but it is explicitly
            # positioned on the operator monitor, never the experimental one.
            try:
                splash.attributes(
                    "-topmost", True
                )
            except Exception:
                pass

            outer = tk.Frame(
                splash,
                bg="#FFFFFF",
                highlightbackground="#C9D5DB",
                highlightthickness=1,
                bd=0
            )
            outer.pack(
                fill="both",
                expand=True
            )

            # Tk PhotoImage can read the embedded PNG directly from base64.
            full_image = tk.PhotoImage(
                data=SPLASH_IMAGE_B64
            )
            self.splash_image = full_image

            image_label = tk.Label(
                outer,
                image=self.splash_image,
                bg="#FFFFFF",
                bd=0
            )
            image_label.pack(
                padx=18,
                pady=(16, 6)
            )

            tk.Label(
                outer,
                text="MEA Visual Stimuli",
                bg="#FFFFFF",
                fg="#1F2D33",
                font=(
                    "Segoe UI Semibold",
                    15
                )
            ).pack(
                pady=(1, 0)
            )

            tk.Label(
                outer,
                text=f"Versión {APP_VERSION}",
                bg="#FFFFFF",
                fg="#677983",
                font=("Segoe UI", 9)
            ).pack(
                pady=(1, 10)
            )

            status_label = tk.Label(
                outer,
                textvariable=self.splash_status_var,
                bg="#FFFFFF",
                fg="#255A73",
                font=(
                    "Segoe UI Semibold",
                    10
                )
            )
            status_label.pack(
                padx=20,
                pady=(0, 3)
            )
            self.splash_status_label = (
                status_label
            )

            detail_label = tk.Label(
                outer,
                textvariable=self.splash_detail_var,
                bg="#FFFFFF",
                fg="#677983",
                font=("Segoe UI", 9),
                wraplength=500,
                justify="center"
            )
            detail_label.pack(
                padx=24,
                pady=(0, 9)
            )
            self.splash_detail_label = (
                detail_label
            )

            self.splash_progress = ttk.Progressbar(
                outer,
                orient="horizontal",
                mode="indeterminate",
                length=360
            )
            self.splash_progress.pack(
                pady=(0, 13)
            )
            self.splash_progress.start(12)

            self.splash_open_btn = ttk.Button(
                outer,
                text="Abrir interfaz",
                command=self._force_open_interface,
                style="Secondary.TButton"
            )

            splash.update_idletasks()

            sw = splash.winfo_reqwidth()
            sh = splash.winfo_reqheight()

            operator_screen = (
                self._get_operator_screen_for_splash()
            )

            if operator_screen is not None:
                x = int(
                    operator_screen["x"]
                    + (
                        operator_screen["width"]
                        - sw
                    ) / 2
                )
                y = int(
                    operator_screen["y"]
                    + (
                        operator_screen["height"]
                        - sh
                    ) / 2
                )
            else:
                x = int(
                    (
                        splash.winfo_screenwidth()
                        - sw
                    ) / 2
                )
                y = int(
                    (
                        splash.winfo_screenheight()
                        - sh
                    ) / 2
                )

            splash.geometry(
                f"{sw}x{sh}{x:+d}{y:+d}"
            )
            splash.update_idletasks()
            splash.update()

        except Exception:
            # A splash failure must never prevent the experiment software
            # from opening. Continue with the normal interface.
            self.splash = None
            self.splash_image = None
            self.after(
                1,
                self._force_open_interface
            )

    def _splash_update(
        self, status_text, detail_text=""
    ):
        if (
            self.splash is None
            or not self.splash.winfo_exists()
        ):
            return

        try:
            self.splash_status_var.set(
                str(status_text)
            )
            self.splash_detail_var.set(
                str(detail_text)
            )
            self.splash.update_idletasks()
        except Exception:
            pass

    def _splash_show_error(
        self, message
    ):
        self.splash_error = True

        if (
            self.splash is None
            or not self.splash.winfo_exists()
        ):
            self._force_open_interface()
            return

        try:
            self.splash_progress.stop()
            self.splash_progress.pack_forget()

            self.splash_status_label.configure(
                fg="#B65353"
            )

            self.splash_status_var.set(
                "No se pudo completar la inicialización"
            )
            self.splash_detail_var.set(
                str(message)
            )

            if not self.splash_open_btn.winfo_manager():
                self.splash_open_btn.pack(
                    pady=(0, 14)
                )
        except Exception:
            self._force_open_interface()

    def _request_splash_close(
        self, hz=None
    ):
        if self.splash_ready_requested:
            return

        self.splash_ready_requested = True

        if hz is not None:
            self._splash_update(
                "Sistema preparado",
                (
                    "Pantalla experimental estable · "
                    f"{float(hz):.3f} Hz · espera segura"
                )
            )
        else:
            self._splash_update(
                "Sistema preparado",
                "Abriendo interfaz del operador"
            )

        elapsed_ms = int(
            (
                time.perf_counter()
                - self.splash_started
            ) * 1000
        )
        wait_ms = max(
            0,
            SPLASH_MIN_VISIBLE_MS
            - elapsed_ms
        )

        self.after(
            wait_ms,
            self._finish_splash
        )

    def _finish_splash(self):
        if self.splash_error:
            return

        try:
            if self.splash is not None:
                try:
                    self.splash_progress.stop()
                except Exception:
                    pass
                try:
                    self.splash.destroy()
                except Exception:
                    pass
        finally:
            self.splash = None
            self.deiconify()
            try:
                if int(self.winfo_screenheight()) <= 900:
                    self.state("zoomed")
            except Exception:
                pass
            try:
                self.lift()
                self.focus_force()
            except Exception:
                pass

    def _force_open_interface(self):
        self.splash_error = False

        try:
            if self.splash is not None:
                try:
                    self.splash_progress.stop()
                except Exception:
                    pass
                try:
                    self.splash.destroy()
                except Exception:
                    pass
        finally:
            self.splash = None
            self.deiconify()
            try:
                if int(self.winfo_screenheight()) <= 900:
                    self.state("zoomed")
            except Exception:
                pass
            try:
                self.lift()
                self.focus_force()
            except Exception:
                pass

    def _configure_styles(self):
        self.palette = {
            "bg": "#F3F6F8",
            "panel": "#FFFFFF",
            "panel_alt": "#F8FAFB",
            "border": "#D9E2E7",
            "text": "#1F2D33",
            "muted": "#677983",
            "accent": "#2F6F8F",
            "accent_dark": "#255A73",
            "success": "#3B7D62",
            "danger": "#B65353",
            "danger_dark": "#914141",
            "preview_bg": "#EEF2F4",
            "gray50": "#808080",
        }

        self.configure(bg=self.palette["bg"])
        self.option_add("*Font", ("Segoe UI", 10))

        style = ttk.Style(self)
        try:
            style.theme_use("clam")
        except Exception:
            pass

        style.configure(
            "TFrame",
            background=self.palette["panel"]
        )
        style.configure(
            "TLabel",
            background=self.palette["panel"],
            foreground=self.palette["text"],
            font=("Segoe UI Semibold", 11)
        )
        style.configure(
            "TLabelframe",
            background=self.palette["panel"],
            bordercolor=self.palette["border"],
            relief="solid",
            borderwidth=1
        )
        style.configure(
            "TLabelframe.Label",
            background=self.palette["panel"],
            foreground=self.palette["text"],
            font=("Segoe UI Semibold", 11)
        )

        style.configure(
            "App.TFrame",
            background=self.palette["bg"]
        )
        style.configure(
            "Card.TFrame",
            background=self.palette["panel"]
        )
        style.configure(
            "TabBody.TFrame",
            background=self.palette["panel"]
        )

        style.configure(
            "Title.TLabel",
            background=self.palette["bg"],
            foreground=self.palette["text"],
            font=("Segoe UI Semibold", 24)
        )
        style.configure(
            "Subtitle.TLabel",
            background=self.palette["bg"],
            foreground=self.palette["muted"],
            font=("Segoe UI Semibold", 11)
        )
        style.configure(
            "Section.TLabel",
            background=self.palette["panel"],
            foreground=self.palette["text"],
            font=("Segoe UI Semibold", 10)
        )
        style.configure(
            "Muted.TLabel",
            background=self.palette["panel"],
            foreground=self.palette["muted"],
            font=("Segoe UI", 9)
        )
        style.configure(
            "Status.TLabel",
            background=self.palette["panel"],
            foreground=self.palette["text"],
            font=("Segoe UI Semibold", 10)
        )
        style.configure(
            "Badge.TLabel",
            background="#E8F2F6",
            foreground=self.palette["accent_dark"],
            padding=(10, 5),
            font=("Segoe UI Semibold", 10)
        )
        style.configure(
            "ProtocolDuration.TLabel",
            background=self.palette["panel"],
            foreground=self.palette["accent_dark"],
            font=("Segoe UI Semibold", 18)
        )
        style.configure(
            "ProtocolDurationCaption.TLabel",
            background=self.palette["panel"],
            foreground=self.palette["muted"],
            font=("Segoe UI", 9)
        )

        style.configure(
            "Card.TLabelframe",
            background=self.palette["panel"],
            bordercolor=self.palette["border"],
            relief="solid",
            borderwidth=1
        )
        style.configure(
            "Card.TLabelframe.Label",
            background=self.palette["panel"],
            foreground=self.palette["text"],
            font=("Segoe UI Semibold", 10)
        )

        style.configure(
            "TNotebook",
            background=self.palette["bg"],
            borderwidth=0
        )
        style.configure(
            "TNotebook.Tab",
            background="#E8EEF1",
            foreground=self.palette["muted"],
            padding=(13, 7),
            font=("Segoe UI Semibold", 10),
            borderwidth=0
        )
        style.map(
            "TNotebook.Tab",
            background=[
                ("selected", self.palette["panel"]),
                ("active", "#F4F7F9"),
            ],
            foreground=[
                ("selected", self.palette["accent_dark"]),
                ("active", self.palette["text"]),
            ]
        )

        style.configure(
            "Accent.TButton",
            background=self.palette["accent"],
            foreground="white",
            padding=(15, 8),
            borderwidth=0,
            font=("Segoe UI Semibold", 10)
        )
        style.map(
            "Accent.TButton",
            background=[
                ("active", self.palette["accent_dark"]),
                ("disabled", "#A7BBC5"),
            ],
            foreground=[
                ("disabled", "#EDF2F4"),
            ]
        )

        style.configure(
            "Danger.TButton",
            background=self.palette["danger"],
            foreground="white",
            padding=(13, 8),
            borderwidth=0,
            font=("Segoe UI Semibold", 9)
        )
        style.map(
            "Danger.TButton",
            background=[
                ("active", self.palette["danger_dark"]),
                ("disabled", "#C9B0B0"),
            ],
            foreground=[
                ("disabled", "#F3EDED"),
            ]
        )

        style.configure(
            "Secondary.TButton",
            background="#E6EDF1",
            foreground=self.palette["text"],
            padding=(11, 7),
            borderwidth=0,
            font=("Segoe UI Semibold", 10)
        )
        style.map(
            "Secondary.TButton",
            background=[
                ("active", "#D7E3E8")
            ]
        )

        style.configure(
            "TEntry",
            padding=6,
            fieldbackground="#FBFCFD",
            bordercolor=self.palette["border"],
            lightcolor=self.palette["border"],
            darkcolor=self.palette["border"]
        )
        style.configure(
            "TCombobox",
            padding=5,
            fieldbackground="#FBFCFD"
        )

    def _build(self):
        # ----------------------------------------------------------
        # CABECERA
        # ----------------------------------------------------------
        header = ttk.Frame(
            self, style="App.TFrame"
        )
        header.pack(
            fill="x",
            padx=20,
            pady=(12, 6)
        )

        header_left = ttk.Frame(
            header, style="App.TFrame"
        )
        header_left.pack(side="left")

        ttk.Label(
            header_left,
            text="MEA Visual Stimuli",
            style="Title.TLabel"
        ).pack(anchor="w")

        ttk.Label(
            header_left,
            text=(
                "Estimulación visual para MEA · PsychoPy · "
                "pantalla experimental protegida"
            ),
            style="Subtitle.TLabel"
        ).pack(
            anchor="w",
            pady=(1, 0)
        )

        header_right = ttk.Frame(
            header, style="App.TFrame"
        )
        header_right.pack(
            side="right",
            anchor="e",
            pady=(4, 0)
        )

        self.language_combo = ttk.Combobox(
            header_right,
            textvariable=self.language_display_var,
            values=["Español", "English"],
            state="readonly",
            width=9
        )
        self.language_combo.pack(side="right", padx=(8, 0))
        self.language_combo.bind(
            "<<ComboboxSelected>>", self._on_language_selected
        )
        ttk.Label(
            header_right,
            text="Idioma:",
            style="Muted.TLabel"
        ).pack(side="right", padx=(8, 0))

        ttk.Label(
            header_right,
            text=f"v{APP_VERSION}",
            style="Badge.TLabel"
        ).pack(side="right")

        self.session_button = ttk.Button(
            header_right,
            text="Sesión / metadatos…",
            command=self._open_session_metadata_dialog,
            style="Secondary.TButton"
        )
        self.session_button.pack(
            side="right",
            padx=(0, 8)
        )

        ttk.Label(
            header_right,
            textvariable=self.session_summary_var,
            style="Muted.TLabel"
        ).pack(
            side="right",
            padx=(0, 10)
        )

        self.manual_button = ttk.Button(
            header_right,
            text="? Manual / Ayuda",
            command=self._open_manual,
            style="Secondary.TButton"
        )
        self.manual_button.pack(
            side="right",
            padx=(0, 8)
        )

        self.red_mode_btn = ttk.Button(
            header_right,
            text="🔴 MODO ROJO",
            command=self.toggle_red_mode,
            style="Secondary.TButton"
        )
        self.red_mode_btn.pack(side="right", padx=(0, 8))

        ttk.Label(
            header_right,
            textvariable=self.ttl_header_text,
            style="Badge.TLabel"
        ).pack(side="right", padx=(0, 8))

        # ----------------------------------------------------------
        # CONFIGURACIÓN DEL MONTAJE
        # ----------------------------------------------------------
        common = ttk.LabelFrame(
            self,
            text="Configuración del montaje",
            style="Card.TLabelframe"
        )
        common.pack(
            fill="x",
            padx=20,
            pady=(0, 6)
        )

        row = ttk.Frame(
            common, style="Card.TFrame"
        )
        row.pack(
            fill="x",
            padx=12,
            pady=(7, 3)
        )

        ttk.Label(
            row,
            text="Pantalla de estimulación:",
            style="Section.TLabel"
        ).pack(side="left")

        self.combo = ttk.Combobox(
            row,
            textvariable=self.screen_var,
            state="readonly",
            width=52
        )
        self.combo.pack(
            side="left",
            padx=(10, 8),
            fill="x",
            expand=True
        )

        ttk.Button(
            row,
            text="Detectar",
            command=self.refresh_screens,
            style="Secondary.TButton"
        ).pack(side="left")

        ttk.Button(
            row,
            text="Reiniciar pantalla",
            command=self.restart_display_stack,
            style="Secondary.TButton"
        ).pack(
            side="left",
            padx=(6, 0)
        )

        grid = ttk.Frame(
            common, style="Card.TFrame"
        )
        grid.pack(
            fill="x",
            padx=12,
            pady=(0, 7)
        )

        self._entry(
            grid, "Anchura pantalla",
            self.width_cm, 0, "cm"
        )
        self._entry(
            grid, "Altura pantalla",
            self.height_cm, 0, "cm",
            column_offset=3
        )
        self._entry(
            grid, "Distancia a retina",
            self.distance_cm, 0, "cm",
            column_offset=6
        )
        # TTL controls moved to the dedicated “Sistema / TTL” tab in v1.22.
        # Only a compact status badge remains in the global header.

        # ----------------------------------------------------------
        # ZONA INFERIOR FIJA
        #
        # Se empaqueta ANTES de las pestañas y desde abajo. De esta
        # forma siempre conserva su espacio; si falta altura, es la
        # zona central la que se adapta.
        # ----------------------------------------------------------
        status_line = ttk.Frame(
            self, style="App.TFrame"
        )
        status_line.pack(
            side="bottom",
            fill="x",
            padx=22,
            pady=(0, 8)
        )

        # Tiny hidden easter-egg trigger.
        self.easter_button = tk.Canvas(
            status_line,
            width=14,
            height=14,
            bg=self.palette["bg"],
            highlightthickness=0,
            bd=0,
            cursor="arrow"
        )
        self.easter_button.pack(
            side="right",
            padx=(8, 0),
            pady=(1, 0)
        )
        self.easter_button.create_oval(
            4, 4, 10, 10,
            fill="#C2CACE",
            outline=""
        )
        self.easter_button.bind(
            "<Button-1>",
            self._trigger_easter_egg
        )

        ttk.Label(
            status_line,
            textvariable=self.status,
            style="Subtitle.TLabel",
            wraplength=1150
        ).pack(
            side="left",
            fill="x",
            expand=True,
            anchor="w"
        )

        footer = ttk.Frame(
            self, style="App.TFrame"
        )
        footer.pack(
            side="bottom",
            fill="x",
            padx=20,
            pady=(4, 8)
        )

        status_card = ttk.LabelFrame(
            footer,
            text="Estado experimental",
            style="Card.TLabelframe"
        )
        status_card.pack(
            side="left",
            fill="x",
            expand=True,
            padx=(0, 10)
        )

        status_inner = ttk.Frame(
            status_card, style="Card.TFrame"
        )
        status_inner.pack(
            fill="x",
            expand=True,
            padx=10,
            pady=6
        )

        ttk.Label(
            status_inner,
            textvariable=self.refresh_info,
            style="Status.TLabel"
        ).pack(anchor="w")

        ttk.Label(
            status_inner,
            textvariable=self.timing_info,
            style="Muted.TLabel",
            wraplength=700
        ).pack(
            anchor="w",
            pady=(2, 0)
        )

        action_card = ttk.Frame(
            footer, style="App.TFrame"
        )
        action_card.pack(
            side="right",
            anchor="se"
        )

        self.open_results_btn = ttk.Button(
            action_card,
            text="Abrir logs",
            command=self._open_logs_folder,
            state="normal",
            style="Secondary.TButton"
        )
        self.open_results_btn.pack(
            side="left",
            padx=(0, 8)
        )

        ttk.Button(
            action_card,
            text="Guardar",
            command=self.save_current,
            style="Secondary.TButton"
        ).pack(
            side="left",
            padx=(0, 8)
        )

        self.abort_btn = ttk.Button(
            action_card,
            text="ABORTAR → SEGURO",
            command=self.abort_test,
            state="disabled",
            style="Danger.TButton"
        )
        self.abort_btn.pack(
            side="left",
            padx=(0, 8)
        )

        self.start_btn = ttk.Button(
            action_card,
            text="▶  INICIAR ESTÍMULO",
            command=self.start_test,
            state="disabled",
            style="Accent.TButton"
        )
        self.start_btn.pack(side="left")

        # ----------------------------------------------------------
        # PESTAÑAS — ocupan únicamente el espacio restante
        # ----------------------------------------------------------
        self.tabs = ttk.Notebook(self)
        self.tabs.pack(
            fill="both",
            expand=True,
            padx=20,
            pady=(2, 4)
        )

        self.tab_on = ttk.Frame(
            self.tabs, style="TabBody.TFrame"
        )
        self.tab_motion = ttk.Frame(
            self.tabs, style="TabBody.TFrame"
        )
        self.tab_grating = ttk.Frame(
            self.tabs, style="TabBody.TFrame"
        )
        self.tab_dense = ttk.Frame(
            self.tabs, style="TabBody.TFrame"
        )
        self.tab_rf = ttk.Frame(
            self.tabs, style="TabBody.TFrame"
        )
        self.tab_chirp = ttk.Frame(
            self.tabs, style="TabBody.TFrame"
        )
        self.tab_protocols = ttk.Frame(
            self.tabs, style="TabBody.TFrame"
        )
        self.tab_system = ttk.Frame(
            self.tabs, style="TabBody.TFrame"
        )

        self.tabs.add(
            self.tab_on, text="ON / OFF"
        )
        self.tabs.add(
            self.tab_motion, text="Motion"
        )
        self.tabs.add(
            self.tab_grating, text="Grating"
        )
        self.tabs.add(
            self.tab_dense, text="Noise"
        )
        self.tabs.add(
            self.tab_rf, text="RF Mapping"
        )
        self.tabs.add(
            self.tab_chirp, text="Temporal"
        )
        self.tabs.add(
            self.tab_protocols, text="Protocolos"
        )
        self.tabs.add(
            self.tab_system, text="Sistema / TTL"
        )

        self._build_on()
        self._build_motion()
        self._build_grating()
        self._build_dense()
        self._build_rf_mapping()
        self._build_chirp()
        self._build_protocols()
        self._build_system_ttl()
        self.bind_all(
            "<F1>",
            lambda _event: self._open_manual()
        )

        self.tabs.bind(
            "<<NotebookTabChanged>>",
            self._on_tab_changed
        )

        self.after(
            450,
            self.refresh_active_preview
        )

    @staticmethod
    def _entry(
        parent, label, var, row, unit,
        column_offset=0
    ):
        ttk.Label(
            parent, text=label + ":"
        ).grid(
            row=row,
            column=column_offset,
            sticky="w",
            padx=(0, 7),
            pady=4
        )

        entry = ttk.Entry(
            parent,
            textvariable=var,
            width=13
        )
        entry.grid(
            row=row,
            column=column_offset + 1,
            sticky="w",
            pady=4
        )

        ttk.Label(
            parent, text=unit
        ).grid(
            row=row,
            column=column_offset + 2,
            sticky="w",
            padx=(5, 18),
            pady=4
        )
        return entry

    def _color_selector(
        self, parent, label, variable, row
    ):
        ttk.Label(
            parent, text=label + ":"
        ).grid(
            row=row, column=0,
            sticky="w", padx=(0, 8), pady=5
        )

        btn = tk.Button(
            parent,
            text="  Seleccionar  ",
            width=14,
            bg=variable.get(),
            fg=best_text_color(variable.get()),
            activebackground=variable.get(),
            activeforeground=best_text_color(
                variable.get()
            ),
            relief="solid",
            bd=1,
            highlightthickness=0,
            padx=8,
            pady=5,
            font=("Segoe UI Semibold", 10),
            cursor="hand2"
        )
        btn.grid(
            row=row, column=1,
            sticky="w", pady=5
        )

        ttk.Label(
            parent,
            textvariable=variable,
            width=10
        ).grid(
            row=row, column=2,
            sticky="w",
            padx=(8, 0),
            pady=5
        )

        def choose():
            _, chosen = colorchooser.askcolor(
                color=variable.get(),
                parent=self,
                title=_tr_text(label)
            )
            if chosen:
                variable.set(chosen.upper())
                btn.configure(
                    bg=variable.get(),
                    fg=best_text_color(
                        variable.get()
                    ),
                    activebackground=variable.get(),
                    activeforeground=best_text_color(
                        variable.get()
                    )
                )
                self.refresh_active_preview()

        btn.configure(command=choose)
        self._color_selector_widgets.append((btn, variable))
        return btn

    def _build_on(self):
        s = self.saved.get("on_off", {})

        self.on_baseline = tk.StringVar(
            value=str(
                s.get("baseline_s", 2.0)
            )
        )
        self.on_on = tk.StringVar(
            value=str(
                s.get(
                    "on_s",
                    s.get("white_s", 2.0)
                )
            )
        )
        self.on_gray = tk.StringVar(
            value=str(
                s.get("gray_interval_s", 2.0)
            )
        )
        self.on_off = tk.StringVar(
            value=str(
                s.get(
                    "off_s",
                    s.get("black_s", 2.0)
                )
            )
        )
        self.on_final = tk.StringVar(
            value=str(
                s.get("final_gray_s", 2.0)
            )
        )
        self.on_reps = tk.StringVar(
            value=str(
                s.get("repetitions", 5)
            )
        )

        # A standard ON/OFF experiment is simply the default:
        # ON = white and OFF = black. Changing either selector turns
        # the same stimulus into a chromatic ON/OFF paradigm.
        self.on_on_hex = tk.StringVar(
            value=str(
                s.get("on_hex", "#FFFFFF")
            ).upper()
        )
        self.on_off_hex = tk.StringVar(
            value=str(
                s.get("off_hex", "#000000")
            ).upper()
        )

        controls = self._make_tab_layout(
            self.tab_on,
            "on_off",
            "ON / OFF"
        )

        g = ttk.Frame(
            controls, style="Card.TFrame"
        )
        g.pack(
            anchor="nw",
            fill="x",
            padx=16,
            pady=16
        )

        ttk.Label(
            g,
            text=(
                "adaptación → ON → adaptación → "
                "OFF → adaptación"
            ),
            font=("Segoe UI Semibold", 11)
        ).grid(
            row=0,
            column=0,
            columnspan=3,
            sticky="w",
            pady=(0, 10)
        )

        ttk.Label(
            g,
            text=(
                "Blanco y negro son los colores predeterminados. "
                "Puedes cambiarlos directamente para realizar el mismo "
                "paradigma con cualquier pareja de colores."
            ),
            style="Muted.TLabel",
            wraplength=720
        ).grid(
            row=1,
            column=0,
            columnspan=3,
            sticky="w",
            pady=(0, 10)
        )

        self._entry(
            g, "Adaptación inicial",
            self.on_baseline, 2, "s"
        )
        self._entry(
            g, "Duración ON",
            self.on_on, 3, "s"
        )
        self._entry(
            g, "Adaptación intermedia",
            self.on_gray, 4, "s"
        )
        self._entry(
            g, "Duración OFF",
            self.on_off, 5, "s"
        )
        self._entry(
            g, "Adaptación final",
            self.on_final, 6, "s"
        )
        self._entry(
            g, "Repeticiones",
            self.on_reps, 7, ""
        )

        self._color_selector(
            g, "Color ON",
            self.on_on_hex, 8
        )
        self._color_selector(
            g, "Color OFF",
            self.on_off_hex, 9
        )

        ttk.Label(
            g,
            text=(
                "Estado basal y periodos intermedios: fondo de adaptación. "
                "TTL: un pulso al inicio de ON y otro al inicio de OFF."
            ),
            style="Muted.TLabel",
            wraplength=720
        ).grid(
            row=10,
            column=0,
            columnspan=3,
            sticky="w",
            pady=(8, 0)
        )

    def _build_motion(self):
        old_bar = self.saved.get("bar", {})
        s = self.saved.get("motion", old_bar)

        self.motion_mode = tk.StringVar(
            value=str(
                s.get("mode", "Moving Bar")
            )
        )
        self.motion_baseline = tk.StringVar(
            value=str(
                s.get("baseline_s", 2.0)
            )
        )
        self.motion_duration = tk.StringVar(
            value=str(
                s.get("duration_s", 3.0)
            )
        )
        self.motion_isi = tk.StringVar(
            value=str(
                s.get("isi_s", 2.0)
            )
        )
        self.motion_reps = tk.StringVar(
            value=str(
                s.get("repetitions", 3)
            )
        )
        self.motion_dirs = tk.StringVar(
            value=str(
                s.get(
                    "directions",
                    "0,45,90,135,180,225,270,315"
                )
            )
        )

        self.motion_background_hex = tk.StringVar(
            value=str(
                s.get(
                    "background_hex",
                    "#000000"
                )
            )
        )
        self.motion_stimulus_hex = tk.StringVar(
            value=str(
                s.get(
                    "stimulus_hex",
                    s.get(
                        "bar_hex",
                        "#FFFFFF"
                    )
                )
            )
        )

        self.motion_bar_width = tk.StringVar(
            value=str(
                s.get(
                    "bar_width_deg",
                    s.get("width_deg", 5.0)
                )
            )
        )
        self.motion_bar_length = tk.StringVar(
            value=str(
                s.get(
                    "bar_length_deg",
                    s.get("length_deg", 40.0)
                )
            )
        )
        self.motion_spot_diameter = tk.StringVar(
            value=str(
                s.get(
                    "spot_diameter_deg",
                    8.0
                )
            )
        )
        self.motion_small_diameter = tk.StringVar(
            value=str(
                s.get(
                    "small_diameter_deg",
                    2.0
                )
            )
        )
        self.motion_large_diameter = tk.StringVar(
            value=str(
                s.get(
                    "large_diameter_deg",
                    40.0
                )
            )
        )
        self.motion_annulus_small_diameter = tk.StringVar(
            value=str(
                s.get(
                    "annulus_small_diameter_deg",
                    8.0
                )
            )
        )
        self.motion_annulus_large_diameter = tk.StringVar(
            value=str(
                s.get(
                    "annulus_large_diameter_deg",
                    40.0
                )
            )
        )
        self.motion_annulus_thickness = tk.StringVar(
            value=str(
                s.get(
                    "annulus_thickness_deg",
                    2.0
                )
            )
        )
        self.motion_center_x = tk.StringVar(
            value=str(
                s.get("center_x_deg", 0.0)
            )
        )
        self.motion_center_y = tk.StringVar(
            value=str(
                s.get("center_y_deg", 0.0)
            )
        )

        controls = self._make_tab_layout(
            self.tab_motion,
            "motion",
            "Motion"
        )

        g = ttk.Frame(
            controls, style="Card.TFrame"
        )
        g.pack(
            fill="both",
            expand=True,
            padx=14,
            pady=12
        )

        ttk.Label(
            g,
            text=(
                "Familia de estímulos de movimiento. "
                "0° = derecha, 90° = arriba; "
                "ángulos positivos en sentido antihorario."
            ),
            font=("Segoe UI Semibold", 11),
            wraplength=700
        ).pack(
            anchor="w",
            pady=(0, 8)
        )

        mode_row = ttk.Frame(
            g, style="Card.TFrame"
        )
        mode_row.pack(
            fill="x",
            pady=(0, 8)
        )

        ttk.Label(
            mode_row,
            text="Estímulo:"
        ).pack(side="left")

        self.motion_mode_combo = ttk.Combobox(
            mode_row,
            textvariable=self.motion_mode,
            state="readonly",
            values=[
                "Moving Bar",
                "Moving Spot",
                "Looming",
                "Receding",
            ],
            width=18
        )
        self.motion_mode_combo.pack(
            side="left",
            padx=(8, 0)
        )
        self.motion_mode_combo.bind(
            "<<ComboboxSelected>>",
            self._update_motion_mode
        )

        columns = ttk.Frame(
            g, style="Card.TFrame"
        )
        columns.pack(
            fill="both",
            expand=True
        )
        columns.columnconfigure(0, weight=1)
        columns.columnconfigure(1, weight=1)

        common = ttk.LabelFrame(
            columns,
            text="Parámetros comunes"
        )
        common.grid(
            row=0,
            column=0,
            sticky="nsew",
            padx=(0, 6)
        )

        cg = ttk.Frame(common)
        cg.pack(
            fill="x",
            padx=9,
            pady=7
        )

        self._entry(
            cg, "Adaptación previa",
            self.motion_baseline, 0, "s"
        )
        self._entry(
            cg, "Duración del estímulo",
            self.motion_duration, 1, "s"
        )
        self._entry(
            cg, "Adaptación entre ensayos",
            self.motion_isi, 2, "s"
        )
        self._entry(
            cg, "Repeticiones",
            self.motion_reps, 3, ""
        )

        self._color_selector(
            cg, "Color del fondo",
            self.motion_background_hex, 4
        )
        self._color_selector(
            cg, "Color del estímulo",
            self.motion_stimulus_hex, 5
        )

        modes = ttk.LabelFrame(
            columns,
            text="Parámetros del estímulo"
        )
        modes.grid(
            row=0,
            column=1,
            sticky="nsew",
            padx=(6, 0)
        )

        self.motion_mode_host = ttk.Frame(modes)
        self.motion_mode_host.pack(
            fill="both",
            expand=True,
            padx=9,
            pady=7
        )

        self.motion_bar_frame = ttk.Frame(
            self.motion_mode_host
        )
        self.motion_spot_frame = ttk.Frame(
            self.motion_mode_host
        )
        self.motion_loom_frame = ttk.Frame(
            self.motion_mode_host
        )
        self.motion_annulus_frame = ttk.Frame(
            self.motion_mode_host
        )

        self._entry(
            self.motion_bar_frame,
            "Anchura de barra",
            self.motion_bar_width, 0, "°"
        )
        self._entry(
            self.motion_bar_frame,
            "Longitud de barra",
            self.motion_bar_length, 1, "°"
        )
        self._entry(
            self.motion_bar_frame,
            "Direcciones",
            self.motion_dirs, 2,
            "°"
        )
        ttk.Label(
            self.motion_bar_frame,
            text=(
                "La barra atraviesa completamente la pantalla "
                "en cada dirección."
            ),
            style="Muted.TLabel",
            wraplength=320
        ).grid(
            row=3, column=0,
            columnspan=3,
            sticky="w",
            pady=(8, 0)
        )

        self._entry(
            self.motion_spot_frame,
            "Diámetro del spot",
            self.motion_spot_diameter, 0, "°"
        )
        self._entry(
            self.motion_spot_frame,
            "Direcciones",
            self.motion_dirs, 1,
            "°"
        )
        ttk.Label(
            self.motion_spot_frame,
            text=(
                "El spot circular entra desde fuera del campo visual "
                "y cruza completamente la pantalla."
            ),
            style="Muted.TLabel",
            wraplength=320
        ).grid(
            row=2, column=0,
            columnspan=3,
            sticky="w",
            pady=(8, 0)
        )

        self._entry(
            self.motion_loom_frame,
            "Diámetro pequeño",
            self.motion_small_diameter, 0, "°"
        )
        self._entry(
            self.motion_loom_frame,
            "Diámetro grande",
            self.motion_large_diameter, 1, "°"
        )
        self._entry(
            self.motion_loom_frame,
            "Centro X",
            self.motion_center_x, 2, "°"
        )
        self._entry(
            self.motion_loom_frame,
            "Centro Y",
            self.motion_center_y, 3, "°"
        )

        self.motion_loom_note = ttk.Label(
            self.motion_loom_frame,
            text="",
            style="Muted.TLabel",
            wraplength=320
        )
        self.motion_loom_note.grid(
            row=4, column=0,
            columnspan=3,
            sticky="w",
            pady=(8, 0)
        )

        self._entry(
            self.motion_annulus_frame,
            "Diámetro exterior pequeño",
            self.motion_annulus_small_diameter, 0, "°"
        )
        self._entry(
            self.motion_annulus_frame,
            "Diámetro exterior grande",
            self.motion_annulus_large_diameter, 1, "°"
        )
        self._entry(
            self.motion_annulus_frame,
            "Grosor radial del anillo",
            self.motion_annulus_thickness, 2, "°"
        )
        self._entry(
            self.motion_annulus_frame,
            "Centro X",
            self.motion_center_x, 3, "°"
        )
        self._entry(
            self.motion_annulus_frame,
            "Centro Y",
            self.motion_center_y, 4, "°"
        )

        self.motion_annulus_note = ttk.Label(
            self.motion_annulus_frame,
            text="",
            style="Muted.TLabel",
            wraplength=320
        )
        self.motion_annulus_note.grid(
            row=5, column=0,
            columnspan=3,
            sticky="w",
            pady=(8, 0)
        )

        self._update_motion_mode()

    def _update_motion_mode(self, event=None):
        for frame in (
            self.motion_bar_frame,
            self.motion_spot_frame,
            self.motion_loom_frame,
            self.motion_annulus_frame,
        ):
            frame.pack_forget()

        mode = self.motion_mode.get()

        # v1.13 compatibility: continuous annulus drafts are no longer
        # exposed and are mapped back to the equivalent disk motion.
        if mode == "Expanding Annulus":
            mode = "Looming"
            self.motion_mode.set(mode)
        elif mode == "Receding Annulus":
            mode = "Receding"
            self.motion_mode.set(mode)

        if mode == "Moving Bar":
            selected = self.motion_bar_frame
        elif mode == "Moving Spot":
            selected = self.motion_spot_frame
        else:
            selected = self.motion_loom_frame
            if mode == "Looming":
                note = (
                    "Expansión: diámetro pequeño → grande. "
                    "Simula un objeto que se aproxima."
                )
            else:
                note = (
                    "Contracción: diámetro grande → pequeño. "
                    "Simula un objeto que se aleja."
                )
            self.motion_loom_note.configure(text=note)

        selected.pack(fill="both", expand=True)
        self.after(20, self.refresh_active_preview)


    def _build_grating(self):
        s = self.saved.get("grating", {})

        saved_grating_mode = str(
            s.get("mode", "Drifting")
        )
        if saved_grating_mode == "Gabor":
            saved_grating_mode = "Static Gabor"

        self.gr_mode = tk.StringVar(
            value=saved_grating_mode
        )
        self.gr_baseline = tk.StringVar(
            value=str(
                s.get("baseline_s", 2.0)
            )
        )
        self.gr_duration = tk.StringVar(
            value=str(
                s.get("duration_s", 3.0)
            )
        )
        self.gr_isi = tk.StringVar(
            value=str(
                s.get("isi_s", 2.0)
            )
        )
        self.gr_sf = tk.StringVar(
            value=str(
                s.get("sf_cpd", 0.10)
            )
        )
        self.gr_tf = tk.StringVar(
            value=str(
                s.get("tf_hz", 1.0)
            )
        )
        self.gr_reversal_hz = tk.StringVar(
            value=str(
                s.get(
                    "reversal_hz",
                    2.0
                )
            )
        )
        self.gr_phase = tk.StringVar(
            value=str(
                s.get(
                    "phase_cycles",
                    0.0
                )
            )
        )

        # "angles" replaces the old "directions" setting internally.
        # Existing saved GUI settings are accepted as fallback.
        self.gr_angles = tk.StringVar(
            value=str(
                s.get(
                    "angles",
                    s.get(
                        "directions",
                        "0,45,90,135,180,225,270,315"
                    )
                )
            )
        )

        self.gr_contrast = tk.StringVar(
            value=str(
                s.get("contrast", 1.0)
            )
        )
        self.gr_reps = tk.StringVar(
            value=str(
                s.get("repetitions", 3)
            )
        )
        self.gr_absolute = tk.BooleanVar(
            value=bool(
                s.get(
                    "absolute_contrast",
                    False
                )
            )
        )
        self.gr_waveform = tk.StringVar(
            value=str(
                s.get(
                    "waveform",
                    "Sinusoidal"
                )
            )
        )
        self.gr_asym_polarity = tk.StringVar(
            value=str(
                s.get(
                    "asymmetric_polarity",
                    "A_TO_B"
                )
            )
        )

        self.gr_color_a_hex = tk.StringVar(
            value=str(
                s.get(
                    "color_a_hex",
                    "#FFFFFF"
                )
            )
        )
        self.gr_color_b_hex = tk.StringVar(
            value=str(
                s.get(
                    "color_b_hex",
                    "#000000"
                )
            )
        )

        self.gr_gabor_size = tk.StringVar(
            value=str(
                s.get(
                    "gabor_size_deg",
                    30.0
                )
            )
        )
        self.gr_gabor_sigma = tk.StringVar(
            value=str(
                s.get(
                    "gabor_sigma_deg",
                    6.0
                )
            )
        )
        self.gr_gabor_x = tk.StringVar(
            value=str(
                s.get(
                    "gabor_x_deg",
                    0.0
                )
            )
        )
        self.gr_gabor_y = tk.StringVar(
            value=str(
                s.get(
                    "gabor_y_deg",
                    0.0
                )
            )
        )
        self.gr_gabor_bg_hex = tk.StringVar(
            value=str(
                s.get(
                    "gabor_background_hex",
                    "#808080"
                )
            )
        )

        controls = self._make_tab_layout(
            self.tab_grating,
            "grating",
            "Grating"
        )

        g = ttk.Frame(
            controls,
            style="Card.TFrame"
        )
        g.pack(
            fill="both",
            expand=True,
            padx=14,
            pady=12
        )

        ttk.Label(
            g,
            text=(
                "Familia de gratings para caracterización espacial "
                "y temporal: drifting, rampa asimétrica, estático, "
                "inversión de fase, Static Gabor y Drifting Gabor."
            ),
            font=("Segoe UI Semibold", 11),
            wraplength=700
        ).pack(
            anchor="w",
            pady=(0, 8)
        )

        mode_row = ttk.Frame(
            g,
            style="Card.TFrame"
        )
        mode_row.pack(
            fill="x",
            pady=(0, 8)
        )

        ttk.Label(
            mode_row,
            text="Estímulo:"
        ).pack(side="left")

        self.gr_mode_combo = ttk.Combobox(
            mode_row,
            textvariable=self.gr_mode,
            state="readonly",
            values=[
                "Drifting",
                "Asymmetric Drift",
                "Static",
                "Phase Reversal",
                "Static Gabor",
                "Drifting Gabor",
            ],
            width=20
        )
        self.gr_mode_combo.pack(
            side="left",
            padx=(8, 0)
        )
        self.gr_mode_combo.bind(
            "<<ComboboxSelected>>",
            self._update_grating_mode
        )

        columns = ttk.Frame(
            g,
            style="Card.TFrame"
        )
        columns.pack(
            fill="both",
            expand=True
        )
        columns.columnconfigure(
            0, weight=1
        )
        columns.columnconfigure(
            1, weight=1
        )

        common = ttk.LabelFrame(
            columns,
            text="Parámetros comunes"
        )
        common.grid(
            row=0,
            column=0,
            sticky="nsew",
            padx=(0, 6)
        )

        cg = ttk.Frame(common)
        cg.pack(
            fill="x",
            padx=9,
            pady=7
        )

        self._entry(
            cg,
            "Adaptación previa",
            self.gr_baseline,
            0,
            "s"
        )
        self._entry(
            cg,
            "Duración estímulo",
            self.gr_duration,
            1,
            "s"
        )
        self._entry(
            cg,
            "Adaptación entre ensayos",
            self.gr_isi,
            2,
            "s"
        )
        self._entry(
            cg,
            "Repeticiones",
            self.gr_reps,
            3,
            ""
        )
        self._entry(
            cg,
            "Frecuencia espacial",
            self.gr_sf,
            4,
            "ciclos/°"
        )

        self.gr_contrast_entry = self._entry(
            cg,
            "Contraste entre barras",
            self.gr_contrast,
            5,
            "0–1"
        )

        ttk.Label(
            cg,
            text="Perfil:"
        ).grid(
            row=6,
            column=0,
            sticky="w",
            pady=4
        )

        self.gr_waveform_combo = ttk.Combobox(
            cg,
            textvariable=self.gr_waveform,
            state="readonly",
            values=[
                "Sinusoidal",
                "Cuadrado"
            ],
            width=14
        )
        self.gr_waveform_combo.grid(
            row=6,
            column=1,
            sticky="w",
            pady=4
        )

        ttk.Checkbutton(
            cg,
            text=(
                "Contraste absoluto "
                "(usar los dos colores al 100 %)"
            ),
            variable=self.gr_absolute,
            command=self.update_contrast_state
        ).grid(
            row=7,
            column=0,
            columnspan=3,
            sticky="w",
            pady=(7, 4)
        )

        self._color_selector(
            cg,
            "Color A",
            self.gr_color_a_hex,
            8
        )
        self._color_selector(
            cg,
            "Color B",
            self.gr_color_b_hex,
            9
        )

        specific = ttk.LabelFrame(
            columns,
            text="Parámetros del estímulo"
        )
        specific.grid(
            row=0,
            column=1,
            sticky="nsew",
            padx=(6, 0)
        )

        self.gr_mode_host = ttk.Frame(
            specific
        )
        self.gr_mode_host.pack(
            fill="both",
            expand=True,
            padx=9,
            pady=7
        )

        self.gr_drifting_frame = ttk.Frame(
            self.gr_mode_host
        )
        self.gr_asymmetric_frame = ttk.Frame(
            self.gr_mode_host
        )
        self.gr_static_frame = ttk.Frame(
            self.gr_mode_host
        )
        self.gr_reversal_frame = ttk.Frame(
            self.gr_mode_host
        )
        self.gr_gabor_frame = ttk.Frame(
            self.gr_mode_host
        )
        self.gr_drifting_gabor_frame = ttk.Frame(
            self.gr_mode_host
        )

        # Drifting
        self._entry(
            self.gr_drifting_frame,
            "Frecuencia temporal",
            self.gr_tf,
            0,
            "Hz"
        )
        self._entry(
            self.gr_drifting_frame,
            "Direcciones movimiento",
            self.gr_angles,
            1,
            "°"
        )
        ttk.Label(
            self.gr_drifting_frame,
            text=(
                "0° = movimiento de izquierda a derecha. "
                "Los ángulos positivos avanzan en sentido antihorario."
            ),
            style="Muted.TLabel",
            wraplength=330
        ).grid(
            row=2,
            column=0,
            columnspan=3,
            sticky="w",
            pady=(8, 0)
        )

        # Asymmetric Drift: half-cosine transition + abrupt reset.
        self._entry(
            self.gr_asymmetric_frame,
            "Frecuencia temporal",
            self.gr_tf,
            0,
            "Hz"
        )
        self._entry(
            self.gr_asymmetric_frame,
            "Direcciones movimiento",
            self.gr_angles,
            1,
            "°"
        )
        ttk.Label(
            self.gr_asymmetric_frame,
            text="Polaridad espacial:"
        ).grid(
            row=2,
            column=0,
            sticky="w",
            pady=4
        )
        self.gr_asym_polarity_combo = ttk.Combobox(
            self.gr_asymmetric_frame,
            textvariable=self.gr_asym_polarity,
            state="readonly",
            values=[
                "A_TO_B",
                "B_TO_A",
            ],
            width=14
        )
        self.gr_asym_polarity_combo.grid(
            row=2,
            column=1,
            sticky="w",
            pady=4
        )
        self.gr_asym_polarity_combo.bind(
            "<<ComboboxSelected>>",
            lambda _event: self.refresh_active_preview()
        )
        ttk.Label(
            self.gr_asymmetric_frame,
            text=(
                "A_TO_B: Color A → Color B con transición de medio coseno "
                "y salto brusco de vuelta a A. B_TO_A invierte la polaridad. "
                "0° = movimiento de izquierda a derecha."
            ),
            style="Muted.TLabel",
            wraplength=330
        ).grid(
            row=3,
            column=0,
            columnspan=3,
            sticky="w",
            pady=(8, 0)
        )

        # Static
        self._entry(
            self.gr_static_frame,
            "Orientaciones",
            self.gr_angles,
            0,
            "°"
        )
        self._entry(
            self.gr_static_frame,
            "Fase",
            self.gr_phase,
            1,
            "ciclos"
        )
        ttk.Label(
            self.gr_static_frame,
            text=(
                "0° = barras verticales. "
                "Las orientaciones positivas giran en sentido antihorario."
            ),
            style="Muted.TLabel",
            wraplength=330
        ).grid(
            row=2,
            column=0,
            columnspan=3,
            sticky="w",
            pady=(8, 0)
        )

        # Phase reversal
        self._entry(
            self.gr_reversal_frame,
            "Orientaciones",
            self.gr_angles,
            0,
            "°"
        )
        self._entry(
            self.gr_reversal_frame,
            "Frecuencia inversión",
            self.gr_reversal_hz,
            1,
            "Hz"
        )
        self._entry(
            self.gr_reversal_frame,
            "Fase inicial",
            self.gr_phase,
            2,
            "ciclos"
        )
        ttk.Label(
            self.gr_reversal_frame,
            text=(
                "La fase alterna 180° (0,5 ciclos) a la "
                "frecuencia indicada. TTL sólo marca el inicio "
                "de cada condición."
            ),
            style="Muted.TLabel",
            wraplength=330
        ).grid(
            row=3,
            column=0,
            columnspan=3,
            sticky="w",
            pady=(8, 0)
        )

        # Static Gabor
        self._entry(
            self.gr_gabor_frame,
            "Orientaciones",
            self.gr_angles,
            0,
            "°"
        )
        self._entry(
            self.gr_gabor_frame,
            "Fase",
            self.gr_phase,
            1,
            "ciclos"
        )
        self._entry(
            self.gr_gabor_frame,
            "Tamaño Gabor",
            self.gr_gabor_size,
            2,
            "°"
        )
        self._entry(
            self.gr_gabor_frame,
            "Sigma gaussiana",
            self.gr_gabor_sigma,
            3,
            "°"
        )
        self._entry(
            self.gr_gabor_frame,
            "Centro X",
            self.gr_gabor_x,
            4,
            "°"
        )
        self._entry(
            self.gr_gabor_frame,
            "Centro Y",
            self.gr_gabor_y,
            5,
            "°"
        )
        self._color_selector(
            self.gr_gabor_frame,
            "Fondo Gabor",
            self.gr_gabor_bg_hex,
            6
        )
        ttk.Label(
            self.gr_gabor_frame,
            text=(
                "Static Gabor: patrón localizado e inmóvil mediante "
                "envolvente gaussiana. El fondo se reaplica "
                "explícitamente en cada frame."
            ),
            style="Muted.TLabel",
            wraplength=330
        ).grid(
            row=7,
            column=0,
            columnspan=3,
            sticky="w",
            pady=(8, 0)
        )

        # Drifting Gabor
        self._entry(
            self.gr_drifting_gabor_frame,
            "Frecuencia temporal",
            self.gr_tf,
            0,
            "Hz"
        )
        self._entry(
            self.gr_drifting_gabor_frame,
            "Direcciones movimiento",
            self.gr_angles,
            1,
            "°"
        )
        self._entry(
            self.gr_drifting_gabor_frame,
            "Fase inicial",
            self.gr_phase,
            2,
            "ciclos"
        )
        self._entry(
            self.gr_drifting_gabor_frame,
            "Tamaño Gabor",
            self.gr_gabor_size,
            3,
            "°"
        )
        self._entry(
            self.gr_drifting_gabor_frame,
            "Sigma gaussiana",
            self.gr_gabor_sigma,
            4,
            "°"
        )
        self._entry(
            self.gr_drifting_gabor_frame,
            "Centro X",
            self.gr_gabor_x,
            5,
            "°"
        )
        self._entry(
            self.gr_drifting_gabor_frame,
            "Centro Y",
            self.gr_gabor_y,
            6,
            "°"
        )
        self._color_selector(
            self.gr_drifting_gabor_frame,
            "Fondo Gabor",
            self.gr_gabor_bg_hex,
            7
        )
        ttk.Label(
            self.gr_drifting_gabor_frame,
            text=(
                "Drifting Gabor: el parche permanece fijo, pero "
                "las barras se desplazan dentro de la envolvente "
                "gaussiana a la frecuencia temporal indicada."
            ),
            style="Muted.TLabel",
            wraplength=330
        ).grid(
            row=8,
            column=0,
            columnspan=3,
            sticky="w",
            pady=(8, 0)
        )

        self.update_contrast_state()
        self._update_grating_mode()

    def _update_grating_mode(
        self,
        event=None
    ):
        for frame in (
            self.gr_drifting_frame,
            self.gr_asymmetric_frame,
            self.gr_static_frame,
            self.gr_reversal_frame,
            self.gr_gabor_frame,
            self.gr_drifting_gabor_frame,
        ):
            frame.pack_forget()

        mode = self.gr_mode.get()

        # Backward compatibility with protocols/settings from v1.10.
        if mode == "Gabor":
            mode = "Static Gabor"
            self.gr_mode.set(
                mode
            )

        if mode == "Drifting":
            selected = (
                self.gr_drifting_frame
            )
        elif mode == "Asymmetric Drift":
            selected = (
                self.gr_asymmetric_frame
            )
        elif mode == "Static":
            selected = (
                self.gr_static_frame
            )
        elif mode == "Phase Reversal":
            selected = (
                self.gr_reversal_frame
            )
        elif mode == "Static Gabor":
            selected = (
                self.gr_gabor_frame
            )
        else:
            selected = (
                self.gr_drifting_gabor_frame
            )

        selected.pack(
            fill="both",
            expand=True
        )

        # The asymmetric profile is fixed by definition.
        try:
            self.gr_waveform_combo.configure(
                state=(
                    "disabled"
                    if mode == "Asymmetric Drift"
                    else "readonly"
                )
            )
        except Exception:
            pass

        self.after(
            20,
            self.refresh_active_preview
        )


    def _build_dense(self):
        # Keep the historical "dense_noise" settings key internally so
        # existing laboratory configurations and saved protocols continue
        # to load without migration.
        s = self.saved.get(
            "dense_noise",
            {}
        )

        self.dn_family = tk.StringVar(
            value=str(
                s.get(
                    "family_mode",
                    "Dense White Noise"
                )
            )
        )
        self.dn_mode = tk.StringVar(
            value=str(
                s.get(
                    "distribution",
                    s.get(
                        "mode",
                        "Binary"
                    )
                )
            )
        )
        self.dn_baseline = tk.StringVar(
            value=str(
                s.get("baseline_s", 2.0)
            )
        )
        self.dn_duration = tk.StringVar(
            value=str(
                s.get("duration_s", 5.0)
            )
        )
        self.dn_isi = tk.StringVar(
            value=str(
                s.get("isi_s", 2.0)
            )
        )
        self.dn_update_hz = tk.StringVar(
            value=str(
                s.get("update_hz", 10.0)
            )
        )
        self.dn_check_size = tk.StringVar(
            value=str(
                s.get("check_size_deg", 5.0)
            )
        )
        self.dn_contrast = tk.StringVar(
            value=str(
                s.get("contrast", 1.0)
            )
        )
        self.dn_reps = tk.StringVar(
            value=str(
                s.get("repetitions", 3)
            )
        )
        self.dn_seed = tk.StringVar(
            value=str(
                s.get("seed", 12345)
            )
        )

        self.dn_low_hex = tk.StringVar(
            value=str(
                s.get(
                    "color_low_hex",
                    "#000000"
                )
            )
        )
        self.dn_mid_hex = tk.StringVar(
            value=str(
                s.get(
                    "color_mid_hex",
                    "#808080"
                )
            )
        )
        self.dn_high_hex = tk.StringVar(
            value=str(
                s.get(
                    "color_high_hex",
                    "#FFFFFF"
                )
            )
        )

        controls = self._make_tab_layout(
            self.tab_dense,
            "dense_noise",
            "Noise"
        )

        g = ttk.Frame(
            controls,
            style="Card.TFrame"
        )
        g.pack(
            anchor="nw",
            fill="x",
            padx=16,
            pady=16
        )

        ttk.Label(
            g,
            text=(
                "Familia de ruido espaciotemporal de pantalla completa. "
                "Sparse Noise permanece en RF Mapping."
            ),
            font=(
                "Segoe UI Semibold",
                11
            ),
            wraplength=760
        ).grid(
            row=0,
            column=0,
            columnspan=3,
            sticky="w",
            pady=(0, 10)
        )

        ttk.Label(
            g,
            text="Estímulo:"
        ).grid(
            row=1,
            column=0,
            sticky="w",
            pady=4
        )

        self.dn_family_combo = ttk.Combobox(
            g,
            textvariable=self.dn_family,
            state="readonly",
            values=[
                "Dense White Noise",
                "Gaussian White Noise",
                "Frozen Noise",
            ],
            width=23
        )
        self.dn_family_combo.grid(
            row=1,
            column=1,
            sticky="w",
            pady=4
        )
        self.dn_family_combo.bind(
            "<<ComboboxSelected>>",
            self._update_noise_mode
        )

        self.dn_distribution_label = ttk.Label(
            g,
            text="Distribución:"
        )
        self.dn_distribution_label.grid(
            row=2,
            column=0,
            sticky="w",
            pady=4
        )

        self.dn_distribution_combo = ttk.Combobox(
            g,
            textvariable=self.dn_mode,
            state="readonly",
            values=[
                "Binary",
                "Ternary",
            ],
            width=14
        )
        self.dn_distribution_combo.grid(
            row=2,
            column=1,
            sticky="w",
            pady=4
        )
        self.dn_distribution_combo.bind(
            "<<ComboboxSelected>>",
            self._update_noise_mode
        )

        self._entry(
            g,
            "Adaptación previa",
            self.dn_baseline,
            3,
            "s"
        )
        self._entry(
            g,
            "Duración noise",
            self.dn_duration,
            4,
            "s"
        )
        self._entry(
            g,
            "Adaptación entre repeticiones",
            self.dn_isi,
            5,
            "s"
        )
        self._entry(
            g,
            "Frecuencia de actualización",
            self.dn_update_hz,
            6,
            "Hz"
        )
        self._entry(
            g,
            "Tamaño de check",
            self.dn_check_size,
            7,
            "°"
        )
        self._entry(
            g,
            "Contraste",
            self.dn_contrast,
            8,
            "0–1"
        )
        self._entry(
            g,
            "Repeticiones",
            self.dn_reps,
            9,
            ""
        )
        self._entry(
            g,
            "Semilla base",
            self.dn_seed,
            10,
            ""
        )

        self._color_selector(
            g,
            "Color bajo",
            self.dn_low_hex,
            11
        )
        self._color_selector(
            g,
            "Color medio",
            self.dn_mid_hex,
            12
        )
        self._color_selector(
            g,
            "Color alto",
            self.dn_high_hex,
            13
        )

        self.dn_help_label = ttk.Label(
            g,
            text="",
            style="Muted.TLabel",
            wraplength=760
        )
        self.dn_help_label.grid(
            row=14,
            column=0,
            columnspan=3,
            sticky="w",
            pady=(8, 0)
        )

        self._update_noise_mode()

    def _update_noise_mode(
        self,
        event=None
    ):
        family = self.dn_family.get()

        if family == "Dense White Noise":
            self.dn_distribution_label.grid()
            self.dn_distribution_combo.grid()
            self.dn_distribution_combo.configure(
                values=[
                    "Binary",
                    "Ternary",
                ]
            )
            if self.dn_mode.get() not in (
                "Binary",
                "Ternary",
            ):
                self.dn_mode.set(
                    "Binary"
                )

            help_text = (
                "Dense White Noise: muchos checks cambian de forma "
                "independiente en cada actualización. Binary usa Color "
                "bajo/alto; Ternary usa bajo/medio/alto. Cada repetición "
                "genera una secuencia nueva y reproducible (semilla + rep)."
            )

        elif family == "Gaussian White Noise":
            self.dn_distribution_label.grid_remove()
            self.dn_distribution_combo.grid_remove()

            help_text = (
                "Gaussian White Noise: cada check toma un valor continuo "
                "e independiente de una distribución gaussiana centrada "
                "en Color medio. Con contraste 1, bajo/alto corresponden "
                "aproximadamente a ±3σ; los extremos se recortan."
            )

        else:
            self.dn_distribution_label.grid()
            self.dn_distribution_combo.grid()
            self.dn_distribution_combo.configure(
                values=[
                    "Binary",
                    "Ternary",
                    "Gaussian",
                ]
            )
            if self.dn_mode.get() not in (
                "Binary",
                "Ternary",
                "Gaussian",
            ):
                self.dn_mode.set(
                    "Binary"
                )

            help_text = (
                "Frozen Noise: una única secuencia aleatoria se genera "
                "con la semilla base y se reproduce exactamente igual en "
                "todas las repeticiones. Útil para medir fiabilidad "
                "trial-to-trial. Puede ser Binary, Ternary o Gaussian."
            )

        self.dn_help_label.configure(
            text=help_text
        )

        self.after(
            20,
            self.refresh_active_preview
        )

    def _build_rf_mapping(self):
        s = self.saved.get("rf_mapping", {})

        self.rf_mode = tk.StringVar(
            value=str(s.get("mode", "Spot"))
        )
        self.rf_baseline = tk.StringVar(
            value=str(s.get("baseline_s", 1.0))
        )
        self.rf_duration = tk.StringVar(
            value=str(s.get("duration_s", 1.0))
        )
        self.rf_isi = tk.StringVar(
            value=str(s.get("isi_s", 1.0))
        )
        self.rf_reps = tk.StringVar(
            value=str(s.get("repetitions", 1))
        )
        self.rf_x = tk.StringVar(
            value=str(s.get("x_deg", 0.0))
        )
        self.rf_y = tk.StringVar(
            value=str(s.get("y_deg", 0.0))
        )

        self.rf_stim_hex = tk.StringVar(
            value=str(s.get("stimulus_hex", "#FFFFFF"))
        )
        self.rf_bg_hex = tk.StringVar(
            value=str(s.get("background_hex", "#808080"))
        )

        self.rf_spot_diameter = tk.StringVar(
            value=str(s.get("spot_diameter_deg", 8.0))
        )
        self.rf_annulus_inner = tk.StringVar(
            value=str(s.get("annulus_inner_deg", 8.0))
        )
        self.rf_annulus_outer = tk.StringVar(
            value=str(s.get("annulus_outer_deg", 20.0))
        )
        self.rf_size_series = tk.StringVar(
            value=str(
                s.get(
                    "size_series_deg",
                    "2, 4, 8, 16, 24, 32"
                )
            )
        )
        self.rf_annulus_size_series = tk.StringVar(
            value=str(
                s.get(
                    "annulus_size_series_deg",
                    "8, 12, 16, 24, 32, 40"
                )
            )
        )
        self.rf_annulus_series_thickness = tk.StringVar(
            value=str(
                s.get(
                    "annulus_series_thickness_deg",
                    2.0
                )
            )
        )

        self.rf_sparse_square = tk.StringVar(
            value=str(s.get("sparse_square_deg", 4.0))
        )
        self.rf_sparse_step = tk.StringVar(
            value=str(s.get("sparse_grid_step_deg", 6.0))
        )
        self.rf_sparse_trials = tk.StringVar(
            value=str(s.get("sparse_presentations", 120))
        )
        self.rf_sparse_polarity = tk.StringVar(
            value=str(s.get("sparse_polarity", "Both"))
        )
        self.rf_sparse_seed = tk.StringVar(
            value=str(s.get("sparse_seed", 24680))
        )

        controls = self._make_tab_layout(
            self.tab_rf,
            "rf_mapping",
            "Receptive Field Mapping"
        )

        g = ttk.Frame(
            controls, style="Card.TFrame"
        )
        g.pack(
            fill="both",
            expand=True,
            padx=14,
            pady=12
        )

        ttk.Label(
            g,
            text=(
                "Una sola familia para caracterizar posición, "
                "tamaño y organización centro-periferia del campo receptivo."
            ),
            font=("Segoe UI Semibold", 11),
            wraplength=660
        ).pack(
            anchor="w",
            pady=(0, 8)
        )

        mode_row = ttk.Frame(
            g, style="Card.TFrame"
        )
        mode_row.pack(
            fill="x",
            pady=(0, 8)
        )

        ttk.Label(
            mode_row,
            text="Modo:"
        ).pack(side="left")

        self.rf_mode_combo = ttk.Combobox(
            mode_row,
            textvariable=self.rf_mode,
            state="readonly",
            values=[
                "Spot",
                "Annulus",
                "Size series",
                "Annulus Size series",
                "Sparse Noise",
            ],
            width=20
        )
        self.rf_mode_combo.pack(
            side="left",
            padx=(8, 0)
        )
        self.rf_mode_combo.bind(
            "<<ComboboxSelected>>",
            self._update_rf_mode
        )

        columns = ttk.Frame(
            g, style="Card.TFrame"
        )
        columns.pack(
            fill="both",
            expand=True
        )
        columns.columnconfigure(0, weight=1)
        columns.columnconfigure(1, weight=1)

        common = ttk.LabelFrame(
            columns,
            text="Parámetros comunes"
        )
        common.grid(
            row=0, column=0,
            sticky="nsew",
            padx=(0, 6)
        )

        cg = ttk.Frame(common)
        cg.pack(
            fill="x",
            padx=9,
            pady=7
        )

        self._entry(
            cg, "Adaptación previa",
            self.rf_baseline, 0, "s"
        )
        self._entry(
            cg, "Duración / condición",
            self.rf_duration, 1, "s"
        )
        self._entry(
            cg, "Adaptación entre condiciones",
            self.rf_isi, 2, "s"
        )
        self._entry(
            cg, "Repeticiones",
            self.rf_reps, 3, ""
        )
        self._entry(
            cg, "Posición X",
            self.rf_x, 4, "°"
        )
        self._entry(
            cg, "Posición Y",
            self.rf_y, 5, "°"
        )

        self._color_selector(
            cg, "Color estímulo",
            self.rf_stim_hex, 6
        )
        self._color_selector(
            cg, "Fondo estímulo",
            self.rf_bg_hex, 7
        )

        modes = ttk.LabelFrame(
            columns,
            text="Parámetros del modo"
        )
        modes.grid(
            row=0, column=1,
            sticky="nsew",
            padx=(6, 0)
        )

        self.rf_mode_host = ttk.Frame(modes)
        self.rf_mode_host.pack(
            fill="both",
            expand=True,
            padx=9,
            pady=7
        )

        self.rf_spot_frame = ttk.Frame(
            self.rf_mode_host
        )
        self.rf_annulus_frame = ttk.Frame(
            self.rf_mode_host
        )
        self.rf_size_frame = ttk.Frame(
            self.rf_mode_host
        )
        self.rf_annulus_size_frame = ttk.Frame(
            self.rf_mode_host
        )
        self.rf_sparse_frame = ttk.Frame(
            self.rf_mode_host
        )

        self._entry(
            self.rf_spot_frame,
            "Diámetro del spot",
            self.rf_spot_diameter, 0, "°"
        )
        ttk.Label(
            self.rf_spot_frame,
            text=(
                "Spot circular estático en la posición X/Y. "
                "Útil para polaridad, latencia y respuesta local."
            ),
            style="Muted.TLabel",
            wraplength=310
        ).grid(
            row=1, column=0,
            columnspan=3,
            sticky="w",
            pady=(8, 0)
        )

        self._entry(
            self.rf_annulus_frame,
            "Diámetro interno",
            self.rf_annulus_inner, 0, "°"
        )
        self._entry(
            self.rf_annulus_frame,
            "Diámetro externo",
            self.rf_annulus_outer, 1, "°"
        )
        ttk.Label(
            self.rf_annulus_frame,
            text=(
                "Anillo centrado en X/Y para estimular preferentemente "
                "la periferia del campo receptivo."
            ),
            style="Muted.TLabel",
            wraplength=310
        ).grid(
            row=2, column=0,
            columnspan=3,
            sticky="w",
            pady=(8, 0)
        )

        ttk.Label(
            self.rf_size_frame,
            text="Diámetros:"
        ).grid(
            row=0, column=0,
            sticky="w",
            padx=(0, 7),
            pady=4
        )
        ttk.Entry(
            self.rf_size_frame,
            textvariable=self.rf_size_series,
            width=25
        ).grid(
            row=0, column=1,
            sticky="w",
            pady=4
        )
        ttk.Label(
            self.rf_size_frame,
            text="°"
        ).grid(
            row=0, column=2,
            sticky="w",
            padx=(5, 0)
        )
        ttk.Label(
            self.rf_size_frame,
            text=(
                "Ejemplo: 2, 4, 8, 16, 24, 32. "
                "Cada diámetro constituye una condición principal."
            ),
            style="Muted.TLabel",
            wraplength=310
        ).grid(
            row=1, column=0,
            columnspan=3,
            sticky="w",
            pady=(8, 0)
        )

        ttk.Label(
            self.rf_annulus_size_frame,
            text="Diámetros exteriores:"
        ).grid(
            row=0, column=0,
            sticky="w",
            padx=(0, 7),
            pady=4
        )
        ttk.Entry(
            self.rf_annulus_size_frame,
            textvariable=self.rf_annulus_size_series,
            width=25
        ).grid(
            row=0, column=1,
            sticky="w",
            pady=4
        )
        ttk.Label(
            self.rf_annulus_size_frame,
            text="°"
        ).grid(
            row=0, column=2,
            sticky="w",
            padx=(5, 0)
        )
        self._entry(
            self.rf_annulus_size_frame,
            "Grosor radial",
            self.rf_annulus_series_thickness,
            1, "°"
        )
        ttk.Label(
            self.rf_annulus_size_frame,
            text=(
                "Funciona como Size series, pero cada condición es un "
                "anillo. El grosor radial permanece constante para todos "
                "los diámetros exteriores."
            ),
            style="Muted.TLabel",
            wraplength=310
        ).grid(
            row=2, column=0,
            columnspan=3,
            sticky="w",
            pady=(8, 0)
        )

        self._entry(
            self.rf_sparse_frame,
            "Tamaño del cuadrado",
            self.rf_sparse_square, 0, "°"
        )
        self._entry(
            self.rf_sparse_frame,
            "Paso de rejilla",
            self.rf_sparse_step, 1, "°"
        )
        self._entry(
            self.rf_sparse_frame,
            "Presentaciones / rep",
            self.rf_sparse_trials, 2, ""
        )

        ttk.Label(
            self.rf_sparse_frame,
            text="Polaridad:"
        ).grid(
            row=3, column=0,
            sticky="w",
            padx=(0, 7),
            pady=4
        )
        ttk.Combobox(
            self.rf_sparse_frame,
            textvariable=self.rf_sparse_polarity,
            state="readonly",
            values=["Bright", "Dark", "Both"],
            width=13
        ).grid(
            row=3, column=1,
            sticky="w",
            pady=4
        )

        self._entry(
            self.rf_sparse_frame,
            "Semilla",
            self.rf_sparse_seed, 4, ""
        )

        ttk.Label(
            self.rf_sparse_frame,
            text=(
                "Sparse Noise usa el fondo de adaptación y cuadrados "
                "blancos/negros sobre posiciones de una rejilla reproducible."
            ),
            style="Muted.TLabel",
            wraplength=310
        ).grid(
            row=5, column=0,
            columnspan=3,
            sticky="w",
            pady=(8, 0)
        )

        self._update_rf_mode()

    def _update_rf_mode(self, event=None):
        for frame in (
            self.rf_spot_frame,
            self.rf_annulus_frame,
            self.rf_size_frame,
            self.rf_annulus_size_frame,
            self.rf_sparse_frame,
        ):
            frame.pack_forget()

        mode = self.rf_mode.get()
        selected = {
            "Spot": self.rf_spot_frame,
            "Annulus": self.rf_annulus_frame,
            "Size series": self.rf_size_frame,
            "Annulus Size series": self.rf_annulus_size_frame,
            "Sparse Noise": self.rf_sparse_frame,
        }.get(
            mode,
            self.rf_spot_frame
        )

        selected.pack(
            fill="both",
            expand=True
        )

        self.after(
            20,
            self.refresh_active_preview
        )


    def _build_chirp(self):
        # Keep the historical "chirp_intensity" settings/config key
        # internally so old saved protocols remain compatible.
        s = self.saved.get(
            "chirp_intensity",
            {}
        )

        self.temporal_mode = tk.StringVar(
            value=str(
                s.get(
                    "temporal_mode",
                    "Chirp + Intensity"
                )
            )
        )

        # Shared parameters.
        self.ch_basal = tk.StringVar(
            value=str(
                s.get(
                    "basal_s",
                    2.0
                )
            )
        )
        self.ch_final = tk.StringVar(
            value=str(
                s.get(
                    "final_gray_s",
                    2.0
                )
            )
        )
        self.ch_reps = tk.StringVar(
            value=str(
                s.get(
                    "repetitions",
                    1
                )
            )
        )
        self.ch_on_hex = tk.StringVar(
            value=str(
                s.get(
                    "on_hex",
                    "#FFFFFF"
                )
            )
        )
        self.ch_off_hex = tk.StringVar(
            value=str(
                s.get(
                    "off_hex",
                    "#000000"
                )
            )
        )

        # Historical Chirp + Intensity parameters.
        self.ch_step_on = tk.StringVar(
            value=str(
                s.get(
                    "step_on_s",
                    1.0
                )
            )
        )
        self.ch_step_off = tk.StringVar(
            value=str(
                s.get(
                    "step_off_s",
                    1.0
                )
            )
        )
        self.ch_pause1 = tk.StringVar(
            value=str(
                s.get(
                    "pause1_s",
                    2.0
                )
            )
        )
        self.ch_fmin = tk.StringVar(
            value=str(
                s.get(
                    "frequency_min_hz",
                    0.5
                )
            )
        )
        self.ch_fmax = tk.StringVar(
            value=str(
                s.get(
                    "frequency_max_hz",
                    15.0
                )
            )
        )
        self.ch_fdur = tk.StringVar(
            value=str(
                s.get(
                    "frequency_duration_s",
                    10.0
                )
            )
        )
        self.ch_pause2 = tk.StringVar(
            value=str(
                s.get(
                    "pause2_s",
                    2.0
                )
            )
        )
        self.ch_ifreq = tk.StringVar(
            value=str(
                s.get(
                    "intensity_frequency_hz",
                    2.0
                )
            )
        )
        self.ch_idur = tk.StringVar(
            value=str(
                s.get(
                    "intensity_duration_s",
                    10.0
                )
            )
        )
        self.ch_istart = tk.StringVar(
            value=str(
                s.get(
                    "intensity_start_pct",
                    10.0
                )
            )
        )

        # New Temporal-family parameters.
        self.ch_temp_duration = tk.StringVar(
            value=str(
                s.get(
                    "temporal_duration_s",
                    5.0
                )
            )
        )
        self.ch_temp_frequency = tk.StringVar(
            value=str(
                s.get(
                    "temporal_frequency_hz",
                    2.0
                )
            )
        )
        self.ch_temp_contrast = tk.StringVar(
            value=str(
                s.get(
                    "temporal_contrast_pct",
                    100.0
                )
            )
        )
        self.ch_temp_update_hz = tk.StringVar(
            value=str(
                s.get(
                    "temporal_update_hz",
                    20.0
                )
            )
        )
        self.ch_temp_seed = tk.StringVar(
            value=str(
                s.get(
                    "temporal_seed",
                    54321
                )
            )
        )
        self.ch_contrast_levels = tk.StringVar(
            value=str(
                s.get(
                    "contrast_levels_pct",
                    "5,10,20,40,80,100"
                )
            )
        )
        self.ch_contrast_isi = tk.StringVar(
            value=str(
                s.get(
                    "contrast_isi_s",
                    1.0
                )
            )
        )

        controls = self._make_tab_layout(
            self.tab_chirp,
            "chirp_intensity",
            "Temporal"
        )

        root = ttk.Frame(
            controls,
            style="Card.TFrame"
        )
        root.pack(
            fill="both",
            expand=True,
            padx=14,
            pady=12
        )

        ttk.Label(
            root,
            text=(
                "Familia de estimulación temporal de campo completo: "
                "Chirp + Intensity, flicker sinusoidal, flicker gaussiano "
                "y serie de contraste."
            ),
            font=("Segoe UI Semibold", 11),
            wraplength=760
        ).pack(
            anchor="w",
            pady=(0, 8)
        )

        mode_row = ttk.Frame(
            root,
            style="Card.TFrame"
        )
        mode_row.pack(
            fill="x",
            pady=(0, 8)
        )

        ttk.Label(
            mode_row,
            text="Estímulo:"
        ).pack(side="left")

        self.temporal_mode_combo = ttk.Combobox(
            mode_row,
            textvariable=self.temporal_mode,
            state="readonly",
            values=[
                "Chirp + Intensity",
                "Sinusoidal Flicker",
                "Gaussian Flicker",
                "Contrast Series",
            ],
            width=22
        )
        self.temporal_mode_combo.pack(
            side="left",
            padx=(8, 0)
        )
        self.temporal_mode_combo.bind(
            "<<ComboboxSelected>>",
            self._update_temporal_mode
        )

        columns = ttk.Frame(
            root,
            style="Card.TFrame"
        )
        columns.pack(
            fill="both",
            expand=True
        )
        columns.columnconfigure(
            0,
            weight=1
        )
        columns.columnconfigure(
            1,
            weight=2
        )

        common = ttk.LabelFrame(
            columns,
            text="Parámetros comunes"
        )
        common.grid(
            row=0,
            column=0,
            sticky="nsew",
            padx=(0, 6)
        )

        cg = ttk.Frame(common)
        cg.pack(
            fill="x",
            padx=9,
            pady=7
        )

        self._entry(
            cg,
            "Adaptación basal",
            self.ch_basal,
            0,
            "s"
        )
        self._entry(
            cg,
            "Adaptación final",
            self.ch_final,
            1,
            "s"
        )
        self._entry(
            cg,
            "Repeticiones",
            self.ch_reps,
            2,
            ""
        )
        self._color_selector(
            cg,
            "Color alto / ON",
            self.ch_on_hex,
            3
        )
        self._color_selector(
            cg,
            "Color bajo / OFF",
            self.ch_off_hex,
            4
        )

        ttk.Label(
            common,
            text=(
                "En los modos continuos, los dos colores definen los "
                "extremos de modulación. El basal/final usa el fondo de adaptación "
                "configurado en Sistema / TTL."
            ),
            style="Muted.TLabel",
            wraplength=300
        ).pack(
            anchor="w",
            padx=9,
            pady=(0, 9)
        )

        specific = ttk.LabelFrame(
            columns,
            text="Parámetros del estímulo"
        )
        specific.grid(
            row=0,
            column=1,
            sticky="nsew",
            padx=(6, 0)
        )

        self.temporal_mode_host = ttk.Frame(
            specific
        )
        self.temporal_mode_host.pack(
            fill="both",
            expand=True,
            padx=9,
            pady=7
        )

        self.temporal_chirp_frame = ttk.Frame(
            self.temporal_mode_host
        )
        self.temporal_sine_frame = ttk.Frame(
            self.temporal_mode_host
        )
        self.temporal_gaussian_frame = ttk.Frame(
            self.temporal_mode_host
        )
        self.temporal_contrast_frame = ttk.Frame(
            self.temporal_mode_host
        )

        # ------------------------------------------------------
        # Chirp + Intensity — historical behavior preserved.
        # ------------------------------------------------------
        chirp_cols = ttk.Frame(
            self.temporal_chirp_frame
        )
        chirp_cols.pack(
            fill="x"
        )
        chirp_cols.columnconfigure(
            0,
            weight=1
        )
        chirp_cols.columnconfigure(
            1,
            weight=1
        )

        left = ttk.LabelFrame(
            chirp_cols,
            text="B · ON/OFF + C · Frequency Chirp"
        )
        left.grid(
            row=0,
            column=0,
            sticky="nsew",
            padx=(0, 5)
        )
        lg = ttk.Frame(left)
        lg.pack(
            fill="x",
            padx=7,
            pady=5
        )
        self._entry(
            lg,
            "Duración ON",
            self.ch_step_on,
            0,
            "s"
        )
        self._entry(
            lg,
            "Duración OFF",
            self.ch_step_off,
            1,
            "s"
        )
        self._entry(
            lg,
            "Pausa B → C",
            self.ch_pause1,
            2,
            "s"
        )
        self._entry(
            lg,
            "Frecuencia mínima",
            self.ch_fmin,
            3,
            "Hz"
        )
        self._entry(
            lg,
            "Frecuencia máxima",
            self.ch_fmax,
            4,
            "Hz"
        )
        self._entry(
            lg,
            "Duración chirp",
            self.ch_fdur,
            5,
            "s"
        )

        right = ttk.LabelFrame(
            chirp_cols,
            text="D · Intensity Ramp"
        )
        right.grid(
            row=0,
            column=1,
            sticky="nsew",
            padx=(5, 0)
        )
        rg = ttk.Frame(right)
        rg.pack(
            fill="x",
            padx=7,
            pady=5
        )
        self._entry(
            rg,
            "Pausa C → D",
            self.ch_pause2,
            0,
            "s"
        )
        self._entry(
            rg,
            "Frecuencia ON/OFF",
            self.ch_ifreq,
            1,
            "Hz"
        )
        self._entry(
            rg,
            "Duración",
            self.ch_idur,
            2,
            "s"
        )
        self._entry(
            rg,
            "Intensidad ON inicial",
            self.ch_istart,
            3,
            "%"
        )

        ttk.Label(
            self.temporal_chirp_frame,
            text=(
                "Secuencia histórica A → B → C → D: adaptación basal, ON→OFF, "
                "barrido lineal de frecuencia y rampa de intensidad."
            ),
            style="Muted.TLabel",
            wraplength=440
        ).pack(
            anchor="w",
            pady=(8, 0)
        )

        # ------------------------------------------------------
        # Sinusoidal Flicker.
        # ------------------------------------------------------
        sg = ttk.Frame(
            self.temporal_sine_frame
        )
        sg.pack(
            fill="x"
        )
        self._entry(
            sg,
            "Duración",
            self.ch_temp_duration,
            0,
            "s"
        )
        self._entry(
            sg,
            "Frecuencia",
            self.ch_temp_frequency,
            1,
            "Hz"
        )
        self._entry(
            sg,
            "Contraste",
            self.ch_temp_contrast,
            2,
            "%"
        )

        ttk.Label(
            self.temporal_sine_frame,
            text=(
                "Modulación sinusoidal continua entre el punto medio de "
                "los dos colores y sus extremos. 100 % alcanza los colores "
                "alto/bajo; 0 % permanece en el punto medio."
            ),
            style="Muted.TLabel",
            wraplength=440
        ).pack(
            anchor="w",
            pady=(8, 0)
        )

        # ------------------------------------------------------
        # Gaussian Flicker.
        # ------------------------------------------------------
        gg = ttk.Frame(
            self.temporal_gaussian_frame
        )
        gg.pack(
            fill="x"
        )
        self._entry(
            gg,
            "Duración",
            self.ch_temp_duration,
            0,
            "s"
        )
        self._entry(
            gg,
            "Frecuencia actualización",
            self.ch_temp_update_hz,
            1,
            "Hz"
        )
        self._entry(
            gg,
            "Contraste",
            self.ch_temp_contrast,
            2,
            "%"
        )
        self._entry(
            gg,
            "Semilla base",
            self.ch_temp_seed,
            3,
            ""
        )

        ttk.Label(
            self.temporal_gaussian_frame,
            text=(
                "Ruido temporal gaussiano de campo completo. Cada "
                "actualización toma un nivel independiente, centrado en el "
                "punto medio de los colores. A 100 %, bajo/alto corresponden "
                "aproximadamente a ±3σ. Si se pide más Hz que el refresco, "
                "se registra la frecuencia real alcanzable."
            ),
            style="Muted.TLabel",
            wraplength=440
        ).pack(
            anchor="w",
            pady=(8, 0)
        )

        # ------------------------------------------------------
        # Contrast Series.
        # ------------------------------------------------------
        cs = ttk.Frame(
            self.temporal_contrast_frame
        )
        cs.pack(
            fill="x"
        )
        self._entry(
            cs,
            "Duración por contraste",
            self.ch_temp_duration,
            0,
            "s"
        )
        self._entry(
            cs,
            "Frecuencia flicker",
            self.ch_temp_frequency,
            1,
            "Hz"
        )
        self._entry(
            cs,
            "Contrastes",
            self.ch_contrast_levels,
            2,
            "%"
        )
        self._entry(
            cs,
            "Adaptación entre contrastes",
            self.ch_contrast_isi,
            3,
            "s"
        )

        ttk.Label(
            self.temporal_contrast_frame,
            text=(
                "Serie de flicker sinusoidal a frecuencia fija. Los "
                "porcentajes se presentan en el orden escrito; cada nivel "
                "genera su propio onset TTL."
            ),
            style="Muted.TLabel",
            wraplength=440
        ).pack(
            anchor="w",
            pady=(8, 0)
        )

        self._update_temporal_mode()

    def _update_temporal_mode(
        self,
        event=None
    ):
        for frame in (
            self.temporal_chirp_frame,
            self.temporal_sine_frame,
            self.temporal_gaussian_frame,
            self.temporal_contrast_frame,
        ):
            frame.pack_forget()

        mode = self.temporal_mode.get()

        selected = {
            "Chirp + Intensity":
                self.temporal_chirp_frame,
            "Sinusoidal Flicker":
                self.temporal_sine_frame,
            "Gaussian Flicker":
                self.temporal_gaussian_frame,
            "Contrast Series":
                self.temporal_contrast_frame,
        }.get(
            mode,
            self.temporal_chirp_frame
        )

        selected.pack(
            fill="both",
            expand=True
        )

        self.after(
            20,
            self.refresh_active_preview
        )

    def _load_easter_frames(self):
        if self.easter_frame_images:
            return True

        try:
            self.easter_frame_images = [
                tk.PhotoImage(data=data)
                for data in EASTER_EGG_FRAMES_B64
            ]
            return bool(
                self.easter_frame_images
            )
        except Exception:
            self.easter_frame_images = []
            return False

    def _stop_easter_egg(
        self, redraw=True
    ):
        if self.easter_after_id is not None:
            try:
                self.after_cancel(
                    self.easter_after_id
                )
            except Exception:
                pass

        key = self.easter_active_key
        self.easter_after_id = None
        self.easter_active_key = None
        self.easter_sequence = []
        self.easter_position = 0

        if redraw and key:
            self._draw_preview_static(key)

    def _trigger_easter_egg(
        self, event=None
    ):
        key = self._preview_key_from_tab()
        if not key:
            return

        canvas = self.preview_canvases.get(
            key
        )
        if canvas is None:
            return

        self._stop_preview(
            redraw=False
        )
        self._stop_easter_egg(
            redraw=False
        )

        if not self._load_easter_frames():
            return

        # Forward + backward gives a smoother little dance loop.
        one_loop = (
            list(range(
                len(self.easter_frame_images)
            ))
            + list(range(
                len(self.easter_frame_images) - 2,
                0,
                -1
            ))
        )

        self.easter_sequence = (
            one_loop * EASTER_EGG_LOOPS
        )
        self.easter_active_key = key
        self.easter_position = 0
        self._animate_easter_egg()

    def _animate_easter_egg(self):
        key = self.easter_active_key
        if not key:
            return

        canvas = self.preview_canvases.get(
            key
        )
        if canvas is None:
            self._stop_easter_egg(
                redraw=False
            )
            return

        if (
            self.easter_position
            >= len(self.easter_sequence)
        ):
            # Tiny final wink before restoring the real stimulus preview.
            canvas.delete("all")
            canvas.create_rectangle(
                0, 0, 330, 245,
                fill="#111820",
                outline=""
            )
            canvas.create_text(
                165, 105,
                text="The stimulus must go on.",
                fill="#E8C98B",
                font=("Segoe UI Semibold", 13)
            )
            canvas.create_text(
                165, 137,
                text="SMN-Systems",
                fill="#D7DEE2",
                font=("Segoe UI", 9)
            )

            self.easter_after_id = self.after(
                750,
                lambda:
                    self._stop_easter_egg(
                        redraw=True
                    )
            )
            return

        frame_index = self.easter_sequence[
            self.easter_position
        ]
        image = self.easter_frame_images[
            frame_index
        ]

        canvas.delete("all")
        canvas.create_rectangle(
            0, 0, 330, 245,
            fill="#111820",
            outline=""
        )
        canvas.create_image(
            165, 104,
            image=image,
            anchor="center"
        )
        canvas.create_text(
            165, 222,
            text="The stimulus must go on.",
            fill="#E8C98B",
            font=("Segoe UI Semibold", 9)
        )

        self.easter_position += 1
        self.easter_after_id = self.after(
            EASTER_EGG_FRAME_MS,
            self._animate_easter_egg
        )

    def _make_tab_layout(
        self, tab, key, preview_title
    ):
        body = ttk.Frame(
            tab, style="TabBody.TFrame"
        )
        body.pack(
            fill="both",
            expand=True,
            padx=12,
            pady=12
        )
        # Do not let the preview's natural requested height enlarge this body
        # beyond the space granted by the Notebook. The grid will shrink its
        # preview canvas first while preserving the action buttons.
        body.grid_propagate(False)

        # ------------------------------------------------------
        # LEFT: scrollable parameter area.
        # Only this region scrolls; preview/footer remain fixed.
        # ------------------------------------------------------
        controls_shell = ttk.LabelFrame(
            body,
            text="Parámetros del estímulo",
            style="Card.TLabelframe"
        )
        controls_shell.grid(
            row=0, column=0, sticky="nsew", padx=(0, 10)
        )

        scroll_host = ttk.Frame(
            controls_shell,
            style="Card.TFrame"
        )
        scroll_host.pack(
            fill="both",
            expand=True,
            padx=3,
            pady=3
        )

        controls_canvas = tk.Canvas(
            scroll_host,
            bg=self.palette["panel"],
            highlightthickness=0,
            bd=0
        )
        controls_canvas.pack(
            side="left",
            fill="both",
            expand=True
        )

        scrollbar = ttk.Scrollbar(
            scroll_host,
            orient="vertical",
            command=controls_canvas.yview
        )
        scrollbar.pack(
            side="right",
            fill="y"
        )

        controls_canvas.configure(
            yscrollcommand=scrollbar.set
        )

        controls = ttk.Frame(
            controls_canvas,
            style="Card.TFrame"
        )

        window_id = controls_canvas.create_window(
            (0, 0),
            window=controls,
            anchor="nw"
        )

        def _sync_scrollregion(event=None):
            try:
                controls_canvas.configure(
                    scrollregion=
                        controls_canvas.bbox("all")
                )
            except Exception:
                pass

        def _fit_inner_width(event):
            try:
                controls_canvas.itemconfigure(
                    window_id,
                    width=max(
                        1,
                        event.width
                    )
                )
            except Exception:
                pass

        controls.bind(
            "<Configure>",
            _sync_scrollregion
        )
        controls_canvas.bind(
            "<Configure>",
            _fit_inner_width
        )

        # Mouse wheel only scrolls while pointer is over the
        # parameter panel, so the rest of the app behaves normally.
        def _wheel(event):
            try:
                delta = int(
                    -1 * (
                        event.delta / 120
                    )
                )
                if delta == 0:
                    delta = (
                        -1
                        if event.delta > 0
                        else 1
                    )
                controls_canvas.yview_scroll(
                    delta,
                    "units"
                )
            except Exception:
                pass
            return "break"

        def _wheel_linux_up(event):
            controls_canvas.yview_scroll(
                -1, "units"
            )
            return "break"

        def _wheel_linux_down(event):
            controls_canvas.yview_scroll(
                1, "units"
            )
            return "break"

        def _bind_wheel(event=None):
            controls_canvas.bind_all(
                "<MouseWheel>",
                _wheel
            )
            controls_canvas.bind_all(
                "<Button-4>",
                _wheel_linux_up
            )
            controls_canvas.bind_all(
                "<Button-5>",
                _wheel_linux_down
            )

        def _unbind_wheel(event=None):
            controls_canvas.unbind_all(
                "<MouseWheel>"
            )
            controls_canvas.unbind_all(
                "<Button-4>"
            )
            controls_canvas.unbind_all(
                "<Button-5>"
            )

        controls_canvas.bind(
            "<Enter>",
            _bind_wheel
        )
        controls_canvas.bind(
            "<Leave>",
            _unbind_wheel
        )
        controls.bind(
            "<Enter>",
            _bind_wheel
        )
        controls.bind(
            "<Leave>",
            _unbind_wheel
        )

        # Keep references alive and available for future UI work.
        if not hasattr(
            self,
            "tab_scroll_canvases"
        ):
            self.tab_scroll_canvases = {}
        self.tab_scroll_canvases[key] = (
            controls_canvas
        )

        # ------------------------------------------------------
        # RIGHT: fixed preview area.
        # ------------------------------------------------------
        preview = ttk.LabelFrame(
            body,
            text="Vista del estímulo",
            style="Card.TLabelframe"
        )
        preview.grid(
            row=0, column=1, sticky="ns", padx=(10, 0)
        )

        preview_inner = ttk.Frame(
            preview, style="Card.TFrame"
        )
        preview_inner.pack(
            fill="both",
            expand=True,
            padx=10,
            pady=10
        )

        preview_inner.columnconfigure(0, weight=1)
        preview_inner.rowconfigure(2, weight=1, minsize=80)

        ttk.Label(
            preview_inner,
            text=preview_title,
            style="Section.TLabel"
        ).grid(row=0, column=0, sticky="w")

        ttk.Label(
            preview_inner,
            text=(
                "Esquema para el operador · "
                "no se presenta en el monitor experimental"
            ),
            style="Muted.TLabel",
            wraplength=320
        ).grid(
            row=1, column=0, sticky="w", pady=(2, 8)
        )

        canvas = tk.Canvas(
            preview_inner,
            width=330,
            height=245,
            bg=self.palette["preview_bg"],
            highlightthickness=1,
            highlightbackground=self.palette["border"],
            bd=0
        )
        canvas.grid(row=2, column=0, sticky="nsew")

        summary_var = tk.StringVar(
            value="Parámetros: —"
        )
        ttk.Label(
            preview_inner,
            textvariable=summary_var,
            style="Muted.TLabel",
            wraplength=320,
            justify="left"
        ).grid(
            row=3, column=0, sticky="ew", pady=(8, 7)
        )

        buttons = ttk.Frame(
            preview_inner, style="Card.TFrame"
        )
        buttons.grid(row=4, column=0, sticky="ew")

        ttk.Button(
            buttons,
            text="↻ Actualizar",
            command=lambda k=key:
                self._draw_preview_static(k),
            style="Secondary.TButton"
        ).pack(side="left")

        ttk.Button(
            buttons,
            text="▶ Vista previa",
            command=lambda k=key:
                self._start_preview(k),
            style="Accent.TButton"
        ).pack(
            side="right"
        )

        protocol_button = ttk.Button(
            preview_inner,
            text="＋ Añadir al protocolo",
            command=self._add_current_stimulus_to_protocol,
            style="Secondary.TButton"
        )
        protocol_button.grid(
            row=5, column=0, sticky="ew", pady=(8, 0)
        )
        self.protocol_add_buttons[key] = (
            protocol_button
        )

        self.preview_canvases[key] = canvas
        self.preview_summary_vars[key] = summary_var

        # Responsive layout: on wide windows preview stays at the right; on
        # narrow windows it moves below the parameters. The parameter area is
        # the part that shrinks and scrolls, so preview / protocol buttons
        # remain visible instead of falling outside the screen.
        body.columnconfigure(0, weight=1)
        body.rowconfigure(0, weight=1)

        layout_state = {"narrow": None}
        def _responsive_layout(event=None):
            width = body.winfo_width() if event is None else int(event.width)
            narrow = width < 700
            if layout_state["narrow"] == narrow:
                return
            layout_state["narrow"] = narrow
            controls_shell.grid_forget()
            preview.grid_forget()
            if narrow:
                body.columnconfigure(1, weight=0)
                body.rowconfigure(0, weight=1)
                body.rowconfigure(1, weight=0)
                controls_shell.grid(row=0, column=0, sticky="nsew", pady=(0, 8))
                preview.grid(row=1, column=0, sticky="ew")
            else:
                body.columnconfigure(0, weight=1)
                body.columnconfigure(1, weight=0)
                body.rowconfigure(0, weight=1)
                body.rowconfigure(1, weight=0)
                controls_shell.grid(row=0, column=0, sticky="nsew", padx=(0, 10))
                preview.grid(row=0, column=1, sticky="ns", padx=(10, 0))

        body.bind("<Configure>", _responsive_layout, add="+")
        self.after(20, _responsive_layout)

        return controls

    def _build_protocols(self):
        self.protocol_name_var = tk.StringVar(
            value=self.protocol_draft["name"]
        )
        self.protocol_reps_var = tk.StringVar(
            value=str(
                self.protocol_draft["repetitions"]
            )
        )
        self.protocol_pause_var = tk.StringVar(
            value="10.0"
        )
        self.protocol_pause_color = tk.StringVar(
            value="Fondo de adaptación"
        )
        self.protocol_duration_var = tk.StringVar(
            value="00:00:00"
        )
        self.protocol_duration_detail_var = tk.StringVar(
            value="1 vuelta: 00:00:00 · ×1 repetición"
        )
        self.protocol_structure_var = tk.StringVar(
            value="0 pasos · 0 estímulos · 0 pausas"
        )
        self.protocol_builder_status = tk.StringVar(
            value=(
                "Crea un protocolo nuevo o carga uno guardado. "
                "Configura un estímulo en su pestaña y pulsa "
                "«Añadir al protocolo»."
            )
        )

        body = ttk.Frame(
            self.tab_protocols,
            style="TabBody.TFrame"
        )
        body.pack(
            fill="both",
            expand=True,
            padx=10,
            pady=10
        )
        body.columnconfigure(0, weight=0, minsize=220)
        body.columnconfigure(1, weight=1)
        body.rowconfigure(0, weight=1)

        # --------------------------------------------------
        # Saved protocol library - permanently visible.
        # The list absorbs all spare vertical room while the
        # action buttons stay reachable at the bottom.
        # --------------------------------------------------
        library = ttk.LabelFrame(
            body,
            text="Protocolos guardados",
            style="Card.TLabelframe"
        )
        library.grid(
            row=0, column=0,
            sticky="nsew",
            padx=(0, 7)
        )
        library.columnconfigure(0, weight=1)
        library.rowconfigure(0, weight=1, minsize=90)

        list_host = ttk.Frame(
            library,
            style="Card.TFrame"
        )
        list_host.grid(
            row=0, column=0,
            sticky="nsew",
            padx=7,
            pady=7
        )
        list_host.columnconfigure(0, weight=1)
        list_host.rowconfigure(0, weight=1)

        self.protocol_listbox = tk.Listbox(
            list_host,
            width=24,
            height=10,
            exportselection=False,
            activestyle="none",
            font=("Segoe UI", 10),
            bg="#FBFCFD",
            fg=self.palette["text"],
            selectbackground="#D9E8EF",
            selectforeground=self.palette["text"],
            highlightthickness=1,
            highlightbackground=self.palette["border"],
            bd=0
        )
        self.protocol_listbox.grid(
            row=0, column=0,
            sticky="nsew"
        )

        protocol_scroll = ttk.Scrollbar(
            list_host,
            orient="vertical",
            command=self.protocol_listbox.yview
        )
        protocol_scroll.grid(
            row=0, column=1,
            sticky="ns"
        )
        self.protocol_listbox.configure(
            yscrollcommand=protocol_scroll.set
        )
        self.protocol_listbox.bind(
            "<Double-Button-1>",
            lambda event:
                self._load_selected_protocol()
        )

        lib_buttons = ttk.Frame(
            library,
            style="Card.TFrame"
        )
        lib_buttons.grid(
            row=1, column=0,
            sticky="ew",
            padx=7,
            pady=(0, 7)
        )

        ttk.Button(
            lib_buttons,
            text="Nuevo",
            command=self._new_protocol,
            style="Secondary.TButton"
        ).grid(
            row=0, column=0,
            sticky="ew",
            padx=(0, 3),
            pady=2
        )
        ttk.Button(
            lib_buttons,
            text="Cargar",
            command=self._load_selected_protocol,
            style="Secondary.TButton"
        ).grid(
            row=0, column=1,
            sticky="ew",
            padx=(3, 0),
            pady=2
        )
        ttk.Button(
            lib_buttons,
            text="Duplicar",
            command=self._duplicate_selected_protocol,
            style="Secondary.TButton"
        ).grid(
            row=1, column=0,
            sticky="ew",
            padx=(0, 3),
            pady=2
        )
        ttk.Button(
            lib_buttons,
            text="Eliminar",
            command=self._delete_selected_protocol,
            style="Danger.TButton"
        ).grid(
            row=1, column=1,
            sticky="ew",
            padx=(3, 0),
            pady=2
        )

        self.protocol_export_btn = ttk.Button(
            lib_buttons,
            text="Exportar…",
            command=self._export_selected_protocol,
            style="Secondary.TButton"
        )
        self.protocol_export_btn.grid(
            row=2, column=0,
            sticky="ew",
            padx=(0, 3),
            pady=(4, 2)
        )

        self.protocol_import_btn = ttk.Button(
            lib_buttons,
            text="Importar…",
            command=self._import_protocol_file,
            style="Secondary.TButton"
        )
        self.protocol_import_btn.grid(
            row=2, column=1,
            sticky="ew",
            padx=(3, 0),
            pady=(4, 2)
        )

        lib_buttons.columnconfigure(0, weight=1)
        lib_buttons.columnconfigure(1, weight=1)

        # --------------------------------------------------
        # Protocol editor.  v1.26 deliberately spends width
        # to recover vertical room: compact header, a large
        # central table and a narrow action rail on the right.
        # --------------------------------------------------
        editor = ttk.LabelFrame(
            body,
            text="Editor del protocolo",
            style="Card.TLabelframe"
        )
        editor.grid(
            row=0, column=1,
            sticky="nsew",
            padx=(7, 0)
        )
        editor.columnconfigure(0, weight=1)
        editor.rowconfigure(1, weight=1, minsize=150)

        header = ttk.Frame(
            editor,
            style="Card.TFrame"
        )
        header.grid(
            row=0, column=0,
            sticky="ew",
            padx=9,
            pady=(8, 5)
        )
        header.columnconfigure(1, weight=1)

        ttk.Label(
            header,
            text="Nombre:"
        ).grid(
            row=0, column=0,
            sticky="w"
        )
        name_entry = ttk.Entry(
            header,
            textvariable=self.protocol_name_var
        )
        name_entry.grid(
            row=0, column=1,
            sticky="ew",
            padx=(6, 12)
        )
        name_entry.bind(
            "<KeyRelease>",
            lambda event:
                self._protocol_mark_dirty()
        )

        ttk.Label(
            header,
            text="Repetir:"
        ).grid(
            row=0, column=2,
            sticky="w"
        )
        reps_entry = ttk.Entry(
            header,
            textvariable=self.protocol_reps_var,
            width=6
        )
        reps_entry.grid(
            row=0, column=3,
            sticky="w",
            padx=(6, 4)
        )
        reps_entry.bind(
            "<KeyRelease>",
            lambda event:
                self._protocol_mark_dirty()
        )
        ttk.Label(
            header,
            text="veces"
        ).grid(
            row=0, column=4,
            sticky="w"
        )

        summary_strip = ttk.Frame(
            header,
            style="Card.TFrame"
        )
        summary_strip.grid(
            row=1, column=0,
            columnspan=5,
            sticky="ew",
            pady=(5, 0)
        )

        ttk.Label(
            summary_strip,
            text="Duración:",
            style="ProtocolDurationCaption.TLabel"
        ).pack(side="left")
        ttk.Label(
            summary_strip,
            textvariable=self.protocol_duration_var,
            style="ProtocolDuration.TLabel"
        ).pack(side="left", padx=(5, 12))
        ttk.Label(
            summary_strip,
            textvariable=self.protocol_duration_detail_var,
            style="Section.TLabel"
        ).pack(side="left")
        ttk.Label(
            summary_strip,
            text="  ·  ",
            style="Muted.TLabel"
        ).pack(side="left")
        ttk.Label(
            summary_strip,
            textvariable=self.protocol_structure_var,
            style="Muted.TLabel"
        ).pack(side="left")

        # Main workspace = large step table + fixed-width action rail.
        workspace = ttk.Frame(
            editor,
            style="Card.TFrame"
        )
        workspace.grid(
            row=1, column=0,
            sticky="nsew",
            padx=9,
            pady=(2, 5)
        )
        workspace.columnconfigure(0, weight=1)
        workspace.columnconfigure(1, weight=0, minsize=136)
        workspace.rowconfigure(0, weight=1)

        tree_host = ttk.Frame(
            workspace,
            style="Card.TFrame"
        )
        tree_host.grid(
            row=0, column=0,
            sticky="nsew",
            padx=(0, 7)
        )
        tree_host.columnconfigure(0, weight=1)
        tree_host.rowconfigure(0, weight=1)

        columns = (
            "number",
            "type",
            "summary",
            "duration"
        )
        self.protocol_tree = ttk.Treeview(
            tree_host,
            columns=columns,
            show="headings",
            height=14,
            selectmode="browse"
        )
        self.protocol_tree.heading(
            "number", text="#"
        )
        self.protocol_tree.heading(
            "type", text=self._tr("Estímulo")
        )
        self.protocol_tree.heading(
            "summary", text=self._tr("Configuración")
        )
        self.protocol_tree.heading(
            "duration", text=self._tr("Duración")
        )

        self.protocol_tree.column(
            "number",
            width=36,
            minwidth=36,
            stretch=False,
            anchor="center"
        )
        self.protocol_tree.column(
            "type",
            width=112,
            minwidth=88,
            stretch=False
        )
        self.protocol_tree.column(
            "summary",
            width=430,
            minwidth=220,
            stretch=True
        )
        self.protocol_tree.column(
            "duration",
            width=82,
            minwidth=74,
            stretch=False,
            anchor="e"
        )

        tree_scroll_y = ttk.Scrollbar(
            tree_host,
            orient="vertical",
            command=self.protocol_tree.yview
        )
        tree_scroll_x = ttk.Scrollbar(
            tree_host,
            orient="horizontal",
            command=self.protocol_tree.xview
        )
        self.protocol_tree.configure(
            yscrollcommand=tree_scroll_y.set,
            xscrollcommand=tree_scroll_x.set
        )
        self.protocol_tree.grid(
            row=0, column=0,
            sticky="nsew"
        )
        tree_scroll_y.grid(
            row=0, column=1,
            sticky="ns"
        )
        tree_scroll_x.grid(
            row=1, column=0,
            sticky="ew"
        )

        # Right-side protocol actions remain permanently available, but the
        # rail itself can scroll vertically on short displays.  This preserves
        # the useful saved-protocol library and the large central step table
        # without allowing "Añadir pausa" to fall below the visible area.
        action_shell = ttk.Frame(
            workspace,
            style="Card.TFrame"
        )
        action_shell.grid(
            row=0, column=1,
            sticky="nsew"
        )
        action_shell.columnconfigure(0, weight=1)
        action_shell.rowconfigure(0, weight=1)

        action_canvas = tk.Canvas(
            action_shell,
            highlightthickness=0,
            bd=0,
            bg=self.palette["panel"]
        )
        action_scroll = ttk.Scrollbar(
            action_shell,
            orient="vertical",
            command=action_canvas.yview
        )
        action_canvas.configure(
            yscrollcommand=action_scroll.set
        )
        action_canvas.grid(
            row=0, column=0,
            sticky="nsew"
        )
        action_scroll.grid(
            row=0, column=1,
            sticky="ns"
        )

        action_rail = ttk.Frame(
            action_canvas,
            style="Card.TFrame"
        )
        # Compact two-column action grid. This keeps the useful permanent
        # right-hand rail while freeing enough vertical space for the pause
        # controls on shorter operator displays.
        action_rail.columnconfigure(0, weight=1, uniform="protocol_actions")
        action_rail.columnconfigure(1, weight=1, uniform="protocol_actions")
        action_window = action_canvas.create_window(
            (0, 0),
            window=action_rail,
            anchor="nw"
        )

        def _sync_protocol_action_scroll(event=None):
            try:
                bbox = action_canvas.bbox("all")
                if bbox:
                    action_canvas.configure(
                        scrollregion=bbox
                    )
            except Exception:
                pass

        def _fit_protocol_action_width(event):
            try:
                action_canvas.itemconfigure(
                    action_window,
                    width=max(1, int(event.width))
                )
            except Exception:
                pass

        action_rail.bind(
            "<Configure>",
            _sync_protocol_action_scroll,
            add="+"
        )
        action_canvas.bind(
            "<Configure>",
            _fit_protocol_action_width,
            add="+"
        )

        ttk.Label(
            action_rail,
            text="Paso seleccionado",
            style="Section.TLabel"
        ).grid(
            row=0, column=0, columnspan=2,
            sticky="w",
            pady=(0, 4)
        )

        # Two columns instead of one long vertical stack. The existing
        # button styles are retained so normal and red-mode theming remain
        # unchanged.
        action_buttons = (
            ("↑ Subir", lambda: self._move_protocol_step(-1), "Secondary.TButton"),
            ("↓ Bajar", lambda: self._move_protocol_step(1), "Secondary.TButton"),
            ("Editar", self._edit_protocol_step, "Secondary.TButton"),
            ("Duplicar", self._duplicate_protocol_step, "Secondary.TButton"),
        )
        for idx, (text, command, style_name) in enumerate(action_buttons):
            ttk.Button(
                action_rail,
                text=text,
                command=command,
                style=style_name
            ).grid(
                row=1 + idx // 2,
                column=idx % 2,
                sticky="ew",
                padx=(0, 3) if idx % 2 == 0 else (3, 0),
                pady=2
            )

        ttk.Button(
            action_rail,
            text="Eliminar",
            command=self._delete_protocol_step,
            style="Danger.TButton"
        ).grid(
            row=3, column=0, columnspan=2,
            sticky="ew",
            pady=(2, 3)
        )

        ttk.Separator(
            action_rail,
            orient="horizontal"
        ).grid(
            row=4, column=0, columnspan=2,
            sticky="ew",
            pady=(5, 5)
        )

        ttk.Label(
            action_rail,
            text="Añadir pausa",
            style="Section.TLabel"
        ).grid(
            row=5, column=0, columnspan=2,
            sticky="w",
            pady=(0, 3)
        )

        pause_time_row = ttk.Frame(
            action_rail,
            style="Card.TFrame"
        )
        pause_time_row.grid(
            row=6, column=0, columnspan=2,
            sticky="ew",
            pady=(0, 3)
        )
        pause_time_row.columnconfigure(0, weight=1)
        ttk.Entry(
            pause_time_row,
            textvariable=self.protocol_pause_var,
            width=6
        ).grid(
            row=0, column=0,
            sticky="ew"
        )
        ttk.Label(
            pause_time_row,
            text="s"
        ).grid(
            row=0, column=1,
            sticky="w",
            padx=(4, 0)
        )

        ttk.Combobox(
            action_rail,
            textvariable=self.protocol_pause_color,
            values=["Fondo de adaptación", "Negro", "Gris medio"],
            state="readonly",
            width=17
        ).grid(
            row=7, column=0, columnspan=2,
            sticky="ew",
            pady=(0, 3)
        )

        ttk.Button(
            action_rail,
            text="＋ Añadir pausa",
            command=self._add_protocol_pause,
            style="Secondary.TButton"
        ).grid(
            row=8, column=0, columnspan=2,
            sticky="ew"
        )

        # Bottom command bar stays visible while the tree consumes the
        # remaining height.  Runtime pause is separated from save/execute.
        bottom = ttk.Frame(
            editor,
            style="Card.TFrame"
        )
        bottom.grid(
            row=2, column=0,
            sticky="ew",
            padx=9,
            pady=(1, 4)
        )
        bottom.columnconfigure(1, weight=1)

        self.protocol_pause_runtime_btn = ttk.Button(
            bottom,
            text="⏸ Pausar protocolo",
            command=self._toggle_protocol_pause,
            state="disabled",
            style="Secondary.TButton"
        )
        self.protocol_pause_runtime_btn.grid(
            row=0, column=0,
            sticky="w"
        )

        command_box = ttk.Frame(
            bottom,
            style="Card.TFrame"
        )
        command_box.grid(
            row=0, column=2,
            sticky="e"
        )

        ttk.Button(
            command_box,
            text="Guardar protocolo",
            command=self._save_protocol_draft,
            style="Accent.TButton"
        ).pack(
            side="left",
            padx=(0, 7)
        )

        self.protocol_execute_btn = ttk.Button(
            command_box,
            text="▶ Ejecutar protocolo",
            command=self._execute_protocol,
            state="disabled",
            style="Accent.TButton"
        )
        self.protocol_execute_btn.pack(
            side="left"
        )

        ttk.Label(
            editor,
            textvariable=self.protocol_builder_status,
            style="Muted.TLabel",
            wraplength=760
        ).grid(
            row=3, column=0,
            sticky="ew",
            padx=9,
            pady=(0, 7)
        )

        self._refresh_protocol_library()
        self._refresh_protocol_editor()

    def _protocol_clone(self, value):
        return json.loads(
            json.dumps(value)
        )

    def _new_protocol_id(self):
        return (
            "protocol_"
            + datetime.now().strftime(
                "%Y%m%d_%H%M%S_%f"
            )
        )

    def _new_step_id(self):
        return (
            "step_"
            + datetime.now().strftime(
                "%Y%m%d_%H%M%S_%f"
            )
        )

    def _protocol_mark_dirty(self):
        self.protocol_dirty = True
        self._sync_protocol_header_from_vars()
        self._refresh_protocol_editor()

    def _sync_protocol_header_from_vars(self):
        if not hasattr(
            self,
            "protocol_name_var"
        ):
            return

        self.protocol_draft["name"] = (
            self.protocol_name_var.get().strip()
            or "Nuevo protocolo"
        )

        try:
            reps = int(
                self.protocol_reps_var.get()
            )
            if reps >= 1:
                self.protocol_draft[
                    "repetitions"
                ] = reps
        except Exception:
            pass

    def _refresh_protocol_library(
        self, select_id=None
    ):
        if not hasattr(
            self,
            "protocol_listbox"
        ):
            return

        self.protocol_listbox.delete(
            0, "end"
        )
        self.protocol_list_ids = []

        protocols = sorted(
            self.protocol_library.get(
                "protocols", []
            ),
            key=lambda p:
                str(p.get("name", "")).lower()
        )

        for protocol in protocols:
            pid = protocol.get("id")
            name = protocol.get(
                "name",
                "Sin nombre"
            )
            version = int(
                protocol.get("version", 1)
            )
            steps = len(
                protocol.get("steps", [])
            )

            self.protocol_listbox.insert(
                "end",
                f"{name}  ·  v{version}  ·  {steps} {self._tr('pasos')}"
            )
            self.protocol_list_ids.append(
                pid
            )

        if select_id is not None:
            try:
                idx = self.protocol_list_ids.index(
                    select_id
                )
                self.protocol_listbox.selection_clear(
                    0, "end"
                )
                self.protocol_listbox.selection_set(
                    idx
                )
                self.protocol_listbox.see(
                    idx
                )
            except Exception:
                pass

    def _selected_protocol_id(self):
        try:
            selection = (
                self.protocol_listbox.curselection()
            )
            if not selection:
                return None
            idx = int(selection[0])
            return self.protocol_list_ids[idx]
        except Exception:
            return None

    def _find_protocol(self, protocol_id):
        for protocol in self.protocol_library.get(
            "protocols", []
        ):
            if protocol.get("id") == protocol_id:
                return protocol
        return None

    def _confirm_discard_draft(self):
        if not self.protocol_dirty:
            return True

        return messagebox.askyesno(
            "Cambios sin guardar",
            (
                "El protocolo actual contiene cambios sin guardar.\n\n"
                "¿Quieres descartarlos?"
            ),
            parent=self
        )

    def _new_protocol(self):
        if not self._confirm_discard_draft():
            return

        self.protocol_draft = {
            "id": None,
            "name": self._tr("Nuevo protocolo"),
            "version": 0,
            "repetitions": 1,
            "steps": [],
        }
        self.protocol_dirty = False
        self.protocol_edit_step_index = None

        self.protocol_name_var.set(
            "Nuevo protocolo"
        )
        self.protocol_reps_var.set(
            "1"
        )
        self._refresh_protocol_edit_buttons()
        self._refresh_protocol_editor()

        self.protocol_builder_status.set(
            "Nuevo protocolo preparado. "
            "Configura cualquier estímulo y pulsa "
            "«Añadir al protocolo»."
        )

    def _load_selected_protocol(self):
        pid = self._selected_protocol_id()
        if pid is None:
            messagebox.showinfo(
                "Protocolos",
                "Selecciona un protocolo guardado."
            )
            return

        if not self._confirm_discard_draft():
            return

        protocol = self._find_protocol(
            pid
        )
        if protocol is None:
            return

        self.protocol_draft = (
            self._protocol_clone(
                protocol
            )
        )
        self.protocol_dirty = False
        self.protocol_edit_step_index = None

        self.protocol_name_var.set(
            self.protocol_draft.get(
                "name", "Sin nombre"
            )
        )
        self.protocol_reps_var.set(
            str(
                self.protocol_draft.get(
                    "repetitions", 1
                )
            )
        )

        self._refresh_protocol_edit_buttons()
        self._refresh_protocol_editor()

        self.protocol_builder_status.set(
            (
                f"Protocolo cargado · v"
                f"{self.protocol_draft.get('version', 1)}. "
                "Puedes modificarlo y guardarlo como una nueva revisión."
            )
        )

    def _duplicate_selected_protocol(self):
        pid = self._selected_protocol_id()
        if pid is None:
            messagebox.showinfo(
                "Protocolos",
                "Selecciona un protocolo guardado."
            )
            return

        protocol = self._find_protocol(
            pid
        )
        if protocol is None:
            return

        if not self._confirm_discard_draft():
            return

        duplicate = self._protocol_clone(
            protocol
        )
        duplicate["id"] = None
        duplicate["version"] = 0
        duplicate["name"] = (
            str(
                duplicate.get(
                    "name", "Protocolo"
                )
            )
            + " (copia)"
        )
        duplicate.pop(
            "created_at", None
        )
        duplicate.pop(
            "modified_at", None
        )

        for step in duplicate.get(
            "steps", []
        ):
            step["id"] = (
                self._new_step_id()
            )

        self.protocol_draft = duplicate
        self.protocol_dirty = True
        self.protocol_edit_step_index = None

        self.protocol_name_var.set(
            duplicate["name"]
        )
        self.protocol_reps_var.set(
            str(
                duplicate.get(
                    "repetitions", 1
                )
            )
        )

        self._refresh_protocol_edit_buttons()
        self._refresh_protocol_editor()

        self.protocol_builder_status.set(
            "Copia creada como borrador. "
            "Guárdala para añadirla a la biblioteca."
        )

    def _delete_selected_protocol(self):
        pid = self._selected_protocol_id()
        if pid is None:
            messagebox.showinfo(
                "Protocolos",
                "Selecciona un protocolo guardado."
            )
            return

        protocol = self._find_protocol(
            pid
        )
        if protocol is None:
            return

        if not messagebox.askyesno(
            "Eliminar protocolo",
            (
                f"¿Eliminar «{protocol.get('name', 'Sin nombre')}»?\n\n"
                "Esta acción elimina la definición guardada, "
                "no los CSV de experimentos anteriores."
            ),
            parent=self
        ):
            return

        self.protocol_library["protocols"] = [
            p
            for p in self.protocol_library.get(
                "protocols", []
            )
            if p.get("id") != pid
        ]
        save_protocol_library(
            self.protocol_library
        )
        self._refresh_protocol_library()

        self.protocol_builder_status.set(
            "Protocolo eliminado de la biblioteca."
        )

    def _save_protocol_draft(self):
        self._sync_protocol_header_from_vars()

        name = self.protocol_draft.get(
            "name", ""
        ).strip()
        if not name:
            messagebox.showerror(
                "Protocolos",
                "Escribe un nombre para el protocolo."
            )
            return

        try:
            reps = int(
                self.protocol_reps_var.get()
            )
        except Exception:
            reps = 0

        if reps < 1:
            messagebox.showerror(
                "Protocolos",
                "Las repeticiones del protocolo deben ser >= 1."
            )
            return

        steps = self.protocol_draft.get(
            "steps", []
        )
        if not steps:
            messagebox.showerror(
                "Protocolos",
                "Añade al menos un paso antes de guardar."
            )
            return

        now = datetime.now().isoformat(
            timespec="seconds"
        )

        pid = self.protocol_draft.get(
            "id"
        )
        existing = (
            self._find_protocol(pid)
            if pid
            else None
        )

        if existing is None:
            pid = self._new_protocol_id()
            version = 1
            created_at = now
        else:
            version = int(
                existing.get(
                    "version", 0
                )
            ) + 1
            created_at = existing.get(
                "created_at", now
            )

        saved_protocol = (
            self._protocol_clone(
                self.protocol_draft
            )
        )
        saved_protocol.update({
            "id": pid,
            "name": name,
            "version": version,
            "repetitions": reps,
            "created_at": created_at,
            "modified_at": now,
        })

        protocols = self.protocol_library.get(
            "protocols", []
        )

        if existing is None:
            protocols.append(
                saved_protocol
            )
        else:
            for i, protocol in enumerate(
                protocols
            ):
                if protocol.get("id") == pid:
                    protocols[i] = (
                        saved_protocol
                    )
                    break

        self.protocol_library[
            "protocols"
        ] = protocols

        save_protocol_library(
            self.protocol_library
        )

        self.protocol_draft = (
            self._protocol_clone(
                saved_protocol
            )
        )
        self.protocol_dirty = False
        self.protocol_edit_step_index = None

        self.protocol_name_var.set(
            name
        )
        self.protocol_reps_var.set(
            str(reps)
        )

        self._refresh_protocol_library(
            select_id=pid
        )
        self._refresh_protocol_edit_buttons()
        self._refresh_protocol_editor()

        self.protocol_builder_status.set(
            (
                f"Protocolo guardado · {name} · "
                f"versión {version} · "
                f"{len(steps)} pasos."
            )
        )

    @staticmethod
    def _protocol_export_safe_filename(name):
        safe = "".join(
            c
            if (
                c.isalnum()
                or c in "-_ "
            )
            else "_"
            for c in str(name)
        ).strip()

        safe = "_".join(
            safe.split()
        )

        return (
            safe[:80]
            if safe
            else "protocolo"
        )

    def _portable_protocol_copy(
        self, protocol
    ):
        """
        Build the shareable representation of one protocol.

        Local library IDs, step IDs and machine-specific screen/TTL
        parameters are deliberately excluded. The exported definition
        describes WHAT to present, not the computer on which it was made.
        """
        portable = {
            "name": str(
                protocol.get(
                    "name",
                    "Protocolo"
                )
            ).strip(),
            "version": int(
                protocol.get(
                    "version",
                    1
                )
            ),
            "repetitions": int(
                protocol.get(
                    "repetitions",
                    1
                )
            ),
            "steps": [],
        }

        for source_step in protocol.get(
            "steps", []
        ):
            step = self._protocol_clone(
                source_step
            )
            step.pop(
                "id",
                None
            )

            if step.get(
                "kind"
            ) == "stimulus":
                cfg = self._protocol_strip_common_config(
                    step.get(
                        "config",
                        {}
                    )
                )

                # App version is provenance of a local snapshot rather
                # than a requirement of the portable protocol itself.
                cfg.pop(
                    "app_version",
                    None
                )
                step[
                    "config"
                ] = cfg

            portable[
                "steps"
            ].append(
                step
            )

        return portable

    def _export_selected_protocol(self):
        pid = self._selected_protocol_id()

        # Helpful fallback: a saved protocol may be loaded in the editor
        # even if the list selection has been cleared.
        if (
            pid is None
            and self.protocol_draft.get(
                "id"
            )
        ):
            pid = self.protocol_draft.get(
                "id"
            )

        if pid is None:
            messagebox.showinfo(
                "Exportar protocolo",
                (
                    "Selecciona primero un protocolo guardado "
                    "en la biblioteca."
                ),
                parent=self
            )
            return

        protocol = self._find_protocol(
            pid
        )
        if protocol is None:
            messagebox.showerror(
                "Exportar protocolo",
                "No se ha podido localizar el protocolo seleccionado.",
                parent=self
            )
            return

        portable = self._portable_protocol_copy(
            protocol
        )

        payload = {
            "format":
                PROTOCOL_EXPORT_FORMAT,
            "schema_version":
                PROTOCOL_EXPORT_SCHEMA_VERSION,
            "exported_with_app_version":
                APP_VERSION,
            "exported_at":
                datetime.now().isoformat(
                    timespec="seconds"
                ),
            "protocol":
                portable,
        }

        initial_name = (
            self._protocol_export_safe_filename(
                portable.get(
                    "name",
                    "protocolo"
                )
            )
            + ".mea_protocol.json"
        )

        selected_path = filedialog.asksaveasfilename(
            parent=self,
            title="Exportar protocolo",
            initialfile=initial_name,
            defaultextension=".mea_protocol.json",
            filetypes=[
                (
                    "Protocolo MEA Visual Stimuli",
                    "*.mea_protocol.json"
                ),
                (
                    "Archivo JSON",
                    "*.json"
                ),
                (
                    "Todos los archivos",
                    "*.*"
                ),
            ]
        )

        if not selected_path:
            return

        path = Path(
            selected_path
        )

        try:
            path.write_text(
                json.dumps(
                    payload,
                    indent=2,
                    ensure_ascii=False
                ),
                encoding="utf-8"
            )
        except Exception as exc:
            messagebox.showerror(
                "Exportar protocolo",
                (
                    "No se ha podido guardar el archivo.\n\n"
                    f"{exc}"
                ),
                parent=self
            )
            return

        self.protocol_builder_status.set(
            (
                f"Protocolo exportado · "
                f"{portable.get('name', 'Protocolo')} · "
                f"{path.name}"
            )
        )

        messagebox.showinfo(
            "Protocolo exportado",
            (
                "El protocolo se ha exportado correctamente.\n\n"
                f"{path}"
            ),
            parent=self
        )

    def _validate_imported_protocol_payload(
        self, payload
    ):
        if not isinstance(
            payload,
            dict
        ):
            raise ValueError(
                "El archivo no contiene un objeto JSON válido."
            )

        if payload.get(
            "format"
        ) != PROTOCOL_EXPORT_FORMAT:
            raise ValueError(
                "El archivo no es un protocolo de MEA Visual Stimuli."
            )

        try:
            schema_version = int(
                payload.get(
                    "schema_version",
                    -1
                )
            )
        except Exception:
            schema_version = -1

        if (
            schema_version
            != PROTOCOL_EXPORT_SCHEMA_VERSION
        ):
            raise ValueError(
                (
                    "Versión de formato no compatible: "
                    f"{payload.get('schema_version')}. "
                    "Esta versión del programa admite "
                    f"schema {PROTOCOL_EXPORT_SCHEMA_VERSION}."
                )
            )

        source = payload.get(
            "protocol"
        )

        if not isinstance(
            source,
            dict
        ):
            raise ValueError(
                "Falta la definición del protocolo."
            )

        name = str(
            source.get(
                "name",
                ""
            )
        ).strip()

        if not name:
            raise ValueError(
                "El protocolo importado no tiene nombre."
            )

        try:
            version = int(
                source.get(
                    "version",
                    1
                )
            )
            repetitions = int(
                source.get(
                    "repetitions",
                    1
                )
            )
        except Exception:
            raise ValueError(
                "Versión o número de repeticiones no válido."
            )

        if version < 1:
            raise ValueError(
                "La versión del protocolo debe ser >= 1."
            )

        if repetitions < 1:
            raise ValueError(
                "Las repeticiones del protocolo deben ser >= 1."
            )

        steps = source.get(
            "steps"
        )

        if (
            not isinstance(
                steps,
                list
            )
            or not steps
        ):
            raise ValueError(
                "El protocolo no contiene pasos."
            )

        supported_types = {
            "on_off",
            "motion",
            "grating",
            "dense_noise",
            "rf_mapping",
            "chirp_intensity",
        }

        supported_motion = {
            "Moving Bar",
            "Moving Spot",
            "Looming",
            "Receding",
        }

        supported_grating = {
            "Drifting",
            "Asymmetric Drift",
            "Static",
            "Phase Reversal",
            "Static Gabor",
            "Drifting Gabor",
            "Gabor",
        }

        supported_noise_families = {
            "Dense White Noise",
            "Gaussian White Noise",
            "Frozen Noise",
        }

        supported_rf = {
            "Spot",
            "Annulus",
            "Size series",
            "Annulus Size series",
            "Sparse Noise",
        }

        supported_temporal = {
            "Chirp + Intensity",
            "Sinusoidal Flicker",
            "Gaussian Flicker",
            "Contrast Series",
        }

        normalized_steps = []

        for index, source_step in enumerate(
            steps,
            start=1
        ):
            if not isinstance(
                source_step,
                dict
            ):
                raise ValueError(
                    f"Paso {index}: estructura no válida."
                )

            kind = source_step.get(
                "kind"
            )

            if kind == "pause":
                try:
                    duration = float(
                        source_step.get(
                            "duration_s",
                            0
                        )
                    )
                except Exception:
                    duration = 0.0

                if duration <= 0:
                    raise ValueError(
                        f"Paso {index}: la pausa debe durar > 0 s."
                    )

                color = source_step.get(
                    "color",
                    "gray50"
                )

                if color not in ("gray50", "adaptation", "black"):
                    raise ValueError(
                        (
                            f"Paso {index}: color de pausa no compatible "
                            f"({color})."
                        )
                    )

                normalized_steps.append({
                    "id":
                        self._new_step_id(),
                    "kind":
                        "pause",
                    "duration_s":
                        duration,
                    "color":
                        color,
                })
                continue

            if kind != "stimulus":
                raise ValueError(
                    (
                        f"Paso {index}: tipo de paso desconocido "
                        f"({kind})."
                    )
                )

            stimulus_type = source_step.get(
                "stimulus_type"
            )
            cfg = source_step.get(
                "config"
            )

            if stimulus_type not in supported_types:
                raise ValueError(
                    (
                        f"Paso {index}: estímulo no compatible "
                        f"({stimulus_type})."
                    )
                )

            if not isinstance(
                cfg,
                dict
            ):
                raise ValueError(
                    f"Paso {index}: configuración ausente o inválida."
                )

            cfg = self._protocol_strip_common_config(
                cfg
            )
            cfg.pop(
                "app_version",
                None
            )

            config_type = cfg.get(
                "stimulus_type",
                stimulus_type
            )

            if (
                config_type
                != stimulus_type
            ):
                raise ValueError(
                    (
                        f"Paso {index}: stimulus_type no coincide "
                        "con su configuración."
                    )
                )

            cfg[
                "stimulus_type"
            ] = stimulus_type

            if stimulus_type == "motion":
                mode = cfg.get(
                    "motion_mode"
                )
                if mode not in supported_motion:
                    raise ValueError(
                        (
                            f"Paso {index}: modo Motion no compatible "
                            f"({mode})."
                        )
                    )

            elif stimulus_type == "grating":
                mode = cfg.get(
                    "grating_mode",
                    "Drifting"
                )
                if mode not in supported_grating:
                    raise ValueError(
                        (
                            f"Paso {index}: modo Grating no compatible "
                            f"({mode})."
                        )
                    )

            elif stimulus_type == "dense_noise":
                family = cfg.get(
                    "dense_family_mode",
                    "Dense White Noise"
                )
                if family not in supported_noise_families:
                    raise ValueError(
                        (
                            f"Paso {index}: familia Noise no compatible "
                            f"({family})."
                        )
                    )

            elif stimulus_type == "rf_mapping":
                mode = cfg.get(
                    "rf_mode"
                )
                if mode not in supported_rf:
                    raise ValueError(
                        (
                            f"Paso {index}: modo RF Mapping no compatible "
                            f"({mode})."
                        )
                    )

            elif stimulus_type == "chirp_intensity":
                mode = cfg.get(
                    "temporal_mode",
                    "Chirp + Intensity"
                )
                if mode not in supported_temporal:
                    raise ValueError(
                        (
                            f"Paso {index}: modo Temporal no compatible "
                            f"({mode})."
                        )
                    )

            normalized_steps.append({
                "id":
                    self._new_step_id(),
                "kind":
                    "stimulus",
                "stimulus_type":
                    stimulus_type,
                "config":
                    cfg,
            })

        return {
            "name":
                name,
            "version":
                version,
            "repetitions":
                repetitions,
            "steps":
                normalized_steps,
            "source_exported_with_app_version":
                payload.get(
                    "exported_with_app_version"
                ),
            "source_schema_version":
                schema_version,
        }

    def _unique_imported_protocol_name(
        self, requested_name
    ):
        existing = {
            str(
                protocol.get(
                    "name",
                    ""
                )
            ).strip().casefold()
            for protocol in self.protocol_library.get(
                "protocols",
                []
            )
        }

        requested_name = str(
            requested_name
        ).strip()

        if (
            requested_name.casefold()
            not in existing
        ):
            return requested_name

        base = (
            requested_name
            + " (importado)"
        )

        if base.casefold() not in existing:
            return base

        counter = 2

        while True:
            candidate = (
                requested_name
                + f" (importado {counter})"
            )
            if (
                candidate.casefold()
                not in existing
            ):
                return candidate
            counter += 1

    def _import_protocol_file(self):
        selected_path = filedialog.askopenfilename(
            parent=self,
            title="Importar protocolo",
            filetypes=[
                (
                    "Protocolo MEA Visual Stimuli",
                    "*.mea_protocol.json"
                ),
                (
                    "Archivo JSON",
                    "*.json"
                ),
                (
                    "Todos los archivos",
                    "*.*"
                ),
            ]
        )

        if not selected_path:
            return

        path = Path(
            selected_path
        )

        try:
            payload = json.loads(
                path.read_text(
                    encoding="utf-8"
                )
            )
            imported = (
                self._validate_imported_protocol_payload(
                    payload
                )
            )
        except Exception as exc:
            messagebox.showerror(
                "Importar protocolo",
                (
                    "No se puede importar este protocolo.\n\n"
                    f"{exc}"
                ),
                parent=self
            )
            return

        if not self._confirm_discard_draft():
            return

        original_name = imported[
            "name"
        ]
        imported_name = (
            self._unique_imported_protocol_name(
                original_name
            )
        )

        now = datetime.now().isoformat(
            timespec="seconds"
        )
        new_id = self._new_protocol_id()

        saved_protocol = {
            "id":
                new_id,
            "name":
                imported_name,
            "version":
                int(
                    imported.get(
                        "version",
                        1
                    )
                ),
            "repetitions":
                int(
                    imported.get(
                        "repetitions",
                        1
                    )
                ),
            "steps":
                imported[
                    "steps"
                ],
            "created_at":
                now,
            "modified_at":
                now,
            "imported_at":
                now,
            "imported_from_file":
                path.name,
            "source_exported_with_app_version":
                imported.get(
                    "source_exported_with_app_version"
                ),
            "source_schema_version":
                imported.get(
                    "source_schema_version"
                ),
        }

        new_library = self._protocol_clone(
            self.protocol_library
        )
        new_library.setdefault(
            "protocols",
            []
        ).append(
            saved_protocol
        )

        try:
            save_protocol_library(
                new_library
            )
        except Exception as exc:
            messagebox.showerror(
                "Importar protocolo",
                (
                    "El protocolo es válido, pero no se ha podido "
                    "guardar en la biblioteca local.\n\n"
                    f"{exc}"
                ),
                parent=self
            )
            return

        self.protocol_library = (
            new_library
        )
        self.protocol_draft = (
            self._protocol_clone(
                saved_protocol
            )
        )
        self.protocol_dirty = False
        self.protocol_edit_step_index = None

        self.protocol_name_var.set(
            imported_name
        )
        self.protocol_reps_var.set(
            str(
                saved_protocol[
                    "repetitions"
                ]
            )
        )

        self._refresh_protocol_library(
            select_id=new_id
        )
        self._refresh_protocol_edit_buttons()
        self._refresh_protocol_editor()

        if imported_name != original_name:
            rename_note = (
                f" Ya existía ese nombre; se ha guardado como "
                f"«{imported_name}»."
            )
        else:
            rename_note = ""

        self.protocol_builder_status.set(
            (
                f"Protocolo importado · {imported_name} · "
                f"{len(saved_protocol['steps'])} pasos."
                + rename_note
            )
        )

        messagebox.showinfo(
            "Protocolo importado",
            (
                f"«{original_name}» se ha importado correctamente "
                "a la biblioteca local."
                + rename_note
            ),
            parent=self
        )

    def _protocol_strip_common_config(
        self, cfg
    ):
        cfg = self._protocol_clone(cfg)

        for key in (
            "screen_index",
            "screen_width_px",
            "screen_height_px",
            "screen_x",
            "screen_y",
            "physical_width_cm",
            "physical_height_cm",
            "distance_cm",
            "ttl_mode",
            "ttl_pin",
            "session_metadata",
            "ttl_port",
        ):
            cfg.pop(key, None)

        return cfg

    def _add_current_stimulus_to_protocol(self):
        try:
            tab = self.tabs.index(
                self.tabs.select()
            )
        except Exception:
            tab = -1

        if not 0 <= tab <= 5:
            messagebox.showinfo(
                "Protocolos",
                (
                    "Abre primero una pestaña de estímulo, "
                    "configúrala y pulsa «Añadir al protocolo»."
                )
            )
            return

        try:
            cfg = self.collect_stimulus_config()
        except Exception as exc:
            messagebox.showerror(
                "No se puede añadir",
                str(exc)
            )
            return

        cfg = self._protocol_strip_common_config(
            cfg
        )

        step = {
            "id": self._new_step_id(),
            "kind": "stimulus",
            "stimulus_type":
                cfg.get("stimulus_type"),
            "config": cfg,
        }

        edit_idx = (
            self.protocol_edit_step_index
        )

        if (
            edit_idx is not None
            and 0 <= edit_idx < len(
                self.protocol_draft["steps"]
            )
        ):
            old_id = (
                self.protocol_draft[
                    "steps"
                ][edit_idx].get(
                    "id",
                    step["id"]
                )
            )
            step["id"] = old_id

            self.protocol_draft[
                "steps"
            ][edit_idx] = step

            self.protocol_edit_step_index = None
            self.protocol_dirty = True
            self._refresh_protocol_edit_buttons()
            self._refresh_protocol_editor()

            self.protocol_builder_status.set(
                (
                    f"Paso {edit_idx + 1} actualizado. "
                    "Vuelve a guardarlo para crear una nueva revisión."
                )
            )

            # Editing began from Protocols; return there when finished.
            self.tabs.select(
                self.tab_protocols
            )
            return

        self.protocol_draft[
            "steps"
        ].append(step)
        self.protocol_dirty = True
        self._refresh_protocol_editor()

        self.status.set(
            (
                f"Añadido a «{self.protocol_draft.get('name', 'Nuevo protocolo')}»: "
                f"{self._protocol_type_name(step)}"
            )
        )

    def _add_protocol_pause(self):
        try:
            duration = float(
                self.protocol_pause_var.get().replace(
                    ",", "."
                )
            )
        except Exception:
            duration = -1

        if duration <= 0:
            messagebox.showerror(
                "Pausa",
                "La duración de la pausa debe ser > 0 s."
            )
            return

        self.protocol_draft[
            "steps"
        ].append({
            "id": self._new_step_id(),
            "kind": "pause",
            "duration_s": duration,
            "color": {
                "Fondo de adaptación": "adaptation",
                "Negro": "black",
                "Gris medio": "gray50",
            }.get(self.protocol_pause_color.get(), "adaptation"),
        })

        self.protocol_dirty = True
        self._refresh_protocol_editor()

    def _selected_protocol_step_index(self):
        try:
            selection = (
                self.protocol_tree.selection()
            )
            if not selection:
                return None

            iid = selection[0]
            return int(
                iid.split("_", 1)[1]
            )
        except Exception:
            return None

    def _move_protocol_step(self, direction):
        idx = self._selected_protocol_step_index()
        if idx is None:
            return

        steps = self.protocol_draft[
            "steps"
        ]
        new_idx = idx + int(direction)

        if not (
            0 <= new_idx < len(steps)
        ):
            return

        steps[idx], steps[new_idx] = (
            steps[new_idx],
            steps[idx]
        )
        self.protocol_dirty = True
        self._refresh_protocol_editor(
            select_index=new_idx
        )

    def _duplicate_protocol_step(self):
        idx = self._selected_protocol_step_index()
        if idx is None:
            return

        steps = self.protocol_draft[
            "steps"
        ]
        duplicate = self._protocol_clone(
            steps[idx]
        )
        duplicate["id"] = (
            self._new_step_id()
        )
        steps.insert(
            idx + 1,
            duplicate
        )

        self.protocol_dirty = True
        self._refresh_protocol_editor(
            select_index=idx + 1
        )

    def _delete_protocol_step(self):
        idx = self._selected_protocol_step_index()
        if idx is None:
            return

        steps = self.protocol_draft[
            "steps"
        ]
        if not (
            0 <= idx < len(steps)
        ):
            return

        del steps[idx]

        if (
            self.protocol_edit_step_index
            == idx
        ):
            self.protocol_edit_step_index = None
            self._refresh_protocol_edit_buttons()

        self.protocol_dirty = True
        self._refresh_protocol_editor(
            select_index=min(
                idx,
                len(steps) - 1
            )
            if steps
            else None
        )

    def _edit_protocol_step(self):
        idx = self._selected_protocol_step_index()
        if idx is None:
            return

        steps = self.protocol_draft[
            "steps"
        ]
        if not (
            0 <= idx < len(steps)
        ):
            return

        step = steps[idx]

        if step.get("kind") == "pause":
            current = float(
                step.get(
                    "duration_s", 1.0
                )
            )
            value = simpledialog.askfloat(
                "Editar pausa",
                "Duración de la pausa (s):",
                initialvalue=current,
                minvalue=0.001,
                parent=self
            )
            if value is None:
                return

            step["duration_s"] = float(value)
            self.protocol_dirty = True
            self._refresh_protocol_editor(
                select_index=idx
            )
            return

        cfg = step.get(
            "config", {}
        )

        if not self._load_protocol_step_into_ui(
            cfg
        ):
            return

        self.protocol_edit_step_index = idx
        self._refresh_protocol_edit_buttons()

        self.status.set(
            (
                f"Editando paso {idx + 1} del protocolo. "
                "Modifica los parámetros y pulsa «Actualizar paso»."
            )
        )

    def _refresh_protocol_edit_buttons(self):
        for button in self.protocol_add_buttons.values():
            try:
                button.configure(
                    text="＋ Añadir al protocolo"
                )
            except Exception:
                pass

        idx = self.protocol_edit_step_index
        if idx is None:
            return

        steps = self.protocol_draft.get(
            "steps", []
        )
        if not (
            0 <= idx < len(steps)
        ):
            self.protocol_edit_step_index = None
            return

        step = steps[idx]
        if step.get("kind") != "stimulus":
            return

        key = {
            "on_off": "on_off",
            "motion": "motion",
            "grating": "grating",
            "dense_noise":
                "dense_noise",
            "rf_mapping":
                "rf_mapping",
            "chirp_intensity":
                "chirp_intensity",
        }.get(
            step.get("stimulus_type")
        )

        if key in self.protocol_add_buttons:
            self.protocol_add_buttons[
                key
            ].configure(
                text="✓ Actualizar paso"
            )

    def _load_protocol_step_into_ui(
        self, cfg
    ):
        stimulus_type = cfg.get(
            "stimulus_type"
        )

        try:
            if stimulus_type == "on_off":
                self.on_baseline.set(
                    cfg["baseline_s"]
                )
                self.on_on.set(
                    cfg["on_s"]
                )
                self.on_gray.set(
                    cfg["gray_interval_s"]
                )
                self.on_off.set(
                    cfg["off_s"]
                )
                self.on_final.set(
                    cfg["final_gray_s"]
                )
                self.on_reps.set(
                    cfg["repetitions"]
                )
                self.on_on_hex.set(
                    rgb255_to_hex(
                        cfg["on_color_rgb"]
                    )
                )
                self.on_off_hex.set(
                    rgb255_to_hex(
                        cfg["off_color_rgb"]
                    )
                )
                target = 0

            elif stimulus_type == "motion":
                loaded_motion_mode = cfg[
                    "motion_mode"
                ]
                if loaded_motion_mode == "Expanding Annulus":
                    loaded_motion_mode = "Looming"
                elif loaded_motion_mode == "Receding Annulus":
                    loaded_motion_mode = "Receding"

                self.motion_mode.set(
                    loaded_motion_mode
                )
                self.motion_baseline.set(
                    cfg["motion_baseline_s"]
                )
                self.motion_duration.set(
                    cfg["motion_duration_s"]
                )
                self.motion_isi.set(
                    cfg["motion_isi_s"]
                )
                self.motion_reps.set(
                    cfg["motion_repetitions"]
                )
                self.motion_dirs.set(
                    cfg["motion_directions"]
                )
                self.motion_background_hex.set(
                    rgb255_to_hex(
                        cfg["motion_background_rgb"]
                    )
                )
                self.motion_stimulus_hex.set(
                    rgb255_to_hex(
                        cfg["motion_stimulus_rgb"]
                    )
                )
                self.motion_bar_width.set(
                    cfg["motion_bar_width_deg"]
                )
                self.motion_bar_length.set(
                    cfg["motion_bar_length_deg"]
                )
                self.motion_spot_diameter.set(
                    cfg["motion_spot_diameter_deg"]
                )
                self.motion_small_diameter.set(
                    cfg["motion_small_diameter_deg"]
                )
                self.motion_large_diameter.set(
                    cfg["motion_large_diameter_deg"]
                )
                self.motion_annulus_small_diameter.set(
                    cfg.get(
                        "motion_annulus_small_diameter_deg",
                        8.0
                    )
                )
                self.motion_annulus_large_diameter.set(
                    cfg.get(
                        "motion_annulus_large_diameter_deg",
                        40.0
                    )
                )
                self.motion_annulus_thickness.set(
                    cfg.get(
                        "motion_annulus_thickness_deg",
                        2.0
                    )
                )
                self.motion_center_x.set(
                    cfg["motion_center_x_deg"]
                )
                self.motion_center_y.set(
                    cfg["motion_center_y_deg"]
                )
                self._update_motion_mode()
                target = 1

            elif stimulus_type == "grating":
                loaded_grating_mode = cfg.get(
                    "grating_mode",
                    "Drifting"
                )
                if loaded_grating_mode == "Gabor":
                    loaded_grating_mode = (
                        "Static Gabor"
                    )
                self.gr_mode.set(
                    loaded_grating_mode
                )
                self.gr_baseline.set(
                    cfg["grating_baseline_s"]
                )
                self.gr_duration.set(
                    cfg["grating_duration_s"]
                )
                self.gr_isi.set(
                    cfg["grating_isi_s"]
                )
                self.gr_sf.set(
                    cfg["grating_sf_cpd"]
                )
                self.gr_tf.set(
                    cfg.get(
                        "grating_tf_hz",
                        1.0
                    )
                )
                self.gr_reversal_hz.set(
                    cfg.get(
                        "grating_reversal_hz",
                        2.0
                    )
                )
                self.gr_phase.set(
                    cfg.get(
                        "grating_phase_cycles",
                        0.0
                    )
                )
                self.gr_angles.set(
                    cfg.get(
                        "grating_angles",
                        cfg.get(
                            "grating_directions",
                            "0"
                        )
                    )
                )
                self.gr_contrast.set(
                    cfg["grating_contrast"]
                )
                self.gr_reps.set(
                    cfg["grating_repetitions"]
                )
                self.gr_absolute.set(
                    bool(
                        cfg[
                            "grating_absolute_contrast"
                        ]
                    )
                )
                self.gr_waveform.set(
                    cfg["grating_waveform"]
                )
                self.gr_asym_polarity.set(
                    cfg.get(
                        "grating_asymmetric_polarity",
                        "A_TO_B"
                    )
                )
                self.gr_color_a_hex.set(
                    rgb255_to_hex(
                        cfg[
                            "grating_color_a_rgb"
                        ]
                    )
                )
                self.gr_color_b_hex.set(
                    rgb255_to_hex(
                        cfg[
                            "grating_color_b_rgb"
                        ]
                    )
                )
                self.gr_gabor_size.set(
                    cfg.get(
                        "grating_gabor_size_deg",
                        30.0
                    )
                )
                self.gr_gabor_sigma.set(
                    cfg.get(
                        "grating_gabor_sigma_deg",
                        6.0
                    )
                )
                self.gr_gabor_x.set(
                    cfg.get(
                        "grating_gabor_x_deg",
                        0.0
                    )
                )
                self.gr_gabor_y.set(
                    cfg.get(
                        "grating_gabor_y_deg",
                        0.0
                    )
                )
                self.gr_gabor_bg_hex.set(
                    rgb255_to_hex(
                        cfg.get(
                            "grating_gabor_background_rgb",
                            [128, 128, 128]
                        )
                    )
                )
                self.update_contrast_state()
                self._update_grating_mode()
                target = 2

            elif stimulus_type == "dense_noise":
                family = cfg.get(
                    "dense_family_mode",
                    "Dense White Noise"
                )
                distribution = cfg.get(
                    "dense_distribution",
                    cfg.get(
                        "dense_mode",
                        "Binary"
                    )
                )

                self.dn_family.set(
                    family
                )
                self.dn_mode.set(
                    distribution
                )
                self.dn_baseline.set(
                    cfg["dense_baseline_s"]
                )
                self.dn_duration.set(
                    cfg["dense_duration_s"]
                )
                self.dn_isi.set(
                    cfg["dense_isi_s"]
                )
                self.dn_update_hz.set(
                    cfg["dense_update_hz"]
                )
                self.dn_check_size.set(
                    cfg["dense_check_size_deg"]
                )
                self.dn_contrast.set(
                    cfg["dense_contrast"]
                )
                self.dn_reps.set(
                    cfg["dense_repetitions"]
                )
                self.dn_seed.set(
                    cfg["dense_seed"]
                )
                self.dn_low_hex.set(
                    rgb255_to_hex(
                        cfg["dense_color_low_rgb"]
                    )
                )
                self.dn_mid_hex.set(
                    rgb255_to_hex(
                        cfg["dense_color_mid_rgb"]
                    )
                )
                self.dn_high_hex.set(
                    rgb255_to_hex(
                        cfg["dense_color_high_rgb"]
                    )
                )
                self._update_noise_mode()
                target = 3

            elif stimulus_type == "rf_mapping":
                self.rf_mode.set(
                    cfg["rf_mode"]
                )
                self.rf_baseline.set(
                    cfg["rf_baseline_s"]
                )
                self.rf_duration.set(
                    cfg["rf_duration_s"]
                )
                self.rf_isi.set(
                    cfg["rf_isi_s"]
                )
                self.rf_reps.set(
                    cfg["rf_repetitions"]
                )
                self.rf_x.set(
                    cfg["rf_x_deg"]
                )
                self.rf_y.set(
                    cfg["rf_y_deg"]
                )
                self.rf_stim_hex.set(
                    rgb255_to_hex(
                        cfg["rf_stimulus_rgb"]
                    )
                )
                self.rf_bg_hex.set(
                    rgb255_to_hex(
                        cfg["rf_background_rgb"]
                    )
                )
                self.rf_spot_diameter.set(
                    cfg[
                        "rf_spot_diameter_deg"
                    ]
                )
                self.rf_annulus_inner.set(
                    cfg[
                        "rf_annulus_inner_deg"
                    ]
                )
                self.rf_annulus_outer.set(
                    cfg[
                        "rf_annulus_outer_deg"
                    ]
                )
                self.rf_size_series.set(
                    cfg[
                        "rf_size_series_deg"
                    ]
                )
                self.rf_annulus_size_series.set(
                    cfg.get(
                        "rf_annulus_size_series_deg",
                        "8, 12, 16, 24, 32, 40"
                    )
                )
                self.rf_annulus_series_thickness.set(
                    cfg.get(
                        "rf_annulus_series_thickness_deg",
                        2.0
                    )
                )
                self.rf_sparse_square.set(
                    cfg[
                        "rf_sparse_square_deg"
                    ]
                )
                self.rf_sparse_step.set(
                    cfg[
                        "rf_sparse_grid_step_deg"
                    ]
                )
                self.rf_sparse_trials.set(
                    cfg[
                        "rf_sparse_presentations"
                    ]
                )
                self.rf_sparse_polarity.set(
                    cfg[
                        "rf_sparse_polarity"
                    ]
                )
                self.rf_sparse_seed.set(
                    cfg[
                        "rf_sparse_seed"
                    ]
                )
                self._update_rf_mode()
                target = 4

            elif stimulus_type == "chirp_intensity":
                self.temporal_mode.set(
                    cfg.get(
                        "temporal_mode",
                        "Chirp + Intensity"
                    )
                )

                self.ch_basal.set(
                    cfg.get(
                        "chirp_basal_s",
                        2.0
                    )
                )
                self.ch_step_on.set(
                    cfg.get(
                        "chirp_step_on_s",
                        1.0
                    )
                )
                self.ch_step_off.set(
                    cfg.get(
                        "chirp_step_off_s",
                        1.0
                    )
                )
                self.ch_pause1.set(
                    cfg.get(
                        "chirp_pause1_s",
                        2.0
                    )
                )
                self.ch_fmin.set(
                    cfg.get(
                        "chirp_frequency_min_hz",
                        0.5
                    )
                )
                self.ch_fmax.set(
                    cfg.get(
                        "chirp_frequency_max_hz",
                        15.0
                    )
                )
                self.ch_fdur.set(
                    cfg.get(
                        "chirp_frequency_duration_s",
                        10.0
                    )
                )
                self.ch_pause2.set(
                    cfg.get(
                        "chirp_pause2_s",
                        2.0
                    )
                )
                self.ch_ifreq.set(
                    cfg.get(
                        "chirp_intensity_frequency_hz",
                        2.0
                    )
                )
                self.ch_idur.set(
                    cfg.get(
                        "chirp_intensity_duration_s",
                        10.0
                    )
                )
                self.ch_istart.set(
                    cfg.get(
                        "chirp_intensity_start_pct",
                        10.0
                    )
                )

                self.ch_temp_duration.set(
                    cfg.get(
                        "temporal_duration_s",
                        5.0
                    )
                )
                self.ch_temp_frequency.set(
                    cfg.get(
                        "temporal_frequency_hz",
                        2.0
                    )
                )
                self.ch_temp_contrast.set(
                    cfg.get(
                        "temporal_contrast_pct",
                        100.0
                    )
                )
                self.ch_temp_update_hz.set(
                    cfg.get(
                        "temporal_update_hz",
                        20.0
                    )
                )
                self.ch_temp_seed.set(
                    cfg.get(
                        "temporal_seed",
                        54321
                    )
                )
                self.ch_contrast_levels.set(
                    cfg.get(
                        "temporal_contrast_levels_pct",
                        "5,10,20,40,80,100"
                    )
                )
                self.ch_contrast_isi.set(
                    cfg.get(
                        "temporal_contrast_isi_s",
                        1.0
                    )
                )

                self.ch_on_hex.set(
                    rgb255_to_hex(
                        cfg.get(
                            "chirp_on_color_rgb",
                            [255, 255, 255]
                        )
                    )
                )
                self.ch_off_hex.set(
                    rgb255_to_hex(
                        cfg.get(
                            "chirp_off_color_rgb",
                            [0, 0, 0]
                        )
                    )
                )
                self.ch_final.set(
                    cfg.get(
                        "chirp_final_gray_s",
                        2.0
                    )
                )
                self.ch_reps.set(
                    cfg.get(
                        "chirp_repetitions",
                        1
                    )
                )

                self._update_temporal_mode()
                target = 5

            else:
                raise ValueError(
                    f"Tipo de estímulo no reconocido: {stimulus_type}"
                )

        except Exception as exc:
            messagebox.showerror(
                "Editar paso",
                (
                    "No se pudo cargar la configuración del paso.\n\n"
                    f"{exc}"
                )
            )
            return False

        self.tabs.select(target)
        self.after(
            30,
            self.refresh_active_preview
        )
        return True

    def _protocol_type_name(
        self, step
    ):
        if step.get("kind") == "pause":
            return "Pausa"

        cfg = step.get(
            "config", {}
        )
        stype = step.get(
            "stimulus_type",
            cfg.get("stimulus_type")
        )

        names = {
            "on_off": "ON / OFF",
            "motion": "Motion",
            "grating": "Grating",
            "dense_noise": "Noise",
            "rf_mapping": "RF Mapping",
            "chirp_intensity":
                "Temporal",
        }
        return names.get(
            stype,
            str(stype)
        )

    def _protocol_list_count(self, text):
        return len([
            x
            for x in str(text).replace(
                ";", ","
            ).split(",")
            if x.strip()
        ])

    def _protocol_step_duration(
        self, step
    ):
        if step.get("kind") == "pause":
            return float(
                step.get(
                    "duration_s", 0.0
                )
            )

        cfg = step.get(
            "config", {}
        )
        stype = cfg.get(
            "stimulus_type"
        )

        try:
            if stype == "on_off":
                return int(
                    cfg["repetitions"]
                ) * sum([
                    float(cfg["baseline_s"]),
                    float(cfg["on_s"]),
                    float(cfg["gray_interval_s"]),
                    float(cfg["off_s"]),
                    float(cfg["final_gray_s"]),
                ])

            if stype == "motion":
                mode = cfg["motion_mode"]
                if mode in (
                    "Moving Bar",
                    "Moving Spot",
                ):
                    n = max(
                        1,
                        self._protocol_list_count(
                            cfg[
                                "motion_directions"
                            ]
                        )
                    )
                else:
                    n = 1

                return (
                    int(
                        cfg[
                            "motion_repetitions"
                        ]
                    )
                    * n
                    * (
                        float(
                            cfg[
                                "motion_baseline_s"
                            ]
                        )
                        + float(
                            cfg[
                                "motion_duration_s"
                            ]
                        )
                        + float(
                            cfg[
                                "motion_isi_s"
                            ]
                        )
                    )
                )

            if stype == "grating":
                angles = cfg.get(
                    "grating_angles",
                    cfg.get(
                        "grating_directions",
                        "0"
                    )
                )
                n = max(
                    1,
                    self._protocol_list_count(
                        angles
                    )
                )
                return (
                    int(
                        cfg[
                            "grating_repetitions"
                        ]
                    )
                    * n
                    * (
                        float(
                            cfg[
                                "grating_baseline_s"
                            ]
                        )
                        + float(
                            cfg[
                                "grating_duration_s"
                            ]
                        )
                        + float(
                            cfg[
                                "grating_isi_s"
                            ]
                        )
                    )
                )

            if stype == "dense_noise":
                return int(
                    cfg["dense_repetitions"]
                ) * (
                    float(
                        cfg["dense_baseline_s"]
                    )
                    + float(
                        cfg["dense_duration_s"]
                    )
                    + float(
                        cfg["dense_isi_s"]
                    )
                )

            if stype == "rf_mapping":
                mode = cfg["rf_mode"]
                if mode == "Size series":
                    n = max(
                        1,
                        self._protocol_list_count(
                            cfg[
                                "rf_size_series_deg"
                            ]
                        )
                    )
                elif mode == "Annulus Size series":
                    n = max(
                        1,
                        self._protocol_list_count(
                            cfg[
                                "rf_annulus_size_series_deg"
                            ]
                        )
                    )
                elif mode == "Sparse Noise":
                    n = max(
                        1,
                        int(
                            cfg[
                                "rf_sparse_presentations"
                            ]
                        )
                    )
                else:
                    n = 1

                return int(
                    cfg["rf_repetitions"]
                ) * (
                    float(
                        cfg["rf_baseline_s"]
                    )
                    + n * (
                        float(
                            cfg[
                                "rf_duration_s"
                            ]
                        )
                        + float(
                            cfg["rf_isi_s"]
                        )
                    )
                )

            if stype == "chirp_intensity":
                mode = cfg.get(
                    "temporal_mode",
                    "Chirp + Intensity"
                )
                reps = int(
                    cfg.get(
                        "chirp_repetitions",
                        1
                    )
                )
                basal = float(
                    cfg.get(
                        "chirp_basal_s",
                        0.0
                    )
                )
                final = float(
                    cfg.get(
                        "chirp_final_gray_s",
                        0.0
                    )
                )

                if mode == "Chirp + Intensity":
                    one = sum([
                        basal,
                        float(
                            cfg[
                                "chirp_step_on_s"
                            ]
                        ),
                        float(
                            cfg[
                                "chirp_step_off_s"
                            ]
                        ),
                        float(
                            cfg[
                                "chirp_pause1_s"
                            ]
                        ),
                        float(
                            cfg[
                                "chirp_frequency_duration_s"
                            ]
                        ),
                        float(
                            cfg[
                                "chirp_pause2_s"
                            ]
                        ),
                        float(
                            cfg[
                                "chirp_intensity_duration_s"
                            ]
                        ),
                        final,
                    ])
                    return reps * one

                duration = float(
                    cfg[
                        "temporal_duration_s"
                    ]
                )

                if mode == "Contrast Series":
                    n = max(
                        1,
                        self._protocol_list_count(
                            cfg[
                                "temporal_contrast_levels_pct"
                            ]
                        )
                    )
                    return reps * (
                        basal
                        + n * (
                            duration
                            + float(
                                cfg[
                                    "temporal_contrast_isi_s"
                                ]
                            )
                        )
                        + final
                    )

                return reps * (
                    basal
                    + duration
                    + final
                )

        except Exception:
            return 0.0

        return 0.0

    def _protocol_step_summary(
        self, step
    ):
        if step.get("kind") == "pause":
            return (
                f"adaptación · "
                f"{float(step.get('duration_s', 0)):.3g} s"
            )

        cfg = step.get(
            "config", {}
        )
        stype = cfg.get(
            "stimulus_type"
        )

        try:
            if stype == "on_off":
                return (
                    f"ON {cfg['on_s']} s "
                    f"{rgb255_to_hex(cfg['on_color_rgb'])} · "
                    f"OFF {cfg['off_s']} s "
                    f"{rgb255_to_hex(cfg['off_color_rgb'])} · "
                    f"{cfg['repetitions']} rep"
                )

            if stype == "motion":
                mode = cfg["motion_mode"]

                if mode == "Moving Bar":
                    detail = (
                        f"{cfg['motion_directions']}° · "
                        f"barra "
                        f"{cfg['motion_bar_width_deg']}×"
                        f"{cfg['motion_bar_length_deg']}°"
                    )
                elif mode == "Moving Spot":
                    detail = (
                        f"{cfg['motion_directions']}° · "
                        f"spot "
                        f"{cfg['motion_spot_diameter_deg']}°"
                    )
                elif mode in (
                    "Looming",
                    "Receding",
                ):
                    detail = (
                        f"{cfg['motion_small_diameter_deg']}↔"
                        f"{cfg['motion_large_diameter_deg']}° · "
                        f"centro "
                        f"({cfg['motion_center_x_deg']}, "
                        f"{cfg['motion_center_y_deg']})°"
                    )
                else:
                    detail = (
                        f"Ø ext. "
                        f"{cfg.get('motion_annulus_small_diameter_deg', 8.0)}↔"
                        f"{cfg.get('motion_annulus_large_diameter_deg', 40.0)}° · "
                        f"grosor "
                        f"{cfg.get('motion_annulus_thickness_deg', 2.0)}° · "
                        f"centro "
                        f"({cfg['motion_center_x_deg']}, "
                        f"{cfg['motion_center_y_deg']})°"
                    )

                bg_hex = rgb255_to_hex(
                    cfg["motion_background_rgb"]
                )
                stim_hex = rgb255_to_hex(
                    cfg["motion_stimulus_rgb"]
                )

                return (
                    f"{mode} · {detail} · "
                    f"fondo {bg_hex} · estímulo {stim_hex} · "
                    f"{cfg['motion_repetitions']} rep"
                )

            if stype == "grating":
                mode = cfg.get(
                    "grating_mode",
                    "Drifting"
                )
                if mode == "Gabor":
                    mode = "Static Gabor"

                angles = cfg.get(
                    "grating_angles",
                    cfg.get(
                        "grating_directions",
                        "0"
                    )
                )

                if mode == "Drifting":
                    detail = (
                        f"{angles}° · "
                        f"{cfg['grating_sf_cpd']} c/° · "
                        f"{cfg.get('grating_tf_hz', 1.0)} Hz"
                    )
                elif mode == "Asymmetric Drift":
                    polarity = cfg.get(
                        "grating_asymmetric_polarity",
                        "A_TO_B"
                    )
                    detail = (
                        f"{angles}° · "
                        f"{cfg['grating_sf_cpd']} c/° · "
                        f"{cfg.get('grating_tf_hz', 1.0)} Hz · "
                        f"{polarity}"
                    )
                elif mode == "Phase Reversal":
                    detail = (
                        f"{angles}° · "
                        f"{cfg['grating_sf_cpd']} c/° · "
                        f"reversal "
                        f"{cfg.get('grating_reversal_hz', 2.0)} Hz"
                    )
                elif mode == "Static Gabor":
                    detail = (
                        f"{angles}° · "
                        f"{cfg['grating_sf_cpd']} c/° · "
                        f"{cfg.get('grating_gabor_size_deg', 30.0)}° "
                        f"σ={cfg.get('grating_gabor_sigma_deg', 6.0)}°"
                    )
                elif mode == "Drifting Gabor":
                    detail = (
                        f"{angles}° · "
                        f"{cfg['grating_sf_cpd']} c/° · "
                        f"{cfg.get('grating_tf_hz', 1.0)} Hz · "
                        f"{cfg.get('grating_gabor_size_deg', 30.0)}° "
                        f"σ={cfg.get('grating_gabor_sigma_deg', 6.0)}°"
                    )
                else:
                    detail = (
                        f"{angles}° · "
                        f"{cfg['grating_sf_cpd']} c/° · "
                        f"fase "
                        f"{cfg.get('grating_phase_cycles', 0.0)}"
                    )

                return (
                    f"{mode} · {detail} · "
                    f"A {rgb255_to_hex(cfg['grating_color_a_rgb'])} · "
                    f"B {rgb255_to_hex(cfg['grating_color_b_rgb'])}"
                )

            if stype == "dense_noise":
                family = cfg.get(
                    "dense_family_mode",
                    "Dense White Noise"
                )
                distribution = cfg.get(
                    "dense_distribution",
                    cfg.get(
                        "dense_mode",
                        "Binary"
                    )
                )

                if family == "Gaussian White Noise":
                    descriptor = (
                        "Gaussian"
                    )
                elif family == "Frozen Noise":
                    descriptor = (
                        f"Frozen {distribution}"
                    )
                else:
                    descriptor = (
                        f"Dense {distribution}"
                    )

                return (
                    f"{descriptor} · "
                    f"{cfg['dense_update_hz']} Hz nominal · "
                    f"check {cfg['dense_check_size_deg']}° · "
                    f"{cfg['dense_repetitions']} rep"
                )

            if stype == "rf_mapping":
                mode = cfg["rf_mode"]
                if mode == "Spot":
                    detail = (
                        f"spot {cfg['rf_spot_diameter_deg']}°"
                    )
                elif mode == "Annulus":
                    detail = (
                        f"{cfg['rf_annulus_inner_deg']}→"
                        f"{cfg['rf_annulus_outer_deg']}°"
                    )
                elif mode == "Size series":
                    detail = cfg[
                        "rf_size_series_deg"
                    ]
                elif mode == "Annulus Size series":
                    detail = (
                        f"{cfg['rf_annulus_size_series_deg']}° · "
                        f"grosor "
                        f"{cfg['rf_annulus_series_thickness_deg']}°"
                    )
                else:
                    detail = (
                        f"{cfg['rf_sparse_presentations']} probes · "
                        f"{cfg['rf_sparse_polarity']}"
                    )
                return (
                    f"{mode} · {detail}"
                )

            if stype == "chirp_intensity":
                mode = cfg.get(
                    "temporal_mode",
                    "Chirp + Intensity"
                )

                if mode == "Chirp + Intensity":
                    return (
                        f"Chirp "
                        f"{cfg['chirp_frequency_min_hz']}→"
                        f"{cfg['chirp_frequency_max_hz']} Hz · "
                        f"Intensity "
                        f"{cfg['chirp_intensity_frequency_hz']} Hz"
                    )

                if mode == "Sinusoidal Flicker":
                    return (
                        f"Sinusoidal · "
                        f"{cfg['temporal_frequency_hz']} Hz · "
                        f"{cfg['temporal_contrast_pct']} %"
                    )

                if mode == "Gaussian Flicker":
                    return (
                        f"Gaussian · "
                        f"{cfg['temporal_update_hz']} Hz nominal · "
                        f"{cfg['temporal_contrast_pct']} % · "
                        f"seed {cfg['temporal_seed']}"
                    )

                return (
                    f"Contrast Series · "
                    f"{cfg['temporal_frequency_hz']} Hz · "
                    f"{cfg['temporal_contrast_levels_pct']} %"
                )

        except Exception:
            pass

        return "Configuración guardada"

    @staticmethod
    def _protocol_format_clock(
        seconds
    ):
        seconds = max(
            0.0,
            float(seconds)
        )
        total = int(
            round(seconds)
        )
        hours, rem = divmod(
            total,
            3600
        )
        minutes, secs = divmod(
            rem,
            60
        )
        return (
            f"{hours:02d}:"
            f"{minutes:02d}:"
            f"{secs:02d}"
        )

    @staticmethod

    def _protocol_format_duration(
        seconds
    ):
        seconds = max(
            0.0,
            float(seconds)
        )

        if seconds < 60:
            return f"{seconds:.1f} s"

        total = int(round(seconds))
        hours, rem = divmod(
            total, 3600
        )
        minutes, secs = divmod(
            rem, 60
        )

        if hours:
            return (
                f"{hours:d}:"
                f"{minutes:02d}:"
                f"{secs:02d}"
            )
        return (
            f"{minutes:d}:"
            f"{secs:02d}"
        )

    def _refresh_protocol_editor(
        self, select_index=None
    ):
        if not hasattr(
            self,
            "protocol_tree"
        ):
            return

        self._sync_protocol_header_from_vars()

        self.protocol_tree.delete(
            *self.protocol_tree.get_children()
        )

        steps = self.protocol_draft.get(
            "steps", []
        )

        total_one = 0.0

        for idx, step in enumerate(
            steps
        ):
            duration = (
                self._protocol_step_duration(
                    step
                )
            )
            total_one += duration

            self.protocol_tree.insert(
                "",
                "end",
                iid=f"step_{idx}",
                values=(
                    idx + 1,
                    self._protocol_type_name(
                        step
                    ),
                    self._protocol_step_summary(
                        step
                    ),
                    self._protocol_format_duration(
                        duration
                    ),
                )
            )

        try:
            reps = max(
                1,
                int(
                    self.protocol_reps_var.get()
                )
            )
        except Exception:
            reps = 1

        total = total_one * reps

        stimulus_count = sum(
            1
            for step in steps
            if step.get("kind") == "stimulus"
        )
        pause_count = sum(
            1
            for step in steps
            if step.get("kind") == "pause"
        )

        self.protocol_duration_var.set(
            self._protocol_format_clock(
                total
            )
        )
        self.protocol_duration_detail_var.set(
            (
                "1 vuelta: "
                f"{self._protocol_format_clock(total_one)}"
                f" · ×{reps} "
                + (
                    "repetición"
                    if reps == 1
                    else "repeticiones"
                )
            )
        )
        self.protocol_structure_var.set(
            (
                f"{len(steps)} "
                + (
                    "paso"
                    if len(steps) == 1
                    else "pasos"
                )
                + f" · {stimulus_count} "
                + (
                    "estímulo"
                    if stimulus_count == 1
                    else "estímulos"
                )
                + f" · {pause_count} "
                + (
                    "pausa"
                    if pause_count == 1
                    else "pausas"
                )
            )
        )

        if (
            select_index is not None
            and 0 <= select_index < len(steps)
        ):
            iid = f"step_{select_index}"
            self.protocol_tree.selection_set(
                iid
            )
            self.protocol_tree.focus(
                iid
            )
            self.protocol_tree.see(
                iid
            )

        self._update_protocol_execute_state()

    def _worker_ready_for_screen(self, screen_index):
        """
        Return True only when the live PsychoPy worker itself reports
        READY on the selected experimental monitor.

        The status file is authoritative. current_worker_screen is kept
        synchronized from that status instead of being a hard requirement,
        avoiding false 'Pantalla no preparada' warnings when the GUI's
        cached screen state lags behind the worker.
        """
        if (
            self.worker_process is None
            or self.worker_process.poll() is not None
            or not self.status_path.exists()
        ):
            return False

        try:
            st = json.loads(
                self.status_path.read_text(
                    encoding="utf-8"
                )
            )
        except Exception:
            return False

        if st.get("state") != "ready":
            return False

        try:
            requested = int(screen_index)
        except Exception:
            return False

        reported = st.get(
            "screen_index",
            self.current_worker_screen
        )

        try:
            reported = int(reported)
        except Exception:
            return False

        if reported != requested:
            return False

        self.current_worker_screen = reported

        if "hz" in st:
            try:
                self.current_measured_hz = float(
                    st["hz"]
                )
            except Exception:
                pass

        return True

    def _write_protocol_pause_request(
        self,
        requested,
        command_id=None
    ):
        if command_id is None:
            command_id = (
                self.active_command_id
                or ""
            )

        write_json_atomic(
            self.protocol_pause_path,
            {
                "command_id":
                    str(command_id),
                "pause_requested":
                    bool(requested),
                "requested_at":
                    datetime.now().isoformat(
                        timespec="milliseconds"
                    ),
            }
        )

    def _update_protocol_pause_button(self):
        if not hasattr(
            self,
            "protocol_pause_runtime_btn"
        ):
            return

        if (
            self.active_command_id is None
            or self.active_run_type != "protocol"
        ):
            text = "⏸ Pausar protocolo"
            state = "disabled"
        elif self.protocol_pause_state == "pause_requested":
            text = "⏸ Pausa solicitada…"
            state = "disabled"
        elif self.protocol_pause_state == "paused":
            text = "▶ Reanudar protocolo"
            state = "normal"
        elif self.protocol_pause_state == "resuming":
            text = "▶ Reanudando…"
            state = "disabled"
        else:
            text = "⏸ Pausar protocolo"
            state = "normal"

        try:
            self.protocol_pause_runtime_btn.configure(
                text=text,
                state=state
            )
        except Exception:
            pass

    def _toggle_protocol_pause(self):
        if (
            self.active_command_id is None
            or self.active_run_type != "protocol"
        ):
            return

        if self.protocol_pause_state == "paused":
            try:
                self._write_protocol_pause_request(
                    False
                )
            except Exception as exc:
                messagebox.showerror(
                    "Reanudar protocolo",
                    (
                        "No se ha podido enviar la orden de "
                        f"reanudación.\n\n{exc}"
                    ),
                    parent=self
                )
                return

            self.protocol_pause_state = "resuming"
            self.protocol_builder_status.set(
                "Reanudando protocolo… se mantiene el fondo de adaptación "
                "hasta que comience el siguiente paso."
            )
            self.status.set(
                "Reanudando protocolo…"
            )
            self._update_protocol_pause_button()
            return

        if self.protocol_pause_state != "running":
            return

        try:
            self._write_protocol_pause_request(
                True
            )
        except Exception as exc:
            messagebox.showerror(
                "Pausar protocolo",
                (
                    "No se ha podido solicitar la pausa.\n\n"
                    f"{exc}"
                ),
                parent=self
            )
            return

        self.protocol_pause_state = "pause_requested"
        self.protocol_builder_status.set(
            (
                "Pausa solicitada · el paso actual terminará "
                "normalmente. El protocolo se detendrá sobre el fondo de adaptación "
                "antes de iniciar el siguiente paso."
            )
        )
        self.status.set(
            "Pausa solicitada · esperando un punto seguro…"
        )
        self._update_protocol_pause_button()

    def _protocol_can_execute(self):
        return bool(
            self.protocol_draft.get("id")
            and self.protocol_draft.get("steps")
            and not self.protocol_dirty
        )

    def _update_protocol_execute_state(self):
        if not hasattr(
            self,
            "protocol_execute_btn"
        ):
            return

        worker_ready = False

        try:
            idx = self.combo.current()
            if idx >= 0:
                worker_ready = self._worker_ready_for_screen(
                    idx
                )
        except Exception:
            worker_ready = False

        state = (
            "normal"
            if (
                self._protocol_can_execute()
                and self.active_command_id is None
                and worker_ready
            )
            else "disabled"
        )

        try:
            self.protocol_execute_btn.configure(
                state=state
            )
        except Exception:
            pass

    def _execute_protocol(self):
        if self._trigger_test_active:
            messagebox.showinfo(
                "Prueba de triggers activa",
                "Termina o detén la prueba de triggers antes de ejecutar el estímulo.",
                parent=self,
            )
            return
        if not self._protocol_can_execute():
            messagebox.showinfo(
                "Protocolos",
                (
                    "Guarda el protocolo antes de ejecutarlo. "
                    "Así el registro experimental queda asociado "
                    "a una versión concreta y reproducible."
                ),
                parent=self
            )
            return

        protocol = self._protocol_clone(
            self.protocol_draft
        )

        steps = protocol.get(
            "steps", []
        )
        if not steps:
            return

        try:
            repetitions = int(
                protocol.get(
                    "repetitions", 1
                )
            )
            if repetitions < 1:
                raise ValueError

            common = self.common_config()
        except Exception as exc:
            messagebox.showerror(
                "No se puede ejecutar",
                str(exc),
                parent=self
            )
            return

        worker_ready = self._worker_ready_for_screen(
            common["screen_index"]
        )

        if not worker_ready:
            self._update_protocol_execute_state()

            state_text = ""
            try:
                if self.status_path.exists():
                    st = json.loads(
                        self.status_path.read_text(
                            encoding="utf-8"
                        )
                    )
                    state_text = str(
                        st.get("state", "")
                    )
            except Exception:
                state_text = ""

            if state_text in (
                "measuring",
                "starting"
            ):
                detail = (
                    "PsychoPy todavía está preparando la pantalla "
                    "o midiendo el refresco. Espera unos segundos; "
                    "el botón se habilitará automáticamente cuando "
                    "el estado sea estable."
                )
            else:
                detail = (
                    "La ventana PsychoPy no está preparada en el "
                    "monitor seleccionado. La capa de seguridad "
                    "permanece activa. Usa «Reiniciar pantalla» sólo "
                    "si el estado no vuelve a preparado."
                )

            messagebox.showwarning(
                "Pantalla no preparada",
                detail,
                parent=self
            )
            return

        # Annotate each step for readable logs without modifying the
        # persistent library definition.
        execution_steps = []
        for index, step in enumerate(
            steps, start=1
        ):
            item = self._protocol_clone(
                step
            )
            item["step_index"] = index
            item["label"] = (
                self._protocol_type_name(
                    step
                )
            )
            execution_steps.append(
                item
            )

        one_pass_s = sum(
            self._protocol_step_duration(
                step
            )
            for step in steps
        )
        total_s = one_pass_s * repetitions

        ttl_label = self.ttl_mode.get()
        hz_label = (
            "—"
            if self.current_measured_hz is None
            else f"{self.current_measured_hz:.3f} Hz"
        )

        # Noise is intentionally allowed to request a nominal update
        # rate above display refresh. Mention it only as metadata here.
        dense_over_refresh = []
        if self.current_measured_hz is not None:
            for idx, step in enumerate(
                steps, start=1
            ):
                cfg = step.get(
                    "config", {}
                )
                if (
                    cfg.get("stimulus_type")
                    == "dense_noise"
                ):
                    try:
                        requested = float(
                            cfg[
                                "dense_update_hz"
                            ]
                        )
                        if (
                            requested
                            > self.current_measured_hz
                        ):
                            dense_over_refresh.append(
                                (
                                    idx,
                                    requested
                                )
                            )
                    except Exception:
                        pass

        extra = ""
        if dense_over_refresh:
            values = ", ".join(
                f"paso {idx}: {freq:g} Hz"
                for idx, freq
                in dense_over_refresh
            )
            extra = (
                "\n\nNoise nominal por encima del refresco "
                f"({values}). Se permitirá y se registrará la "
                "frecuencia realmente presentada."
            )

        confirmed = messagebox.askyesno(
            "Ejecutar protocolo",
            (
                f"{protocol.get('name', 'Protocolo')} "
                f"· v{protocol.get('version', 1)}\n\n"
                f"Pasos: {len(steps)}\n"
                f"Repeticiones del protocolo: {repetitions}\n"
                f"Duración estimada: "
                f"{self._protocol_format_duration(total_s)}\n\n"
                f"Refresco medido: {hz_label}\n"
                f"TTL: {ttl_label}\n\n"
                "Todo el protocolo se enviará al worker como "
                "una única ejecución. Antes del primer frame registrado, "
                "PsychoPy preparará los recursos gráficos manteniendo la "
                "pantalla en el fondo de adaptación. Después generará un único CSV visual "
                "y un único CSV TTL."
                f"{extra}\n\n"
                "¿Iniciar ahora?"
            ),
            parent=self
        )
        if not confirmed:
            return

        cfg = dict(common)
        cfg.update({
            "stimulus_type": "protocol",
            "protocol_id":
                protocol.get("id"),
            "protocol_name":
                protocol.get(
                    "name",
                    "Protocolo"
                ),
            "protocol_version":
                int(
                    protocol.get(
                        "version", 1
                    )
                ),
            "protocol_repetitions":
                repetitions,
            "protocol_steps":
                execution_steps,
        })

        self.protocol_execute_btn.configure(
            state="disabled"
        )
        self.protocol_builder_status.set(
            (
                f"Ejecutando «{cfg['protocol_name']}» "
                f"v{cfg['protocol_version']} · "
                f"{len(steps)} pasos × {repetitions}. "
                "ABORTAR → GRIS detiene el protocolo completo."
            )
        )

        self.send_run_command(cfg)
        self.status.set(
            (
                f"Protocolo enviado · "
                f"{cfg['protocol_name']} v{cfg['protocol_version']}"
            )
        )

    def _current_tab_is_protocols(self):
        try:
            return (
                self.tabs.index(
                    self.tabs.select()
                ) == 6
            )
        except Exception:
            return False

    def _current_tab_is_system(self):
        try:
            return self.tabs.select() == str(self.tab_system)
        except Exception:
            return False

    def _preview_key_from_tab(self):
        try:
            idx = self.tabs.index(
                self.tabs.select()
            )
        except Exception:
            return None

        keys = [
            "on_off",
            "motion",
            "grating",
            "dense_noise",
            "rf_mapping",
            "chirp_intensity",
        ]
        if 0 <= idx < len(keys):
            return keys[idx]
        return None

    def _on_tab_changed(self, event=None):
        self._stop_preview(
            redraw=False
        )
        self._stop_easter_egg(
            redraw=False
        )

        key = self._preview_key_from_tab()
        if key:
            try:
                canvas = self.tab_scroll_canvases.get(
                    key
                )
                if canvas is not None:
                    canvas.yview_moveto(0.0)
            except Exception:
                pass

        if self._current_tab_is_protocols() or self._current_tab_is_system():
            try:
                self.start_btn.configure(state="disabled")
            except Exception:
                pass
            if self._current_tab_is_protocols():
                self._refresh_protocol_editor()

        self.after(
            40,
            self.refresh_active_preview
        )

    def refresh_active_preview(self):
        key = self._preview_key_from_tab()
        if key:
            self._draw_preview_static(key)

    def _safe_float(self, var, default=0.0):
        try:
            return float(
                var.get().replace(",", ".")
            )
        except Exception:
            return float(default)

    def _safe_int(self, var, default=1):
        try:
            return int(var.get())
        except Exception:
            return int(default)

    @staticmethod
    def _preview_list_count(text):
        try:
            items = [
                x.strip()
                for x in str(text).replace(
                    ";", ","
                ).split(",")
                if x.strip()
            ]
            return max(1, len(items))
        except Exception:
            return 1

    @staticmethod
    def _preview_first_angle(text):
        try:
            part = str(text).replace(
                ";", ","
            ).split(",")[0].strip()
            return float(part)
        except Exception:
            return 0.0

    @staticmethod
    def _hex_rgb(value):
        value = str(value).strip().lstrip("#")
        if len(value) != 6:
            return (128, 128, 128)
        try:
            return (
                int(value[0:2], 16),
                int(value[2:4], 16),
                int(value[4:6], 16),
            )
        except Exception:
            return (128, 128, 128)

    @classmethod
    def _blend_hex(cls, a, b, alpha):
        ar, ag, ab = cls._hex_rgb(a)
        br, bg, bb = cls._hex_rgb(b)
        alpha = max(
            0.0, min(1.0, float(alpha))
        )
        r = round(ar + (br - ar) * alpha)
        g = round(ag + (bg - ag) * alpha)
        bl = round(ab + (bb - ab) * alpha)
        return f"#{r:02X}{g:02X}{bl:02X}"

    def _preview_duration(self, key):
        try:
            if key == "on_off":
                one = (
                    self._safe_float(
                        self.on_baseline
                    )
                    + self._safe_float(
                        self.on_on
                    )
                    + self._safe_float(
                        self.on_gray
                    )
                    + self._safe_float(
                        self.on_off
                    )
                    + self._safe_float(
                        self.on_final
                    )
                )
                return one * self._safe_int(
                    self.on_reps
                )

            if key == "motion":
                one = (
                    self._safe_float(
                        self.motion_baseline
                    )
                    + self._safe_float(
                        self.motion_duration
                    )
                    + self._safe_float(
                        self.motion_isi
                    )
                )

                mode = self.motion_mode.get()
                if mode in (
                    "Moving Bar",
                    "Moving Spot",
                ):
                    n_cond = max(
                        1,
                        self._preview_list_count(
                            self.motion_dirs.get()
                        )
                    )
                else:
                    n_cond = 1

                return (
                    one
                    * n_cond
                    * self._safe_int(
                        self.motion_reps
                    )
                )

            if key == "grating":
                one = (
                    self._safe_float(
                        self.gr_baseline
                    )
                    + self._safe_float(
                        self.gr_duration
                    )
                    + self._safe_float(
                        self.gr_isi
                    )
                )
                n = max(
                    1,
                    self._preview_list_count(
                        self.gr_angles.get()
                    )
                )
                return (
                    one
                    * n
                    * self._safe_int(
                        self.gr_reps
                    )
                )

            if key == "dense_noise":
                one = (
                    self._safe_float(
                        self.dn_baseline
                    )
                    + self._safe_float(
                        self.dn_duration
                    )
                    + self._safe_float(
                        self.dn_isi
                    )
                )
                return one * self._safe_int(
                    self.dn_reps
                )

            if key == "rf_mapping":
                mode = self.rf_mode.get()
                baseline = self._safe_float(
                    self.rf_baseline
                )
                duration = self._safe_float(
                    self.rf_duration
                )
                isi = self._safe_float(
                    self.rf_isi
                )
                reps = self._safe_int(
                    self.rf_reps
                )

                if mode == "Size series":
                    n_cond = self._preview_list_count(
                        self.rf_size_series.get()
                    )
                elif mode == "Annulus Size series":
                    n_cond = self._preview_list_count(
                        self.rf_annulus_size_series.get()
                    )
                elif mode == "Sparse Noise":
                    n_cond = max(
                        1,
                        self._safe_int(
                            self.rf_sparse_trials
                        )
                    )
                else:
                    n_cond = 1

                return reps * (
                    baseline
                    + n_cond * (duration + isi)
                )

            if key == "chirp_intensity":
                mode = self.temporal_mode.get()
                reps = self._safe_int(
                    self.ch_reps
                )
                basal = self._safe_float(
                    self.ch_basal
                )
                final = self._safe_float(
                    self.ch_final
                )

                if mode == "Chirp + Intensity":
                    one = (
                        basal
                        + self._safe_float(
                            self.ch_step_on
                        )
                        + self._safe_float(
                            self.ch_step_off
                        )
                        + self._safe_float(
                            self.ch_pause1
                        )
                        + self._safe_float(
                            self.ch_fdur
                        )
                        + self._safe_float(
                            self.ch_pause2
                        )
                        + self._safe_float(
                            self.ch_idur
                        )
                        + final
                    )
                    return one * reps

                duration = self._safe_float(
                    self.ch_temp_duration
                )

                if mode == "Contrast Series":
                    n = max(
                        1,
                        self._preview_list_count(
                            self.ch_contrast_levels.get()
                        )
                    )
                    return reps * (
                        basal
                        + n * (
                            duration
                            + self._safe_float(
                                self.ch_contrast_isi
                            )
                        )
                        + final
                    )

                return reps * (
                    basal
                    + duration
                    + final
                )
        except Exception:
            pass
        return 0.0

    def _set_preview_summary(
        self, key, extra=""
    ):
        var = self.preview_summary_vars.get(
            key
        )
        if var is None:
            return

        duration = self._preview_duration(key)
        text = (
            f"Duración estimada: {duration:.1f} s"
        )
        if extra:
            text += "\n" + extra
        var.set(text)

    def _preview_monitor(
        self, canvas, fill="#808080"
    ):
        canvas.delete("all")
        canvas.create_rectangle(
            18, 18, 312, 175,
            fill=fill,
            outline="#5E6B72",
            width=2
        )
        canvas.create_text(
            165, 190,
            text="monitor experimental",
            fill=self.palette["muted"],
            font=("Segoe UI", 9)
        )

    def _timeline(
        self, canvas, segments
    ):
        x0, x1 = 26, 304
        y0, y1 = 205, 231

        total = sum(
            max(0.01, float(d))
            for d, _, _ in segments
        )
        x = x0

        for i, (duration, color, label) in enumerate(
            segments
        ):
            if i == len(segments) - 1:
                nx = x1
            else:
                nx = x + (
                    (x1 - x0)
                    * max(0.01, float(duration))
                    / total
                )

            canvas.create_rectangle(
                x, y0, nx, y1,
                fill=color,
                outline="#FFFFFF"
            )

            if nx - x >= 30:
                canvas.create_text(
                    (x + nx) / 2,
                    (y0 + y1) / 2,
                    text=label,
                    fill=best_text_color(color),
                    font=("Segoe UI Semibold", 9)
                )
            x = nx

    def _draw_preview_static(self, key):
        canvas = self.preview_canvases.get(
            key
        )
        if canvas is None:
            return

        self._stop_preview(
            redraw=False
        )
        self._stop_easter_egg(
            redraw=False
        )

        if key == "on_off":
            on = self.on_on_hex.get()
            off = self.on_off_hex.get()

            self._preview_monitor(
                canvas, "#808080"
            )
            self._timeline(
                canvas,
                [
                    (
                        self._safe_float(
                            self.on_baseline, 2
                        ),
                        "#808080", "gris"
                    ),
                    (
                        self._safe_float(
                            self.on_on, 2
                        ),
                        on, "ON"
                    ),
                    (
                        self._safe_float(
                            self.on_gray, 2
                        ),
                        "#808080", "gris"
                    ),
                    (
                        self._safe_float(
                            self.on_off, 2
                        ),
                        off, "OFF"
                    ),
                    (
                        self._safe_float(
                            self.on_final, 2
                        ),
                        "#808080", "gris"
                    ),
                ]
            )
            self._set_preview_summary(
                key,
                (
                    f"Full-field · ON {on} · "
                    f"OFF {off}"
                )
            )

        elif key == "motion":
            mode = self.motion_mode.get()
            bg = self.motion_background_hex.get()
            fg = self.motion_stimulus_hex.get()

            self._preview_monitor(
                canvas, bg
            )

            cx, cy = 165, 96

            if mode in (
                "Moving Bar",
                "Moving Spot",
            ):
                angle = self._preview_first_angle(
                    self.motion_dirs.get()
                )
                theta = math.radians(angle)
                dx = math.cos(theta)
                dy = -math.sin(theta)

                path = 105
                canvas.create_line(
                    cx - dx * path,
                    cy - dy * path,
                    cx + dx * path,
                    cy + dy * path,
                    fill="#F0A44B",
                    width=3,
                    arrow="last",
                    arrowshape=(10, 12, 5)
                )

                if mode == "Moving Bar":
                    ax, ay = -dy, dx
                    half_l = 70
                    half_w = 10
                    points = [
                        (cx + ax * half_l + dx * half_w,
                         cy + ay * half_l + dy * half_w),
                        (cx - ax * half_l + dx * half_w,
                         cy - ay * half_l + dy * half_w),
                        (cx - ax * half_l - dx * half_w,
                         cy - ay * half_l - dy * half_w),
                        (cx + ax * half_l - dx * half_w,
                         cy + ay * half_l - dy * half_w),
                    ]
                    flat = [
                        value
                        for point in points
                        for value in point
                    ]
                    canvas.create_polygon(
                        *flat,
                        fill=fg,
                        outline="#5E6B72",
                        width=1
                    )
                    extra = (
                        f"Moving Bar · 1ª dirección {angle:g}° · "
                        f"{self.motion_bar_width.get()}×"
                        f"{self.motion_bar_length.get()}°"
                    )
                else:
                    d = max(
                        14,
                        min(
                            90,
                            self._safe_float(
                                self.motion_spot_diameter,
                                8
                            ) * 4.0
                        )
                    )
                    canvas.create_oval(
                        cx - d / 2,
                        cy - d / 2,
                        cx + d / 2,
                        cy + d / 2,
                        fill=fg,
                        outline="#5E6B72"
                    )
                    extra = (
                        f"Moving Spot · 1ª dirección {angle:g}° · "
                        f"diámetro {self.motion_spot_diameter.get()}°"
                    )

            elif mode in (
                "Looming",
                "Receding",
            ):
                small = max(
                    8.0,
                    min(
                        90.0,
                        self._safe_float(
                            self.motion_small_diameter,
                            2
                        ) * 3.0
                    )
                )
                large = max(
                    small + 8.0,
                    min(
                        150.0,
                        self._safe_float(
                            self.motion_large_diameter,
                            40
                        ) * 3.0
                    )
                )

                canvas.create_oval(
                    cx - large / 2,
                    cy - large / 2,
                    cx + large / 2,
                    cy + large / 2,
                    outline=fg,
                    width=2
                )
                canvas.create_oval(
                    cx - small / 2,
                    cy - small / 2,
                    cx + small / 2,
                    cy + small / 2,
                    fill=fg,
                    outline=fg
                )

                arrow = "last"
                label = "expansión"
                if mode == "Receding":
                    arrow = "first"
                    label = "contracción"

                canvas.create_line(
                    cx + small / 2 + 7,
                    cy,
                    cx + large / 2 - 4,
                    cy,
                    fill="#F0A44B",
                    width=3,
                    arrow=arrow,
                    arrowshape=(10, 12, 5)
                )
                extra = (
                    f"{mode} · {label} · "
                    f"{self.motion_small_diameter.get()}→"
                    f"{self.motion_large_diameter.get()}°"
                )

            else:
                small_deg = max(
                    0.1,
                    self._safe_float(
                        self.motion_annulus_small_diameter,
                        8
                    )
                )
                large_deg = max(
                    small_deg + 0.1,
                    self._safe_float(
                        self.motion_annulus_large_diameter,
                        40
                    )
                )
                thickness_deg = max(
                    0.1,
                    self._safe_float(
                        self.motion_annulus_thickness,
                        2
                    )
                )

                scale = min(
                    3.0,
                    145.0 / large_deg
                )
                small = max(
                    12.0,
                    small_deg * scale
                )
                large = max(
                    small + 8.0,
                    large_deg * scale
                )
                thickness_px = max(
                    2.0,
                    thickness_deg * scale
                )

                # Small annulus.
                canvas.create_oval(
                    cx - small / 2,
                    cy - small / 2,
                    cx + small / 2,
                    cy + small / 2,
                    fill=fg,
                    outline=fg
                )
                inner_small = max(
                    1.0,
                    small - 2.0 * thickness_px
                )
                canvas.create_oval(
                    cx - inner_small / 2,
                    cy - inner_small / 2,
                    cx + inner_small / 2,
                    cy + inner_small / 2,
                    fill=bg,
                    outline=bg
                )

                # Large final/reference annulus.
                canvas.create_oval(
                    cx - large / 2,
                    cy - large / 2,
                    cx + large / 2,
                    cy + large / 2,
                    outline=fg,
                    width=2
                )
                inner_large = max(
                    1.0,
                    large - 2.0 * thickness_px
                )
                canvas.create_oval(
                    cx - inner_large / 2,
                    cy - inner_large / 2,
                    cx + inner_large / 2,
                    cy + inner_large / 2,
                    outline=bg,
                    width=2
                )

                arrow = "last"
                label = "expansión"
                if mode == "Receding Annulus":
                    arrow = "first"
                    label = "contracción"

                canvas.create_line(
                    cx + small / 2 + 7,
                    cy,
                    cx + large / 2 - 4,
                    cy,
                    fill="#F0A44B",
                    width=3,
                    arrow=arrow,
                    arrowshape=(10, 12, 5)
                )

                extra = (
                    f"{mode} · {label} · "
                    f"Ø ext. {self.motion_annulus_small_diameter.get()}→"
                    f"{self.motion_annulus_large_diameter.get()}° · "
                    f"grosor {self.motion_annulus_thickness.get()}°"
                )

            self._set_preview_summary(
                key,
                extra
            )

        elif key == "grating":
            mode = self.gr_mode.get()
            if mode == "Gabor":
                mode = "Static Gabor"

            color_a = self.gr_color_a_hex.get()
            color_b = self.gr_color_b_hex.get()
            angle = self._preview_first_angle(
                self.gr_angles.get()
            )

            if mode in (
                "Static Gabor",
                "Drifting Gabor",
            ):
                bg = self.gr_gabor_bg_hex.get()
            else:
                bg = color_b

            self._preview_monitor(
                canvas,
                bg
            )

            theta = math.radians(angle)
            mx = math.cos(theta)
            my = -math.sin(theta)
            sx, sy = -my, mx
            cx, cy = 165, 96

            if mode == "Drifting":
                for offset in range(
                    -190, 191, 28
                ):
                    ox = mx * offset
                    oy = my * offset
                    canvas.create_line(
                        cx + ox - sx * 150,
                        cy + oy - sy * 150,
                        cx + ox + sx * 150,
                        cy + oy + sy * 150,
                        fill=color_a,
                        width=12
                    )

                canvas.create_line(
                    cx - mx * 85,
                    cy - my * 85,
                    cx + mx * 85,
                    cy + my * 85,
                    fill="#F0A44B",
                    width=3,
                    arrow="last",
                    arrowshape=(10, 12, 5)
                )

                extra = (
                    f"Drifting · 1ª dirección {angle:g}° · "
                    f"{self.gr_sf.get()} c/° · "
                    f"{self.gr_tf.get()} Hz"
                )

            elif mode == "Asymmetric Drift":
                a_rgb = hex_to_rgb255(color_a)
                b_rgb = hex_to_rgb255(color_b)
                contrast = (
                    1.0
                    if bool(self.gr_absolute.get())
                    else max(
                        0.0,
                        min(
                            1.0,
                            self._safe_float(
                                self.gr_contrast,
                                1.0
                            )
                        )
                    )
                )
                midpoint = [
                    0.5 * (a_rgb[i] + b_rgb[i])
                    for i in range(3)
                ]
                a_eff = [
                    midpoint[i]
                    + contrast * (a_rgb[i] - midpoint[i])
                    for i in range(3)
                ]
                b_eff = [
                    midpoint[i]
                    + contrast * (b_rgb[i] - midpoint[i])
                    for i in range(3)
                ]
                polarity = self.gr_asym_polarity.get()
                period_px = 46.0

                for offset in range(-220, 221, 2):
                    u = (float(offset) / period_px) % 1.0
                    weight = 0.5 * (1.0 + math.cos(math.pi * u))
                    if polarity == "B_TO_A":
                        weight = 1.0 - weight
                    rgb = [
                        int(round(
                            b_eff[i]
                            + weight * (a_eff[i] - b_eff[i])
                        ))
                        for i in range(3)
                    ]
                    col = rgb255_to_hex(rgb)
                    ox = mx * offset
                    oy = my * offset
                    canvas.create_line(
                        cx + ox - sx * 155,
                        cy + oy - sy * 155,
                        cx + ox + sx * 155,
                        cy + oy + sy * 155,
                        fill=col,
                        width=3
                    )

                canvas.create_line(
                    cx - mx * 85,
                    cy - my * 85,
                    cx + mx * 85,
                    cy + my * 85,
                    fill="#F0A44B",
                    width=3,
                    arrow="last",
                    arrowshape=(10, 12, 5)
                )
                extra = (
                    f"Asymmetric Drift · 1ª dirección {angle:g}° · "
                    f"{self.gr_sf.get()} c/° · {self.gr_tf.get()} Hz · "
                    f"{polarity}"
                )

            elif mode in (
                "Static",
                "Phase Reversal",
            ):
                for offset in range(
                    -190, 191, 28
                ):
                    ox = mx * offset
                    oy = my * offset
                    canvas.create_line(
                        cx + ox - sx * 150,
                        cy + oy - sy * 150,
                        cx + ox + sx * 150,
                        cy + oy + sy * 150,
                        fill=color_a,
                        width=12
                    )

                if mode == "Static":
                    extra = (
                        f"Static · 1ª orientación {angle:g}° · "
                        f"fase {self.gr_phase.get()} ciclos · "
                        f"{self.gr_sf.get()} c/°"
                    )
                else:
                    canvas.create_text(
                        cx,
                        150,
                        text="fase ↔ fase + 0,5 ciclos",
                        fill="#F0A44B",
                        font=(
                            "Segoe UI Semibold",
                            9
                        )
                    )
                    extra = (
                        f"Phase Reversal · 1ª orientación {angle:g}° · "
                        f"{self.gr_reversal_hz.get()} Hz"
                    )

            else:
                radius = max(
                    28.0,
                    min(
                        72.0,
                        self._safe_float(
                            self.gr_gabor_size,
                            30
                        ) * 2.0
                    )
                )

                canvas.create_oval(
                    cx - radius,
                    cy - radius,
                    cx + radius,
                    cy + radius,
                    outline="#5E6B72",
                    width=1
                )

                for offset in range(
                    -70, 71, 18
                ):
                    if abs(offset) > radius:
                        continue

                    half_chord = math.sqrt(
                        max(
                            0.0,
                            radius * radius
                            - offset * offset
                        )
                    )
                    ox = mx * offset
                    oy = my * offset

                    canvas.create_line(
                        cx + ox - sx * half_chord,
                        cy + oy - sy * half_chord,
                        cx + ox + sx * half_chord,
                        cy + oy + sy * half_chord,
                        fill=color_a,
                        width=8
                    )

                if mode == "Drifting Gabor":
                    canvas.create_line(
                        cx - mx * 46,
                        cy - my * 46,
                        cx + mx * 46,
                        cy + my * 46,
                        fill="#F0A44B",
                        width=3,
                        arrow="last",
                        arrowshape=(10, 12, 5)
                    )
                    extra = (
                        f"Drifting Gabor · 1ª dirección {angle:g}° · "
                        f"{self.gr_tf.get()} Hz · "
                        f"tamaño {self.gr_gabor_size.get()}°"
                    )
                else:
                    extra = (
                        f"Static Gabor · 1ª orientación {angle:g}° · "
                        f"tamaño {self.gr_gabor_size.get()}° · "
                        f"σ {self.gr_gabor_sigma.get()}°"
                    )

            self._set_preview_summary(
                key,
                extra
            )

        elif key == "dense_noise":
            self._preview_monitor(
                canvas,
                "#808080"
            )

            family = self.dn_family.get()
            distribution = self.dn_mode.get()

            if family == "Gaussian White Noise":
                distribution = "Gaussian"

            contrast = max(
                0.0,
                min(
                    1.0,
                    self._safe_float(
                        self.dn_contrast,
                        1.0
                    )
                )
            )

            low_rgb = hex_to_rgb255(
                self.dn_low_hex.get()
            )
            mid_rgb = hex_to_rgb255(
                self.dn_mid_hex.get()
            )
            high_rgb = hex_to_rgb255(
                self.dn_high_hex.get()
            )

            rng = random.Random(
                self._safe_int(
                    self.dn_seed,
                    12345
                )
            )

            def preview_noise_color():
                if distribution == "Binary":
                    return (
                        self.dn_low_hex.get()
                        if rng.randrange(2) == 0
                        else self.dn_high_hex.get()
                    )

                if distribution == "Ternary":
                    return [
                        self.dn_low_hex.get(),
                        self.dn_mid_hex.get(),
                        self.dn_high_hex.get(),
                    ][
                        rng.randrange(3)
                    ]

                # Gaussian: mean = Color medio. With contrast=1,
                # approximately ±3 sigma reaches low/high.
                v = max(
                    -1.0,
                    min(
                        1.0,
                        rng.gauss(
                            0.0,
                            1.0 / 3.0
                        )
                    )
                )
                v *= contrast

                if v < 0:
                    w = -v
                    rgb = [
                        mid_rgb[i]
                        + w
                        * (
                            low_rgb[i]
                            - mid_rgb[i]
                        )
                        for i in range(3)
                    ]
                else:
                    w = v
                    rgb = [
                        mid_rgb[i]
                        + w
                        * (
                            high_rgb[i]
                            - mid_rgb[i]
                        )
                        for i in range(3)
                    ]

                return rgb255_to_hex([
                    int(
                        round(
                            max(
                                0,
                                min(255, x)
                            )
                        )
                    )
                    for x in rgb
                ])

            x0, y0 = 18, 18
            x1, y1 = 312, 175
            cols, rows = 10, 6
            cw = (x1 - x0) / cols
            ch = (y1 - y0) / rows

            for r in range(rows):
                for c in range(cols):
                    color = (
                        preview_noise_color()
                    )
                    canvas.create_rectangle(
                        x0 + c * cw,
                        y0 + r * ch,
                        x0 + (c + 1) * cw,
                        y0 + (r + 1) * ch,
                        fill=color,
                        outline=color
                    )

            canvas.create_rectangle(
                x0, y0, x1, y1,
                outline="#5E6B72",
                width=2
            )

            if family == "Dense White Noise":
                family_label = (
                    f"Dense White Noise · {distribution}"
                )
            elif family == "Gaussian White Noise":
                family_label = (
                    "Gaussian White Noise"
                )
            else:
                family_label = (
                    f"Frozen Noise · {distribution}"
                )

            dense_extra = (
                f"{family_label} · "
                f"{self.dn_update_hz.get()} Hz · "
                f"check {self.dn_check_size.get()}°"
            )

            if family == "Frozen Noise":
                dense_extra += (
                    "\nLa misma secuencia se repite en cada repetición."
                )

            try:
                requested = self._safe_float(
                    self.dn_update_hz
                )
                if (
                    self.current_measured_hz is not None
                    and requested > self.current_measured_hz
                ):
                    dense_extra += (
                        "\n⚠ Limitado por refresco: "
                        f"máx. {self.current_measured_hz:.2f} Hz"
                    )
            except Exception:
                pass

            self._set_preview_summary(
                key,
                dense_extra
            )

        elif key == "rf_mapping":
            mode = self.rf_mode.get()
            bg = self.rf_bg_hex.get()
            stim = self.rf_stim_hex.get()

            if mode == "Sparse Noise":
                bg = "#808080"

            self._preview_monitor(
                canvas, bg
            )

            px = 165 + max(
                -120,
                min(
                    120,
                    self._safe_float(
                        self.rf_x
                    ) * 4.0
                )
            )
            py = 96 - max(
                -60,
                min(
                    60,
                    self._safe_float(
                        self.rf_y
                    ) * 4.0
                )
            )

            if mode == "Spot":
                d = max(
                    12,
                    min(
                        125,
                        self._safe_float(
                            self.rf_spot_diameter,
                            8
                        ) * 5.0
                    )
                )
                canvas.create_oval(
                    px - d / 2, py - d / 2,
                    px + d / 2, py + d / 2,
                    fill=stim,
                    outline="#5E6B72",
                    width=1
                )
                extra = (
                    f"Spot · {self.rf_spot_diameter.get()}° · "
                    f"X {self.rf_x.get()}° / Y {self.rf_y.get()}°"
                )

            elif mode == "Annulus":
                outer = max(
                    30,
                    min(
                        145,
                        self._safe_float(
                            self.rf_annulus_outer,
                            20
                        ) * 4.5
                    )
                )
                inner = max(
                    8,
                    min(
                        outer - 5,
                        self._safe_float(
                            self.rf_annulus_inner,
                            8
                        ) * 4.5
                    )
                )
                canvas.create_oval(
                    px - outer / 2, py - outer / 2,
                    px + outer / 2, py + outer / 2,
                    fill=stim,
                    outline="#5E6B72"
                )
                canvas.create_oval(
                    px - inner / 2, py - inner / 2,
                    px + inner / 2, py + inner / 2,
                    fill=bg,
                    outline=bg
                )
                extra = (
                    f"Annulus · {self.rf_annulus_inner.get()}° → "
                    f"{self.rf_annulus_outer.get()}°"
                )

            elif mode == "Size series":
                try:
                    vals = [
                        float(x.strip())
                        for x in self.rf_size_series.get().split(",")
                        if x.strip()
                    ]
                except Exception:
                    vals = [2, 4, 8, 16]

                vals = vals or [2, 4, 8, 16]
                max_v = max(vals)
                for value in sorted(vals, reverse=True):
                    d = 20 + 105 * value / max_v
                    canvas.create_oval(
                        px - d / 2, py - d / 2,
                        px + d / 2, py + d / 2,
                        outline=stim,
                        width=2
                    )
                extra = (
                    f"Size series · {len(vals)} diámetros"
                )

            elif mode == "Annulus Size series":
                try:
                    vals = [
                        float(x.strip())
                        for x in self.rf_annulus_size_series.get().split(",")
                        if x.strip()
                    ]
                except Exception:
                    vals = [8, 12, 16, 24, 32, 40]

                vals = vals or [8, 12, 16, 24, 32, 40]
                thickness = max(
                    0.1,
                    self._safe_float(
                        self.rf_annulus_series_thickness,
                        2.0
                    )
                )
                max_v = max(vals)

                for value in sorted(vals, reverse=True):
                    outer = 20 + 105 * value / max_v
                    scale = outer / value
                    thick_px = max(
                        2.0,
                        thickness * scale
                    )
                    inner = max(
                        1.0,
                        outer - 2.0 * thick_px
                    )
                    canvas.create_oval(
                        px - outer / 2, py - outer / 2,
                        px + outer / 2, py + outer / 2,
                        outline=stim,
                        width=2
                    )
                    canvas.create_oval(
                        px - inner / 2, py - inner / 2,
                        px + inner / 2, py + inner / 2,
                        outline=bg,
                        width=2
                    )

                extra = (
                    f"Annulus Size series · {len(vals)} diámetros · "
                    f"grosor {self.rf_annulus_series_thickness.get()}°"
                )

            else:
                x0, y0 = 28, 28
                x1, y1 = 302, 163
                step_grid = 34
                for gx in range(x0, x1 + 1, step_grid):
                    canvas.create_line(
                        gx, y0, gx, y1,
                        fill="#9AA6AC",
                        dash=(2, 4)
                    )
                for gy in range(y0, y1 + 1, step_grid):
                    canvas.create_line(
                        x0, gy, x1, gy,
                        fill="#9AA6AC",
                        dash=(2, 4)
                    )

                square = 28
                example_color = (
                    "#FFFFFF"
                    if self.rf_sparse_polarity.get()
                    != "Dark"
                    else "#000000"
                )
                canvas.create_rectangle(
                    151, 69,
                    151 + square,
                    69 + square,
                    fill=example_color,
                    outline="#5E6B72"
                )
                extra = (
                    f"Sparse Noise · {self.rf_sparse_square.get()}° · "
                    f"{self.rf_sparse_polarity.get()} · "
                    f"{self.rf_sparse_trials.get()} presentaciones/rep"
                )

            self._set_preview_summary(
                key,
                extra
            )

        elif key == "chirp_intensity":
            canvas.delete("all")

            mode = self.temporal_mode.get()
            high = self.ch_on_hex.get()
            low = self.ch_off_hex.get()

            if mode == "Chirp + Intensity":
                canvas.create_text(
                    18, 15,
                    anchor="nw",
                    text="A        B             C                    D",
                    fill=self.palette["muted"],
                    font=("Segoe UI Semibold", 9)
                )

                baseline_y = 145
                canvas.create_line(
                    18, baseline_y,
                    312, baseline_y,
                    fill="#8A989F",
                    width=2
                )
                canvas.create_line(
                    20, baseline_y,
                    52, baseline_y,
                    fill="#808080",
                    width=7
                )
                canvas.create_rectangle(
                    58, 58, 82, 145,
                    fill=high,
                    outline=""
                )
                canvas.create_rectangle(
                    82, 122, 105, 145,
                    fill=low,
                    outline=""
                )
                canvas.create_line(
                    106, baseline_y,
                    125, baseline_y,
                    fill="#808080",
                    width=5
                )

                x = 130
                gap = 10.0
                while x < 225:
                    canvas.create_rectangle(
                        x, 72,
                        x + max(
                            2,
                            gap * 0.45
                        ),
                        145,
                        fill=high,
                        outline=""
                    )
                    x += gap
                    gap = max(
                        3.0,
                        gap * 0.90
                    )

                canvas.create_line(
                    226, baseline_y,
                    239, baseline_y,
                    fill="#808080",
                    width=5
                )

                for i in range(6):
                    alpha = (
                        (i + 1) / 6
                    )
                    color = self._blend_hex(
                        low,
                        high,
                        alpha
                    )
                    x0 = (
                        244
                        + i * 11
                    )
                    canvas.create_rectangle(
                        x0,
                        78,
                        x0 + 6,
                        145,
                        fill=color,
                        outline=""
                    )

                canvas.create_text(
                    165,
                    178,
                    text=(
                        "ON/OFF inicial → chirp → "
                        "rampa de intensidad"
                    ),
                    fill=self.palette["text"],
                    font=("Segoe UI Semibold", 9)
                )

                extra = (
                    f"Chirp {self.ch_fmin.get()}→"
                    f"{self.ch_fmax.get()} Hz · "
                    f"Intensity {self.ch_ifreq.get()} Hz"
                )

            elif mode == "Sinusoidal Flicker":
                self._preview_monitor(
                    canvas,
                    "#808080"
                )
                pts = []
                for i in range(0, 281, 4):
                    x = 25 + i
                    y = (
                        96
                        - 48
                        * math.sin(
                            2
                            * math.pi
                            * i
                            / 80.0
                        )
                    )
                    pts.extend(
                        [x, y]
                    )
                canvas.create_line(
                    *pts,
                    fill="#F0A44B",
                    width=3,
                    smooth=True
                )
                canvas.create_text(
                    165,
                    174,
                    text="modulación sinusoidal continua",
                    fill=self.palette["text"],
                    font=("Segoe UI Semibold", 9)
                )
                extra = (
                    f"Sinusoidal · {self.ch_temp_frequency.get()} Hz · "
                    f"{self.ch_temp_contrast.get()} %"
                )

            elif mode == "Gaussian Flicker":
                self._preview_monitor(
                    canvas,
                    "#808080"
                )
                rng = random.Random(
                    self._safe_int(
                        self.ch_temp_seed,
                        54321
                    )
                )
                pts = []
                x0 = 28
                for i in range(42):
                    x = (
                        x0
                        + i * 6.5
                    )
                    y = (
                        96
                        - max(
                            -52,
                            min(
                                52,
                                rng.gauss(
                                    0,
                                    20
                                )
                            )
                        )
                    )
                    pts.extend(
                        [x, y]
                    )
                canvas.create_line(
                    *pts,
                    fill="#F0A44B",
                    width=2
                )
                canvas.create_text(
                    165,
                    174,
                    text="niveles temporales gaussianos independientes",
                    fill=self.palette["text"],
                    font=("Segoe UI Semibold", 9)
                )
                extra = (
                    f"Gaussian · {self.ch_temp_update_hz.get()} Hz nominal · "
                    f"{self.ch_temp_contrast.get()} %"
                )

            else:
                self._preview_monitor(
                    canvas,
                    "#808080"
                )
                try:
                    levels = [
                        float(
                            v.strip()
                        )
                        for v
                        in self.ch_contrast_levels.get().replace(
                            ";", ","
                        ).split(",")
                        if v.strip()
                    ]
                except Exception:
                    levels = [
                        10,
                        25,
                        50,
                        100
                    ]

                levels = (
                    levels[:6]
                    if levels
                    else [
                        10,
                        25,
                        50,
                        100
                    ]
                )

                width = (
                    250
                    / max(
                        1,
                        len(levels)
                    )
                )
                for idx, pct in enumerate(
                    levels
                ):
                    amp = (
                        48
                        * max(
                            0.0,
                            min(
                                100.0,
                                pct
                            )
                        )
                        / 100.0
                    )
                    cx = (
                        40
                        + idx * width
                    )
                    canvas.create_line(
                        cx,
                        96 - amp,
                        cx,
                        96 + amp,
                        fill="#F0A44B",
                        width=max(
                            3,
                            int(
                                width * 0.25
                            )
                        )
                    )
                    canvas.create_text(
                        cx,
                        160,
                        text=f"{pct:g}%",
                        fill=self.palette["text"],
                        font=("Segoe UI", 8)
                    )

                canvas.create_text(
                    165,
                    184,
                    text="amplitud de flicker por contraste",
                    fill=self.palette["text"],
                    font=("Segoe UI Semibold", 9)
                )
                extra = (
                    f"Contrast Series · {self.ch_temp_frequency.get()} Hz · "
                    f"{self.ch_contrast_levels.get()} %"
                )

            self._set_preview_summary(
                key,
                extra
            )


    def _stop_preview(
        self, redraw=True
    ):
        if self.preview_after_id is not None:
            try:
                self.after_cancel(
                    self.preview_after_id
                )
            except Exception:
                pass
        key = self.preview_running_key
        self.preview_after_id = None
        self.preview_running_key = None
        self.preview_step = 0

        if redraw and key:
            self._draw_preview_static(key)

    def _start_preview(self, key):
        self._stop_preview(
            redraw=False
        )
        self._stop_easter_egg(
            redraw=False
        )
        self.preview_running_key = key
        self.preview_step = 0
        self._animate_preview()

    def _animate_preview(self):
        key = self.preview_running_key
        if not key:
            return

        canvas = self.preview_canvases.get(
            key
        )
        if canvas is None:
            self._stop_preview(
                redraw=False
            )
            return

        step = self.preview_step
        delay = 90
        done = False

        if key == "on_off":
            seq = [
                ("#808080", "gris"),
                (
                    self.on_on_hex.get(),
                    "ON"
                ),
                ("#808080", "gris"),
                (
                    self.on_off_hex.get(),
                    "OFF"
                ),
                ("#808080", "gris"),
            ]

            idx = min(
                len(seq) - 1,
                step // 5
            )
            color, label = seq[idx]
            self._preview_monitor(
                canvas, color
            )
            canvas.create_text(
                165, 96,
                text=label,
                fill=best_text_color(color),
                font=("Segoe UI Semibold", 21)
            )
            delay = 110
            done = step >= 24

        elif key == "motion":
            mode = self.motion_mode.get()
            bg = self.motion_background_hex.get()
            fg = self.motion_stimulus_hex.get()

            self._preview_monitor(
                canvas, bg
            )

            if mode in (
                "Moving Bar",
                "Moving Spot",
            ):
                angle = self._preview_first_angle(
                    self.motion_dirs.get()
                )
                theta = math.radians(angle)
                dx = math.cos(theta)
                dy = -math.sin(theta)
                u = min(
                    1.0,
                    step / 34.0
                )
                cx = 165 + (
                    (u - 0.5)
                    * 220
                    * dx
                )
                cy = 96 + (
                    (u - 0.5)
                    * 120
                    * dy
                )

                if mode == "Moving Bar":
                    ax, ay = -dy, dx
                    half_l = 68
                    half_w = 9
                    points = [
                        (cx + ax * half_l + dx * half_w,
                         cy + ay * half_l + dy * half_w),
                        (cx - ax * half_l + dx * half_w,
                         cy - ay * half_l + dy * half_w),
                        (cx - ax * half_l - dx * half_w,
                         cy - ay * half_l - dy * half_w),
                        (cx + ax * half_l - dx * half_w,
                         cy + ay * half_l - dy * half_w),
                    ]
                    flat = [
                        value
                        for point in points
                        for value in point
                    ]
                    canvas.create_polygon(
                        *flat,
                        fill=fg,
                        outline="#5E6B72"
                    )
                else:
                    d = max(
                        14,
                        min(
                            70,
                            self._safe_float(
                                self.motion_spot_diameter,
                                8
                            ) * 3.5
                        )
                    )
                    canvas.create_oval(
                        cx - d / 2,
                        cy - d / 2,
                        cx + d / 2,
                        cy + d / 2,
                        fill=fg,
                        outline="#5E6B72"
                    )

                delay = 45
                done = step >= 34

            elif mode in (
                "Looming",
                "Receding",
            ):
                small = max(
                    10.0,
                    min(
                        45.0,
                        self._safe_float(
                            self.motion_small_diameter,
                            2
                        ) * 3.0
                    )
                )
                large = max(
                    small + 10.0,
                    min(
                        145.0,
                        self._safe_float(
                            self.motion_large_diameter,
                            40
                        ) * 3.0
                    )
                )

                u = min(
                    1.0,
                    step / 34.0
                )
                if mode == "Looming":
                    d = (
                        small
                        + (large - small) * u
                    )
                else:
                    d = (
                        large
                        - (large - small) * u
                    )

                cx, cy = 165, 96
                canvas.create_oval(
                    cx - d / 2,
                    cy - d / 2,
                    cx + d / 2,
                    cy + d / 2,
                    fill=fg,
                    outline="#5E6B72"
                )
                delay = 45
                done = step >= 34

            else:
                small_deg = max(
                    0.1,
                    self._safe_float(
                        self.motion_annulus_small_diameter,
                        8
                    )
                )
                large_deg = max(
                    small_deg + 0.1,
                    self._safe_float(
                        self.motion_annulus_large_diameter,
                        40
                    )
                )
                thickness_deg = max(
                    0.1,
                    self._safe_float(
                        self.motion_annulus_thickness,
                        2
                    )
                )

                scale = min(
                    3.0,
                    145.0 / large_deg
                )
                small = max(
                    12.0,
                    small_deg * scale
                )
                large = max(
                    small + 10.0,
                    large_deg * scale
                )
                thickness_px = max(
                    2.0,
                    thickness_deg * scale
                )

                u = min(
                    1.0,
                    step / 34.0
                )

                if mode == "Expanding Annulus":
                    d = (
                        small
                        + (large - small) * u
                    )
                else:
                    d = (
                        large
                        - (large - small) * u
                    )

                inner_d = max(
                    1.0,
                    d - 2.0 * thickness_px
                )

                cx, cy = 165, 96
                canvas.create_oval(
                    cx - d / 2,
                    cy - d / 2,
                    cx + d / 2,
                    cy + d / 2,
                    fill=fg,
                    outline=fg
                )
                canvas.create_oval(
                    cx - inner_d / 2,
                    cy - inner_d / 2,
                    cx + inner_d / 2,
                    cy + inner_d / 2,
                    fill=bg,
                    outline=bg
                )

                delay = 45
                done = step >= 34

        elif key == "grating":
            mode = self.gr_mode.get()
            if mode == "Gabor":
                mode = "Static Gabor"

            a = self.gr_color_a_hex.get()
            b = self.gr_color_b_hex.get()
            angle = self._preview_first_angle(
                self.gr_angles.get()
            )

            if mode in (
                "Static Gabor",
                "Drifting Gabor",
            ):
                bg = self.gr_gabor_bg_hex.get()
            else:
                bg = b

            self._preview_monitor(
                canvas,
                bg
            )

            theta = math.radians(angle)
            mx = math.cos(theta)
            my = -math.sin(theta)
            sx, sy = -my, mx
            cx, cy = 165, 96

            if mode == "Drifting":
                shift = (
                    (step * 6) % 28
                )

                for offset in range(
                    -210, 211, 28
                ):
                    off = (
                        offset
                        + shift
                    )
                    ox = mx * off
                    oy = my * off
                    canvas.create_line(
                        cx + ox - sx * 150,
                        cy + oy - sy * 150,
                        cx + ox + sx * 150,
                        cy + oy + sy * 150,
                        fill=a,
                        width=12
                    )

                delay = 55
                done = step >= 42

            elif mode == "Asymmetric Drift":
                a_rgb = hex_to_rgb255(a)
                b_rgb = hex_to_rgb255(b)
                contrast = (
                    1.0
                    if bool(self.gr_absolute.get())
                    else max(
                        0.0,
                        min(
                            1.0,
                            self._safe_float(
                                self.gr_contrast,
                                1.0
                            )
                        )
                    )
                )
                midpoint = [
                    0.5 * (a_rgb[i] + b_rgb[i])
                    for i in range(3)
                ]
                a_eff = [
                    midpoint[i]
                    + contrast * (a_rgb[i] - midpoint[i])
                    for i in range(3)
                ]
                b_eff = [
                    midpoint[i]
                    + contrast * (b_rgb[i] - midpoint[i])
                    for i in range(3)
                ]
                polarity = self.gr_asym_polarity.get()
                period_px = 46.0
                shift = (step * 5.0) % period_px

                for offset in range(-220, 221, 2):
                    off = float(offset) + shift
                    u = (off / period_px) % 1.0
                    weight = 0.5 * (1.0 + math.cos(math.pi * u))
                    if polarity == "B_TO_A":
                        weight = 1.0 - weight
                    rgb = [
                        int(round(
                            b_eff[i]
                            + weight * (a_eff[i] - b_eff[i])
                        ))
                        for i in range(3)
                    ]
                    col = rgb255_to_hex(rgb)
                    ox = mx * off
                    oy = my * off
                    canvas.create_line(
                        cx + ox - sx * 155,
                        cy + oy - sy * 155,
                        cx + ox + sx * 155,
                        cy + oy + sy * 155,
                        fill=col,
                        width=3
                    )

                delay = 55
                done = step >= 42

            elif mode == "Phase Reversal":
                shift = (
                    14
                    if (step // 5) % 2
                    else 0
                )

                for offset in range(
                    -210, 211, 28
                ):
                    off = (
                        offset
                        + shift
                    )
                    ox = mx * off
                    oy = my * off
                    canvas.create_line(
                        cx + ox - sx * 150,
                        cy + oy - sy * 150,
                        cx + ox + sx * 150,
                        cy + oy + sy * 150,
                        fill=a,
                        width=12
                    )

                delay = 75
                done = step >= 34

            elif mode == "Static":
                for offset in range(
                    -210, 211, 28
                ):
                    ox = mx * offset
                    oy = my * offset
                    canvas.create_line(
                        cx + ox - sx * 150,
                        cy + oy - sy * 150,
                        cx + ox + sx * 150,
                        cy + oy + sy * 150,
                        fill=a,
                        width=12
                    )

                delay = 90
                done = step >= 24

            else:
                radius = max(
                    28.0,
                    min(
                        72.0,
                        self._safe_float(
                            self.gr_gabor_size,
                            30
                        ) * 2.0
                    )
                )

                canvas.create_oval(
                    cx - radius,
                    cy - radius,
                    cx + radius,
                    cy + radius,
                    outline="#5E6B72",
                    width=1
                )

                shift = (
                    (step * 5) % 18
                    if mode == "Drifting Gabor"
                    else 0
                )

                for offset in range(
                    -70, 71, 18
                ):
                    off = offset + shift
                    if abs(off) > radius:
                        continue
                    half_chord = math.sqrt(
                        max(
                            0.0,
                            radius * radius
                            - off * off
                        )
                    )
                    ox = mx * off
                    oy = my * off
                    canvas.create_line(
                        cx + ox - sx * half_chord,
                        cy + oy - sy * half_chord,
                        cx + ox + sx * half_chord,
                        cy + oy + sy * half_chord,
                        fill=a,
                        width=8
                    )

                delay = (
                    55
                    if mode == "Drifting Gabor"
                    else 90
                )
                done = (
                    step >= 42
                    if mode == "Drifting Gabor"
                    else step >= 24
                )

        elif key == "dense_noise":
            self._preview_monitor(
                canvas,
                "#808080"
            )

            family = self.dn_family.get()
            distribution = self.dn_mode.get()

            if family == "Gaussian White Noise":
                distribution = "Gaussian"

            contrast = max(
                0.0,
                min(
                    1.0,
                    self._safe_float(
                        self.dn_contrast,
                        1.0
                    )
                )
            )

            low_rgb = hex_to_rgb255(
                self.dn_low_hex.get()
            )
            mid_rgb = hex_to_rgb255(
                self.dn_mid_hex.get()
            )
            high_rgb = hex_to_rgb255(
                self.dn_high_hex.get()
            )

            # The seed + temporal step depicts one deterministic sequence.
            # Frozen Noise means that this same sequence is replayed across
            # repetitions, not that the image remains spatially frozen.
            rng = random.Random(
                self._safe_int(
                    self.dn_seed,
                    12345
                )
                + step
            )

            def preview_noise_color():
                if distribution == "Binary":
                    return (
                        self.dn_low_hex.get()
                        if rng.randrange(2) == 0
                        else self.dn_high_hex.get()
                    )

                if distribution == "Ternary":
                    return [
                        self.dn_low_hex.get(),
                        self.dn_mid_hex.get(),
                        self.dn_high_hex.get(),
                    ][
                        rng.randrange(3)
                    ]

                v = max(
                    -1.0,
                    min(
                        1.0,
                        rng.gauss(
                            0.0,
                            1.0 / 3.0
                        )
                    )
                )
                v *= contrast

                if v < 0:
                    w = -v
                    rgb = [
                        mid_rgb[i]
                        + w
                        * (
                            low_rgb[i]
                            - mid_rgb[i]
                        )
                        for i in range(3)
                    ]
                else:
                    w = v
                    rgb = [
                        mid_rgb[i]
                        + w
                        * (
                            high_rgb[i]
                            - mid_rgb[i]
                        )
                        for i in range(3)
                    ]

                return rgb255_to_hex([
                    int(
                        round(
                            max(
                                0,
                                min(255, x)
                            )
                        )
                    )
                    for x in rgb
                ])

            x0, y0 = 18, 18
            x1, y1 = 312, 175
            cols, rows = 10, 6
            cw = (x1 - x0) / cols
            ch = (y1 - y0) / rows

            for r in range(rows):
                for c in range(cols):
                    color = (
                        preview_noise_color()
                    )
                    canvas.create_rectangle(
                        x0 + c * cw,
                        y0 + r * ch,
                        x0 + (c + 1) * cw,
                        y0 + (r + 1) * ch,
                        fill=color,
                        outline=color
                    )

            canvas.create_rectangle(
                x0, y0, x1, y1,
                outline="#5E6B72",
                width=2
            )
            delay = 110
            done = step >= 24

        elif key == "rf_mapping":
            mode = self.rf_mode.get()
            bg = self.rf_bg_hex.get()
            stim = self.rf_stim_hex.get()

            if mode == "Sparse Noise":
                bg = "#808080"

            self._preview_monitor(
                canvas, bg
            )

            px = 165
            py = 96

            if mode == "Spot":
                if (step // 5) % 2 == 0:
                    d = max(
                        16,
                        min(
                            125,
                            self._safe_float(
                                self.rf_spot_diameter,
                                8
                            ) * 5.0
                        )
                    )
                    canvas.create_oval(
                        px - d / 2, py - d / 2,
                        px + d / 2, py + d / 2,
                        fill=stim,
                        outline="#5E6B72"
                    )
                delay = 90
                done = step >= 24

            elif mode == "Annulus":
                outer = max(
                    35,
                    min(
                        145,
                        self._safe_float(
                            self.rf_annulus_outer,
                            20
                        ) * 4.5
                    )
                )
                inner = max(
                    8,
                    min(
                        outer - 5,
                        self._safe_float(
                            self.rf_annulus_inner,
                            8
                        ) * 4.5
                    )
                )
                if (step // 5) % 2 == 0:
                    canvas.create_oval(
                        px - outer / 2, py - outer / 2,
                        px + outer / 2, py + outer / 2,
                        fill=stim,
                        outline="#5E6B72"
                    )
                    canvas.create_oval(
                        px - inner / 2, py - inner / 2,
                        px + inner / 2, py + inner / 2,
                        fill=bg,
                        outline=bg
                    )
                delay = 90
                done = step >= 24

            elif mode == "Size series":
                try:
                    vals = [
                        float(x.strip())
                        for x in self.rf_size_series.get().split(",")
                        if x.strip()
                    ]
                except Exception:
                    vals = [2, 4, 8, 16]
                vals = vals or [2, 4, 8, 16]
                index = min(
                    len(vals) - 1,
                    (step // 5) % len(vals)
                )
                d = 20 + (
                    105
                    * vals[index]
                    / max(vals)
                )
                canvas.create_oval(
                    px - d / 2, py - d / 2,
                    px + d / 2, py + d / 2,
                    fill=stim,
                    outline="#5E6B72"
                )
                delay = 100
                done = step >= max(
                    24,
                    len(vals) * 7
                )

            elif mode == "Annulus Size series":
                try:
                    vals = [
                        float(x.strip())
                        for x in self.rf_annulus_size_series.get().split(",")
                        if x.strip()
                    ]
                except Exception:
                    vals = [8, 12, 16, 24, 32, 40]

                vals = vals or [8, 12, 16, 24, 32, 40]
                thickness = max(
                    0.1,
                    self._safe_float(
                        self.rf_annulus_series_thickness,
                        2.0
                    )
                )
                index = min(
                    len(vals) - 1,
                    (step // 5) % len(vals)
                )
                outer = (
                    20
                    + 105
                    * vals[index]
                    / max(vals)
                )
                scale = outer / vals[index]
                thick_px = max(
                    2.0,
                    thickness * scale
                )
                inner = max(
                    1.0,
                    outer - 2.0 * thick_px
                )

                if (step // 5) % 2 == 0:
                    canvas.create_oval(
                        px - outer / 2, py - outer / 2,
                        px + outer / 2, py + outer / 2,
                        fill=stim,
                        outline="#5E6B72"
                    )
                    canvas.create_oval(
                        px - inner / 2, py - inner / 2,
                        px + inner / 2, py + inner / 2,
                        fill=bg,
                        outline=bg
                    )

                delay = 100
                done = step >= max(
                    24,
                    len(vals) * 7
                )

            else:
                rng = random.Random(
                    self._safe_int(
                        self.rf_sparse_seed,
                        24680
                    ) + step // 2
                )
                gx = rng.randrange(1, 8)
                gy = rng.randrange(1, 4)
                x = 34 + gx * 31
                y = 30 + gy * 31
                polarity = (
                    self.rf_sparse_polarity.get()
                )
                if polarity == "Dark":
                    c = "#000000"
                elif polarity == "Bright":
                    c = "#FFFFFF"
                else:
                    c = (
                        "#FFFFFF"
                        if rng.random() >= 0.5
                        else "#000000"
                    )
                canvas.create_rectangle(
                    x - 14, y - 14,
                    x + 14, y + 14,
                    fill=c,
                    outline="#5E6B72"
                )
                delay = 95
                done = step >= 30

        elif key == "chirp_intensity":
            mode = self.temporal_mode.get()
            high = self.ch_on_hex.get()
            low = self.ch_off_hex.get()

            if mode == "Chirp + Intensity":
                if step < 8:
                    color = "#808080"
                    label = "A · adaptación basal"

                elif step < 14:
                    color = high
                    label = "B · ON"

                elif step < 20:
                    color = low
                    label = "B · OFF"

                elif step < 25:
                    color = "#808080"
                    label = "pausa"

                elif step < 50:
                    local = (
                        step - 25
                    )
                    period = max(
                        2,
                        round(
                            8
                            - 6
                            * (
                                local
                                / 24
                            )
                        )
                    )
                    color = (
                        high
                        if (
                            (
                                local
                                // max(
                                    1,
                                    period
                                    // 2
                                )
                            )
                            % 2
                            == 0
                        )
                        else low
                    )
                    label = (
                        "C · frequency chirp"
                    )

                elif step < 55:
                    color = "#808080"
                    label = "pausa"

                else:
                    local = (
                        step - 55
                    )
                    cycle = (
                        local // 4
                    )
                    is_on = (
                        local % 4
                        < 2
                    )
                    alpha = min(
                        1.0,
                        0.15
                        + cycle
                        * 0.13
                    )
                    color = (
                        self._blend_hex(
                            low,
                            high,
                            alpha
                        )
                        if is_on
                        else low
                    )
                    label = (
                        "D · intensity ramp"
                    )

                delay = 80
                done = (
                    step >= 83
                )

            elif mode == "Sinusoidal Flicker":
                contrast = max(
                    0.0,
                    min(
                        1.0,
                        self._safe_float(
                            self.ch_temp_contrast,
                            100
                        )
                        / 100.0
                    )
                )
                phase = (
                    2
                    * math.pi
                    * step
                    / 18.0
                )
                alpha = (
                    0.5
                    + 0.5
                    * contrast
                    * math.sin(
                        phase
                    )
                )
                color = self._blend_hex(
                    low,
                    high,
                    alpha
                )
                label = (
                    "Sinusoidal Flicker"
                )
                delay = 55
                done = (
                    step >= 54
                )

            elif mode == "Gaussian Flicker":
                rng = random.Random(
                    self._safe_int(
                        self.ch_temp_seed,
                        54321
                    )
                    + step
                )
                contrast = max(
                    0.0,
                    min(
                        1.0,
                        self._safe_float(
                            self.ch_temp_contrast,
                            100
                        )
                        / 100.0
                    )
                )
                z = max(
                    -1.0,
                    min(
                        1.0,
                        rng.gauss(
                            0.0,
                            1.0 / 3.0
                        )
                    )
                )
                alpha = (
                    0.5
                    + 0.5
                    * contrast
                    * z
                )
                color = self._blend_hex(
                    low,
                    high,
                    alpha
                )
                label = (
                    "Gaussian Flicker"
                )
                delay = 75
                done = (
                    step >= 42
                )

            else:
                try:
                    levels = [
                        float(
                            v.strip()
                        )
                        for v
                        in self.ch_contrast_levels.get().replace(
                            ";", ","
                        ).split(",")
                        if v.strip()
                    ]
                except Exception:
                    levels = [
                        10,
                        25,
                        50,
                        100
                    ]
                if not levels:
                    levels = [
                        10,
                        25,
                        50,
                        100
                    ]

                idx = (
                    step // 12
                ) % len(
                    levels
                )
                contrast = max(
                    0.0,
                    min(
                        1.0,
                        levels[idx]
                        / 100.0
                    )
                )
                local = (
                    step % 12
                )
                alpha = (
                    0.5
                    + 0.5
                    * contrast
                    * math.sin(
                        2
                        * math.pi
                        * local
                        / 12.0
                    )
                )
                color = self._blend_hex(
                    low,
                    high,
                    alpha
                )
                label = (
                    f"Contrast {levels[idx]:g} %"
                )
                delay = 60
                done = (
                    step
                    >= max(
                        36,
                        len(
                            levels
                        )
                        * 12
                    )
                )

            self._preview_monitor(
                canvas,
                color
            )
            canvas.create_text(
                165,
                96,
                text=label,
                fill=best_text_color(
                    color
                ),
                font=(
                    "Segoe UI Semibold",
                    14
                )
            )


        if done:
            self.preview_running_key = None
            self.preview_after_id = None
            self.preview_step = 0
            self.after(
                250,
                lambda k=key:
                    self._draw_preview_static(k)
            )
            return

        self.preview_step += 1
        self.preview_after_id = self.after(
            delay,
            self._animate_preview
        )

    def update_contrast_state(self):
        if not hasattr(
            self, "gr_contrast_entry"
        ):
            return

        self.gr_contrast_entry.configure(
            state=(
                "disabled"
                if self.gr_absolute.get()
                else "normal"
            )
        )

    def _num(self, var):
        return float(
            var.get().replace(",", ".")
        )

    def refresh_screens(self):
        try:
            self.screens = detect_screens()
            self.combo["values"] = [
                s["label"]
                for s in self.screens
            ]

            if not self.screens:
                self.status.set(
                    "No se detectaron pantallas."
                )
                return

            idx = int(
                self.saved.get(
                    "screen_index",
                    1 if len(self.screens) > 1 else 0
                )
            )
            if idx >= len(self.screens):
                idx = 0

            self.combo.current(idx)

        except Exception as exc:
            messagebox.showerror(
                "Pantallas", str(exc)
            )

    def _build_system_ttl(self):
        body = ttk.Frame(self.tab_system, style="TabBody.TFrame")
        body.pack(fill="both", expand=True, padx=12, pady=12)
        body.columnconfigure(0, weight=1)
        body.columnconfigure(1, weight=1)
        body.rowconfigure(1, weight=1)

        ttl_card = ttk.LabelFrame(body, text="Hardware de triggers", style="Card.TLabelframe")
        ttl_card.grid(row=0, column=0, columnspan=2, sticky="ew", pady=(0, 10))
        ttl_card.columnconfigure(1, weight=1)

        ttk.Label(ttl_card, text="Modo:").grid(row=0, column=0, sticky="w", padx=(12, 6), pady=(10, 5))
        self.ttl_mode_combo = ttk.Combobox(
            ttl_card, textvariable=self.ttl_mode, state="readonly",
            values=["Simulación", "MaxOne Legacy FTDI", "Trigger Box TTL genérica", "Desactivado"],
            width=28
        )
        self.ttl_mode_combo.grid(row=0, column=1, sticky="w", pady=(10, 5))
        self.ttl_mode_combo.bind("<<ComboboxSelected>>", self.update_ttl_ui)

        ttk.Label(ttl_card, text="Puerto:").grid(row=0, column=2, sticky="w", padx=(18, 6), pady=(10, 5))
        self.ttl_port_entry = ttk.Entry(ttl_card, textvariable=self.ttl_port, width=12)
        self.ttl_port_entry.grid(row=0, column=3, sticky="w", pady=(10, 5))
        self.ttl_port_entry.bind("<KeyRelease>", self.update_ttl_ui)

        ttk.Label(ttl_card, textvariable=self.ttl_status_text, style="Status.TLabel").grid(
            row=1, column=0, columnspan=3, sticky="w", padx=12, pady=(3, 8)
        )
        self.ttl_test_btn = ttk.Button(
            ttl_card, text="Probar triggers", command=self.test_ttl_simulation,
            style="Secondary.TButton"
        )
        self.ttl_test_btn.grid(row=1, column=3, sticky="e", padx=12, pady=(3, 8))

        ttk.Label(
            ttl_card,
            text=("Genérico: UNIVERSAL=FRAME + EVENT ~3 ms después; AUX EVENT muestra estados y marca cada pausa con un pulso corto.  "
                  "MaxOne Legacy conserva exactamente la ruta v1.17."),
            style="Muted.TLabel", wraplength=900
        ).grid(row=2, column=0, columnspan=4, sticky="w", padx=12, pady=(0, 8))

        ttk.Button(
            ttl_card, text="Guardar esta configuración como predeterminada",
            command=self._save_ttl_default, style="Accent.TButton"
        ).grid(row=3, column=0, columnspan=4, sticky="w", padx=12, pady=(0, 10))

        display_card = ttk.LabelFrame(body, text="Pantalla segura y fondo de adaptación", style="Card.TLabelframe")
        display_card.grid(row=1, column=0, sticky="nsew", padx=(0, 5))

        ttk.Label(
            display_card,
            text="Espera segura: NEGRO",
            style="Status.TLabel"
        ).pack(anchor="w", padx=12, pady=(10, 3))
        ttk.Label(
            display_card,
            text=("La pantalla experimental permanece negra mientras no se ejecuta un estímulo. "
                  "El fondo de adaptación se usa en baselines, intervalos y pausas experimentales."),
            style="Muted.TLabel", wraplength=470
        ).pack(anchor="w", padx=12, pady=(0, 10))

        adapt_row = ttk.Frame(display_card, style="Card.TFrame")
        adapt_row.pack(fill="x", padx=12, pady=(0, 10))
        self._color_selector(adapt_row, "Fondo de adaptación", self.adaptation_hex, 0)
        ttk.Button(
            adapt_row, text="Actualizar vista / guardar con «Guardar»",
            command=self.refresh_active_preview, style="Secondary.TButton"
        ).grid(row=1, column=0, columnspan=3, sticky="w", pady=(5, 0))

        red_card = ttk.LabelFrame(body, text="Modo rojo / sala oscura", style="Card.TLabelframe")
        red_card.grid(row=1, column=1, sticky="nsew", padx=(5, 0))
        ttk.Label(
            red_card,
            text=("Modo de sala oscura de la interfaz: convierte únicamente la ventana del operador "
                  "a una paleta roja tenue y muy oscura. No modifica Windows, la gamma del monitor "
                  "ni la pantalla experimental, que permanece negra en espera."),
            style="Muted.TLabel", wraplength=470
        ).pack(anchor="w", padx=12, pady=(10, 8))

        ttk.Label(
            red_card,
            text=("La paleta se mantiene estable mientras se presenta un estímulo. "
                  "Pulsa 🔴 MODO ROJO en la cabecera para activarla o desactivarla."),
            style="Muted.TLabel", wraplength=470
        ).pack(anchor="w", padx=12, pady=(4, 6))
        ttk.Label(red_card, textvariable=self.red_mode_status, style="Status.TLabel", wraplength=470).pack(
            anchor="w", padx=12, pady=(3, 8)
        )
        self.update_ttl_ui()

    def _save_ttl_default(self):
        try:
            saved = load_saved()
            ttl = dict(saved.get("ttl", {}))
            ttl.update({
                "mode": self.ttl_mode.get(),
                "pin": int(self.ttl_pin.get()),
                "port": self.ttl_port.get().strip() or "auto",
                "default_mode": self.ttl_mode.get(),
                "default_pin": int(self.ttl_pin.get()),
                "default_port": self.ttl_port.get().strip() or "auto",
            })
            saved["ttl"] = ttl
            write_json_atomic(CONFIG_FILE, saved)
            self.saved = saved
            self.status.set(f"TTL predeterminado: {self.ttl_mode.get()}.")
            self.update_ttl_ui()
        except Exception as exc:
            messagebox.showerror("TTL predeterminado", str(exc), parent=self)

    def _red_rgb(self):
        try:
            pct = float(self.red_mode_intensity.get().replace(",", "."))
        except Exception:
            pct = 10.0
        pct = max(1.0, min(100.0, pct))
        self.red_mode_intensity.set(f"{pct:g}")
        return [int(round(255.0 * pct / 100.0)), 0, 0]

    def _current_idle_rgb(self):
        # v1.22: dark-room mode never affects the experimental display.
        # Its safe idle state is always black.
        return [0, 0, 0]
    def _set_operator_titlebar_red(self, active):
        """Best-effort dark native Windows title bar for Modo rojo.

        Only the non-client title bar of this operator window is requested to
        change. No desktop colour transform, gamma/LUT change or experimental
        monitor modification is performed. Unsupported Windows versions/drivers
        simply keep their normal native title bar.
        """
        if sys.platform != "win32":
            return False
        try:
            import ctypes
            from ctypes import wintypes

            # Tk uses a small client HWND inside the real top-level wrapper on
            # Windows. DWM attributes must be applied to the outer/root HWND.
            self.update_idletasks()
            user32 = ctypes.windll.user32
            dwm = ctypes.windll.dwmapi

            client_hwnd = wintypes.HWND(int(self.winfo_id()))
            try:
                user32.GetAncestor.argtypes = [wintypes.HWND, wintypes.UINT]
                user32.GetAncestor.restype = wintypes.HWND
                hwnd = user32.GetAncestor(client_hwnd, 2)  # GA_ROOT
            except Exception:
                hwnd = None
            if not hwnd:
                try:
                    hwnd = user32.GetParent(client_hwnd)
                except Exception:
                    hwnd = None
            if not hwnd:
                hwnd = client_hwnd

            changed = False

            # Windows 10/11: request an immersive dark native caption.
            flag = ctypes.c_int(1 if active else 0)
            for attr in (20, 19):  # DWMWA_USE_IMMERSIVE_DARK_MODE variants
                try:
                    hr = dwm.DwmSetWindowAttribute(
                        hwnd, attr, ctypes.byref(flag), ctypes.sizeof(flag)
                    )
                    if hr == 0:
                        changed = True
                        break
                except Exception:
                    pass

            # Windows 11: if supported, make the caption itself very dark red
            # and its text a dim red. Older versions ignore these attributes.
            if active:
                caption = ctypes.c_uint32(0x000012)    # COLORREF: RGB(18,0,0)
                title_text = ctypes.c_uint32(0x4A4AC8) # COLORREF: RGB(200,74,74)
            else:
                caption = ctypes.c_uint32(0xFFFFFFFF)  # DWMWA_COLOR_DEFAULT
                title_text = ctypes.c_uint32(0xFFFFFFFF)

            for attr, value in ((35, caption), (36, title_text)):
                try:
                    hr = dwm.DwmSetWindowAttribute(
                        hwnd, attr, ctypes.byref(value), ctypes.sizeof(value)
                    )
                    if hr == 0:
                        changed = True
                except Exception:
                    pass

            # Ask Windows to repaint only the native frame. This does not alter
            # geometry or activation and is harmless if DWM ignored the request.
            try:
                SWP_NOSIZE = 0x0001
                SWP_NOMOVE = 0x0002
                SWP_NOZORDER = 0x0004
                SWP_NOACTIVATE = 0x0010
                SWP_FRAMECHANGED = 0x0020
                user32.SetWindowPos(
                    hwnd, None, 0, 0, 0, 0,
                    SWP_NOSIZE | SWP_NOMOVE | SWP_NOZORDER |
                    SWP_NOACTIVATE | SWP_FRAMECHANGED,
                )
            except Exception:
                pass

            return changed
        except Exception:
            return False

    def _recolor_native_widgets(self, widget, active, panel, text, muted, border, accent):
        """Recolour every native Tk child so dark-room mode contains no bright islands.

        ttk widgets are covered by the style table. This pass handles classic Tk
        widgets and scrollable canvases, which otherwise may keep an OS/default
        white background.
        """
        try:
            preview_set = set(getattr(self, "preview_canvases", {}).values())
        except Exception:
            preview_set = set()
        try:
            children = widget.winfo_children()
        except Exception:
            children = []
        for child in children:
            try:
                if isinstance(child, tk.Toplevel):
                    child.configure(bg=panel)
                elif isinstance(child, tk.Canvas):
                    child.configure(
                        bg=("#100000" if active and child in preview_set else panel),
                        highlightbackground=border, highlightcolor=border,
                    )
                elif isinstance(child, (tk.Frame, tk.LabelFrame)):
                    child.configure(bg=panel, highlightbackground=border)
                elif isinstance(child, tk.Label):
                    child.configure(bg=panel, fg=text)
                elif isinstance(child, tk.Listbox):
                    child.configure(
                        bg=("#140000" if active else "#FBFCFD"), fg=text,
                        selectbackground=accent,
                        selectforeground=("#E86A6A" if active else "#FFFFFF"),
                        highlightbackground=border, highlightcolor=border,
                    )
                elif isinstance(child, tk.Text):
                    child.configure(
                        bg=("#140000" if active else "#FBFCFD"), fg=text,
                        insertbackground=text, selectbackground=accent,
                        selectforeground=("#E86A6A" if active else "#FFFFFF"),
                        highlightbackground=border, highlightcolor=border,
                    )
                elif isinstance(child, (tk.Entry, tk.Spinbox)):
                    kwargs = dict(
                        bg=("#160000" if active else "#FBFCFD"), fg=text,
                        insertbackground=text,
                        highlightbackground=border, highlightcolor=border,
                    )
                    try:
                        kwargs["readonlybackground"] = ("#160000" if active else "#FBFCFD")
                    except Exception:
                        pass
                    child.configure(**kwargs)
                elif isinstance(child, tk.Button):
                    if not any(child is pair[0] for pair in getattr(self, "_color_selector_widgets", [])):
                        child.configure(
                            bg=("#1B0202" if active else self.palette["panel_alt"]),
                            fg=text,
                            activebackground=("#260404" if active else self.palette["panel_alt"]),
                            activeforeground=text,
                            highlightbackground=border,
                        )
                elif isinstance(child, tk.Checkbutton):
                    child.configure(
                        bg=panel, fg=text, activebackground=panel, activeforeground=text,
                        selectcolor=("#160000" if active else self.palette["panel_alt"]),
                        highlightbackground=border,
                    )
                elif isinstance(child, tk.Radiobutton):
                    child.configure(
                        bg=panel, fg=text, activebackground=panel, activeforeground=text,
                        selectcolor=("#160000" if active else self.palette["panel_alt"]),
                        highlightbackground=border,
                    )
                elif isinstance(child, tk.Scale):
                    child.configure(
                        bg=panel, fg=text, troughcolor=("#170000" if active else self.palette["panel_alt"]),
                        activebackground=accent, highlightbackground=border,
                    )
                elif isinstance(child, tk.Scrollbar):
                    child.configure(
                        bg=("#220303" if active else self.palette["panel_alt"]),
                        troughcolor=panel, activebackground=accent,
                        highlightbackground=border,
                    )
            except Exception:
                pass
            self._recolor_native_widgets(child, active, panel, text, muted, border, accent)
    def _set_app_red_palette(self, active):
        """Switch only the operator application between normal and dim-red palettes.

        This deliberately does not touch the Windows desktop or either monitor's
        gamma/LUT. It is therefore safe to leave enabled throughout stimulation.
        """
        style = ttk.Style(self)
        if active:
            # Deliberately dim values: no pure white and no bright full-scale red.
            bg, panel, text, muted = "#070000", "#0E0000", "#CF4545", "#863030"
            accent, accent_dark, border = "#461010", "#310808", "#290707"
            panel_alt, field_bg = "#150101", "#120000"
            disabled_fg = "#6E2929"
            select_fg = "#E05A5A"
        else:
            bg, panel = self.palette["bg"], self.palette["panel"]
            text, muted = self.palette["text"], self.palette["muted"]
            accent, accent_dark, border = self.palette["accent"], self.palette["accent_dark"], self.palette["border"]
            panel_alt, field_bg = self.palette["panel_alt"], "#FBFCFD"
            disabled_fg = self.palette["muted"]
            select_fg = "#FFFFFF"

        self.configure(bg=bg)
        self._set_operator_titlebar_red(active)

        # Option database also colours classic pop-down/list controls created by Tk.
        try:
            self.option_add("*TCombobox*Listbox.background", field_bg)
            self.option_add("*TCombobox*Listbox.foreground", text)
            self.option_add("*TCombobox*Listbox.selectBackground", accent)
            self.option_add("*TCombobox*Listbox.selectForeground", select_fg)
            self.option_add("*Menu.background", panel)
            self.option_add("*Menu.foreground", text)
            self.option_add("*Menu.activeBackground", accent)
            self.option_add("*Menu.activeForeground", select_fg)
        except Exception:
            pass

        style.configure("App.TFrame", background=bg)
        for sty in ("TFrame", "Card.TFrame", "TabBody.TFrame"):
            style.configure(sty, background=panel)
        for sty in ("TLabel", "Section.TLabel", "Status.TLabel"):
            style.configure(sty, background=panel, foreground=text)
        style.configure("Muted.TLabel", background=panel, foreground=muted)
        style.configure("Title.TLabel", background=bg, foreground=text)
        style.configure("Subtitle.TLabel", background=bg, foreground=muted)
        style.configure("Badge.TLabel", background=("#180202" if active else "#E8F2F6"), foreground=(text if active else accent_dark))
        style.configure("ProtocolDuration.TLabel", background=panel, foreground=text)
        style.configure("ProtocolDurationCaption.TLabel", background=panel, foreground=muted)

        for sty in ("TLabelframe", "Card.TLabelframe"):
            style.configure(sty, background=panel, bordercolor=border)
        for sty in ("TLabelframe.Label", "Card.TLabelframe.Label"):
            style.configure(sty, background=panel, foreground=text)

        style.configure("TNotebook", background=bg, borderwidth=0)
        style.configure("TNotebook.Tab", background=("#160202" if active else "#E8EEF1"), foreground=muted)
        style.map(
            "TNotebook.Tab",
            background=[("selected", panel), ("active", "#1D0303" if active else "#F4F7F9")],
            foreground=[("selected", text if active else accent_dark), ("active", text)],
        )

        # Buttons: explicitly map disabled/pressed/active states so the theme never
        # falls back to a bright platform colour.
        button_defs = {
            "TButton": (panel_alt, text),
            "Secondary.TButton": (("#180202" if active else "#E6EDF1"), text),
            "Accent.TButton": (accent, (text if active else "#FFFFFF")),
            "Danger.TButton": (("#3B0808" if active else self.palette["danger"]), (text if active else "#FFFFFF")),
        }
        for sty, (button_bg, button_fg) in button_defs.items():
            style.configure(sty, background=button_bg, foreground=button_fg, bordercolor=border)
            style.map(
                sty,
                background=[
                    ("disabled", "#100000" if active else button_bg),
                    ("pressed", "#250404" if active else button_bg),
                    ("active", "#210303" if active else button_bg),
                ],
                foreground=[("disabled", disabled_fg), ("pressed", button_fg), ("active", button_fg)],
            )

        for sty in ("TCheckbutton", "TRadiobutton"):
            style.configure(sty, background=panel, foreground=text)
            style.map(sty, background=[("active", panel)], foreground=[("disabled", disabled_fg)])

        for sty in ("TEntry", "TSpinbox"):
            style.configure(sty, fieldbackground=field_bg, background=field_bg, foreground=text, bordercolor=border)
            style.map(
                sty,
                fieldbackground=[("readonly", field_bg), ("disabled", field_bg)],
                foreground=[("disabled", disabled_fg)],
            )
        style.configure("TCombobox", fieldbackground=field_bg, background=field_bg, foreground=text, arrowcolor=text, bordercolor=border)
        style.map(
            "TCombobox",
            fieldbackground=[("readonly", field_bg), ("disabled", field_bg)],
            background=[("readonly", field_bg), ("disabled", field_bg)],
            foreground=[("readonly", text), ("disabled", disabled_fg)],
            selectbackground=[("readonly", field_bg)],
            selectforeground=[("readonly", text)],
            arrowcolor=[("disabled", disabled_fg)],
        )

        style.configure("Treeview", background=field_bg, fieldbackground=field_bg, foreground=text, bordercolor=border)
        style.map("Treeview", background=[("selected", accent)], foreground=[("selected", select_fg)])
        style.configure("Treeview.Heading", background=panel_alt, foreground=text, bordercolor=border)
        style.map("Treeview.Heading", background=[("active", "#210303" if active else panel_alt)])

        # Scrollbars/progress/separators are common sources of bright strips.
        style.configure(
            "TScrollbar", background=("#1A0202" if active else panel_alt),
            troughcolor=panel, arrowcolor=text, bordercolor=border, darkcolor=border, lightcolor=border
        )
        style.map("TScrollbar", background=[("active", accent), ("pressed", accent)])
        style.configure("Horizontal.TScrollbar", background=("#1A0202" if active else panel_alt), troughcolor=panel, arrowcolor=text)
        style.configure("Vertical.TScrollbar", background=("#1A0202" if active else panel_alt), troughcolor=panel, arrowcolor=text)
        style.configure("TSeparator", background=border)
        style.configure("Horizontal.TProgressbar", background=accent, troughcolor=panel, bordercolor=border)
        style.configure("Vertical.TProgressbar", background=accent, troughcolor=panel, bordercolor=border)

        self._recolor_native_widgets(self, active, panel, text, muted, border, accent)

        # Stimulus-colour selectors remain useful in normal mode, but a white swatch
        # is intentionally suppressed while dark-room mode is active.
        for btn, variable in getattr(self, "_color_selector_widgets", []):
            try:
                if active:
                    btn.configure(
                        bg="#180202", fg=text, activebackground="#210303",
                        activeforeground=text, highlightbackground=border
                    )
                else:
                    c = variable.get()
                    btn.configure(
                        bg=c, fg=best_text_color(c),
                        activebackground=c, activeforeground=best_text_color(c)
                    )
            except Exception:
                pass

        try:
            self.update_idletasks()
        except Exception:
            pass
    def _mag_restore(self):
        if sys.platform != "win32" or not self._mag_initialized:
            self._mag_filter_active = False
            return False
        ok = False
        try:
            import ctypes
            mag = ctypes.WinDLL("Magnification.dll")
            Effect = ctypes.c_float * 25
            if self._mag_original_effect is not None:
                effect = Effect(*self._mag_original_effect)
                ok = bool(mag.MagSetFullscreenColorEffect(ctypes.byref(effect)))
        except Exception:
            ok = False
        self._mag_filter_active = False
        try:
            import ctypes
            ctypes.WinDLL("Magnification.dll").MagUninitialize()
        except Exception:
            pass
        self._mag_initialized = False
        self._mag_original_effect = None
        return ok

    def _apply_windows_color_matrix(self):
        """Disabled in v1.22: never apply a desktop-wide Windows colour matrix."""
        return False
    def _capture_global_gamma(self):
        """Capture gamma ramp from the monitor containing the operator window."""
        if sys.platform != "win32" or self._gamma_original is not None:
            return self._gamma_original is not None
        try:
            import ctypes
            from ctypes import wintypes
            user32 = ctypes.windll.user32
            gdi32 = ctypes.windll.gdi32

            class MONITORINFOEXW(ctypes.Structure):
                _fields_ = [
                    ("cbSize", wintypes.DWORD),
                    ("rcMonitor", wintypes.RECT),
                    ("rcWork", wintypes.RECT),
                    ("dwFlags", wintypes.DWORD),
                    ("szDevice", wintypes.WCHAR * 32),
                ]

            hwnd = wintypes.HWND(self.winfo_id())
            monitor = user32.MonitorFromWindow(hwnd, 2)  # MONITOR_DEFAULTTONEAREST
            info = MONITORINFOEXW()
            info.cbSize = ctypes.sizeof(MONITORINFOEXW)
            if not user32.GetMonitorInfoW(monitor, ctypes.byref(info)):
                return False
            device = str(info.szDevice)
            dc = gdi32.CreateDCW("DISPLAY", device, None, None)
            if not dc:
                return False
            Ramp = wintypes.WORD * (256 * 3)
            ramp = Ramp()
            if not gdi32.GetDeviceGammaRamp(dc, ctypes.byref(ramp)):
                gdi32.DeleteDC(dc)
                return False
            self._gamma_dc = dc
            self._gamma_device_name = device
            self._gamma_original = [int(v) for v in ramp]
            return True
        except Exception:
            return False

    def _apply_gamma_fallback(self):
        """Disabled in v1.22: never alter monitor gamma from the application."""
        return False
    def _apply_global_red_gamma(self, allow_full_desktop=True):
        """Disabled since v1.22. Retained only for backwards-safe cleanup/API compatibility."""
        self._global_red_backend = ""
        return False
    def _restore_global_gamma_safely(self):
        restored = False
        if self._mag_initialized:
            restored = self._mag_restore() or restored
        if self._gamma_original is not None and self._gamma_dc is not None:
            try:
                import ctypes
                from ctypes import wintypes
                Ramp = wintypes.WORD * (256 * 3)
                ramp = Ramp(*self._gamma_original)
                ok = bool(ctypes.windll.gdi32.SetDeviceGammaRamp(self._gamma_dc, ctypes.byref(ramp)))
                restored = ok or restored
            except Exception:
                pass
            try:
                import ctypes
                ctypes.windll.gdi32.DeleteDC(self._gamma_dc)
            except Exception:
                pass
        self._gamma_original = None
        self._gamma_dc = None
        self._gamma_device_name = None
        self._gamma_filter_active = False
        self._mag_filter_active = False
        self._global_red_backend = ""
        return restored

    def _push_idle_color_to_worker(self):
        rgb = self._current_idle_rgb()
        # Safety process is underneath PsychoPy; restart it with the same color
        # so a worker restart never flashes gray/white.
        try:
            self.stop_safety_window()
            self.start_safety_window()
        except Exception:
            pass
        if self.worker_process is None or self.worker_process.poll() is not None:
            return
        try:
            command_id = "idle_" + datetime.now().strftime("%Y%m%d_%H%M%S_%f")
            write_json_atomic(self.command_path, {
                "command_id": command_id,
                "action": "set_idle_color",
                "rgb": rgb,
            })
        except Exception:
            pass

    def toggle_red_mode(self):
        # v1.22 intentionally has no OS-wide filtering. If a legacy v1.21 filter
        # were left active by an abnormal previous run, restore it first.
        self._restore_global_gamma_safely()
        self.red_mode_active = not self.red_mode_active
        active = self.red_mode_active

        self._set_app_red_palette(active)
        self._red_filter_suspended_for_run = False

        if active:
            self.red_mode_btn.configure(text="⚫ SALIR MODO ROJO")
            self.red_mode_status.set(
                "Modo rojo ACTIVO · solo interfaz del operador · pantalla experimental sin cambios"
            )
        else:
            self.red_mode_btn.configure(text="🔴 MODO ROJO")
            self.red_mode_status.set("Modo rojo: inactivo")

        self.status.set(self.red_mode_status.get())
    def _suspend_red_filter_for_stimulus(self):
        # No-op in v1.22: dark-room mode is application-only and therefore safe
        # to keep enabled while the experimental display presents true colours.
        return
    def _resume_red_filter_after_stimulus(self):
        # No-op in v1.22. Re-apply the palette defensively in case a native widget
        # was created during the run.
        if self.red_mode_active:
            self._set_app_red_palette(True)
        return
    def update_ttl_ui(self, event=None):
        busy = self._trigger_test_active
        mode = self.ttl_mode.get()
        if not busy:
            labels = {
                "Simulación": "● SIMULACIÓN · EVENT por condición",
                "MaxOne Legacy FTDI": "● MAXONE LEGACY · FRAME + EVENT v1.17",
                "Trigger Box TTL genérica": "● TTL GENÉRICO · UNIVERSAL + AUX ESTADO/PAUSA",
                "Desactivado": "○ TRIGGERS DESACTIVADOS",
            }
            self.ttl_status_text.set(labels.get(mode, "Modo de trigger no reconocido"))
        short = {
            "Simulación": "Simulación",
            "MaxOne Legacy FTDI": "MaxOne Legacy",
            "Trigger Box TTL genérica": "Genérico",
            "Desactivado": "Desactivado",
        }.get(mode, mode)
        port = self.ttl_port.get().strip() or "auto"
        self.ttl_header_text.set(
            f"TTL: {short}" + (f" · {port}" if mode in ("MaxOne Legacy FTDI", "Trigger Box TTL genérica") else "")
        )
        if hasattr(self, "ttl_mode_combo"):
            self.ttl_mode_combo.configure(state="disabled" if busy else "readonly")
        if hasattr(self, "ttl_port_entry"):
            self.ttl_port_entry.configure(
                state=(
                    "normal"
                    if mode in ("MaxOne Legacy FTDI", "Trigger Box TTL genérica") and not busy
                    else "disabled"
                )
            )
        if hasattr(self, "ttl_test_btn"):
            self.ttl_test_btn.configure(
                text="Detener prueba" if busy else "Probar triggers",
                state=(
                    "normal"
                    if busy or mode in (
                        "Simulación", "MaxOne Legacy FTDI", "Trigger Box TTL genérica"
                    )
                    else "disabled"
                ),
            )


    def _finish_trigger_test(self, error="", cancelled=False, silent=False):
        import csv
        self._trigger_test_active = False
        for aid in self.ttl_test_after_ids:
            try:
                self.after_cancel(aid)
            except Exception:
                pass
        self.ttl_test_after_ids = []
        backend, self._trigger_test_backend = self._trigger_test_backend, None
        if backend is not None:
            close_errors = backend.close()
            if close_errors:
                error = (error + "; " if error else "") + "; ".join(close_errors)
        path = self._trigger_test_path
        if path is not None:
            try:
                outcome = "error" if error else "cancelled" if cancelled else "completed"
                with path.open("w", newline="", encoding="utf-8") as fh:
                    writer = csv.DictWriter(fh, fieldnames=[
                        "elapsed_s", "event_logical", "frame_logical",
                        "expected_bits", "mode", "port", "label",
                        "event_offset_measured_ms", "event_width_measured_ms",
                        "test_outcome",
                    ])
                    writer.writeheader()
                    for row in self._trigger_test_events:
                        writer.writerow(dict(row, test_outcome=outcome))
            except Exception as exc:
                error = (error + "; " if error else "") + f"Guardar diagnóstico: {exc}"
        self._trigger_test_path = None
        if silent:
            return
        self.update_ttl_ui()
        if error:
            self.ttl_status_text.set("● TRIGGERS · error de diagnóstico")
            messagebox.showerror("Prueba de triggers", error, parent=self)
        elif cancelled:
            self.status.set("Prueba de triggers detenida · salidas devueltas a reposo.")
        elif self._trigger_test_mode == "MaxOne Legacy FTDI":
            self.status.set(
                "Prueba MaxOne Legacy enviada · comportamiento v1.17 conservado. "
                f"Registro: {path}"
            )
        elif self._trigger_test_mode == "Trigger Box TTL genérica":
            self.status.set(
                "Prueba TTL genérica enviada · contrasta FRAME/EVENT/UNIVERSAL. "
                f"Registro: {path}"
            )
        else:
            self.status.set(f"Prueba simulada completada · Registro: {path}")


    def test_ttl_simulation(self):
        if self._trigger_test_active:
            self._finish_trigger_test(cancelled=True)
            return
        if self.active_command_id is not None:
            messagebox.showinfo(
                "Prueba de triggers", "Finaliza o aborta el estímulo antes de probar los triggers.",
                parent=self,
            )
            return
        mode = self.ttl_mode.get()
        if mode not in (
            "Simulación", "MaxOne Legacy FTDI", "Trigger Box TTL genérica"
        ):
            return
        self._trigger_test_mode = mode
        self._trigger_test_events = []
        self._trigger_test_path = None
        self._trigger_test_backend = None
        self._trigger_test_active = True
        port = ""
        try:
            if mode == "MaxOne Legacy FTDI":
                # Exactly the legacy backend used by v1.17.
                self._trigger_test_backend = MaxOneFTDI(self.ttl_port.get())
                port = self._trigger_test_backend.open()
            elif mode == "Trigger Box TTL genérica":
                self._trigger_test_backend = GenericTTLFTDI(self.ttl_port.get())
                port = self._trigger_test_backend.open()
            stamp = datetime.now().strftime("%Y%m%d_%H%M%S_%f")
            if mode == "MaxOne Legacy FTDI":
                prefix = "maxone_ftdi_test"
            elif mode == "Trigger Box TTL genérica":
                prefix = "generic_ftdi_test"
            else:
                prefix = "ttl_simulation_test"
            self._trigger_test_path = LOG_DIR / f"{prefix}_{stamp}.csv"
            self.update_ttl_ui()
        except Exception as exc:
            self._finish_trigger_test(error=f"{type(exc).__name__}: {exc}")
            return

        if mode == "Trigger Box TTL genérica" and port:
            # Independent-output plateaus plus three real UNIVERSAL pairs.
            sequence = [
                (0, 0, 1000, "REST"), (1, 0, 1000, "EVENT"),
                (0, 0, 500, "REST"), (0, 1, 1000, "FRAME"),
                (0, 0, 500, "REST"), (1, 1, 1000, "EVENT+FRAME"),
                (0, 0, 500, "REST"),
                (0, 0, 250, "UNIVERSAL_PAIR_1"),
                (0, 0, 250, "UNIVERSAL_PAIR_2"),
                (0, 0, 250, "UNIVERSAL_PAIR_3"),
                (0, 0, 750, "REST"),
            ]
        elif port:
            # MaxOne Legacy: exact v1.17 diagnostic sequence.
            sequence = [
                (0, 0, 1000, "REST"), (1, 0, 1000, "EVENT"),
                (0, 0, 500, "REST"), (0, 1, 1000, "FRAME"),
                (0, 0, 500, "REST"), (1, 1, 1000, "EVENT+FRAME"),
                (0, 0, 1000, "REST"),
            ]
        else:
            sequence = [
                (0, 0, 400, "REST"), (1, 0, 150, "EVENT_1"),
                (0, 0, 650, "REST"), (1, 0, 150, "EVENT_2"),
                (0, 0, 650, "REST"), (1, 0, 150, "EVENT_3"),
                (0, 0, 550, "REST"),
            ]
        started = time.perf_counter()

        def advance(index):
            self.ttl_test_after_ids = []
            if not self._trigger_test_active:
                return
            if index == len(sequence):
                self._finish_trigger_test()
                return
            event_value, frame_value, duration_ms, label = sequence[index]
            try:
                metrics = {}
                if self._trigger_test_backend is not None:
                    if (
                        mode == "Trigger Box TTL genérica"
                        and label.startswith("UNIVERSAL_PAIR")
                    ):
                        self._trigger_test_backend.idle(strict=True)
                        self._trigger_test_backend.frame_high()
                        self._trigger_test_backend.frame_low()
                        metrics = self._trigger_test_backend.pulse_event_after_frame()
                    else:
                        self._trigger_test_backend.set_frame(0)
                        self._trigger_test_backend.set_event(event_value)
                        self._trigger_test_backend.set_frame(frame_value)
                self._trigger_test_events.append({
                    "elapsed_s": f"{time.perf_counter() - started:.9f}",
                    "event_logical": event_value,
                    "frame_logical": frame_value,
                    "expected_bits": (
                        16 + event_value + 8 * frame_value
                        if mode == "MaxOne Legacy FTDI" and port else ""
                    ),
                    "mode": mode,
                    "port": port,
                    "label": label,
                    "event_offset_measured_ms": (
                        "" if "event_offset_s" not in metrics
                        else f'{1000.0 * metrics["event_offset_s"]:.6f}'
                    ),
                    "event_width_measured_ms": (
                        "" if "event_width_s" not in metrics
                        else f'{1000.0 * metrics["event_width_s"]:.6f}'
                    ),
                })
                self.ttl_status_text.set(f"● PRUEBA · {label}")
                aid = self.after(duration_ms, lambda: advance(index + 1))
                self.ttl_test_after_ids = [aid]
            except Exception as exc:
                self._finish_trigger_test(error=f"{type(exc).__name__}: {exc}")

        advance(0)


    def common_config(self):
        idx = self.combo.current()
        if idx < 0:
            raise ValueError(
                "Selecciona la pantalla de estimulación."
            )

        s = self.screens[idx]

        ui_mode = self.ttl_mode.get()
        mode_map = {
            "Simulación": "simulation",
            "MaxOne Legacy FTDI": "maxone_ftdi",
            "Trigger Box TTL genérica": "generic_ftdi",
            "Desactivado": "disabled",
        }
        if ui_mode not in mode_map:
            raise ValueError(
                f"Modo de trigger desconocido: {ui_mode}"
            )

        ttl_mode = mode_map[ui_mode]

        if ttl_mode == "simulation":
            ttl_pin = int(self.ttl_pin.get())
            if ttl_pin < 0:
                raise ValueError(
                    "El pin simulado debe ser un entero >= 0."
                )
        else:
            ttl_pin = 0

        ttl_port = self.ttl_port.get().strip()
        if not ttl_port:
            ttl_port = "auto"

        cfg = {
            "app_version": APP_VERSION,
            "screen_index": idx,
            "screen_width_px": s["width"],
            "screen_height_px": s["height"],
            "screen_x": s["x"],
            "screen_y": s["y"],
            "physical_width_cm": self._num(self.width_cm),
            "physical_height_cm": self._num(self.height_cm),
            "distance_cm": self._num(self.distance_cm),
            "ttl_mode": ttl_mode,
            "ttl_pin": ttl_pin,
            "ttl_port": ttl_port,
            "adaptation_rgb": hex_to_rgb255(self.adaptation_hex.get()),
            "idle_rgb": self._current_idle_rgb(),
            "session_metadata": self._session_metadata_snapshot(),
        }

        if min(
            cfg["physical_width_cm"],
            cfg["physical_height_cm"],
            cfg["distance_cm"]
        ) <= 0:
            raise ValueError(
                "Tamaño de pantalla y distancia deben ser > 0."
            )

        return cfg


    def collect_stimulus_config(self):
        cfg = self.common_config()
        tab = self.tabs.index(
            self.tabs.select()
        )

        if tab == 0:
            cfg.update({
                "stimulus_type": "on_off",
                "baseline_s":
                    self._num(self.on_baseline),
                "on_s":
                    self._num(self.on_on),
                "gray_interval_s":
                    self._num(self.on_gray),
                "off_s":
                    self._num(self.on_off),
                "final_gray_s":
                    self._num(self.on_final),
                "repetitions":
                    int(self.on_reps.get()),
                "on_color_rgb":
                    hex_to_rgb255(
                        self.on_on_hex.get()
                    ),
                "off_color_rgb":
                    hex_to_rgb255(
                        self.on_off_hex.get()
                    ),
            })

        elif tab == 1:
            mode = self.motion_mode.get()

            baseline = self._num(
                self.motion_baseline
            )
            duration = self._num(
                self.motion_duration
            )
            isi = self._num(
                self.motion_isi
            )
            reps = int(
                self.motion_reps.get()
            )
            directions = self.motion_dirs.get()

            bar_width = self._num(
                self.motion_bar_width
            )
            bar_length = self._num(
                self.motion_bar_length
            )
            spot_diameter = self._num(
                self.motion_spot_diameter
            )
            small_diameter = self._num(
                self.motion_small_diameter
            )
            large_diameter = self._num(
                self.motion_large_diameter
            )
            annulus_small_diameter = self._num(
                self.motion_annulus_small_diameter
            )
            annulus_large_diameter = self._num(
                self.motion_annulus_large_diameter
            )
            annulus_thickness = self._num(
                self.motion_annulus_thickness
            )
            center_x = self._num(
                self.motion_center_x
            )
            center_y = self._num(
                self.motion_center_y
            )

            if (
                baseline < 0
                or duration <= 0
                or isi < 0
            ):
                raise ValueError(
                    "Motion: duración > 0 y "
                    "tiempos de adaptación >= 0."
                )
            if reps < 1:
                raise ValueError(
                    "Motion: repeticiones debe ser >= 1."
                )

            if mode in (
                "Moving Bar",
                "Moving Spot",
            ):
                if not [
                    x
                    for x in directions.replace(
                        ";", ","
                    ).split(",")
                    if x.strip()
                ]:
                    raise ValueError(
                        "Introduce al menos una dirección."
                    )

            if mode == "Moving Bar":
                if (
                    bar_width <= 0
                    or bar_length <= 0
                ):
                    raise ValueError(
                        "Anchura y longitud de barra "
                        "deben ser > 0."
                    )

            elif mode == "Moving Spot":
                if spot_diameter <= 0:
                    raise ValueError(
                        "El diámetro del spot debe ser > 0."
                    )

            elif mode in (
                "Looming",
                "Receding",
            ):
                if (
                    small_diameter <= 0
                    or large_diameter
                    <= small_diameter
                ):
                    raise ValueError(
                        "Motion: diámetro grande > "
                        "diámetro pequeño > 0."
                    )

            elif mode in (
                "Expanding Annulus",
                "Receding Annulus",
            ):
                if (
                    annulus_small_diameter <= 0
                    or annulus_large_diameter
                    <= annulus_small_diameter
                ):
                    raise ValueError(
                        "Annulus: diámetro exterior grande > "
                        "diámetro exterior pequeño > 0."
                    )

                if annulus_thickness <= 0:
                    raise ValueError(
                        "Annulus: el grosor radial debe ser > 0."
                    )

                if (
                    2.0
                    * annulus_thickness
                    >= annulus_small_diameter
                ):
                    raise ValueError(
                        "Annulus: el grosor radial debe ser menor "
                        "que la mitad del diámetro exterior pequeño."
                    )

            else:
                raise ValueError(
                    f"Modo Motion desconocido: {mode}"
                )

            cfg.update({
                "stimulus_type":
                    "motion",
                "motion_mode":
                    mode,
                "motion_baseline_s":
                    baseline,
                "motion_duration_s":
                    duration,
                "motion_isi_s":
                    isi,
                "motion_repetitions":
                    reps,
                "motion_directions":
                    directions,
                "motion_background_rgb":
                    hex_to_rgb255(
                        self.motion_background_hex.get()
                    ),
                "motion_stimulus_rgb":
                    hex_to_rgb255(
                        self.motion_stimulus_hex.get()
                    ),
                "motion_bar_width_deg":
                    bar_width,
                "motion_bar_length_deg":
                    bar_length,
                "motion_spot_diameter_deg":
                    spot_diameter,
                "motion_small_diameter_deg":
                    small_diameter,
                "motion_large_diameter_deg":
                    large_diameter,
                "motion_annulus_small_diameter_deg":
                    annulus_small_diameter,
                "motion_annulus_large_diameter_deg":
                    annulus_large_diameter,
                "motion_annulus_thickness_deg":
                    annulus_thickness,
                "motion_center_x_deg":
                    center_x,
                "motion_center_y_deg":
                    center_y,
            })

        elif tab == 2:
            mode = self.gr_mode.get()

            baseline = self._num(
                self.gr_baseline
            )
            duration = self._num(
                self.gr_duration
            )
            isi = self._num(
                self.gr_isi
            )
            sf = self._num(
                self.gr_sf
            )
            tf = self._num(
                self.gr_tf
            )
            reversal_hz = self._num(
                self.gr_reversal_hz
            )
            phase = self._num(
                self.gr_phase
            )
            contrast = self._num(
                self.gr_contrast
            )
            reps = int(
                self.gr_reps.get()
            )
            angles = self.gr_angles.get()

            gabor_size = self._num(
                self.gr_gabor_size
            )
            gabor_sigma = self._num(
                self.gr_gabor_sigma
            )
            gabor_x = self._num(
                self.gr_gabor_x
            )
            gabor_y = self._num(
                self.gr_gabor_y
            )

            if (
                baseline < 0
                or duration <= 0
                or isi < 0
            ):
                raise ValueError(
                    "Grating: duración > 0 y "
                    "tiempos de adaptación >= 0."
                )

            if sf <= 0:
                raise ValueError(
                    "Grating: la frecuencia espacial "
                    "debe ser > 0."
                )

            if not 0.0 <= contrast <= 1.0:
                raise ValueError(
                    "El contraste debe estar "
                    "entre 0 y 1."
                )

            if reps < 1:
                raise ValueError(
                    "Grating: repeticiones debe ser >= 1."
                )

            if not [
                x
                for x in angles.replace(
                    ";", ","
                ).split(",")
                if x.strip()
            ]:
                raise ValueError(
                    "Grating: introduce al menos "
                    "una dirección/orientación."
                )

            if mode in (
                "Drifting",
                "Drifting Gabor",
                "Asymmetric Drift",
            ):
                if tf <= 0:
                    raise ValueError(
                        f"{mode}: la frecuencia temporal "
                        "debe ser > 0."
                    )

            elif mode == "Phase Reversal":
                if reversal_hz <= 0:
                    raise ValueError(
                        "Phase Reversal: la frecuencia "
                        "de inversión debe ser > 0."
                    )

            asymmetric_polarity = str(
                self.gr_asym_polarity.get()
            ).strip().upper()
            if (
                mode == "Asymmetric Drift"
                and asymmetric_polarity not in (
                    "A_TO_B",
                    "B_TO_A",
                )
            ):
                raise ValueError(
                    "Asymmetric Drift: selecciona A_TO_B o B_TO_A."
                )

            if mode in (
                "Static Gabor",
                "Drifting Gabor",
            ):
                if (
                    gabor_size <= 0
                    or gabor_sigma <= 0
                    or gabor_sigma
                    > gabor_size / 2.0
                ):
                    raise ValueError(
                        f"{mode}: tamaño > 0 y "
                        "0 < sigma <= tamaño/2."
                    )

            cfg.update({
                "stimulus_type":
                    "grating",
                "grating_mode":
                    mode,
                "grating_baseline_s":
                    baseline,
                "grating_duration_s":
                    duration,
                "grating_isi_s":
                    isi,
                "grating_sf_cpd":
                    sf,
                "grating_tf_hz":
                    tf,
                "grating_reversal_hz":
                    reversal_hz,
                "grating_phase_cycles":
                    phase,
                "grating_angles":
                    angles,
                "grating_contrast":
                    contrast,
                "grating_repetitions":
                    reps,
                "grating_absolute_contrast":
                    bool(
                        self.gr_absolute.get()
                    ),
                "grating_waveform":
                    self.gr_waveform.get(),
                "grating_asymmetric_polarity":
                    asymmetric_polarity,
                "grating_color_a_rgb":
                    hex_to_rgb255(
                        self.gr_color_a_hex.get()
                    ),
                "grating_color_b_rgb":
                    hex_to_rgb255(
                        self.gr_color_b_hex.get()
                    ),
                "grating_gabor_size_deg":
                    gabor_size,
                "grating_gabor_sigma_deg":
                    gabor_sigma,
                "grating_gabor_x_deg":
                    gabor_x,
                "grating_gabor_y_deg":
                    gabor_y,
                "grating_gabor_background_rgb":
                    hex_to_rgb255(
                        self.gr_gabor_bg_hex.get()
                    ),
            })

        elif tab == 3:
            family = self.dn_family.get()
            distribution = self.dn_mode.get()

            if family == "Gaussian White Noise":
                distribution = "Gaussian"

            baseline = self._num(
                self.dn_baseline
            )
            duration = self._num(
                self.dn_duration
            )
            isi = self._num(
                self.dn_isi
            )
            update_hz = self._num(
                self.dn_update_hz
            )
            check_size = self._num(
                self.dn_check_size
            )
            contrast = self._num(
                self.dn_contrast
            )
            reps = int(
                self.dn_reps.get()
            )
            seed = int(
                self.dn_seed.get()
            )

            if family not in (
                "Dense White Noise",
                "Gaussian White Noise",
                "Frozen Noise",
            ):
                raise ValueError(
                    "Noise: familia desconocida."
                )

            if family == "Dense White Noise":
                if distribution not in (
                    "Binary",
                    "Ternary",
                ):
                    raise ValueError(
                        "Dense White Noise: usa Binary o Ternary."
                    )

            if family == "Frozen Noise":
                if distribution not in (
                    "Binary",
                    "Ternary",
                    "Gaussian",
                ):
                    raise ValueError(
                        "Frozen Noise: distribución no válida."
                    )

            if (
                baseline < 0
                or duration <= 0
                or isi < 0
            ):
                raise ValueError(
                    "Noise: duración > 0 y tiempos de adaptación >= 0."
                )

            if update_hz <= 0:
                raise ValueError(
                    "Noise: la frecuencia de actualización "
                    "debe ser > 0."
                )

            if check_size <= 0:
                raise ValueError(
                    "Noise: el tamaño de check debe ser > 0."
                )

            if not 0.0 <= contrast <= 1.0:
                raise ValueError(
                    "El contraste debe estar entre 0 y 1."
                )

            if reps < 1:
                raise ValueError(
                    "Noise: repeticiones debe ser >= 1."
                )

            cfg.update({
                "stimulus_type":
                    "dense_noise",
                "dense_family_mode":
                    family,
                "dense_distribution":
                    distribution,
                # Legacy mirror retained so older analysis/scripts can
                # still identify Binary/Ternary where applicable.
                "dense_mode":
                    distribution,
                "dense_baseline_s":
                    baseline,
                "dense_duration_s":
                    duration,
                "dense_isi_s":
                    isi,
                "dense_update_hz":
                    update_hz,
                "dense_check_size_deg":
                    check_size,
                "dense_contrast":
                    contrast,
                "dense_repetitions":
                    reps,
                "dense_seed":
                    seed,
                "dense_color_low_rgb":
                    hex_to_rgb255(
                        self.dn_low_hex.get()
                    ),
                "dense_color_mid_rgb":
                    hex_to_rgb255(
                        self.dn_mid_hex.get()
                    ),
                "dense_color_high_rgb":
                    hex_to_rgb255(
                        self.dn_high_hex.get()
                    ),
            })

        elif tab == 4:
            mode = self.rf_mode.get()

            baseline = self._num(
                self.rf_baseline
            )
            duration = self._num(
                self.rf_duration
            )
            isi = self._num(
                self.rf_isi
            )
            reps = int(
                self.rf_reps.get()
            )

            if baseline < 0 or duration <= 0 or isi < 0:
                raise ValueError(
                    "RF Mapping: duración > 0 y tiempos de adaptación >= 0."
                )
            if reps < 1:
                raise ValueError(
                    "RF Mapping: repeticiones debe ser >= 1."
                )

            spot_d = self._num(
                self.rf_spot_diameter
            )
            ann_in = self._num(
                self.rf_annulus_inner
            )
            ann_out = self._num(
                self.rf_annulus_outer
            )
            annulus_series_thickness = self._num(
                self.rf_annulus_series_thickness
            )
            sparse_square = self._num(
                self.rf_sparse_square
            )
            sparse_step = self._num(
                self.rf_sparse_step
            )
            sparse_trials = int(
                self.rf_sparse_trials.get()
            )

            if mode == "Spot" and spot_d <= 0:
                raise ValueError(
                    "El diámetro del spot debe ser > 0."
                )
            if mode == "Annulus":
                if ann_in <= 0 or ann_out <= ann_in:
                    raise ValueError(
                        "Annulus: diámetro externo > interno > 0."
                    )
            if mode == "Size series":
                try:
                    sizes = [
                        float(v.strip())
                        for v in self.rf_size_series.get().split(",")
                        if v.strip()
                    ]
                except Exception:
                    raise ValueError(
                        "Los diámetros deben ser números separados por comas."
                    )
                if not sizes:
                    raise ValueError(
                        "Introduce al menos un diámetro."
                    )
                if any(v <= 0 for v in sizes):
                    raise ValueError(
                        "Todos los diámetros deben ser > 0."
                    )

            if mode == "Annulus Size series":
                try:
                    annulus_sizes = [
                        float(v.strip())
                        for v in self.rf_annulus_size_series.get().split(",")
                        if v.strip()
                    ]
                except Exception:
                    raise ValueError(
                        "Annulus Size series: los diámetros exteriores "
                        "deben ser números separados por comas."
                    )
                if not annulus_sizes:
                    raise ValueError(
                        "Annulus Size series: introduce al menos un diámetro."
                    )
                if any(v <= 0 for v in annulus_sizes):
                    raise ValueError(
                        "Annulus Size series: todos los diámetros deben ser > 0."
                    )
                if annulus_series_thickness <= 0:
                    raise ValueError(
                        "Annulus Size series: el grosor radial debe ser > 0."
                    )
                if any(
                    2.0 * annulus_series_thickness >= v
                    for v in annulus_sizes
                ):
                    raise ValueError(
                        "Annulus Size series: el grosor radial debe ser menor "
                        "que la mitad de todos los diámetros exteriores."
                    )

            if mode == "Sparse Noise":
                if sparse_square <= 0 or sparse_step <= 0:
                    raise ValueError(
                        "Sparse Noise: tamaño y paso deben ser > 0."
                    )
                if sparse_trials < 1:
                    raise ValueError(
                        "Sparse Noise: presentaciones debe ser >= 1."
                    )

            cfg.update({
                "stimulus_type": "rf_mapping",
                "rf_mode": mode,
                "rf_baseline_s": baseline,
                "rf_duration_s": duration,
                "rf_isi_s": isi,
                "rf_repetitions": reps,
                "rf_x_deg":
                    self._num(self.rf_x),
                "rf_y_deg":
                    self._num(self.rf_y),
                "rf_stimulus_rgb":
                    hex_to_rgb255(
                        self.rf_stim_hex.get()
                    ),
                "rf_background_rgb":
                    hex_to_rgb255(
                        self.rf_bg_hex.get()
                    ),
                "rf_spot_diameter_deg": spot_d,
                "rf_annulus_inner_deg": ann_in,
                "rf_annulus_outer_deg": ann_out,
                "rf_size_series_deg":
                    self.rf_size_series.get(),
                "rf_annulus_size_series_deg":
                    self.rf_annulus_size_series.get(),
                "rf_annulus_series_thickness_deg":
                    annulus_series_thickness,
                "rf_sparse_square_deg":
                    sparse_square,
                "rf_sparse_grid_step_deg":
                    sparse_step,
                "rf_sparse_presentations":
                    sparse_trials,
                "rf_sparse_polarity":
                    self.rf_sparse_polarity.get(),
                "rf_sparse_seed":
                    int(self.rf_sparse_seed.get()),
            })

        elif tab == 5:
            mode = self.temporal_mode.get()

            basal = self._num(
                self.ch_basal
            )
            final = self._num(
                self.ch_final
            )
            reps = int(
                self.ch_reps.get()
            )

            if (
                basal < 0
                or final < 0
            ):
                raise ValueError(
                    "Temporal: los tiempos de adaptación deben ser >= 0."
                )
            if reps < 1:
                raise ValueError(
                    "Temporal: repeticiones debe ser >= 1."
                )

            # Historical Chirp + Intensity values are always retained
            # in the config so old protocols remain fully editable.
            fmin = self._num(
                self.ch_fmin
            )
            fmax = self._num(
                self.ch_fmax
            )
            istart = self._num(
                self.ch_istart
            )

            temporal_duration = self._num(
                self.ch_temp_duration
            )
            temporal_frequency = self._num(
                self.ch_temp_frequency
            )
            temporal_contrast = self._num(
                self.ch_temp_contrast
            )
            temporal_update = self._num(
                self.ch_temp_update_hz
            )
            temporal_seed = int(
                self.ch_temp_seed.get()
            )
            contrast_isi = self._num(
                self.ch_contrast_isi
            )
            contrast_levels_text = (
                self.ch_contrast_levels.get()
            )

            if mode == "Chirp + Intensity":
                if (
                    fmin <= 0
                    or fmax <= 0
                ):
                    raise ValueError(
                        "Las frecuencias del chirp deben ser > 0."
                    )
                if fmax < fmin:
                    raise ValueError(
                        "La frecuencia máxima debe ser >= a la mínima."
                    )
                if not (
                    0.0
                    <= istart
                    <= 100.0
                ):
                    raise ValueError(
                        "La intensidad ON inicial debe estar entre 0 y 100 %."
                    )

            elif mode == "Sinusoidal Flicker":
                if temporal_duration <= 0:
                    raise ValueError(
                        "Sinusoidal Flicker: duración debe ser > 0."
                    )
                if temporal_frequency <= 0:
                    raise ValueError(
                        "Sinusoidal Flicker: frecuencia debe ser > 0."
                    )
                if not (
                    0.0
                    <= temporal_contrast
                    <= 100.0
                ):
                    raise ValueError(
                        "Sinusoidal Flicker: contraste debe estar entre 0 y 100 %."
                    )

            elif mode == "Gaussian Flicker":
                if temporal_duration <= 0:
                    raise ValueError(
                        "Gaussian Flicker: duración debe ser > 0."
                    )
                if temporal_update <= 0:
                    raise ValueError(
                        "Gaussian Flicker: frecuencia de actualización debe ser > 0."
                    )
                if not (
                    0.0
                    <= temporal_contrast
                    <= 100.0
                ):
                    raise ValueError(
                        "Gaussian Flicker: contraste debe estar entre 0 y 100 %."
                    )

            elif mode == "Contrast Series":
                if temporal_duration <= 0:
                    raise ValueError(
                        "Contrast Series: duración por contraste debe ser > 0."
                    )
                if temporal_frequency <= 0:
                    raise ValueError(
                        "Contrast Series: frecuencia debe ser > 0."
                    )
                if contrast_isi < 0:
                    raise ValueError(
                        "Contrast Series: la adaptación entre contrastes debe ser >= 0."
                    )

                try:
                    levels = [
                        float(
                            x.strip()
                        )
                        for x
                        in contrast_levels_text.replace(
                            ";", ","
                        ).split(",")
                        if x.strip()
                    ]
                except Exception:
                    raise ValueError(
                        "Contrast Series: introduce porcentajes separados por comas."
                    )

                if not levels:
                    raise ValueError(
                        "Contrast Series: introduce al menos un contraste."
                    )
                if any(
                    x < 0.0
                    or x > 100.0
                    for x in levels
                ):
                    raise ValueError(
                        "Contrast Series: todos los contrastes deben estar entre 0 y 100 %."
                    )

            else:
                raise ValueError(
                    f"Modo Temporal desconocido: {mode}"
                )

            cfg.update({
                "stimulus_type":
                    "chirp_intensity",
                "temporal_mode":
                    mode,

                "chirp_basal_s":
                    basal,
                "chirp_step_on_s":
                    self._num(
                        self.ch_step_on
                    ),
                "chirp_step_off_s":
                    self._num(
                        self.ch_step_off
                    ),
                "chirp_pause1_s":
                    self._num(
                        self.ch_pause1
                    ),
                "chirp_frequency_min_hz":
                    fmin,
                "chirp_frequency_max_hz":
                    fmax,
                "chirp_frequency_duration_s":
                    self._num(
                        self.ch_fdur
                    ),
                "chirp_pause2_s":
                    self._num(
                        self.ch_pause2
                    ),
                "chirp_intensity_frequency_hz":
                    self._num(
                        self.ch_ifreq
                    ),
                "chirp_intensity_duration_s":
                    self._num(
                        self.ch_idur
                    ),
                "chirp_intensity_start_pct":
                    istart,

                "temporal_duration_s":
                    temporal_duration,
                "temporal_frequency_hz":
                    temporal_frequency,
                "temporal_contrast_pct":
                    temporal_contrast,
                "temporal_update_hz":
                    temporal_update,
                "temporal_seed":
                    temporal_seed,
                "temporal_contrast_levels_pct":
                    contrast_levels_text,
                "temporal_contrast_isi_s":
                    contrast_isi,

                "chirp_on_color_rgb":
                    hex_to_rgb255(
                        self.ch_on_hex.get()
                    ),
                "chirp_off_color_rgb":
                    hex_to_rgb255(
                        self.ch_off_hex.get()
                    ),
                "chirp_final_gray_s":
                    final,
                "chirp_repetitions":
                    reps,
            })

        else:
            raise ValueError(
                "La pestaña Protocolos es un editor. "
                "La ejecución automática se activará en la siguiente fase."
            )

        return cfg

    def save_current(self):
        try:
            data = {
                "screen_index":
                    self.combo.current(),
                "physical_width_cm":
                    self._num(self.width_cm),
                "physical_height_cm":
                    self._num(self.height_cm),
                "distance_cm":
                    self._num(self.distance_cm),

                "ttl": {
                    "mode": self.ttl_mode.get(),
                    "pin": int(self.ttl_pin.get()),
                    "port": (self.ttl_port.get().strip() or "auto"),
                    "default_mode": self.saved.get("ttl", {}).get(
                        "default_mode", self.ttl_mode.get()
                    ),
                    "default_pin": self.saved.get("ttl", {}).get(
                        "default_pin", int(self.ttl_pin.get())
                    ),
                    "default_port": self.saved.get("ttl", {}).get(
                        "default_port", self.ttl_port.get().strip() or "auto"
                    ),
                },

                "display_safety": {
                    "standby": "black",
                    "adaptation_hex": self.adaptation_hex.get().upper(),
                    "red_intensity_pct": float(self.red_mode_intensity.get().replace(",", ".")),
                    "red_global_filter": False,
                },

                "on_off": {
                    "baseline_s":
                        self._num(self.on_baseline),
                    "on_s":
                        self._num(self.on_on),
                    "gray_interval_s":
                        self._num(self.on_gray),
                    "off_s":
                        self._num(self.on_off),
                    "final_gray_s":
                        self._num(self.on_final),
                    "repetitions":
                        int(self.on_reps.get()),
                    "on_hex":
                        self.on_on_hex.get(),
                    "off_hex":
                        self.on_off_hex.get(),
                },

                "motion": {
                    "mode":
                        self.motion_mode.get(),
                    "baseline_s":
                        self._num(
                            self.motion_baseline
                        ),
                    "duration_s":
                        self._num(
                            self.motion_duration
                        ),
                    "isi_s":
                        self._num(
                            self.motion_isi
                        ),
                    "repetitions":
                        int(
                            self.motion_reps.get()
                        ),
                    "directions":
                        self.motion_dirs.get(),
                    "background_hex":
                        self.motion_background_hex.get(),
                    "stimulus_hex":
                        self.motion_stimulus_hex.get(),
                    "bar_width_deg":
                        self._num(
                            self.motion_bar_width
                        ),
                    "bar_length_deg":
                        self._num(
                            self.motion_bar_length
                        ),
                    "spot_diameter_deg":
                        self._num(
                            self.motion_spot_diameter
                        ),
                    "small_diameter_deg":
                        self._num(
                            self.motion_small_diameter
                        ),
                    "large_diameter_deg":
                        self._num(
                            self.motion_large_diameter
                        ),
                    "annulus_small_diameter_deg":
                        self._num(
                            self.motion_annulus_small_diameter
                        ),
                    "annulus_large_diameter_deg":
                        self._num(
                            self.motion_annulus_large_diameter
                        ),
                    "annulus_thickness_deg":
                        self._num(
                            self.motion_annulus_thickness
                        ),
                    "center_x_deg":
                        self._num(
                            self.motion_center_x
                        ),
                    "center_y_deg":
                        self._num(
                            self.motion_center_y
                        ),
                },

                "grating": {
                    "mode":
                        self.gr_mode.get(),
                    "baseline_s":
                        self._num(
                            self.gr_baseline
                        ),
                    "duration_s":
                        self._num(
                            self.gr_duration
                        ),
                    "isi_s":
                        self._num(
                            self.gr_isi
                        ),
                    "sf_cpd":
                        self._num(
                            self.gr_sf
                        ),
                    "tf_hz":
                        self._num(
                            self.gr_tf
                        ),
                    "reversal_hz":
                        self._num(
                            self.gr_reversal_hz
                        ),
                    "phase_cycles":
                        self._num(
                            self.gr_phase
                        ),
                    "angles":
                        self.gr_angles.get(),
                    "contrast":
                        self._num(
                            self.gr_contrast
                        ),
                    "repetitions":
                        int(
                            self.gr_reps.get()
                        ),
                    "absolute_contrast":
                        bool(
                            self.gr_absolute.get()
                        ),
                    "waveform":
                        self.gr_waveform.get(),
                    "asymmetric_polarity":
                        self.gr_asym_polarity.get(),
                    "color_a_hex":
                        self.gr_color_a_hex.get(),
                    "color_b_hex":
                        self.gr_color_b_hex.get(),
                    "gabor_size_deg":
                        self._num(
                            self.gr_gabor_size
                        ),
                    "gabor_sigma_deg":
                        self._num(
                            self.gr_gabor_sigma
                        ),
                    "gabor_x_deg":
                        self._num(
                            self.gr_gabor_x
                        ),
                    "gabor_y_deg":
                        self._num(
                            self.gr_gabor_y
                        ),
                    "gabor_background_hex":
                        self.gr_gabor_bg_hex.get(),
                },

                "dense_noise": {
                    "family_mode":
                        self.dn_family.get(),
                    "distribution":
                        (
                            "Gaussian"
                            if self.dn_family.get()
                            == "Gaussian White Noise"
                            else self.dn_mode.get()
                        ),
                    "mode":
                        (
                            "Gaussian"
                            if self.dn_family.get()
                            == "Gaussian White Noise"
                            else self.dn_mode.get()
                        ),
                    "baseline_s":
                        self._num(
                            self.dn_baseline
                        ),
                    "duration_s":
                        self._num(
                            self.dn_duration
                        ),
                    "isi_s":
                        self._num(
                            self.dn_isi
                        ),
                    "update_hz":
                        self._num(
                            self.dn_update_hz
                        ),
                    "check_size_deg":
                        self._num(
                            self.dn_check_size
                        ),
                    "contrast":
                        self._num(
                            self.dn_contrast
                        ),
                    "repetitions":
                        int(
                            self.dn_reps.get()
                        ),
                    "seed":
                        int(
                            self.dn_seed.get()
                        ),
                    "color_low_hex":
                        self.dn_low_hex.get(),
                    "color_mid_hex":
                        self.dn_mid_hex.get(),
                    "color_high_hex":
                        self.dn_high_hex.get(),
                },

                "rf_mapping": {
                    "mode":
                        self.rf_mode.get(),
                    "baseline_s":
                        self._num(self.rf_baseline),
                    "duration_s":
                        self._num(self.rf_duration),
                    "isi_s":
                        self._num(self.rf_isi),
                    "repetitions":
                        int(self.rf_reps.get()),
                    "x_deg":
                        self._num(self.rf_x),
                    "y_deg":
                        self._num(self.rf_y),
                    "stimulus_hex":
                        self.rf_stim_hex.get(),
                    "background_hex":
                        self.rf_bg_hex.get(),
                    "spot_diameter_deg":
                        self._num(self.rf_spot_diameter),
                    "annulus_inner_deg":
                        self._num(self.rf_annulus_inner),
                    "annulus_outer_deg":
                        self._num(self.rf_annulus_outer),
                    "size_series_deg":
                        self.rf_size_series.get(),
                    "annulus_size_series_deg":
                        self.rf_annulus_size_series.get(),
                    "annulus_series_thickness_deg":
                        self._num(
                            self.rf_annulus_series_thickness
                        ),
                    "sparse_square_deg":
                        self._num(self.rf_sparse_square),
                    "sparse_grid_step_deg":
                        self._num(self.rf_sparse_step),
                    "sparse_presentations":
                        int(self.rf_sparse_trials.get()),
                    "sparse_polarity":
                        self.rf_sparse_polarity.get(),
                    "sparse_seed":
                        int(self.rf_sparse_seed.get()),
                },

                "chirp_intensity": {
                    "temporal_mode":
                        self.temporal_mode.get(),
                    "basal_s":
                        self._num(
                            self.ch_basal
                        ),
                    "step_on_s":
                        self._num(
                            self.ch_step_on
                        ),
                    "step_off_s":
                        self._num(
                            self.ch_step_off
                        ),
                    "pause1_s":
                        self._num(
                            self.ch_pause1
                        ),
                    "frequency_min_hz":
                        self._num(
                            self.ch_fmin
                        ),
                    "frequency_max_hz":
                        self._num(
                            self.ch_fmax
                        ),
                    "frequency_duration_s":
                        self._num(
                            self.ch_fdur
                        ),
                    "pause2_s":
                        self._num(
                            self.ch_pause2
                        ),
                    "intensity_frequency_hz":
                        self._num(
                            self.ch_ifreq
                        ),
                    "intensity_duration_s":
                        self._num(
                            self.ch_idur
                        ),
                    "intensity_start_pct":
                        self._num(
                            self.ch_istart
                        ),
                    "temporal_duration_s":
                        self._num(
                            self.ch_temp_duration
                        ),
                    "temporal_frequency_hz":
                        self._num(
                            self.ch_temp_frequency
                        ),
                    "temporal_contrast_pct":
                        self._num(
                            self.ch_temp_contrast
                        ),
                    "temporal_update_hz":
                        self._num(
                            self.ch_temp_update_hz
                        ),
                    "temporal_seed":
                        int(
                            self.ch_temp_seed.get()
                        ),
                    "contrast_levels_pct":
                        self.ch_contrast_levels.get(),
                    "contrast_isi_s":
                        self._num(
                            self.ch_contrast_isi
                        ),
                    "on_hex":
                        self.ch_on_hex.get(),
                    "off_hex":
                        self.ch_off_hex.get(),
                    "final_gray_s":
                        self._num(
                            self.ch_final
                        ),
                    "repetitions":
                        int(
                            self.ch_reps.get()
                        ),
                },
            }

            CONFIG_FILE.write_text(
                json.dumps(
                    data,
                    indent=2,
                    ensure_ascii=False
                ),
                encoding="utf-8"
            )
            self.saved = data
            self.status.set(
                f"Configuración guardada en "
                f"{CONFIG_FILE}"
            )

        except Exception as exc:
            messagebox.showerror(
                "Configuración", str(exc)
            )

    # --------------------------------------------------------------
    # Safety background process
    # --------------------------------------------------------------

    def make_safety_file(self):
        if (
            self.safety_file is not None
            and self.safety_file.exists()
        ):
            return

        tmp = tempfile.NamedTemporaryFile(
            mode="w",
            suffix="_mea_safety.py",
            delete=False,
            encoding="utf-8"
        )
        tmp.write(SAFETY_SOURCE_EMBEDDED)
        tmp.close()
        self.safety_file = Path(tmp.name)

    def start_safety_window(self):
        if (
            self.safety_process is not None
            and self.safety_process.poll() is None
        ):
            return

        cfg = self.common_config()
        self.make_safety_file()

        self.safety_process = subprocess.Popen(
            [
                sys.executable,
                str(self.safety_file),
                str(cfg["screen_x"]),
                str(cfg["screen_y"]),
                str(cfg["screen_width_px"]),
                str(cfg["screen_height_px"]),
                rgb255_to_hex(self._current_idle_rgb()),
            ],
            stdout=subprocess.DEVNULL,
            stderr=subprocess.DEVNULL
        )

        self.status.set(
            "Capa de seguridad activa."
        )

    def stop_safety_window(self):
        if (
            self.safety_process is not None
            and self.safety_process.poll() is None
        ):
            try:
                self.safety_process.terminate()
                self.safety_process.wait(
                    timeout=1.0
                )
            except Exception:
                try:
                    self.safety_process.kill()
                except Exception:
                    pass
        self.safety_process = None

    # --------------------------------------------------------------
    # Persistent PsychoPy worker
    # --------------------------------------------------------------
    def make_worker_file(self):
        if (
            self.worker_file is not None
            and self.worker_file.exists()
        ):
            return

        tmp = tempfile.NamedTemporaryFile(
            mode="w",
            suffix="_mea_persistent_worker.py",
            delete=False,
            encoding="utf-8"
        )
        tmp.write(WORKER_SOURCE_EMBEDDED)
        tmp.close()
        self.worker_file = Path(tmp.name)

    def create_abort_map(self):
        self.close_abort_map()

        self.abort_path.write_bytes(b"\x00")
        self.abort_file_handle = (
            self.abort_path.open("r+b")
        )
        self.abort_map = mmap.mmap(
            self.abort_file_handle.fileno(), 1
        )

    def close_abort_map(self):
        try:
            if self.abort_map is not None:
                self.abort_map.close()
        except Exception:
            pass

        try:
            if self.abort_file_handle is not None:
                self.abort_file_handle.close()
        except Exception:
            pass

        self.abort_map = None
        self.abort_file_handle = None

    def clear_abort(self):
        if self.abort_map is not None:
            self.abort_map[0] = 0
            self.abort_map.flush()

    def set_abort(self):
        if self.abort_map is not None:
            self.abort_map[0] = 1
            self.abort_map.flush()

    def start_display_worker(self):
        if (
            self.worker_process is not None
            and self.worker_process.poll() is None
        ):
            return

        try:
            startup = self.common_config()
        except Exception as exc:
            self.status.set(
                f"No se pudo abrir PsychoPy: {exc}"
            )
            self._splash_show_error(
                f"No se pudo abrir PsychoPy: {exc}"
            )
            return

        self.make_worker_file()
        self.create_abort_map()

        try:
            self._write_protocol_pause_request(
                False,
                command_id=""
            )
        except Exception:
            pass

        for path in (
            self.status_path,
            self.command_path,
            self.result_path
        ):
            try:
                path.unlink(missing_ok=True)
            except Exception:
                pass

        write_json_atomic(
            self.startup_config_path,
            startup
        )

        console_path = (
            RUN_DIR / "worker_console.txt"
        )
        self.worker_console = (
            console_path.open(
                "w",
                encoding="utf-8",
                buffering=1
            )
        )

        self.intentional_worker_stop = False

        self.worker_process = subprocess.Popen(
            [
                sys.executable,
                str(self.worker_file),
                str(self.startup_config_path),
                str(self.command_path),
                str(self.status_path),
                str(self.result_path),
                str(LOG_DIR),
                str(self.abort_path),
            ],
            stdout=self.worker_console,
            stderr=subprocess.STDOUT
        )

        self.current_worker_screen = (
            startup["screen_index"]
        )
        self.status.set(
            "Capa segura activa · abriendo PsychoPy..."
        )
        self._splash_update(
            "Iniciando PsychoPy...",
            "Abriendo la ventana experimental sobre la capa segura"
        )
        self.start_btn.configure(
            state="disabled"
        )
        self._update_protocol_execute_state()

    def stop_display_worker(
        self, intentional=True
    ):
        if self.worker_process is not None:
            if (
                self.worker_process.poll() is None
                and intentional
            ):
                # Let an active/paused run execute its FTDI cleanup before shutdown.
                self.set_abort()
                try:
                    cmd_id = (
                        "shutdown_"
                        + datetime.now().strftime(
                            "%Y%m%d_%H%M%S_%f"
                        )
                    )
                    write_json_atomic(
                        self.command_path,
                        {
                            "command_id": cmd_id,
                            "action": "shutdown"
                        }
                    )
                    self.worker_process.wait(
                        timeout=1.0
                    )
                except Exception:
                    try:
                        self.worker_process.terminate()
                        self.worker_process.wait(
                            timeout=1.0
                        )
                    except Exception:
                        try:
                            self.worker_process.kill()
                        except Exception:
                            pass

        try:
            if self.worker_console:
                self.worker_console.close()
        except Exception:
            pass

        self.worker_console = None
        self.worker_process = None
        self.current_worker_screen = None
        self.close_abort_map()

        self.protocol_pause_state = "idle"
        self.active_run_type = None
        self._update_protocol_pause_button()
        self._update_protocol_execute_state()

    # --------------------------------------------------------------
    # Display stack lifecycle
    # --------------------------------------------------------------
    def initialize_display_stack(self):
        try:
            self._splash_update(
                "Activando pantalla de seguridad...",
                "El monitor experimental se mantiene en estado seguro"
            )

            self.start_safety_window()

            self._splash_update(
                "Pantalla de seguridad activa",
                "Iniciando el motor PsychoPy"
            )

            self.after(
                350,
                self.start_display_worker
            )
        except Exception as exc:
            self._splash_show_error(
                f"Pantalla experimental: {exc}"
            )

    def restart_display_stack(self):
        # Explicit operation only. Safety layer masks the transition.
        self.pending_config = None
        self.active_command_id = None
        self.start_btn.configure(
            state="disabled"
        )
        self.abort_btn.configure(
            state="disabled"
        )

        self.stop_display_worker(
            intentional=True
        )
        self.stop_safety_window()

        self.after(
            100,
            self.initialize_display_stack
        )

    def on_screen_changed(self, event=None):
        # Changing monitor necessarily requires recreating PsychoPy,
        # but the safety layer is created first.
        self.restart_display_stack()

    # --------------------------------------------------------------
    # Stimulus commands
    # --------------------------------------------------------------
    def send_run_command(self, cfg):
        self._suspend_red_filter_for_stimulus()
        command_id = datetime.now().strftime(
            "%Y%m%d_%H%M%S_%f"
        )
        self.active_command_id = command_id
        self.active_run_type = str(
            cfg.get(
                "stimulus_type",
                ""
            )
        )
        self.clear_abort()

        try:
            self._write_protocol_pause_request(
                False,
                command_id=command_id
            )
        except Exception:
            pass

        if self.active_run_type == "protocol":
            self.protocol_pause_state = "running"
        else:
            self.protocol_pause_state = "idle"
        self._update_protocol_pause_button()

        write_json_atomic(
            self.command_path,
            {
                "command_id": command_id,
                "action": "run",
                "config": cfg
            }
        )

        self.start_btn.configure(
            state="disabled"
        )
        self.abort_btn.configure(
            state="normal"
        )
        self.status.set(
            "Orden de estímulo enviada..."
        )

    def start_test(self):
        if self._trigger_test_active:
            messagebox.showinfo(
                "Prueba de triggers activa",
                "Termina o detén la prueba de triggers antes de ejecutar el estímulo.",
                parent=self,
            )
            return
        try:
            cfg = self.collect_stimulus_config()

            if (
                cfg.get("stimulus_type") == "dense_noise"
                and self.current_measured_hz is not None
            ):
                requested_hz = float(
                    cfg["dense_update_hz"]
                )
                if requested_hz > self.current_measured_hz:
                    frames_per_update = max(
                        1,
                        int(round(
                            self.current_measured_hz
                            / requested_hz
                        ))
                    )
                    actual_hz = (
                        self.current_measured_hz
                        / frames_per_update
                    )

                    messagebox.showwarning(
                        "Noise · frecuencia limitada",
                        (
                            f"Has solicitado {requested_hz:.3f} Hz de "
                            "actualización, pero la pantalla refresca a "
                            f"{self.current_measured_hz:.3f} Hz.\n\n"
                            "No es posible presentar más de un patrón "
                            "nuevo por frame. El estímulo continuará "
                            f"aproximadamente a {actual_hz:.3f} Hz y "
                            "la frecuencia real quedará registrada en "
                            "los archivos de la sesión."
                        )
                    )

            self.save_current()
        except Exception as exc:
            messagebox.showerror(
                "No se puede iniciar",
                str(exc)
            )
            return

        worker_ready = self._worker_ready_for_screen(
            cfg["screen_index"]
        )

        if not worker_ready:
            messagebox.showwarning(
                "Pantalla no preparada",
                "La ventana PsychoPy no está preparada. "
                "La capa de seguridad permanece activa. "
                "Usa «Reiniciar pantalla» antes de presentar "
                "un estímulo."
            )
            return

        self.send_run_command(cfg)

    def abort_test(self):
        if self._trigger_test_active:
            self._finish_trigger_test(cancelled=True)
            return
        # Crucially: do NOT close/reopen PsychoPy.
        if (
            self.worker_process is not None
            and self.worker_process.poll() is None
        ):
            try:
                self._write_protocol_pause_request(
                    False
                )
            except Exception:
                pass
            self.set_abort()
            self.status.set(
                "Abortando estímulo → retorno a estado seguro..."
            )

    # --------------------------------------------------------------
    # Monitoring
    # --------------------------------------------------------------
    def poll_worker(self):
        try:
            # Safety process failure: recreate only the backing layer.
            if (
                self.safety_process is not None
                and self.safety_process.poll() is not None
            ):
                self.safety_process = None
                try:
                    self.start_safety_window()
                except Exception:
                    pass

            # PsychoPy failure: DO NOT auto-restart.
            # Safety layer remains visible.
            if (
                self.worker_process is not None
                and self.worker_process.poll() is not None
            ):
                console = ""
                try:
                    console_path = (
                        RUN_DIR /
                        "worker_console.txt"
                    )
                    if console_path.exists():
                        console = (
                            console_path.read_text(
                                encoding="utf-8",
                                errors="replace"
                            )
                        )
                except Exception:
                    pass

                self.stop_display_worker(
                    intentional=False
                )
                self._resume_red_filter_after_stimulus()
                self.active_command_id = None
                self.active_run_type = None
                self.protocol_pause_state = "idle"
                self._update_protocol_pause_button()
                self.pending_config = None
                self.start_btn.configure(
                    state="disabled"
                )
                self.abort_btn.configure(
                    state="disabled"
                )
                self.status.set(
                    "Motor PsychoPy detenido. "
                    "La capa de seguridad mantiene el monitor "
                    "en estado seguro. Pulsa «Reiniciar pantalla»."
                )

                if self.splash is not None:
                    self._splash_show_error(
                        "Motor PsychoPy detenido. La capa segura de "
                        "seguridad permanece activa. Puedes abrir la "
                        "interfaz y usar «Reiniciar pantalla»."
                    )

            if self.status_path.exists():
                try:
                    st = json.loads(
                        self.status_path.read_text(
                            encoding="utf-8"
                        )
                    )
                except Exception:
                    st = None

                if st:
                    signature = (
                        st.get("state"),
                        st.get("message"),
                        st.get("last_command_id")
                    )

                    if (
                        signature
                        != self.last_status_signature
                    ):
                        self.status.set(
                            st.get(
                                "message", "Preparado."
                            )
                        )
                        self.last_status_signature = (
                            signature
                        )

                    if "hz" in st:
                        self.current_measured_hz = float(
                            st["hz"]
                        )
                        self.refresh_info.set(
                            "Refresco medido: "
                            f"{self.current_measured_hz:.6f} Hz"
                        )

                    if st.get("state") == "ready":
                        self._resume_red_filter_after_stimulus()
                        self.start_btn.configure(
                            state=(
                                "disabled"
                                if (self._current_tab_is_protocols() or self._current_tab_is_system())
                                else "normal"
                            )
                        )
                        self._update_protocol_execute_state()

                        # On initial startup the splash closes only after
                        # PsychoPy reports a stable, measured refresh.
                        if self.splash is not None:
                            self._request_splash_close(
                                st.get("hz")
                            )

                        if st.get(
                            "last_command_id"
                        ) == self.active_command_id:
                            result = None
                            if self.result_path.exists():
                                try:
                                    result = json.loads(
                                        self.result_path.read_text(
                                            encoding="utf-8"
                                        )
                                    )
                                except Exception:
                                    pass

                            if result:
                                if result.get(
                                    "run_folder"
                                ):
                                    self._set_last_run_folder(
                                        result.get(
                                            "run_folder"
                                        )
                                    )

                                if result.get("ok"):
                                    if (
                                        result.get("stimulus_type")
                                        == "protocol"
                                    ):
                                        pname = result.get(
                                            "protocol_name",
                                            "Protocolo"
                                        )
                                        pver = result.get(
                                            "protocol_version",
                                            ""
                                        )
                                        self.timing_info.set(
                                            "Último protocolo · "
                                            f"{pname}"
                                            + (
                                                f" v{pver}"
                                                if pver != ""
                                                else ""
                                            )
                                            + " · "
                                            f"frames: "
                                            f"{result['total_frames_logged']} · "
                                            f"posibles perdidos: "
                                            f"{result['possible_dropped_frames']} · "
                                            f"TTL: {result.get('ttl_pulses', 0)} "
                                            f"pulsos/marcadores TTL"
                                        )
                                        self.protocol_builder_status.set(
                                            (
                                                f"Protocolo completado · "
                                                f"{pname}"
                                                + (
                                                    f" v{pver}"
                                                    if pver != ""
                                                    else ""
                                                )
                                                + " · carpeta de prueba + CSV + TTL + configuración guardados."
                                            )
                                        )
                                    else:
                                        self.timing_info.set(
                                            "Último estímulo · "
                                            f"{result['stimulus_type']} · "
                                            f"frames: "
                                            f"{result['total_frames_logged']} · "
                                            f"posibles perdidos: "
                                            f"{result['possible_dropped_frames']} · "
                                            f"TTL: {result.get('ttl_pulses', 0)} "
                                            f"pulsos/marcadores TTL"
                                        )
                                elif result.get("aborted"):
                                    if (
                                        result.get("stimulus_type")
                                        == "protocol"
                                    ):
                                        self.timing_info.set(
                                            "Último protocolo abortado "
                                            "con retorno al estado seguro."
                                        )
                                        self.protocol_builder_status.set(
                                            "Protocolo abortado · espera segura estable."
                                        )
                                    else:
                                        self.timing_info.set(
                                            "Último estímulo abortado "
                                            "con retorno al estado seguro."
                                        )
                                else:
                                    messagebox.showerror(
                                        "Error en estímulo",
                                        result.get(
                                            "error",
                                            "Error desconocido"
                                        )
                                    )

                            self.active_command_id = None
                            self.active_run_type = None
                            self.protocol_pause_state = "idle"
                            self._update_protocol_pause_button()

                            try:
                                self._write_protocol_pause_request(
                                    False,
                                    command_id=""
                                )
                            except Exception:
                                pass

                            self.abort_btn.configure(
                                state="disabled"
                            )

                    elif st.get("state") == "protocol_paused":
                        self.start_btn.configure(
                            state="disabled"
                        )
                        self.abort_btn.configure(
                            state="normal"
                        )
                        if hasattr(
                            self,
                            "protocol_execute_btn"
                        ):
                            self.protocol_execute_btn.configure(
                                state="disabled"
                            )

                        if (
                            self.active_run_type == "protocol"
                            and st.get("command_id")
                            == self.active_command_id
                        ):
                            self.protocol_pause_state = "paused"
                            self._update_protocol_pause_button()
                            self.protocol_builder_status.set(
                                (
                                    "PROTOCOLO EN PAUSA · fondo de adaptación "
                                    "estable · siguiente: paso "
                                    f"{st.get('protocol_step', '')}/"
                                    f"{st.get('protocol_total_steps', '')} · "
                                    f"{st.get('next_step_name', 'siguiente paso')}. "
                                    "Pulsa «Reanudar protocolo» para continuar."
                                )
                            )

                    elif st.get("state") == "running":
                        self.start_btn.configure(
                            state="disabled"
                        )
                        self.abort_btn.configure(
                            state="normal"
                        )
                        if hasattr(
                            self,
                            "protocol_execute_btn"
                        ):
                            self.protocol_execute_btn.configure(
                                state="disabled"
                            )

                        if (
                            self.active_run_type == "protocol"
                            and st.get("command_id")
                            == self.active_command_id
                        ):
                            if self.protocol_pause_state == "resuming":
                                self.protocol_pause_state = "running"
                                self.protocol_builder_status.set(
                                    "Protocolo reanudado · ejecución en curso."
                                )
                            elif self.protocol_pause_state not in (
                                "pause_requested",
                                "paused",
                            ):
                                self.protocol_pause_state = "running"

                            self._update_protocol_pause_button()

                    elif st.get("state") in (
                        "opening", "measuring"
                    ):
                        self.start_btn.configure(
                            state="disabled"
                        )

                        if st.get("state") == "opening":
                            self._splash_update(
                                "Abriendo PsychoPy...",
                                (
                                    "Preparando la ventana de estimulación "
                                    "en estado seguro"
                                )
                            )
                        else:
                            self._splash_update(
                                "Midiendo frecuencia de refresco...",
                                (
                                    "PsychoPy está comprobando la "
                                    "temporización real del monitor"
                                )
                            )


                    elif st.get("state") == "fatal":
                        self.start_btn.configure(
                            state="disabled"
                        )
                        self.abort_btn.configure(
                            state="disabled"
                        )

                        if self.splash is not None:
                            self._splash_show_error(
                                st.get(
                                    "message",
                                    "Motor PsychoPy detenido."
                                )
                            )

        finally:
            self.after(
                120,
                self.poll_worker
            )

    def on_close(self):
        self._restore_global_gamma_safely()
        if self._trigger_test_active:
            self._finish_trigger_test(cancelled=True, silent=True)
        self.active_command_id = None
        self.pending_config = None

        self._stop_easter_egg(
            redraw=False
        )

        try:
            if self.splash is not None:
                try:
                    self.splash_progress.stop()
                except Exception:
                    pass
                try:
                    self.splash.destroy()
                except Exception:
                    pass
                self.splash = None
        except Exception:
            pass

        for aid in self.ttl_test_after_ids:
            try:
                self.after_cancel(aid)
            except Exception:
                pass
        self.ttl_test_after_ids = []

        # Keep the safety layer until PsychoPy is fully stopped.
        self.stop_display_worker(
            intentional=True
        )
        self.stop_safety_window()

        for path in (
            self.worker_file,
            self.safety_file
        ):
            try:
                if (
                    path is not None
                    and path.exists()
                ):
                    path.unlink(missing_ok=True)
            except Exception:
                pass

        try:
            self.abort_path.unlink(
                missing_ok=True
            )
        except Exception:
            pass

        try:
            self.protocol_pause_path.unlink(
                missing_ok=True
            )
        except Exception:
            pass

        self.destroy()

SAFETY_SOURCE_EMBEDDED = 'import sys\nimport tkinter as tk\n\nx = int(sys.argv[1])\ny = int(sys.argv[2])\nw = int(sys.argv[3])\nh = int(sys.argv[4])\ncolor = str(sys.argv[5]) if len(sys.argv) > 5 else "#000000"\n\nroot = tk.Tk()\nroot.configure(bg=color)\nroot.overrideredirect(True)\nroot.geometry(f"{w}x{h}+0+0")\nroot.update_idletasks()\nroot.update()\n\ntry:\n    import ctypes\n\n    user32 = ctypes.windll.user32\n    user32.GetParent.argtypes = [ctypes.c_void_p]\n    user32.GetParent.restype = ctypes.c_void_p\n    user32.SetWindowPos.argtypes = [\n        ctypes.c_void_p, ctypes.c_void_p,\n        ctypes.c_int, ctypes.c_int, ctypes.c_int, ctypes.c_int,\n        ctypes.c_uint\n    ]\n    user32.SetWindowPos.restype = ctypes.c_int\n\n    # Tkinter\'s winfo_id() can refer to the client child window on Windows.\n    # GetParent gives us the real top-level HWND which can be positioned at\n    # negative virtual-desktop coordinates such as x=-1440.\n    client_hwnd = ctypes.c_void_p(root.winfo_id())\n    top_hwnd = user32.GetParent(client_hwnd)\n    if not top_hwnd:\n        top_hwnd = client_hwnd\n\n    HWND_NOTOPMOST = ctypes.c_void_p(-2 & 0xFFFFFFFFFFFFFFFF)\n    SWP_NOACTIVATE = 0x0010\n    SWP_SHOWWINDOW = 0x0040\n\n    ok = user32.SetWindowPos(\n        top_hwnd,\n        HWND_NOTOPMOST,\n        x, y, w, h,\n        SWP_NOACTIVATE | SWP_SHOWWINDOW\n    )\n\n    if not ok:\n        raise OSError("SetWindowPos no pudo situar la capa de seguridad.")\n\nexcept Exception:\n    # The safety process remains useful as a safe backing layer even if the\n    # native placement fails, but on Windows the block above is the intended path.\n    pass\n\nroot.mainloop()'
WORKER_SOURCE_EMBEDDED = 'import time as _ftdi_time\n\n\nclass MaxOneFTDI:\n    """Shared FTDI backend for the operator diagnostic and display worker.\n\n    Logical signals use the validated inverted VisExpMan polarity.\n    USB/driver latency is measured by the experiment, not assumed zero.\n    """\n\n    FRAME_PULSE_MIN_S = 0.001\n\n    def __init__(self, port="auto"):\n        self.requested_port = str(port or "auto").strip() or "auto"\n        self.resolved_port = ""\n        self.serial = None\n        self._frame_high_at = None\n\n    def open(self):\n        if self.serial is not None:\n            raise RuntimeError("El puerto FTDI ya está abierto.")\n        try:\n            import serial\n        except ImportError as exc:\n            raise RuntimeError(\n                "MaxOne Legacy FTDI requiere PySerial. Instálalo en el "\n                "entorno de este notebook: %pip install pyserial"\n            ) from exc\n\n        port = self.requested_port\n        if port.lower() in ("auto", "automatico", "automático"):\n            from serial.tools import list_ports\n            matches = [\n                p for p in list_ports.comports()\n                if getattr(p, "vid", None) == 0x0403\n                and getattr(p, "pid", None) == 0x6001\n            ]\n            if not matches:\n                raise RuntimeError("No se encuentra el FTDI MaxOne (0403:6001).")\n            if len(matches) != 1:\n                devices = ", ".join(str(p.device) for p in matches)\n                raise RuntimeError(\n                    f"Hay varios FTDI 0403:6001 ({devices}). "\n                    "Indica el puerto del montaje en el campo Puerto."\n                )\n            port = str(matches[0].device)\n\n        try:\n            # Configure the requested rest state before opening where the\n            # driver permits it; reapply both outputs immediately afterwards.\n            self.serial = serial.Serial(\n                port=None, baudrate=9600, timeout=0, write_timeout=0,\n                rtscts=False, dsrdtr=False, xonxoff=False,\n            )\n            self.serial.rts = True\n            self.serial.break_condition = True\n            self.serial.port = port\n            self.serial.open()\n            self.resolved_port = port\n            self.idle(strict=True)\n        except Exception as exc:\n            self.close()\n            raise RuntimeError(\n                f"No se pudo abrir/configurar {port}: {type(exc).__name__}: {exc}"\n            ) from exc\n        return self.resolved_port\n\n    def _set(self, signal, logical):\n        if self.serial is None:\n            raise RuntimeError("El backend MaxOne FTDI no está abierto.")\n        physical = not bool(logical)\n        if signal == "frame":\n            self.serial.rts = physical\n        else:\n            self.serial.break_condition = physical\n\n    def set_frame(self, logical):\n        self._set("frame", logical)\n        self._frame_high_at = _ftdi_time.perf_counter() if logical else None\n\n    def set_event(self, logical):\n        self._set("event", logical)\n\n    def frame_high(self):\n        self.set_frame(1)\n\n    def frame_low(self):\n        # Keep the requested RTS HIGH for at least 1 ms after the driver call.\n        # A bounded short wait avoids scheduling a Timer across later frames.\n        # Physical width can be longer; validate it in /bits on the real setup.\n        if self._frame_high_at is not None:\n            deadline = self._frame_high_at + self.FRAME_PULSE_MIN_S\n            while _ftdi_time.perf_counter() < deadline:\n                pass\n        self.set_frame(0)\n\n    def idle(self, strict=False):\n        errors = []\n        if self.serial is not None:\n            # Attempt BOTH lines even if one write fails.\n            for name, action in (("FRAME", self.set_frame), ("EVENT", self.set_event)):\n                try:\n                    action(0)\n                except Exception as exc:\n                    errors.append(f"{name}: {type(exc).__name__}: {exc}")\n        self._frame_high_at = None\n        if errors and strict:\n            raise RuntimeError("; ".join(errors))\n        return errors\n\n    def close(self):\n        errors = self.idle()\n        ser, self.serial = self.serial, None\n        if ser is not None:\n            try:\n                ser.close()\n            except Exception as exc:\n                errors.append(f"Cerrar FTDI: {type(exc).__name__}: {exc}")\n        return errors\n\n\nclass GenericTTLFTDI:\n    """Shared FTDI backend for the generic trigger box and display worker.\n\n    Validated hardware mapping (active-low FTDI control-line polarity):\n      UNIVERSAL / FRAME_SYNC -> RTS#  (ser.rts = False)\n      AUX EVENT              -> TX/BREAK (break_condition = False)\n\n    Encoding on UNIVERSAL is generated in software:\n      visual flip -> FRAME pulse on RTS\n      condition onset -> RTS EVENT pulse ~3 ms later + simultaneous BREAK EVENT\n    The EVENT replication onto RTS uses _set() directly so it never overwrites\n    the visual FRAME anchor used for timing and logging.\n    """\n\n    FRAME_PULSE_MIN_S = 0.001\n    EVENT_PULSE_MIN_S = 0.001\n    EVENT_OFFSET_S = 0.003\n\n    def __init__(self, port="auto"):\n        self.requested_port = str(port or "auto").strip() or "auto"\n        self.resolved_port = ""\n        self.serial = None\n        self._frame_high_at = None\n        self._last_frame_anchor = None\n        self._event_high_at = None\n\n    @staticmethod\n    def _wait_until(deadline):\n        """Hybrid sleep/spin wait, matching the validated UNIVERSAL tester."""\n        while True:\n            remaining = float(deadline) - _ftdi_time.perf_counter()\n            if remaining <= 0:\n                return\n            if remaining > 0.002:\n                _ftdi_time.sleep(max(0.0, remaining - 0.001))\n            # Final sub-millisecond section intentionally busy-waits.\n\n    def open(self):\n        if self.serial is not None:\n            raise RuntimeError("El puerto FTDI ya está abierto.")\n        try:\n            import serial\n        except ImportError as exc:\n            raise RuntimeError(\n                "Trigger Box TTL genérica requiere PySerial. Instálalo en el "\n                "entorno de este notebook: %pip install pyserial"\n            ) from exc\n\n        port = self.requested_port\n        if port.lower() in ("auto", "automatico", "automático"):\n            from serial.tools import list_ports\n            matches = [\n                p for p in list_ports.comports()\n                if getattr(p, "vid", None) == 0x0403\n                and getattr(p, "pid", None) == 0x6001\n            ]\n            if not matches:\n                raise RuntimeError("No se encuentra el FTDI de la Trigger Box (0403:6001).")\n            if len(matches) != 1:\n                devices = ", ".join(str(p.device) for p in matches)\n                raise RuntimeError(\n                    f"Hay varios FTDI 0403:6001 ({devices}). "\n                    "Indica manualmente el puerto de la Trigger Box."\n                )\n            port = str(matches[0].device)\n\n        try:\n            self.serial = serial.Serial(\n                port=None, baudrate=9600, timeout=0, write_timeout=0,\n                rtscts=False, dsrdtr=False, xonxoff=False,\n            )\n            # Validated idle polarity: both FTDI lines physically inactive.\n            self.serial.rts = True\n            self.serial.break_condition = True\n            self.serial.port = port\n            self.serial.open()\n            self.resolved_port = port\n            self.idle(strict=True)\n        except Exception as exc:\n            self.close()\n            raise RuntimeError(\n                f"No se pudo abrir/configurar {port}: {type(exc).__name__}: {exc}"\n            ) from exc\n        return self.resolved_port\n\n    def _set(self, signal, logical):\n        if self.serial is None:\n            raise RuntimeError("El backend FTDI genérico no está abierto.")\n        physical = not bool(logical)\n        if signal == "frame":\n            self.serial.rts = physical\n        elif signal == "event":\n            self.serial.break_condition = physical\n        else:\n            raise ValueError(f"Señal FTDI desconocida: {signal}")\n\n    def set_frame(self, logical):\n        self._set("frame", logical)\n        now = _ftdi_time.perf_counter()\n        if logical:\n            self._frame_high_at = now\n            self._last_frame_anchor = now\n        else:\n            self._frame_high_at = None\n        return now\n\n    def set_event(self, logical):\n        self._set("event", logical)\n        now = _ftdi_time.perf_counter()\n        self._event_high_at = now if logical else None\n        return now\n\n    def frame_high(self):\n        return self.set_frame(1)\n\n    def frame_low(self):\n        # Keep FRAME HIGH for at least 1 ms. The validated hardware produced\n        # ~1.1-1.35 ms physical pulses with this requested width.\n        if self._frame_high_at is not None:\n            self._wait_until(self._frame_high_at + self.FRAME_PULSE_MIN_S)\n        return self.set_frame(0)\n\n    def _event_pair_high(self):\n        """Assert EVENT on AUX EVENT and replicate it onto UNIVERSAL."""\n        if self.serial is None:\n            raise RuntimeError("El backend FTDI genérico no está abierto.")\n        errors = []\n        # RTS carries UNIVERSAL. BREAK carries the independent AUX EVENT.\n        # Use _set() directly for RTS so the visual FRAME anchor is NOT changed.\n        for signal in ("frame", "event"):\n            try:\n                self._set(signal, 1)\n            except Exception as exc:\n                errors.append(f"{signal}: {type(exc).__name__}: {exc}")\n        if errors:\n            # Best-effort fail-safe: release both outputs before propagating.\n            for signal in ("event", "frame"):\n                try:\n                    self._set(signal, 0)\n                except Exception:\n                    pass\n            raise RuntimeError("; ".join(errors))\n        now = _ftdi_time.perf_counter()\n        self._event_high_at = now\n        return now\n\n    def _event_pair_low(self):\n        """Release AUX EVENT and UNIVERSAL after an EVENT pulse."""\n        errors = []\n        # Release UNIVERSAL first, then AUX EVENT; both operations are attempted.\n        for signal in ("frame", "event"):\n            try:\n                self._set(signal, 0)\n            except Exception as exc:\n                errors.append(f"{signal}: {type(exc).__name__}: {exc}")\n        now = _ftdi_time.perf_counter()\n        self._event_high_at = None\n        if errors:\n            raise RuntimeError("; ".join(errors))\n        return now\n\n    def pulse_event_now(self, width_s=None):\n        """Generate EVENT simultaneously on UNIVERSAL (RTS) and AUX EVENT (BREAK)."""\n        width = self.EVENT_PULSE_MIN_S if width_s is None else max(\n            self.EVENT_PULSE_MIN_S, float(width_s)\n        )\n        high_at = self._event_pair_high()\n        try:\n            self._wait_until(high_at + width)\n        finally:\n            low_at = self._event_pair_low()\n        return {\n            "event_high_perf_s": high_at,\n            "event_low_perf_s": low_at,\n            "event_width_s": max(0.0, low_at - high_at),\n        }\n\n    def pulse_event_after_frame(self, offset_s=None, width_s=None):\n        """Emit EVENT as the second UNIVERSAL pulse after the last visual FRAME."""\n        if self._last_frame_anchor is None:\n            raise RuntimeError(\n                "No hay un FRAME_SYNC previo al que asociar el EVENT."\n            )\n        offset = self.EVENT_OFFSET_S if offset_s is None else max(0.0, float(offset_s))\n        width = self.EVENT_PULSE_MIN_S if width_s is None else max(\n            self.EVENT_PULSE_MIN_S, float(width_s)\n        )\n        frame_anchor = float(self._last_frame_anchor)\n        self._wait_until(frame_anchor + offset)\n\n        # IMPORTANT: this pulses RTS directly instead of set_frame(), so the\n        # original visual FRAME anchor remains unchanged for timing/logging.\n        high_at = self._event_pair_high()\n        try:\n            self._wait_until(high_at + width)\n        finally:\n            low_at = self._event_pair_low()\n\n        return {\n            "frame_anchor_perf_s": frame_anchor,\n            "event_high_perf_s": high_at,\n            "event_low_perf_s": low_at,\n            "event_offset_s": max(0.0, high_at - frame_anchor),\n            "event_width_s": max(0.0, low_at - high_at),\n        }\n\n    def pulse_universal_event_after_frame(self, aux_state, offset_s=None, width_s=None):\n        """Pulse only UNIVERSAL ~3 ms after FRAME while setting AUX EVENT state.\n\n        This is the v1.27 experimental encoding used by the display worker:\n        RTS/UNIVERSAL keeps the validated short EVENT pulse, whereas TX/BREAK\n        becomes a held condition-state channel. The visual FRAME anchor is never\n        overwritten.\n        """\n        if self._last_frame_anchor is None:\n            raise RuntimeError(\n                "No hay un FRAME_SYNC previo al que asociar el EVENT."\n            )\n        offset = self.EVENT_OFFSET_S if offset_s is None else max(0.0, float(offset_s))\n        width = self.EVENT_PULSE_MIN_S if width_s is None else max(\n            self.EVENT_PULSE_MIN_S, float(width_s)\n        )\n        frame_anchor = float(self._last_frame_anchor)\n        desired_aux = 1 if int(aux_state) else 0\n        self._wait_until(frame_anchor + offset)\n\n        universal_high_at = None\n        aux_transition_at = None\n        try:\n            # UNIVERSAL receives the precise second pulse; use _set() directly so\n            # the FRAME anchor remains the original visual flip.\n            self._set("frame", 1)\n            universal_high_at = _ftdi_time.perf_counter()\n            # AUX EVENT is a held state, changed at the same condition onset.\n            self._set("event", desired_aux)\n            aux_transition_at = _ftdi_time.perf_counter()\n            self._event_high_at = aux_transition_at if desired_aux else None\n            self._wait_until(universal_high_at + width)\n        finally:\n            # Release ONLY UNIVERSAL. AUX EVENT deliberately remains at its\n            # requested condition state until the next onset or explicit reset.\n            self._set("frame", 0)\n            universal_low_at = _ftdi_time.perf_counter()\n\n        return {\n            "frame_anchor_perf_s": frame_anchor,\n            "event_high_perf_s": universal_high_at,\n            "event_low_perf_s": universal_low_at,\n            "event_offset_s": max(0.0, universal_high_at - frame_anchor),\n            "event_width_s": max(0.0, universal_low_at - universal_high_at),\n            "aux_transition_perf_s": aux_transition_at,\n            "aux_state": desired_aux,\n        }\n\n    def pulse_aux_marker(self, width_s=None):\n        """Emit a short marker only on AUX EVENT (TX/BREAK).\n\n        UNIVERSAL/RTS is deliberately untouched. This is used for protocol\n        pause markers in v1.28. AUX always returns LOW when the pulse ends.\n        """\n        width = self.EVENT_PULSE_MIN_S if width_s is None else max(\n            self.EVENT_PULSE_MIN_S, float(width_s)\n        )\n        # Start from a defined LOW state, then generate a clean HIGH pulse.\n        self._set("event", 0)\n        self._event_high_at = None\n        self._set("event", 1)\n        high_at = _ftdi_time.perf_counter()\n        self._event_high_at = high_at\n        try:\n            self._wait_until(high_at + width)\n        finally:\n            self._set("event", 0)\n            low_at = _ftdi_time.perf_counter()\n            self._event_high_at = None\n        return {\n            "aux_high_perf_s": high_at,\n            "aux_low_perf_s": low_at,\n            "aux_width_s": max(0.0, low_at - high_at),\n        }\n\n    def idle(self, strict=False):\n        errors = []\n        if self.serial is not None:\n            # Attempt BOTH lines even if one write fails.\n            for name, action in (("FRAME", self.set_frame), ("EVENT", self.set_event)):\n                try:\n                    action(0)\n                except Exception as exc:\n                    errors.append(f"{name}: {type(exc).__name__}: {exc}")\n        self._frame_high_at = None\n        self._last_frame_anchor = None\n        self._event_high_at = None\n        if errors and strict:\n            raise RuntimeError("; ".join(errors))\n        return errors\n\n    def close(self):\n        errors = self.idle()\n        ser, self.serial = self.serial, None\n        if ser is not None:\n            try:\n                ser.close()\n            except Exception as exc:\n                errors.append(f"Cerrar FTDI: {type(exc).__name__}: {exc}")\n        return errors\n\n\nimport csv\nimport json\nimport math\nimport mmap\nimport os\nimport sys\nimport time\nimport traceback\nfrom datetime import datetime\nfrom pathlib import Path\n\nimport numpy as np\n\nGRAY = (0.0, 0.0, 0.0)\nWHITE = (1.0, 1.0, 1.0)\nBLACK = (-1.0, -1.0, -1.0)\n\nstartup_config_path = Path(sys.argv[1])\ncommand_path = Path(sys.argv[2])\nstatus_path = Path(sys.argv[3])\nresult_path = Path(sys.argv[4])\nlog_dir = Path(sys.argv[5])\nabort_path = Path(sys.argv[6])\nprotocol_pause_path = (\n    abort_path.with_name(\n        "protocol_pause.json"\n    )\n)\n\nlog_dir.mkdir(parents=True, exist_ok=True)\n\ndef write_json_atomic(path, data):\n    """\n    Atomic JSON write with protection against short-lived Windows file\n    locks (WinError 5/32), which can occur when another process briefly\n    reads or scans the destination file.\n\n    status.json is operational telemetry, not stimulus data. If Windows\n    keeps it locked after all retries, the visual experiment must continue\n    safely rather than aborting because of a GUI/status-file race.\n    """\n    path = Path(path)\n\n    # A process-specific temporary name avoids collisions with a stale\n    # status.json.tmp left behind by a previous interrupted process.\n    tmp = path.with_name(\n        path.name\n        + f".{os.getpid()}.tmp"\n    )\n\n    payload = json.dumps(\n        data,\n        indent=2,\n        ensure_ascii=False\n    )\n\n    tmp.write_text(\n        payload,\n        encoding="utf-8"\n    )\n\n    last_exc = None\n\n    for attempt in range(30):\n        try:\n            os.replace(\n                tmp,\n                path\n            )\n            return True\n\n        except PermissionError as exc:\n            last_exc = exc\n            time.sleep(\n                0.01\n                + 0.002 * attempt\n            )\n\n        except OSError as exc:\n            # Common transient Windows sharing/access errors:\n            # 5 = ACCESS_DENIED, 32 = SHARING_VIOLATION.\n            if getattr(\n                exc,\n                "winerror",\n                None\n            ) in (\n                5,\n                32\n            ):\n                last_exc = exc\n                time.sleep(\n                    0.01\n                    + 0.002 * attempt\n                )\n            else:\n                raise\n\n    # Clean the temporary file when possible.\n    try:\n        if tmp.exists():\n            tmp.unlink()\n    except Exception:\n        pass\n\n    # A status-file lock must never stop a running stimulus/protocol.\n    try:\n        is_status_path = (\n            path.resolve()\n            == status_path.resolve()\n        )\n    except Exception:\n        is_status_path = (\n            str(path)\n            == str(status_path)\n        )\n\n    if is_status_path:\n        return False\n\n    if last_exc is not None:\n        raise last_exc\n\n    raise OSError(\n        f"No se pudo guardar JSON: {path}"\n    )\n\ndef seconds_to_frames(seconds, hz):\n    seconds = float(seconds)\n    if seconds <= 0:\n        return 0\n    return max(1, int(round(seconds * hz)))\n\ndef parse_float_list(text):\n    values = []\n    for part in str(text).replace(";", ",").split(","):\n        part = part.strip()\n        if part:\n            values.append(float(part))\n    return values\n\ndef rgb255_to_psychopy(rgb):\n    return tuple((float(v) / 127.5) - 1.0 for v in rgb)\n\ndef get_visual_size_deg(width_cm, height_cm, distance_cm):\n    width_deg = math.degrees(\n        2.0 * math.atan(width_cm / (2.0 * distance_cm))\n    )\n    height_deg = math.degrees(\n        2.0 * math.atan(height_cm / (2.0 * distance_cm))\n    )\n    return width_deg, height_deg\n\ndef colored_grating_texture(color_a, color_b, waveform, contrast):\n    """\n    One-cycle RGB texture. At contrast=1 the extrema are exactly the\n    selected colors. At contrast=0 both converge to their midpoint.\n    """\n    a = np.asarray(rgb255_to_psychopy(color_a), dtype=np.float32)\n    b = np.asarray(rgb255_to_psychopy(color_b), dtype=np.float32)\n    midpoint = (a + b) / 2.0\n\n    a_eff = midpoint + float(contrast) * (a - midpoint)\n    b_eff = midpoint + float(contrast) * (b - midpoint)\n\n    res = 256\n    phase = np.linspace(0.0, 2.0 * np.pi, res, endpoint=False)\n\n    if str(waveform).lower() == "cuadrado":\n        weight = (np.sin(phase) >= 0.0).astype(np.float32)\n    else:\n        weight = ((np.sin(phase) + 1.0) / 2.0).astype(np.float32)\n\n    line = (\n        b_eff[None, :]\n        + weight[:, None] * (a_eff - b_eff)[None, :]\n    )\n\n    texture = np.empty((res, res, 3), dtype=np.float32)\n    texture[:] = line[None, :, :]\n    return np.ascontiguousarray(texture)\n\n\ndef asymmetric_grating_texture(\n    color_a, color_b, contrast, polarity="A_TO_B", res=256\n):\n    """\n    One-cycle asymmetric RGB texture.\n\n    A_TO_B: Color A starts at the reset edge and follows a smooth half-cosine\n    transition to Color B. The periodic texture boundary then jumps abruptly\n    back to A. B_TO_A is the exact inverse polarity.\n\n    The profile is sinusoidal in digital RGB space (not gamma-linearized\n    luminance unless the display has been calibrated separately).\n    """\n    a = np.asarray(rgb255_to_psychopy(color_a), dtype=np.float32)\n    b = np.asarray(rgb255_to_psychopy(color_b), dtype=np.float32)\n    midpoint = (a + b) / 2.0\n\n    a_eff = midpoint + float(contrast) * (a - midpoint)\n    b_eff = midpoint + float(contrast) * (b - midpoint)\n\n    u = (\n        np.arange(int(res), dtype=np.float32)\n        / float(int(res))\n    )\n    # 1 at u=0, approaches 0 at u->1; wrap is the abrupt reset.\n    weight = 0.5 * (1.0 + np.cos(np.pi * u))\n    if str(polarity).strip().upper() in (\n        "B_TO_A",\n        "B→A",\n        "B-A",\n    ):\n        weight = 1.0 - weight\n\n    line = (\n        b_eff[None, :]\n        + weight[:, None] * (a_eff - b_eff)[None, :]\n    )\n\n    texture = np.empty((int(res), int(res), 3), dtype=np.float32)\n    texture[:] = line[None, :, :]\n    return np.ascontiguousarray(texture)\n\n\ndef gaussian_mask_deg(size_deg, sigma_deg, res=256):\n    """\n    PsychoPy custom mask in the documented -1..1 range.\n    The Gaussian sigma is expressed in visual degrees.\n    """\n    size_deg = float(size_deg)\n    sigma_deg = float(sigma_deg)\n\n    coords = np.linspace(\n        -size_deg / 2.0,\n        size_deg / 2.0,\n        int(res),\n        endpoint=False,\n        dtype=np.float32\n    )\n    # Center samples in their bins.\n    coords += (\n        size_deg\n        / (2.0 * float(res))\n    )\n\n    xx, yy = np.meshgrid(\n        coords,\n        coords\n    )\n    alpha = np.exp(\n        -(\n            xx * xx\n            + yy * yy\n        )\n        / (\n            2.0\n            * sigma_deg\n            * sigma_deg\n        )\n    ).astype(\n        np.float32\n    )\n\n    mask = (\n        2.0 * alpha\n        - 1.0\n    )\n    return np.ascontiguousarray(\n        mask,\n        dtype=np.float32\n    )\n\nwin = None\nabort_file = None\nabort_map = None\n\ntry:\n    from psychopy import visual, monitors, event, core\n\n    startup = json.loads(\n        startup_config_path.read_text(encoding="utf-8")\n    )\n\n    screen_index = int(startup["screen_index"])\n    width_px = int(startup["screen_width_px"])\n    height_px = int(startup["screen_height_px"])\n    screen_x = int(startup.get("screen_x", 0))\n    screen_y = int(startup.get("screen_y", 0))\n    physical_width_cm = float(startup["physical_width_cm"])\n    physical_height_cm = float(startup["physical_height_cm"])\n    distance_cm = float(startup["distance_cm"])\n    GRAY = rgb255_to_psychopy(startup.get("adaptation_rgb", [128, 128, 128]))\n    IDLE_COLOR = rgb255_to_psychopy(startup.get("idle_rgb", [0, 0, 0]))\n\n    abort_file = abort_path.open("r+b")\n    abort_map = mmap.mmap(abort_file.fileno(), 1)\n\n    write_json_atomic(status_path, {\n        "state": "opening",\n        "message": "Abriendo pantalla PsychoPy..."\n    })\n\n    mon = monitors.Monitor(\n        name="MEA_Persistent_Stimulus_Display",\n        width=physical_width_cm,\n        distance=distance_cm\n    )\n    mon.setSizePix((width_px, height_px))\n\n    def keep_psychopy_on_stimulus_monitor():\n        """Keep PsychoPy above the safe background layer without stealing focus."""\n        try:\n            import ctypes\n            user32 = ctypes.windll.user32\n            user32.SetWindowPos.argtypes = [\n                ctypes.c_void_p, ctypes.c_void_p,\n                ctypes.c_int, ctypes.c_int, ctypes.c_int, ctypes.c_int,\n                ctypes.c_uint\n            ]\n            user32.SetWindowPos.restype = ctypes.c_int\n\n            hwnd_value = getattr(win.winHandle, "_hwnd", None)\n            if hwnd_value is None:\n                return\n\n            HWND_TOPMOST = ctypes.c_void_p(-1 & 0xFFFFFFFFFFFFFFFF)\n            SWP_NOMOVE = 0x0002\n            SWP_NOSIZE = 0x0001\n            SWP_NOACTIVATE = 0x0010\n            SWP_SHOWWINDOW = 0x0040\n\n            user32.SetWindowPos(\n                ctypes.c_void_p(hwnd_value),\n                HWND_TOPMOST,\n                0, 0, 0, 0,\n                SWP_NOMOVE | SWP_NOSIZE | SWP_NOACTIVATE | SWP_SHOWWINDOW\n            )\n        except Exception:\n            pass\n\n    win = visual.Window(\n        size=(width_px, height_px),\n        screen=screen_index,\n        fullscr=True,\n        allowGUI=False,\n        color=IDLE_COLOR,\n        colorSpace="rgb",\n        units="deg",\n        monitor=mon,\n        waitBlanking=True,\n        checkTiming=True,\n        winType="pyglet",\n        autoLog=False\n    )\n\n    # Keep the experimental window above the safety layer even when the\n    # operator clicks the GUI on the other monitor. Do not steal focus.\n    keep_psychopy_on_stimulus_monitor()\n\n    width_deg, height_deg = get_visual_size_deg(\n        physical_width_cm, physical_height_cm, distance_cm\n    )\n    cover_size = 2.2 * math.hypot(width_deg, height_deg)\n\n    fullfield = visual.Rect(\n        win=win,\n        width=cover_size,\n        height=cover_size,\n        units="deg",\n        fillColor=IDLE_COLOR,\n        lineColor=IDLE_COLOR,\n        colorSpace="rgb",\n        pos=(0, 0)\n    )\n\n    def show_solid(color, do_flip=True):\n        fullfield.fillColor = color\n        fullfield.lineColor = color\n        fullfield.draw()\n        if do_flip:\n            return win.flip()\n        return None\n\n    # Start and remain on the safe idle background (black by default).\n    for _ in range(30):\n        show_solid(IDLE_COLOR)\n\n    write_json_atomic(status_path, {\n        "state": "measuring",\n        "message": "Midiendo refresco de forma silenciosa..."\n    })\n\n    # v1.32: robust refresh calibration from the actual PsychoPy flips.\n    # getActualFrameRate() can report a spurious rate on mixed-refresh\n    # multi-monitor Windows systems (e.g. ~91 Hz while the stimulus screen\n    # is physically flipping at ~60 Hz).  Time the real blocking win.flip()\n    # calls with perf_counter instead, reject outliers, and use the median\n    # interval as the experimental frame period.\n    def measure_refresh_from_real_flips(\n        warmup_frames=45,\n        sample_frames=180\n    ):\n        for _ in range(int(warmup_frames)):\n            show_solid(IDLE_COLOR)\n\n        intervals = []\n        last = time.perf_counter()\n        for _ in range(int(sample_frames)):\n            show_solid(IDLE_COLOR)\n            now_perf = time.perf_counter()\n            dt = now_perf - last\n            last = now_perf\n            if math.isfinite(dt) and dt > 0:\n                intervals.append(float(dt))\n\n        if len(intervals) < 30:\n            raise RuntimeError(\n                "No se pudieron obtener suficientes intervalos de refresco reales."\n            )\n\n        arr = np.asarray(intervals, dtype=np.float64)\n        median0 = float(np.median(arr))\n        if not math.isfinite(median0) or median0 <= 0:\n            raise RuntimeError(\n                "La medición monotónica del refresco no produjo un intervalo válido."\n            )\n\n        # First remove gross half/double-frame outliers. Then use MAD for\n        # transient OS scheduling excursions without biasing the central rate.\n        coarse = arr[(arr >= 0.55 * median0) & (arr <= 1.80 * median0)]\n        if coarse.size < 20:\n            coarse = arr\n        med = float(np.median(coarse))\n        abs_dev = np.abs(coarse - med)\n        mad = float(np.median(abs_dev))\n        if mad > 0:\n            robust_sigma = 1.4826 * mad\n            fine = coarse[abs_dev <= max(4.0 * robust_sigma, 0.00035)]\n            if fine.size >= 20:\n                coarse = fine\n\n        interval_s = float(np.median(coarse))\n        hz_value = 1.0 / interval_s\n        spread_ms = 1000.0 * float(np.percentile(coarse, 95) - np.percentile(coarse, 5))\n\n        if (\n            not math.isfinite(hz_value)\n            or hz_value < 20.0\n            or hz_value > 500.0\n        ):\n            raise RuntimeError(\n                f"Frecuencia de refresco real no plausible: {hz_value:.3f} Hz."\n            )\n\n        return {\n            "hz": float(hz_value),\n            "interval_s": float(interval_s),\n            "n_raw": int(arr.size),\n            "n_used": int(coarse.size),\n            "p05_p95_spread_ms": float(spread_ms),\n        }\n\n    refresh_measurement = measure_refresh_from_real_flips()\n    hz = float(refresh_measurement["hz"])\n    expected_interval = float(refresh_measurement["interval_s"])\n\n    # Return to the safe idle image after the silent calibration.\n    show_solid(IDLE_COLOR)\n\n    write_json_atomic(status_path, {\n        "state": "ready",\n        "message": "Pantalla experimental estable · espera segura",\n        "hz": hz,\n        "screen_index": screen_index,\n        "resolution": [width_px, height_px]\n    })\n\n    last_command_id = None\n    last_idle_flip = time.perf_counter()\n\n    while True:\n        now = time.perf_counter()\n\n        # Refresh the safe idle image periodically to keep the pyglet\n        # window responsive without changing visual content.\n        if now - last_idle_flip >= 0.25:\n            show_solid(IDLE_COLOR)\n            keep_psychopy_on_stimulus_monitor()\n            last_idle_flip = now\n\n        command = None\n        if command_path.exists():\n            try:\n                command = json.loads(\n                    command_path.read_text(encoding="utf-8")\n                )\n            except Exception:\n                command = None\n\n        if not command or command.get("command_id") == last_command_id:\n            core.wait(0.01)\n            continue\n\n        command_id = command.get("command_id")\n        action = command.get("action")\n\n        if action == "shutdown":\n            show_solid(IDLE_COLOR)\n            break\n\n        if action == "set_idle_color":\n            try:\n                IDLE_COLOR = rgb255_to_psychopy(command.get("rgb", [0, 0, 0]))\n                show_solid(IDLE_COLOR)\n                keep_psychopy_on_stimulus_monitor()\n                write_json_atomic(status_path, {\n                    "state": "ready",\n                    "message": "Pantalla experimental estable · espera segura",\n                    "hz": hz,\n                    "screen_index": screen_index,\n                    "resolution": [width_px, height_px],\n                    "last_command_id": command_id,\n                })\n            finally:\n                last_command_id = command_id\n            core.wait(0.005)\n            continue\n\n        if action != "run":\n            last_command_id = command_id\n            core.wait(0.01)\n            continue\n\n        last_command_id = command_id\n        cfg = command["config"]\n        # v1.22: adaptation background is a run-level experimental setting;\n        # safe idle remains black; operator dark-room mode never changes this display.\n        GRAY = rgb255_to_psychopy(cfg.get("adaptation_rgb", [128, 128, 128]))\n        IDLE_COLOR = rgb255_to_psychopy(cfg.get("idle_rgb", [0, 0, 0]))\n        stimulus_type = cfg["stimulus_type"]\n        command_stimulus_type = stimulus_type\n\n        safe_run_prefix = stimulus_type\n        if stimulus_type == "protocol":\n            safe_protocol_name = "".join(\n                c\n                if (\n                    c.isalnum()\n                    or c in "-_"\n                )\n                else "_"\n                for c in str(\n                    cfg.get(\n                        "protocol_name",\n                        "protocol"\n                    )\n                )\n            ).strip("_")[:60]\n\n            if not safe_protocol_name:\n                safe_protocol_name = "protocol"\n\n            safe_run_prefix = (\n                f"protocol_"\n                f"{safe_protocol_name}_"\n                f"v{cfg.get(\'protocol_version\', 1)}"\n            )\n\n        run_stamp = datetime.now().strftime(\n            "%Y%m%d_%H%M%S"\n        )\n\n        # Historical base retained for filenames so CSV/TTL/JSON names\n        # remain fully compatible with previous versions.\n        run_base_name = (\n            f"{safe_run_prefix}_{run_stamp}"\n        )\n\n        # Date/time FIRST only for the containing run folder. This makes\n        # Windows Explorer sort experiment folders chronologically by name:\n        # YYYYMMDD_HHMMSS_stimulus-or-protocol\n        run_folder_name = (\n            f"{run_stamp}_{safe_run_prefix}"\n        )\n        run_dir = log_dir / run_folder_name\n        run_dir.mkdir(\n            parents=True,\n            exist_ok=True\n        )\n\n        # Clear abort flag without recreating the experimental window.\n        abort_map.seek(0)\n        abort_map.write(b"\\x00")\n        abort_map.flush()\n\n        # Geometry may be edited by the operator between stimuli.\n        physical_width_cm = float(cfg["physical_width_cm"])\n        physical_height_cm = float(cfg["physical_height_cm"])\n        distance_cm = float(cfg["distance_cm"])\n\n        try:\n            win.monitor.setWidth(physical_width_cm)\n            win.monitor.setDistance(distance_cm)\n            win.monitor.setSizePix((width_px, height_px))\n        except Exception:\n            pass\n\n        width_deg, height_deg = get_visual_size_deg(\n            physical_width_cm, physical_height_cm, distance_cm\n        )\n        cover_size = 2.2 * math.hypot(width_deg, height_deg)\n        fullfield.width = cover_size\n        fullfield.height = cover_size\n\n        keep_psychopy_on_stimulus_monitor()\n\n        write_json_atomic(status_path, {\n            "state": "running",\n            "message": f"Ejecutando {stimulus_type}...",\n            "hz": hz,\n            "command_id": command_id\n        })\n\n        rows = []\n        previous_flip = None\n        global_frame = 0\n        possible_drops = 0\n        details = {}\n\n        # v1.32: the absolute monotonic pacer is now driven by a refresh\n        # period measured directly from real win.flip() calls on this exact\n        # stimulus monitor. This avoids the erroneous getActualFrameRate()\n        # estimates seen on mixed-refresh multi-monitor Windows systems.\n        # Frame-locked stimuli keep their deterministic frame sequence while\n        # durations, speeds and temporal frequencies are referenced to the\n        # empirically observed display cadence.\n        pacing_anchor_perf = None\n        pacing_anchor_frame = 0\n        pacing_wait_total_s = 0.0\n        pacing_late_frame_count = 0\n        pacing_max_lateness_s = 0.0\n\n        # Prepared resources are used only by Protocol Builder runs.\n        # They are created before the first logged protocol frame so\n        # expensive PsychoPy/OpenGL object construction cannot interrupt\n        # an already-running timed sequence.\n        prepared_step = None\n        protocol_prepared_steps = {}\n\n        # Protocol context is blank for ordinary single-stimulus runs.\n        # flip_and_log() and ttl_commit() read these values so a protocol\n        # can remain one continuous visual/TTL record.\n        protocol_name_context = ""\n        protocol_version_context = ""\n        protocol_rep_context = ""\n        protocol_step_index_context = ""\n        protocol_step_name_context = ""\n        protocol_step_kind_context = ""\n\n        # ----------------------------------------------------------\n        # Trigger layer — v1.29.\n        #\n        # IMPORTANT: MaxOne Legacy FTDI preserves the v1.17 behaviour\n        # unchanged: FRAME_SYNC on every logged flip and EVENT HIGH for one\n        # complete display frame at each principal-condition onset.\n        #\n        # Trigger Box TTL genérica is an additional, independent mode:\n        # FRAME_SYNC remains attached to the real PsychoPy flip and UNIVERSAL\n        # receives a second >=1 ms EVENT pulse ~3 ms later. AUX EVENT is held\n        # as a readable condition-state channel; protocol pauses add an AUX-only short marker (v1.28).\n        ttl_mode = str(cfg.get("ttl_mode", "simulation")).strip().lower()\n        ttl_pin = int(cfg.get("ttl_pin", 8))\n        ttl_port_requested = str(cfg.get("ttl_port", "auto")).strip() or "auto"\n        ttl_port_resolved = ""\n        ttl_backend = None\n        trigger_backend_fault = ""\n        trigger_stopping = False\n        frame_sync_pulses = 0\n\n        # Legacy/simulation state (kept from v1.17).\n        ttl_state = 0\n        ttl_requested_state = 0\n        ttl_events = []\n        ttl_event_counter = 0\n        ttl_pulse_pending = None\n        ttl_pulse_width_frames = 1\n\n        # Generic trigger-box state (v1.28). UNIVERSAL preserves the validated\n        # short EVENT pulse. AUX EVENT becomes a human-readable held state:\n        # ON/OFF => ON HIGH / OFF LOW; all other principal conditions alternate.\n        generic_event_pending = None\n        generic_aux_state = 0\n        event_offset_requested_s = GenericTTLFTDI.EVENT_OFFSET_S\n        event_pulse_width_s = GenericTTLFTDI.EVENT_PULSE_MIN_S\n        # v1.28: protocol pauses receive one short HIGH marker on AUX EVENT.\n        # Use the same validated minimum width as the former EVENT pulses.\n        pause_marker_width_s = GenericTTLFTDI.EVENT_PULSE_MIN_S\n\n\n        def _trigger_write(action, *args):\n            """Latch I/O errors without interrupting PsychoPy\'s callback queue.\n\n            flip_and_log/check_abort raises outside callOnFlip, so pending callbacks\n            cannot be replayed by the next safety flip after a failed port write.\n            """\n            global trigger_backend_fault\n            try:\n                action(*args)\n                return True\n            except Exception as exc:\n                if not trigger_backend_fault:\n                    trigger_backend_fault = f"{type(exc).__name__}: {exc}"\n                if ttl_backend is not None:\n                    ttl_backend.idle()\n                return False\n\n\n        def check_trigger_error():\n            if trigger_backend_fault:\n                if ttl_mode == "maxone_ftdi":\n                    raise RuntimeError(\n                        f"Error del generador MaxOne FTDI: {trigger_backend_fault}"\n                    )\n                raise RuntimeError(\n                    f"Error de la Trigger Box TTL genérica: {trigger_backend_fault}"\n                )\n\n\n        def open_trigger_backend():\n            global ttl_backend, ttl_port_resolved\n            if ttl_mode not in (\n                "simulation", "disabled", "maxone_ftdi", "generic_ftdi"\n            ):\n                raise ValueError(f"Modo de trigger no soportado: {ttl_mode}")\n            if ttl_mode == "maxone_ftdi":\n                # Legacy path: exactly the v1.17 backend.\n                ttl_backend = MaxOneFTDI(ttl_port_requested)\n                ttl_port_resolved = ttl_backend.open()\n            elif ttl_mode == "generic_ftdi":\n                ttl_backend = GenericTTLFTDI(ttl_port_requested)\n                ttl_port_resolved = ttl_backend.open()\n\n\n        def close_trigger_backend():\n            global ttl_backend, trigger_backend_fault\n            backend, ttl_backend = ttl_backend, None\n            if backend is not None:\n                errors = backend.close()\n                if errors and not trigger_backend_fault:\n                    trigger_backend_fault = "; ".join(errors)\n\n\n        def frame_sync_high():\n            global frame_sync_pulses\n            if ttl_backend is not None and not trigger_stopping and not trigger_backend_fault:\n                if _trigger_write(ttl_backend.frame_high):\n                    frame_sync_pulses += 1\n\n\n        def frame_sync_low():\n            if ttl_backend is not None:\n                _trigger_write(ttl_backend.frame_low)\n\n\n        def finish_trigger_run(reason, suppress_errors=False):\n            """Make pending onset callbacks harmless and always release the port."""\n            global trigger_stopping, ttl_pulse_pending, generic_event_pending\n            global trigger_backend_fault\n            # Before stopping callbacks, guarantee AUX EVENT is physically LOW.\n            # This is independent of UNIVERSAL and is logged when it changes.\n            if ttl_mode == "generic_ftdi":\n                force_generic_aux_low(reason)\n            trigger_stopping = True\n            ttl_pulse_pending = None\n            generic_event_pending = None\n            errors = []\n            # Preserve the v1.17 MaxOne/simulation falling-edge behaviour.\n            if ttl_backend is not None and suppress_errors:\n                errors.extend(ttl_backend.idle())\n            try:\n                if (\n                    ttl_mode in ("simulation", "maxone_ftdi")\n                    and ttl_requested_state != 0\n                ):\n                    queue_ttl(0, reason, 0, stimulus_type)\n                safe_gray_flip_and_stamp()\n            except Exception as exc:\n                errors.append(f"Retorno a gris: {type(exc).__name__}: {exc}")\n            finally:\n                close_trigger_backend()\n            if errors and not trigger_backend_fault:\n                trigger_backend_fault = "; ".join(errors)\n            if not suppress_errors:\n                check_trigger_error()\n\n\n        # ----------------------------------------------------------\n        # Legacy EVENT path — copied from v1.17 semantics.\n        # ----------------------------------------------------------\n        def ttl_commit(\n            state, event_name, trial,\n            condition, scheduled_frame\n        ):\n            global ttl_state, ttl_event_counter\n\n            state = int(state)\n            if trigger_stopping and state:\n                return\n            if ttl_backend is not None:\n                if not _trigger_write(ttl_backend.set_event, state):\n                    return\n\n            ttl_state = state\n            ttl_event_counter += 1\n\n            ttl_events.append({\n                "event_index": ttl_event_counter,\n                "state": ttl_state,\n                "level": "HIGH" if ttl_state else "LOW",\n                "event_name": str(event_name),\n                "trial": trial,\n                "condition": str(condition),\n                "scheduled_frame": int(scheduled_frame),\n                "callback_time_s": f"{core.getTime():.9f}",\n                "flip_time_s": "",\n                "mode": ttl_mode,\n                "pin": (0 if ttl_mode == "maxone_ftdi" else ttl_pin),\n                "port": ttl_port_resolved,\n                "signal": (\n                    "EVENT_TX_BREAK"\n                    if ttl_mode == "maxone_ftdi"\n                    else "EVENT_SIMULATED"\n                ),\n                "maxone_bit": (0 if ttl_mode == "maxone_ftdi" else ""),\n                "event_role": "CONDITION_ONSET" if ttl_state else "PULSE_END",\n                "pulse_width_frames": ttl_pulse_width_frames,\n                "event_offset_requested_ms": "",\n                "pulse_width_requested_ms": "",\n                "frame_anchor_perf_s": "",\n                "transition_perf_s": "",\n                "event_offset_measured_ms": "",\n                "pulse_width_measured_ms": "",\n                "universal_code": "",\n                "aux_event_state": "",\n                "aux_event_level": "",\n                "aux_event_policy": "",\n                "universal_pulse_phase": "",\n                "protocol_name": protocol_name_context,\n                "protocol_version": protocol_version_context,\n                "protocol_rep": protocol_rep_context,\n                "protocol_step_index": protocol_step_index_context,\n                "protocol_step_name": protocol_step_name_context,\n                "protocol_step_kind": protocol_step_kind_context,\n            })\n\n\n        def queue_ttl(\n            state, event_name,\n            trial=0, condition=""\n        ):\n            global ttl_requested_state\n\n            if ttl_mode not in (\n                "simulation",\n                "maxone_ftdi"\n            ):\n                return\n\n            state = 1 if int(state) else 0\n            if state == ttl_requested_state:\n                return\n\n            ttl_requested_state = state\n            scheduled_frame = global_frame + 1\n\n            # EVENT transition is attached to the exact next PsychoPy flip.\n            win.callOnFlip(\n                ttl_commit,\n                state,\n                event_name,\n                trial,\n                condition,\n                scheduled_frame\n            )\n\n\n        def queue_ttl_event(\n            event_name, trial=0, condition=""\n        ):\n            """Mark the onset of one principal condition."""\n            global ttl_pulse_pending, generic_event_pending\n\n            if ttl_mode == "generic_ftdi":\n                if generic_event_pending is not None:\n                    raise ValueError(\n                        "Se han solicitado dos EVENT antes del mismo frame visual. "\n                        "Cada condición principal debe tener un onset diferenciable."\n                    )\n                generic_event_pending = {\n                    "event_name": str(event_name),\n                    "trial": trial,\n                    "condition": str(condition),\n                    "scheduled_frame": int(global_frame + 1),\n                    "protocol_name": protocol_name_context,\n                    "protocol_version": protocol_version_context,\n                    "protocol_rep": protocol_rep_context,\n                    "protocol_step_index": protocol_step_index_context,\n                    "protocol_step_name": protocol_step_name_context,\n                    "protocol_step_kind": protocol_step_kind_context,\n                }\n                return\n\n            # v1.17 behaviour for Simulación and MaxOne Legacy FTDI.\n            if ttl_mode not in (\n                "simulation",\n                "maxone_ftdi"\n            ):\n                return\n\n            if ttl_state != 0 or ttl_requested_state != 0:\n                raise ValueError(\n                    "Dos condiciones principales están demasiado juntas "\n                    "para generar pulsos EVENT separados. Cada condición "\n                    "debe durar al menos 2 frames."\n                )\n\n            ttl_pulse_pending = (\n                str(event_name),\n                trial,\n                str(condition)\n            )\n            queue_ttl(\n                1,\n                str(event_name),\n                trial,\n                str(condition)\n            )\n\n\n        def finish_ttl_pulse_after_flip():\n            """\n            After the first HIGH frame has been displayed, queue EVENT LOW\n            for the immediately following display flip.\n            """\n            global ttl_pulse_pending\n\n            if (\n                ttl_mode in (\n                    "simulation",\n                    "maxone_ftdi"\n                )\n                and ttl_pulse_pending is not None\n                and ttl_state == 1\n                and ttl_requested_state == 1\n            ):\n                event_name, trial, condition = ttl_pulse_pending\n                ttl_pulse_pending = None\n                queue_ttl(\n                    0,\n                    f"{event_name}_PULSE_END",\n                    trial,\n                    condition\n                )\n\n\n        # ----------------------------------------------------------\n        # Generic trigger-box EVENT + AUX state + pause-marker path — v1.28.\n        # ----------------------------------------------------------\n        def _generic_aux_target(event_name):\n            """Return (state, policy) for the readable AUX EVENT condition track."""\n            name = str(event_name).strip().upper()\n            if name == "ON_START":\n                return 1, "ON_OFF_STATE"\n            if name == "OFF_START":\n                return 0, "ON_OFF_STATE"\n            # For moving bars, gratings, RF positions, noise/temporal blocks, etc.,\n            # every principal condition onset creates an unmistakable edge.\n            return (0 if generic_aux_state else 1), "ALTERNATE_BY_CONDITION"\n\n\n        def _append_generic_event_transition(\n            pending, state, role, flip_time, aux_state, aux_policy, pulse_phase,\n            transition_perf_s="", metrics=None\n        ):\n            global ttl_event_counter\n            ttl_event_counter += 1\n            metrics = metrics or {}\n            measured_offset = metrics.get("event_offset_s", "")\n            measured_width = metrics.get("event_width_s", "")\n            frame_anchor = metrics.get("frame_anchor_perf_s", "")\n\n            ttl_events.append({\n                "event_index": ttl_event_counter,\n                # state/level preserve the validated UNIVERSAL event-pulse log.\n                "state": int(state),\n                "level": "HIGH" if state else "LOW",\n                "event_name": (\n                    pending["event_name"]\n                    if state else f\'{pending["event_name"]}_PULSE_END\'\n                ),\n                "trial": pending["trial"],\n                "condition": pending["condition"],\n                "scheduled_frame": pending["scheduled_frame"],\n                "callback_time_s": f"{core.getTime():.9f}",\n                "flip_time_s": (\n                    "" if flip_time is None else f"{float(flip_time):.9f}"\n                ),\n                "mode": ttl_mode,\n                "pin": 0,\n                "port": ttl_port_resolved,\n                "signal": "UNIVERSAL_RTS_EVENT_PULSE + AUX_EVENT_TX_BREAK_STATE",\n                "maxone_bit": "",\n                "event_role": role,\n                "pulse_width_frames": 0,\n                "event_offset_requested_ms":\n                    f"{1000.0 * event_offset_requested_s:.3f}",\n                "pulse_width_requested_ms":\n                    f"{1000.0 * event_pulse_width_s:.3f}",\n                "frame_anchor_perf_s": (\n                    "" if frame_anchor == "" else f"{float(frame_anchor):.9f}"\n                ),\n                "transition_perf_s": (\n                    "" if transition_perf_s == "" else f"{float(transition_perf_s):.9f}"\n                ),\n                "event_offset_measured_ms": (\n                    "" if measured_offset == ""\n                    else f"{1000.0 * float(measured_offset):.6f}"\n                ),\n                "pulse_width_measured_ms": (\n                    "" if measured_width == ""\n                    else f"{1000.0 * float(measured_width):.6f}"\n                ),\n                "universal_code": "SECOND_PULSE_AFTER_FRAME",\n                "aux_event_state": int(aux_state),\n                "aux_event_level": "HIGH" if aux_state else "LOW",\n                "aux_event_policy": str(aux_policy),\n                "universal_pulse_phase": str(pulse_phase),\n                "protocol_name": pending["protocol_name"],\n                "protocol_version": pending["protocol_version"],\n                "protocol_rep": pending["protocol_rep"],\n                "protocol_step_index": pending["protocol_step_index"],\n                "protocol_step_name": pending["protocol_step_name"],\n                "protocol_step_kind": pending["protocol_step_kind"],\n            })\n\n\n        def _append_generic_aux_reset(reason, previous_state, transition_perf_s=""):\n            """Log the explicit end-of-stimulus AUX return to LOW."""\n            global ttl_event_counter\n            ttl_event_counter += 1\n            ttl_events.append({\n                "event_index": ttl_event_counter,\n                "state": 0,\n                "level": "LOW",\n                "event_name": str(reason),\n                "trial": 0,\n                "condition": "AUX_RESET_LOW",\n                "scheduled_frame": int(global_frame),\n                "callback_time_s": f"{core.getTime():.9f}",\n                "flip_time_s": "",\n                "mode": ttl_mode,\n                "pin": 0,\n                "port": ttl_port_resolved,\n                "signal": "AUX_EVENT_TX_BREAK_STATE",\n                "maxone_bit": "",\n                "event_role": "AUX_RESET_LOW",\n                "pulse_width_frames": 0,\n                "event_offset_requested_ms": "",\n                "pulse_width_requested_ms": "",\n                "frame_anchor_perf_s": "",\n                "transition_perf_s": (\n                    "" if transition_perf_s == "" else f"{float(transition_perf_s):.9f}"\n                ),\n                "event_offset_measured_ms": "",\n                "pulse_width_measured_ms": "",\n                "universal_code": "",\n                "aux_event_state": 0,\n                "aux_event_level": "LOW",\n                "aux_event_policy": "FORCED_END_RESET",\n                "universal_pulse_phase": "",\n                "protocol_name": protocol_name_context,\n                "protocol_version": protocol_version_context,\n                "protocol_rep": protocol_rep_context,\n                "protocol_step_index": protocol_step_index_context,\n                "protocol_step_name": protocol_step_name_context,\n                "protocol_step_kind": protocol_step_kind_context,\n            })\n\n\n        def force_generic_aux_low(reason):\n            """Force AUX EVENT LOW without adding an EVENT pulse to UNIVERSAL."""\n            global generic_aux_state\n            if ttl_mode != "generic_ftdi":\n                return\n            previous_state = int(generic_aux_state)\n            transition_perf = ""\n            if ttl_backend is not None:\n                if _trigger_write(ttl_backend.set_event, 0):\n                    transition_perf = _ftdi_time.perf_counter()\n            generic_aux_state = 0\n            if previous_state:\n                _append_generic_aux_reset(\n                    reason, previous_state, transition_perf_s=transition_perf\n                )\n\n\n        def _append_generic_pause_marker(\n            state, transition_perf_s="", measured_width_s="", flip_time_s=""\n        ):\n            """Log the AUX-only short pulse marking the start of a protocol pause."""\n            global ttl_event_counter\n            ttl_event_counter += 1\n            ttl_events.append({\n                "event_index": ttl_event_counter,\n                "state": int(state),\n                "level": "HIGH" if state else "LOW",\n                "event_name": (\n                    "PROTOCOL_PAUSE_START" if state\n                    else "PROTOCOL_PAUSE_MARKER_END"\n                ),\n                "trial": 0,\n                "condition": "PAUSE_MARKER",\n                "scheduled_frame": int(global_frame),\n                "callback_time_s": f"{core.getTime():.9f}",\n                "flip_time_s": str(flip_time_s or ""),\n                "mode": ttl_mode,\n                "pin": 0,\n                "port": ttl_port_resolved,\n                "signal": "AUX_EVENT_TX_BREAK_PAUSE_MARKER",\n                "maxone_bit": "",\n                "event_role": (\n                    "PAUSE_MARKER_START" if state else "PAUSE_MARKER_END"\n                ),\n                "pulse_width_frames": 0,\n                "event_offset_requested_ms": "",\n                "pulse_width_requested_ms": f"{1000.0 * pause_marker_width_s:.3f}",\n                "frame_anchor_perf_s": "",\n                "transition_perf_s": (\n                    "" if transition_perf_s == ""\n                    else f"{float(transition_perf_s):.9f}"\n                ),\n                "event_offset_measured_ms": "",\n                "pulse_width_measured_ms": (\n                    "" if measured_width_s == ""\n                    else f"{1000.0 * float(measured_width_s):.6f}"\n                ),\n                "universal_code": "",\n                "aux_event_state": int(state),\n                "aux_event_level": "HIGH" if state else "LOW",\n                "aux_event_policy": "PAUSE_MARKER_PULSE",\n                "universal_pulse_phase": "",\n                "protocol_name": protocol_name_context,\n                "protocol_version": protocol_version_context,\n                "protocol_rep": protocol_rep_context,\n                "protocol_step_index": protocol_step_index_context,\n                "protocol_step_name": protocol_step_name_context,\n                "protocol_step_kind": protocol_step_kind_context,\n            })\n\n\n        def emit_generic_pause_marker():\n            """Mark protocol-pause onset with one short AUX HIGH pulse.\n\n            The first pause frame is already on screen when this function runs.\n            UNIVERSAL is never touched, and AUX is guaranteed LOW afterwards.\n            """\n            global generic_aux_state\n            if ttl_mode != "generic_ftdi":\n                return False\n\n            # A pause always starts from the unambiguous AUX baseline.\n            force_generic_aux_low("PROTOCOL_PAUSE_PRE_MARKER_LOW")\n            metrics = {}\n            if ttl_backend is not None:\n                try:\n                    metrics = ttl_backend.pulse_aux_marker(\n                        width_s=pause_marker_width_s\n                    )\n                except Exception as exc:\n                    if not trigger_backend_fault:\n                        _trigger_write(lambda: (_ for _ in ()).throw(exc))\n                    generic_aux_state = 0\n                    return False\n\n            generic_aux_state = 0\n            high_at = metrics.get("aux_high_perf_s", "")\n            low_at = metrics.get("aux_low_perf_s", "")\n            measured_width = metrics.get("aux_width_s", "")\n            first_pause_flip = ""\n            if rows:\n                last_row = rows[-1]\n                if (\n                    last_row.get("stimulus_type") == "protocol_pause"\n                    and int(last_row.get("global_frame", -1)) == int(global_frame)\n                ):\n                    first_pause_flip = last_row.get("flip_time_s", "")\n\n            _append_generic_pause_marker(\n                1, transition_perf_s=high_at,\n                measured_width_s=measured_width, flip_time_s=first_pause_flip\n            )\n            _append_generic_pause_marker(\n                0, transition_perf_s=low_at,\n                measured_width_s=measured_width, flip_time_s=first_pause_flip\n            )\n            return True\n\n\n        def emit_generic_event_for_current_frame(flip_time):\n            """Emit the validated UNIVERSAL EVENT pulse and update held AUX state."""\n            global generic_event_pending, ttl_state, generic_aux_state\n\n            if ttl_mode != "generic_ftdi":\n                return False\n            pending = generic_event_pending\n            if pending is None:\n                ttl_state = 0\n                return False\n\n            scheduled = int(pending["scheduled_frame"])\n            if scheduled > global_frame:\n                ttl_state = 0\n                return False\n            if scheduled < global_frame:\n                generic_event_pending = None\n                raise RuntimeError(\n                    "EVENT genérico pendiente fuera de sincronía con el frame visual."\n                )\n\n            generic_event_pending = None\n            if trigger_stopping:\n                ttl_state = 0\n                return False\n\n            aux_target, aux_policy = _generic_aux_target(pending["event_name"])\n            metrics = {}\n            if ttl_backend is not None:\n                try:\n                    metrics = ttl_backend.pulse_universal_event_after_frame(\n                        aux_state=aux_target,\n                        offset_s=event_offset_requested_s,\n                        width_s=event_pulse_width_s,\n                    )\n                except Exception as exc:\n                    if not trigger_backend_fault:\n                        _trigger_write(lambda: (_ for _ in ()).throw(exc))\n                    ttl_state = 0\n                    return False\n\n            # Simulation/no-backend still tracks the intended AUX state in logs.\n            generic_aux_state = int(aux_target)\n            high_perf = metrics.get("event_high_perf_s", "")\n            low_perf = metrics.get("event_low_perf_s", "")\n            ttl_state = 1\n            _append_generic_event_transition(\n                pending, 1, "CONDITION_ONSET", flip_time,\n                generic_aux_state, aux_policy, "START",\n                transition_perf_s=high_perf, metrics=metrics\n            )\n            _append_generic_event_transition(\n                pending, 0, "PULSE_END", flip_time,\n                generic_aux_state, aux_policy, "END",\n                transition_perf_s=low_perf, metrics=metrics\n            )\n            ttl_state = 0\n            return True\n\n\n        def safe_gray_flip_and_stamp():\n            """\n            Return the experimental display to the configured adaptation background.\n\n            Legacy MaxOne/simulation retains the v1.17 behaviour: if an EVENT\n            falling edge was queued for this safety flip, it is committed and\n            timestamped. Generic AUX state is handled explicitly and never by\n            this safety display flip.\n            """\n            next_physical_frame = global_frame + 1\n            flip_time = show_solid(GRAY)\n\n            if flip_time is not None:\n                for ttl_event in reversed(ttl_events):\n                    if ttl_event["scheduled_frame"] < next_physical_frame:\n                        break\n                    if ttl_event["scheduled_frame"] == next_physical_frame:\n                        ttl_event["flip_time_s"] = f"{float(flip_time):.9f}"\n\n            return flip_time\n\n        def abort_requested():\n            try:\n                return abort_map[0] == 1\n            except Exception:\n                return False\n\n        def check_abort():\n            if abort_requested():\n                raise KeyboardInterrupt\n            check_trigger_error()\n            if event.getKeys(keyList=["escape"]):\n                raise KeyboardInterrupt\n\n        def read_protocol_pause_request():\n            if not protocol_pause_path.exists():\n                return {\n                    "command_id": "",\n                    "pause_requested": False,\n                    "requested_at": "",\n                }\n\n            try:\n                return json.loads(\n                    protocol_pause_path.read_text(\n                        encoding="utf-8"\n                    )\n                )\n            except Exception:\n                return None\n\n        def wait_for_protocol_resume(\n            protocol_name,\n            protocol_version,\n            protocol_rep,\n            step_position,\n            total_steps,\n            next_step_name,\n            pause_events\n        ):\n            """\n            Pause is honored only before a protocol step begins.\n            The preceding step always completes normally.\n            """\n            global previous_flip, pacing_anchor_perf, pacing_anchor_frame\n\n            request = None\n            for _ in range(10):\n                request = read_protocol_pause_request()\n                if request is not None:\n                    break\n                core.wait(0.01)\n\n            if not request:\n                return\n\n            if (\n                str(request.get("command_id", ""))\n                != str(command_id)\n                or not bool(\n                    request.get(\n                        "pause_requested",\n                        False\n                    )\n                )\n            ):\n                return\n\n            # Unlogged gray flip: completes any pending one-frame EVENT LOW.\n            safe_gray_flip_and_stamp()\n            check_trigger_error()\n            previous_flip = None\n\n            paused_at_wall = (\n                datetime.now().isoformat(\n                    timespec="milliseconds"\n                )\n            )\n            paused_at_core = core.getTime()\n\n            write_json_atomic(\n                status_path,\n                {\n                    "state": "protocol_paused",\n                    "message":\n                        "PROTOCOLO EN PAUSA · fondo de adaptación",\n                    "hz": hz,\n                    "command_id": command_id,\n                    "protocol_name": protocol_name,\n                    "protocol_version": protocol_version,\n                    "protocol_rep": protocol_rep,\n                    "protocol_step": step_position,\n                    "protocol_total_steps": total_steps,\n                    "next_step_name": next_step_name,\n                    "pause_requested_at":\n                        request.get(\n                            "requested_at",\n                            ""\n                        ),\n                    "paused_at": paused_at_wall,\n                }\n            )\n\n            while True:\n                check_abort()\n                current = read_protocol_pause_request()\n\n                # A transient read error keeps the protocol paused.\n                if current is None:\n                    core.wait(0.05)\n                    continue\n\n                same_command = (\n                    str(\n                        current.get(\n                            "command_id",\n                            ""\n                        )\n                    )\n                    == str(command_id)\n                )\n                still_paused = (\n                    same_command\n                    and bool(\n                        current.get(\n                            "pause_requested",\n                            False\n                        )\n                    )\n                )\n\n                if not still_paused:\n                    break\n\n                core.wait(0.05)\n\n            resumed_at_wall = (\n                datetime.now().isoformat(\n                    timespec="milliseconds"\n                )\n            )\n            pause_duration_s = max(\n                0.0,\n                float(\n                    core.getTime()\n                    - paused_at_core\n                )\n            )\n\n            pause_events.append({\n                "pause_index":\n                    len(pause_events) + 1,\n                "requested_at":\n                    request.get(\n                        "requested_at",\n                        ""\n                    ),\n                "paused_at":\n                    paused_at_wall,\n                "resumed_at":\n                    resumed_at_wall,\n                "duration_s":\n                    pause_duration_s,\n                "before_protocol_rep":\n                    protocol_rep,\n                "before_step_position":\n                    step_position,\n                "before_step_name":\n                    next_step_name,\n                "safe_boundary":\n                    "between_protocol_steps",\n            })\n\n            # Refresh the adaptation background after the wait and start a new timing\n            # segment so manual pause time is never flagged as a frame drop.\n            show_solid(GRAY)\n            previous_flip = None\n            # Manual pauses are deliberately outside the experimental\n            # timeline. Restart absolute pacing at the next logged frame.\n            pacing_anchor_perf = None\n            pacing_anchor_frame = global_frame\n\n            write_json_atomic(\n                status_path,\n                {\n                    "state": "running",\n                    "message":\n                        (\n                            f"Reanudando protocolo {protocol_name} · "\n                            f"paso {step_position}/{total_steps}"\n                        ),\n                    "hz": hz,\n                    "command_id": command_id,\n                    "protocol_name": protocol_name,\n                    "protocol_version": protocol_version,\n                    "protocol_rep": protocol_rep,\n                    "protocol_step": step_position,\n                    "protocol_total_steps": total_steps,\n                    "resumed_from_pause": True,\n                }\n            )\n\n        def _wait_until_pacing_deadline(deadline):\n            """Wait efficiently until an absolute perf_counter deadline."""\n            start_wait = time.perf_counter()\n            while True:\n                remaining = float(deadline) - time.perf_counter()\n                if remaining <= 0:\n                    break\n                if remaining > 0.003:\n                    time.sleep(max(0.0, remaining - 0.0015))\n                # Final ~1.5 ms intentionally spins for timing precision.\n            return max(0.0, time.perf_counter() - start_wait)\n\n        def flip_and_log(stim_type, trial, condition, stage, frame_in_stage):\n            global previous_flip, global_frame, possible_drops\n            global pacing_anchor_perf, pacing_anchor_frame\n            global pacing_wait_total_s, pacing_late_frame_count\n            global pacing_max_lateness_s\n\n            check_trigger_error()\n\n            # Keep every logged frame on an absolute timeline. The first\n            # frame establishes the anchor; from then on frame N cannot be\n            # presented before anchor + N * expected_interval. If processing\n            # or the display makes us late, do not add another delay: this\n            # allows natural catch-up without ever running ahead overall.\n            next_global_frame = global_frame + 1\n            pacing_target_perf = None\n            if pacing_anchor_perf is not None:\n                pacing_target_perf = (\n                    pacing_anchor_perf\n                    + (next_global_frame - pacing_anchor_frame)\n                    * expected_interval\n                )\n                if time.perf_counter() < pacing_target_perf:\n                    pacing_wait_total_s += _wait_until_pacing_deadline(\n                        pacing_target_perf\n                    )\n\n            if ttl_backend is not None and not trigger_stopping:\n                win.callOnFlip(frame_sync_high)\n            try:\n                flip_time = win.flip()\n                flip_return_perf = time.perf_counter()\n            finally:\n                frame_sync_low()\n            # Raise I/O faults after all flip callbacks have been consumed.\n            check_trigger_error()\n\n            global_frame += 1\n\n            if pacing_anchor_perf is None:\n                pacing_anchor_perf = flip_return_perf\n                pacing_anchor_frame = global_frame\n            elif pacing_target_perf is not None:\n                lateness = max(0.0, flip_return_perf - pacing_target_perf)\n                # Ignore sub-0.25-ms call/clock noise in the diagnostic count.\n                if lateness > 0.00025:\n                    pacing_late_frame_count += 1\n                    pacing_max_lateness_s = max(\n                        pacing_max_lateness_s, lateness\n                    )\n\n            # Generic mode emits EVENT as the second pulse ~3 ms after the\n            # FRAME pulse of this same onset frame. Legacy MaxOne is untouched.\n            if ttl_mode == "generic_ftdi":\n                emit_generic_event_for_current_frame(flip_time)\n                check_trigger_error()\n\n            # callOnFlip callbacks have already run for this frame.\n            # Add the actual flip timestamp to every TTL event assigned\n            # to this exact global frame.\n            if flip_time is not None:\n                for ttl_event in reversed(ttl_events):\n                    if ttl_event["scheduled_frame"] < global_frame:\n                        break\n                    if ttl_event["scheduled_frame"] == global_frame:\n                        ttl_event["flip_time_s"] = (\n                            f"{float(flip_time):.9f}"\n                        )\n\n            interval = None\n            error_ms = None\n            dropped = False\n\n            if previous_flip is not None and flip_time is not None:\n                interval = float(flip_time - previous_flip)\n                error_ms = (\n                    interval - expected_interval\n                ) * 1000.0\n                dropped = (\n                    interval > expected_interval * 1.5\n                )\n                if dropped:\n                    possible_drops += 1\n\n            rows.append({\n                "stimulus_type": stim_type,\n                "trial": trial,\n                "condition": condition,\n                "stage": stage,\n                "frame_in_stage": frame_in_stage,\n                "global_frame": global_frame,\n                "flip_time_s":\n                    "" if flip_time is None\n                    else f"{float(flip_time):.9f}",\n                "interval_from_previous_flip_s":\n                    "" if interval is None\n                    else f"{interval:.9f}",\n                "expected_interval_s":\n                    f"{expected_interval:.9f}",\n                "interval_error_ms":\n                    "" if error_ms is None\n                    else f"{error_ms:.6f}",\n                "possible_dropped_frame":\n                    int(dropped),\n                "ttl_state": int(ttl_state),\n                "protocol_name":\n                    protocol_name_context,\n                "protocol_version":\n                    protocol_version_context,\n                "protocol_rep":\n                    protocol_rep_context,\n                "protocol_step_index":\n                    protocol_step_index_context,\n                "protocol_step_name":\n                    protocol_step_name_context,\n                "protocol_step_kind":\n                    protocol_step_kind_context,\n            })\n\n            if flip_time is not None:\n                previous_flip = float(flip_time)\n\n            # Event-pulse simulator: HIGH only on the onset frame.\n            # Queue the falling edge for the very next display flip.\n            finish_ttl_pulse_after_flip()\n\n        def solid_frames(\n            color, n_frames, stim_type, trial, condition, stage,\n            after_first_frame=None\n        ):\n            # Zero frames means no presentation and no hidden graphical\n            # state change.\n            if n_frames <= 0:\n                return\n\n            fullfield.fillColor = color\n            fullfield.lineColor = color\n            for frame_i in range(1, n_frames + 1):\n                check_abort()\n                fullfield.draw()\n                flip_and_log(\n                    stim_type, trial, condition,\n                    stage, frame_i\n                )\n                # v1.28: pause markers are emitted only after the first pause\n                # frame is physically visible. Existing calls remain unchanged.\n                if frame_i == 1 and after_first_frame is not None:\n                    after_first_frame()\n                    check_trigger_error()\n\n        def build_noise_resources(\n            noise_cfg\n        ):\n            """\n            Precompute one complete Noise stimulus in RAM.\n\n            Families:\n            - Dense White Noise: Binary or Ternary; new sequence per rep.\n            - Gaussian White Noise: continuous Gaussian; new sequence per rep.\n            - Frozen Noise: one seeded sequence replayed identically every rep.\n\n            Gaussian values are centered on Color medio. With contrast=1,\n            the requested low/high colors correspond approximately to ±3σ.\n            """\n            stim_total_f = seconds_to_frames(\n                noise_cfg[\n                    "dense_duration_s"\n                ],\n                hz\n            )\n            reps = int(\n                noise_cfg[\n                    "dense_repetitions"\n                ]\n            )\n            update_hz_req = float(\n                noise_cfg[\n                    "dense_update_hz"\n                ]\n            )\n            check_size_deg = float(\n                noise_cfg[\n                    "dense_check_size_deg"\n                ]\n            )\n            contrast = float(\n                noise_cfg[\n                    "dense_contrast"\n                ]\n            )\n            seed_base = int(\n                noise_cfg[\n                    "dense_seed"\n                ]\n            )\n\n            family_mode = str(\n                noise_cfg.get(\n                    "dense_family_mode",\n                    "Dense White Noise"\n                )\n            )\n            distribution = str(\n                noise_cfg.get(\n                    "dense_distribution",\n                    noise_cfg.get(\n                        "dense_mode",\n                        "Binary"\n                    )\n                )\n            ).lower()\n\n            if family_mode == "Gaussian White Noise":\n                distribution = "gaussian"\n                frozen = False\n            elif family_mode == "Dense White Noise":\n                frozen = False\n                if distribution not in (\n                    "binary",\n                    "ternary",\n                ):\n                    raise ValueError(\n                        "Dense White Noise usa Binary o Ternary."\n                    )\n            elif family_mode == "Frozen Noise":\n                frozen = True\n                if distribution not in (\n                    "binary",\n                    "ternary",\n                    "gaussian",\n                ):\n                    raise ValueError(\n                        "Frozen Noise: distribución no válida."\n                    )\n            else:\n                raise ValueError(\n                    f"Familia Noise desconocida: {family_mode}"\n                )\n\n            if reps < 1:\n                raise ValueError(\n                    "Noise: repeticiones debe ser >= 1."\n                )\n            if update_hz_req <= 0:\n                raise ValueError(\n                    "Noise: frecuencia de actualización debe ser > 0."\n                )\n            if check_size_deg <= 0:\n                raise ValueError(\n                    "Noise: tamaño de check debe ser > 0."\n                )\n            if not (\n                0.0\n                <= contrast\n                <= 1.0\n            ):\n                raise ValueError(\n                    "Noise: contraste debe estar entre 0 y 1."\n                )\n\n            frames_per_update = max(\n                1,\n                int(\n                    round(\n                        hz\n                        / update_hz_req\n                    )\n                )\n            )\n            actual_update_hz = (\n                hz\n                / frames_per_update\n            )\n            n_updates = max(\n                1,\n                int(\n                    math.ceil(\n                        stim_total_f\n                        / frames_per_update\n                    )\n                )\n            )\n\n            n_cols = max(\n                1,\n                int(\n                    math.ceil(\n                        width_deg\n                        / check_size_deg\n                    )\n                )\n            )\n            n_rows = max(\n                1,\n                int(\n                    math.ceil(\n                        height_deg\n                        / check_size_deg\n                    )\n                )\n            )\n\n            noise_size_deg = (\n                width_deg,\n                height_deg\n            )\n\n            color_low = np.asarray(\n                rgb255_to_psychopy(\n                    noise_cfg[\n                        "dense_color_low_rgb"\n                    ]\n                ),\n                dtype=np.float32\n            )\n            color_mid = np.asarray(\n                rgb255_to_psychopy(\n                    noise_cfg[\n                        "dense_color_mid_rgb"\n                    ]\n                ),\n                dtype=np.float32\n            )\n            color_high = np.asarray(\n                rgb255_to_psychopy(\n                    noise_cfg[\n                        "dense_color_high_rgb"\n                    ]\n                ),\n                dtype=np.float32\n            )\n\n            if distribution == "binary":\n                midpoint = (\n                    color_low\n                    + color_high\n                ) / 2.0\n                low_eff = (\n                    midpoint\n                    + contrast\n                    * (\n                        color_low\n                        - midpoint\n                    )\n                )\n                high_eff = (\n                    midpoint\n                    + contrast\n                    * (\n                        color_high\n                        - midpoint\n                    )\n                )\n                mid_eff = None\n\n            elif distribution == "ternary":\n                low_eff = (\n                    color_mid\n                    + contrast\n                    * (\n                        color_low\n                        - color_mid\n                    )\n                )\n                mid_eff = color_mid\n                high_eff = (\n                    color_mid\n                    + contrast\n                    * (\n                        color_high\n                        - color_mid\n                    )\n                )\n\n            else:\n                # Gaussian uses the full low/mid/high anchors. Contrast\n                # scales the normalized Gaussian values themselves.\n                low_eff = color_low\n                mid_eff = color_mid\n                high_eff = color_high\n\n            def make_sequence(seed):\n                check_abort()\n\n                rng = np.random.default_rng(\n                    seed\n                )\n\n                if distribution == "binary":\n                    states = rng.integers(\n                        0,\n                        2,\n                        size=(\n                            n_updates,\n                            n_rows,\n                            n_cols\n                        ),\n                        dtype=np.uint8\n                    )\n\n                elif distribution == "ternary":\n                    states = rng.integers(\n                        0,\n                        3,\n                        size=(\n                            n_updates,\n                            n_rows,\n                            n_cols\n                        ),\n                        dtype=np.uint8\n                    )\n\n                else:\n                    # Unit range represents the full low↔high excursion.\n                    # Base SD=1/3 means ±3σ approximately reaches ±1.\n                    states = rng.normal(\n                        loc=0.0,\n                        scale=(\n                            1.0\n                            / 3.0\n                        ),\n                        size=(\n                            n_updates,\n                            n_rows,\n                            n_cols\n                        )\n                    ).astype(\n                        np.float32\n                    )\n                    np.clip(\n                        states,\n                        -1.0,\n                        1.0,\n                        out=states\n                    )\n                    states *= contrast\n\n                rep_images = []\n\n                for upd in range(\n                    n_updates\n                ):\n                    check_abort()\n\n                    st = states[upd]\n                    img = np.empty(\n                        (\n                            n_rows,\n                            n_cols,\n                            3\n                        ),\n                        dtype=np.float32\n                    )\n\n                    if distribution == "binary":\n                        img[\n                            st == 0\n                        ] = low_eff\n                        img[\n                            st == 1\n                        ] = high_eff\n\n                    elif distribution == "ternary":\n                        img[\n                            st == 0\n                        ] = low_eff\n                        img[\n                            st == 1\n                        ] = mid_eff\n                        img[\n                            st == 2\n                        ] = high_eff\n\n                    else:\n                        img[:] = color_mid\n\n                        negative = (\n                            st < 0\n                        )\n                        positive = (\n                            st > 0\n                        )\n\n                        if np.any(\n                            negative\n                        ):\n                            weights = (\n                                -st[\n                                    negative\n                                ]\n                            )[:, None]\n                            img[\n                                negative\n                            ] = (\n                                color_mid\n                                + weights\n                                * (\n                                    color_low\n                                    - color_mid\n                                )\n                            )\n\n                        if np.any(\n                            positive\n                        ):\n                            weights = (\n                                st[\n                                    positive\n                                ]\n                            )[:, None]\n                            img[\n                                positive\n                            ] = (\n                                color_mid\n                                + weights\n                                * (\n                                    color_high\n                                    - color_mid\n                                )\n                            )\n\n                    rep_images.append(\n                        np.ascontiguousarray(\n                            img,\n                            dtype=np.float32\n                        )\n                    )\n\n                return (\n                    states,\n                    rep_images\n                )\n\n            state_sequences = []\n            image_sequences = []\n            used_seeds = []\n\n            if frozen:\n                base_states, base_images = (\n                    make_sequence(\n                        seed_base\n                    )\n                )\n\n                for rep in range(\n                    reps\n                ):\n                    state_sequences.append(\n                        base_states\n                    )\n                    image_sequences.append(\n                        base_images\n                    )\n                    used_seeds.append(\n                        seed_base\n                    )\n\n            else:\n                for rep in range(\n                    reps\n                ):\n                    seed = (\n                        seed_base\n                        + rep\n                    )\n                    states, rep_images = (\n                        make_sequence(\n                            seed\n                        )\n                    )\n                    state_sequences.append(\n                        states\n                    )\n                    image_sequences.append(\n                        rep_images\n                    )\n                    used_seeds.append(\n                        seed\n                    )\n\n            noise_stim = visual.ImageStim(\n                win=win,\n                image=image_sequences[\n                    0\n                ][0],\n                units="deg",\n                size=noise_size_deg,\n                pos=(0, 0),\n                colorSpace="rgb",\n                interpolate=False\n            )\n\n            return {\n                "family_mode":\n                    family_mode,\n                "distribution":\n                    distribution,\n                "frozen":\n                    frozen,\n                "frames_per_update":\n                    frames_per_update,\n                "actual_update_hz":\n                    actual_update_hz,\n                "n_updates":\n                    n_updates,\n                "n_cols":\n                    n_cols,\n                "n_rows":\n                    n_rows,\n                "noise_size_deg":\n                    noise_size_deg,\n                "state_sequences":\n                    state_sequences,\n                "image_sequences":\n                    image_sequences,\n                "used_seeds":\n                    used_seeds,\n                "noise_stim":\n                    noise_stim,\n            }\n\n        def save_noise_sequence_before_timing(\n            resources,\n            noise_cfg\n        ):\n            """\n            Save the exact Noise sequence before timed presentation.\n\n            Protocols call this during preload while neutral gray is visible.\n            Standalone Noise calls it before its first logged frame.\n            """\n            existing = resources.get(\n                "sequence_file"\n            )\n            if existing:\n                return str(existing)\n\n            family_mode = resources[\n                "family_mode"\n            ]\n            distribution_label = (\n                resources[\n                    "distribution"\n                ].capitalize()\n            )\n            frozen = bool(\n                resources[\n                    "frozen"\n                ]\n            )\n\n            stamp_seq = (\n                datetime.now().strftime(\n                    "%Y%m%d_%H%M%S_%f"\n                )\n            )\n            seq_path = run_dir / (\n                f"noise_sequence_"\n                f"{stamp_seq}.npz"\n            )\n\n            np.savez_compressed(\n                seq_path,\n                family_mode=np.asarray(\n                    [family_mode]\n                ),\n                distribution=np.asarray(\n                    [distribution_label]\n                ),\n                frozen=np.asarray(\n                    [frozen],\n                    dtype=np.bool_\n                ),\n                states=np.stack(\n                    resources[\n                        "state_sequences"\n                    ],\n                    axis=0\n                ),\n                used_seeds=np.asarray(\n                    resources[\n                        "used_seeds"\n                    ],\n                    dtype=np.int64\n                ),\n                check_size_deg=np.asarray(\n                    [\n                        float(\n                            noise_cfg[\n                                "dense_check_size_deg"\n                            ]\n                        )\n                    ],\n                    dtype=np.float32\n                ),\n                contrast=np.asarray(\n                    [\n                        float(\n                            noise_cfg[\n                                "dense_contrast"\n                            ]\n                        )\n                    ],\n                    dtype=np.float32\n                ),\n                requested_update_hz=np.asarray(\n                    [\n                        float(\n                            noise_cfg[\n                                "dense_update_hz"\n                            ]\n                        )\n                    ],\n                    dtype=np.float32\n                ),\n                actual_update_hz=np.asarray(\n                    [\n                        resources[\n                            "actual_update_hz"\n                        ]\n                    ],\n                    dtype=np.float32\n                ),\n                grid_rows=np.asarray(\n                    [\n                        resources[\n                            "n_rows"\n                        ]\n                    ],\n                    dtype=np.int32\n                ),\n                grid_cols=np.asarray(\n                    [\n                        resources[\n                            "n_cols"\n                        ]\n                    ],\n                    dtype=np.int32\n                ),\n            )\n\n            resources[\n                "sequence_file"\n            ] = str(\n                seq_path\n            )\n            resources[\n                "sequence_saved_before_timing"\n            ] = True\n\n            return str(\n                seq_path\n            )\n\n        def build_temporal_resources(\n            temporal_cfg\n        ):\n            """\n            Precompute frame/update schedules for the Temporal family.\n            This is used during Protocol Builder preload so no random\n            generation or schedule construction interrupts timed steps.\n            """\n            mode = str(\n                temporal_cfg.get(\n                    "temporal_mode",\n                    "Chirp + Intensity"\n                )\n            )\n\n            resources = {\n                "temporal_mode":\n                    mode\n            }\n\n            if mode == "Chirp + Intensity":\n                chirp_duration_f = seconds_to_frames(\n                    temporal_cfg[\n                        "chirp_frequency_duration_s"\n                    ],\n                    hz\n                )\n                f_min = float(\n                    temporal_cfg[\n                        "chirp_frequency_min_hz"\n                    ]\n                )\n                f_max = float(\n                    temporal_cfg[\n                        "chirp_frequency_max_hz"\n                    ]\n                )\n\n                if (\n                    f_min <= 0\n                    or f_max <= 0\n                ):\n                    raise ValueError(\n                        "Las frecuencias del chirp deben ser > 0."\n                    )\n                if f_max < f_min:\n                    raise ValueError(\n                        "La frecuencia máxima debe ser >= a la mínima."\n                    )\n                if f_max > hz / 2.0:\n                    raise ValueError(\n                        f"La frecuencia máxima ({f_max:.3f} Hz) supera "\n                        f"el límite representable de aproximadamente "\n                        f"{hz/2.0:.3f} Hz para este monitor."\n                    )\n\n                chirp_states = []\n                chirp_inst_freq = []\n\n                if chirp_duration_f > 0:\n                    if chirp_duration_f == 1:\n                        chirp_span_s = (\n                            1.0 / hz\n                        )\n                    else:\n                        chirp_span_s = (\n                            (\n                                chirp_duration_f\n                                - 1\n                            )\n                            / hz\n                        )\n\n                    slope = (\n                        (\n                            f_max\n                            - f_min\n                        )\n                        / chirp_span_s\n                        if chirp_span_s > 0\n                        else 0.0\n                    )\n\n                    for frame_idx in range(\n                        chirp_duration_f\n                    ):\n                        t = (\n                            frame_idx\n                            / hz\n                        )\n                        inst_f = (\n                            f_min\n                            + slope\n                            * t\n                        )\n                        phase_cycles = (\n                            f_min\n                            * t\n                            + 0.5\n                            * slope\n                            * t\n                            * t\n                        )\n                        is_on = (\n                            (\n                                phase_cycles\n                                % 1.0\n                            )\n                            < 0.5\n                        )\n                        chirp_states.append(\n                            is_on\n                        )\n                        chirp_inst_freq.append(\n                            inst_f\n                        )\n\n                intensity_duration_f = seconds_to_frames(\n                    temporal_cfg[\n                        "chirp_intensity_duration_s"\n                    ],\n                    hz\n                )\n                intensity_freq = float(\n                    temporal_cfg[\n                        "chirp_intensity_frequency_hz"\n                    ]\n                )\n                intensity_start_pct = float(\n                    temporal_cfg[\n                        "chirp_intensity_start_pct"\n                    ]\n                )\n\n                if intensity_freq <= 0:\n                    raise ValueError(\n                        "La frecuencia de la rampa de intensidad debe ser > 0."\n                    )\n                if intensity_freq > hz / 2.0:\n                    raise ValueError(\n                        f"La frecuencia de intensidad "\n                        f"({intensity_freq:.3f} Hz) supera el límite "\n                        f"representable de aproximadamente "\n                        f"{hz/2.0:.3f} Hz para este monitor."\n                    )\n                if not (\n                    0.0\n                    <= intensity_start_pct\n                    <= 100.0\n                ):\n                    raise ValueError(\n                        "La intensidad ON inicial debe estar entre 0 y 100 %."\n                    )\n\n                intensity_states = []\n                intensity_levels_pct = []\n\n                if intensity_duration_f > 0:\n                    # Derive the number of cycles from the cycle indices\n                    # that are actually reachable by the rendered frames.\n                    # This avoids a theoretical extra cycle caused by\n                    # ceil(duration * frequency) after seconds→frames\n                    # rounding, and guarantees that the final ON cycle\n                    # reaches exactly 100 %.\n                    last_t = (\n                        (\n                            intensity_duration_f\n                            - 1\n                        )\n                        / hz\n                    )\n                    last_phase_cycles = (\n                        last_t\n                        * intensity_freq\n                    )\n                    max_cycle_index = int(\n                        math.floor(\n                            last_phase_cycles\n                        )\n                    )\n                    n_cycles = max(\n                        1,\n                        max_cycle_index\n                        + 1\n                    )\n\n                    for frame_idx in range(\n                        intensity_duration_f\n                    ):\n                        t = (\n                            frame_idx\n                            / hz\n                        )\n                        phase_cycles = (\n                            t\n                            * intensity_freq\n                        )\n                        cycle_index = min(\n                            n_cycles - 1,\n                            int(\n                                math.floor(\n                                    phase_cycles\n                                )\n                            )\n                        )\n                        is_on = (\n                            (\n                                phase_cycles\n                                % 1.0\n                            )\n                            < 0.5\n                        )\n\n                        if n_cycles <= 1:\n                            pct = 100.0\n                        else:\n                            pct = (\n                                intensity_start_pct\n                                + (\n                                    100.0\n                                    - intensity_start_pct\n                                )\n                                * (\n                                    cycle_index\n                                    / (\n                                        n_cycles\n                                        - 1\n                                    )\n                                )\n                            )\n\n                        intensity_states.append(\n                            is_on\n                        )\n                        intensity_levels_pct.append(\n                            pct\n                        )\n\n                resources.update({\n                    "chirp_states":\n                        chirp_states,\n                    "chirp_inst_freq":\n                        chirp_inst_freq,\n                    "intensity_states":\n                        intensity_states,\n                    "intensity_levels_pct":\n                        intensity_levels_pct,\n                })\n\n                return resources\n\n            duration_f = seconds_to_frames(\n                temporal_cfg[\n                    "temporal_duration_s"\n                ],\n                hz\n            )\n            if duration_f <= 0:\n                raise ValueError(\n                    f"{mode}: duración debe ser > 0."\n                )\n\n            if mode in (\n                "Sinusoidal Flicker",\n                "Contrast Series",\n            ):\n                frequency = float(\n                    temporal_cfg[\n                        "temporal_frequency_hz"\n                    ]\n                )\n                if frequency <= 0:\n                    raise ValueError(\n                        f"{mode}: frecuencia debe ser > 0."\n                    )\n                if frequency >= hz / 2.0:\n                    raise ValueError(\n                        f"{mode}: la frecuencia ({frequency:.3f} Hz) "\n                        f"debe ser menor que Nyquist "\n                        f"({hz/2.0:.3f} Hz) para este monitor."\n                    )\n\n                t = (\n                    np.arange(\n                        duration_f,\n                        dtype=np.float64\n                    )\n                    / hz\n                )\n                resources[\n                    "sine_values"\n                ] = np.sin(\n                    2.0\n                    * math.pi\n                    * frequency\n                    * t\n                ).astype(\n                    np.float32\n                )\n\n            if mode == "Sinusoidal Flicker":\n                contrast_pct = float(\n                    temporal_cfg[\n                        "temporal_contrast_pct"\n                    ]\n                )\n                if not (\n                    0.0\n                    <= contrast_pct\n                    <= 100.0\n                ):\n                    raise ValueError(\n                        "Sinusoidal Flicker: contraste debe estar entre 0 y 100 %."\n                    )\n\n                resources[\n                    "contrast_fraction"\n                ] = (\n                    contrast_pct\n                    / 100.0\n                )\n                return resources\n\n            if mode == "Gaussian Flicker":\n                update_hz_req = float(\n                    temporal_cfg[\n                        "temporal_update_hz"\n                    ]\n                )\n                contrast_pct = float(\n                    temporal_cfg[\n                        "temporal_contrast_pct"\n                    ]\n                )\n                reps = int(\n                    temporal_cfg[\n                        "chirp_repetitions"\n                    ]\n                )\n                seed_base = int(\n                    temporal_cfg[\n                        "temporal_seed"\n                    ]\n                )\n\n                if update_hz_req <= 0:\n                    raise ValueError(\n                        "Gaussian Flicker: frecuencia de actualización debe ser > 0."\n                    )\n                if not (\n                    0.0\n                    <= contrast_pct\n                    <= 100.0\n                ):\n                    raise ValueError(\n                        "Gaussian Flicker: contraste debe estar entre 0 y 100 %."\n                    )\n                if reps < 1:\n                    raise ValueError(\n                        "Temporal: repeticiones debe ser >= 1."\n                    )\n\n                frames_per_update = max(\n                    1,\n                    int(\n                        round(\n                            hz\n                            / update_hz_req\n                        )\n                    )\n                )\n                actual_update_hz = (\n                    hz\n                    / frames_per_update\n                )\n                n_updates = max(\n                    1,\n                    int(\n                        math.ceil(\n                            duration_f\n                            / frames_per_update\n                        )\n                    )\n                )\n\n                sequences = []\n                used_seeds = []\n\n                for rep_idx in range(\n                    reps\n                ):\n                    seed = (\n                        seed_base\n                        + rep_idx\n                    )\n                    rng = np.random.default_rng(\n                        seed\n                    )\n                    vals = rng.normal(\n                        loc=0.0,\n                        scale=(\n                            1.0\n                            / 3.0\n                        ),\n                        size=n_updates\n                    ).astype(\n                        np.float32\n                    )\n                    vals = np.clip(\n                        vals,\n                        -1.0,\n                        1.0\n                    )\n                    sequences.append(\n                        vals\n                    )\n                    used_seeds.append(\n                        seed\n                    )\n\n                resources.update({\n                    "frames_per_update":\n                        frames_per_update,\n                    "actual_update_hz":\n                        actual_update_hz,\n                    "n_updates":\n                        n_updates,\n                    "gaussian_sequences":\n                        sequences,\n                    "used_seeds":\n                        used_seeds,\n                    "contrast_fraction":\n                        (\n                            contrast_pct\n                            / 100.0\n                        ),\n                })\n                return resources\n\n            if mode == "Contrast Series":\n                levels = parse_float_list(\n                    temporal_cfg[\n                        "temporal_contrast_levels_pct"\n                    ]\n                )\n                if not levels:\n                    raise ValueError(\n                        "Contrast Series: no hay niveles de contraste."\n                    )\n                if any(\n                    (\n                        level < 0.0\n                        or level > 100.0\n                    )\n                    for level in levels\n                ):\n                    raise ValueError(\n                        "Contrast Series: todos los contrastes deben estar entre 0 y 100 %."\n                    )\n\n                resources[\n                    "contrast_levels_pct"\n                ] = levels\n                return resources\n\n            raise ValueError(\n                f"Modo Temporal desconocido: {mode}"\n            )\n\n        try:\n            # Open the physical trigger backend only when a run starts.\n            # Simulation/disabled modes are no-ops here.\n            open_trigger_backend()\n\n            def prepare_protocol_step(step):\n                """\n                Build heavy PsychoPy/OpenGL resources before protocol timing\n                starts. Any warm-up draw is covered by the adaptation background before\n                the flip, so the experimental display never shows the\n                stimulus during preparation.\n                """\n                if step.get("kind") != "stimulus":\n                    return {\n                        "stimulus_type": "pause"\n                    }\n\n                step_cfg = dict(\n                    step.get("config", {})\n                )\n                stype = str(\n                    step_cfg.get(\n                        "stimulus_type",\n                        step.get(\n                            "stimulus_type", ""\n                        )\n                    )\n                )\n\n                prepared = {\n                    "stimulus_type": stype\n                }\n\n                def warm_draw(draw_callables):\n                    check_abort()\n                    for draw_callable in draw_callables:\n                        draw_callable()\n\n                    # Cover everything in the back buffer with the adaptation background\n                    # before flipping.\n                    fullfield.fillColor = GRAY\n                    fullfield.lineColor = GRAY\n                    fullfield.draw()\n                    win.flip()\n\n                if stype == "motion":\n                    mode = str(\n                        step_cfg[\n                            "motion_mode"\n                        ]\n                    )\n\n                    stim_rgb = rgb255_to_psychopy(\n                        step_cfg[\n                            "motion_stimulus_rgb"\n                        ]\n                    )\n\n                    prepared[\n                        "motion_mode"\n                    ] = mode\n\n                    if mode == "Moving Bar":\n                        bar = visual.Rect(\n                            win=win,\n                            width=float(\n                                step_cfg[\n                                    "motion_bar_length_deg"\n                                ]\n                            ),\n                            height=float(\n                                step_cfg[\n                                    "motion_bar_width_deg"\n                                ]\n                            ),\n                            units="deg",\n                            fillColor=stim_rgb,\n                            lineColor=stim_rgb,\n                            colorSpace="rgb",\n                            pos=(0, 0)\n                        )\n                        prepared[\n                            "motion_stim"\n                        ] = bar\n                        warm_draw([\n                            bar.draw\n                        ])\n\n                    else:\n                        if mode == "Moving Spot":\n                            diameter = float(\n                                step_cfg[\n                                    "motion_spot_diameter_deg"\n                                ]\n                            )\n                        elif mode in (\n                            "Expanding Annulus",\n                            "Receding Annulus",\n                        ):\n                            diameter = float(\n                                step_cfg.get(\n                                    "motion_annulus_large_diameter_deg",\n                                    40.0\n                                )\n                            )\n                        else:\n                            diameter = float(\n                                step_cfg[\n                                    "motion_large_diameter_deg"\n                                ]\n                            )\n\n                        spot = visual.Circle(\n                            win=win,\n                            radius=diameter / 2.0,\n                            units="deg",\n                            pos=(0, 0),\n                            fillColor=stim_rgb,\n                            lineColor=stim_rgb,\n                            colorSpace="rgb"\n                        )\n                        prepared[\n                            "motion_stim"\n                        ] = spot\n\n                        if mode in (\n                            "Expanding Annulus",\n                            "Receding Annulus",\n                        ):\n                            thickness = float(\n                                step_cfg.get(\n                                    "motion_annulus_thickness_deg",\n                                    2.0\n                                )\n                            )\n                            bg_rgb = rgb255_to_psychopy(\n                                step_cfg[\n                                    "motion_background_rgb"\n                                ]\n                            )\n                            inner_diameter = (\n                                diameter\n                                - 2.0\n                                * thickness\n                            )\n\n                            if inner_diameter <= 0:\n                                raise ValueError(\n                                    "Annulus: grosor radial incompatible "\n                                    "con el diámetro exterior."\n                                )\n\n                            inner = visual.Circle(\n                                win=win,\n                                radius=inner_diameter / 2.0,\n                                units="deg",\n                                pos=(0, 0),\n                                fillColor=bg_rgb,\n                                lineColor=bg_rgb,\n                                colorSpace="rgb"\n                            )\n                            prepared[\n                                "motion_inner_stim"\n                            ] = inner\n\n                            warm_draw([\n                                spot.draw,\n                                inner.draw,\n                            ])\n                        else:\n                            warm_draw([\n                                spot.draw\n                            ])\n\n                elif stype == "grating":\n                    mode = str(\n                        step_cfg.get(\n                            "grating_mode",\n                            "Drifting"\n                        )\n                    )\n                    if mode == "Gabor":\n                        mode = "Static Gabor"\n\n                    sf = float(\n                        step_cfg[\n                            "grating_sf_cpd"\n                        ]\n                    )\n                    adjustable_contrast = float(\n                        step_cfg[\n                            "grating_contrast"\n                        ]\n                    )\n                    absolute = bool(\n                        step_cfg[\n                            "grating_absolute_contrast"\n                        ]\n                    )\n                    waveform = str(\n                        step_cfg[\n                            "grating_waveform"\n                        ]\n                    )\n                    asymmetric_polarity = str(\n                        step_cfg.get(\n                            "grating_asymmetric_polarity",\n                            "A_TO_B"\n                        )\n                    )\n                    contrast_used = (\n                        1.0\n                        if absolute\n                        else adjustable_contrast\n                    )\n\n                    if mode == "Asymmetric Drift":\n                        texture = asymmetric_grating_texture(\n                            step_cfg[\n                                "grating_color_a_rgb"\n                            ],\n                            step_cfg[\n                                "grating_color_b_rgb"\n                            ],\n                            contrast_used,\n                            asymmetric_polarity\n                        )\n                    else:\n                        texture = colored_grating_texture(\n                            step_cfg[\n                                "grating_color_a_rgb"\n                            ],\n                            step_cfg[\n                                "grating_color_b_rgb"\n                            ],\n                            waveform,\n                            contrast_used\n                        )\n\n                    if mode in (\n                        "Static Gabor",\n                        "Drifting Gabor",\n                    ):\n                        grat_size = float(\n                            step_cfg.get(\n                                "grating_gabor_size_deg",\n                                30.0\n                            )\n                        )\n                        sigma = float(\n                            step_cfg.get(\n                                "grating_gabor_sigma_deg",\n                                6.0\n                            )\n                        )\n                        mask = gaussian_mask_deg(\n                            grat_size,\n                            sigma\n                        )\n                        pos = (\n                            float(\n                                step_cfg.get(\n                                    "grating_gabor_x_deg",\n                                    0.0\n                                )\n                            ),\n                            float(\n                                step_cfg.get(\n                                    "grating_gabor_y_deg",\n                                    0.0\n                                )\n                            )\n                        )\n                    else:\n                        grat_size = (\n                            2.2\n                            * math.hypot(\n                                width_deg,\n                                height_deg\n                            )\n                        )\n                        mask = None\n                        pos = (0, 0)\n\n                    grating = visual.GratingStim(\n                        win=win,\n                        tex=texture,\n                        mask=mask,\n                        units="deg",\n                        size=(\n                            grat_size,\n                            grat_size\n                        ),\n                        sf=sf,\n                        contrast=1.0,\n                        pos=pos,\n                        color=(1, 1, 1),\n                        colorSpace="rgb",\n                        interpolate=True\n                    )\n\n                    prepared[\n                        "grating_mode"\n                    ] = mode\n                    prepared[\n                        "texture"\n                    ] = texture\n                    prepared[\n                        "grating"\n                    ] = grating\n\n                    if mode in (\n                        "Static Gabor",\n                        "Drifting Gabor",\n                    ):\n                        prepared[\n                            "gabor_mask"\n                        ] = mask\n\n                    warm_draw([\n                        grating.draw\n                    ])\n\n                elif stype == "dense_noise":\n                    prepared.update(\n                        build_noise_resources(\n                            step_cfg\n                        )\n                    )\n\n                    warm_draw([\n                        prepared[\n                            "noise_stim"\n                        ].draw\n                    ])\n\n                    save_noise_sequence_before_timing(\n                        prepared,\n                        step_cfg\n                    )\n\n                elif stype == "rf_mapping":\n                    mode = str(\n                        step_cfg[\n                            "rf_mode"\n                        ]\n                    ).strip().lower()\n\n                    x_deg = float(\n                        step_cfg["rf_x_deg"]\n                    )\n                    y_deg = float(\n                        step_cfg["rf_y_deg"]\n                    )\n                    stim_rgb = rgb255_to_psychopy(\n                        step_cfg[\n                            "rf_stimulus_rgb"\n                        ]\n                    )\n                    bg_rgb = rgb255_to_psychopy(\n                        step_cfg[\n                            "rf_background_rgb"\n                        ]\n                    )\n\n                    prepared["rf_mode"] = (\n                        mode\n                    )\n\n                    if mode == "spot":\n                        diameter = float(\n                            step_cfg[\n                                "rf_spot_diameter_deg"\n                            ]\n                        )\n\n                        spot = visual.Circle(\n                            win=win,\n                            radius=(\n                                diameter\n                                / 2.0\n                            ),\n                            units="deg",\n                            pos=(\n                                x_deg,\n                                y_deg\n                            ),\n                            fillColor=stim_rgb,\n                            lineColor=stim_rgb,\n                            colorSpace="rgb"\n                        )\n                        prepared[\n                            "spot"\n                        ] = spot\n                        warm_draw([\n                            spot.draw\n                        ])\n\n                    elif mode == "annulus":\n                        inner = float(\n                            step_cfg[\n                                "rf_annulus_inner_deg"\n                            ]\n                        )\n                        outer = float(\n                            step_cfg[\n                                "rf_annulus_outer_deg"\n                            ]\n                        )\n\n                        outer_circle = (\n                            visual.Circle(\n                                win=win,\n                                radius=(\n                                    outer\n                                    / 2.0\n                                ),\n                                units="deg",\n                                pos=(\n                                    x_deg,\n                                    y_deg\n                                ),\n                                fillColor=stim_rgb,\n                                lineColor=stim_rgb,\n                                colorSpace="rgb"\n                            )\n                        )\n                        inner_circle = (\n                            visual.Circle(\n                                win=win,\n                                radius=(\n                                    inner\n                                    / 2.0\n                                ),\n                                units="deg",\n                                pos=(\n                                    x_deg,\n                                    y_deg\n                                ),\n                                fillColor=bg_rgb,\n                                lineColor=bg_rgb,\n                                colorSpace="rgb"\n                            )\n                        )\n\n                        prepared[\n                            "outer_circle"\n                        ] = outer_circle\n                        prepared[\n                            "inner_circle"\n                        ] = inner_circle\n                        warm_draw([\n                            outer_circle.draw,\n                            inner_circle.draw\n                        ])\n\n                    elif mode == "annulus size series":\n                        sizes = parse_float_list(\n                            step_cfg[\n                                "rf_annulus_size_series_deg"\n                            ]\n                        )\n                        thickness = float(\n                            step_cfg[\n                                "rf_annulus_series_thickness_deg"\n                            ]\n                        )\n\n                        if not sizes:\n                            raise ValueError(\n                                "RF Annulus Size Series: no hay diámetros."\n                            )\n\n                        if (\n                            thickness <= 0\n                            or any(\n                                2.0 * thickness >= diameter\n                                for diameter in sizes\n                            )\n                        ):\n                            raise ValueError(\n                                "RF Annulus Size Series: grosor incompatible "\n                                "con los diámetros."\n                            )\n\n                        # Build and prewarm ONE independent outer/inner\n                        # Circle pair for EVERY requested diameter. This\n                        # avoids changing Circle.radius for the first time\n                        # during a timed condition.\n                        annulus_size_pairs = []\n\n                        for diameter in sizes:\n                            inner_diameter = (\n                                diameter\n                                - 2.0 * thickness\n                            )\n\n                            outer_circle = visual.Circle(\n                                win=win,\n                                radius=diameter / 2.0,\n                                units="deg",\n                                pos=(\n                                    x_deg,\n                                    y_deg\n                                ),\n                                fillColor=stim_rgb,\n                                lineColor=stim_rgb,\n                                colorSpace="rgb"\n                            )\n                            inner_circle = visual.Circle(\n                                win=win,\n                                radius=inner_diameter / 2.0,\n                                units="deg",\n                                pos=(\n                                    x_deg,\n                                    y_deg\n                                ),\n                                fillColor=bg_rgb,\n                                lineColor=bg_rgb,\n                                colorSpace="rgb"\n                            )\n\n                            annulus_size_pairs.append(\n                                (\n                                    float(diameter),\n                                    outer_circle,\n                                    inner_circle,\n                                )\n                            )\n\n                            warm_draw([\n                                outer_circle.draw,\n                                inner_circle.draw\n                            ])\n\n                        prepared[\n                            "annulus_size_series"\n                        ] = sizes\n                        prepared[\n                            "annulus_size_thickness"\n                        ] = thickness\n                        prepared[\n                            "annulus_size_pairs"\n                        ] = annulus_size_pairs\n\n                    elif mode == "size series":\n                        sizes = parse_float_list(\n                            step_cfg[\n                                "rf_size_series_deg"\n                            ]\n                        )\n                        size_spots = []\n\n                        for diameter in sizes:\n                            spot = visual.Circle(\n                                win=win,\n                                radius=(\n                                    diameter\n                                    / 2.0\n                                ),\n                                units="deg",\n                                pos=(\n                                    x_deg,\n                                    y_deg\n                                ),\n                                fillColor=stim_rgb,\n                                lineColor=stim_rgb,\n                                colorSpace="rgb"\n                            )\n                            size_spots.append(\n                                spot\n                            )\n\n                        prepared[\n                            "size_spots"\n                        ] = size_spots\n\n                        if size_spots:\n                            warm_draw([\n                                spot.draw\n                                for spot\n                                in size_spots\n                            ])\n\n                    elif mode == "sparse noise":\n                        square_size = float(\n                            step_cfg[\n                                "rf_sparse_square_deg"\n                            ]\n                        )\n                        grid_step = float(\n                            step_cfg[\n                                "rf_sparse_grid_step_deg"\n                            ]\n                        )\n\n                        half = (\n                            square_size\n                            / 2.0\n                        )\n                        x_min = (\n                            -width_deg\n                            / 2.0\n                            + half\n                        )\n                        x_max = (\n                            width_deg\n                            / 2.0\n                            - half\n                        )\n                        y_min = (\n                            -height_deg\n                            / 2.0\n                            + half\n                        )\n                        y_max = (\n                            height_deg\n                            / 2.0\n                            - half\n                        )\n\n                        xs = np.arange(\n                            x_min,\n                            x_max\n                            + grid_step\n                            * 0.25,\n                            grid_step\n                        )\n                        ys = np.arange(\n                            y_min,\n                            y_max\n                            + grid_step\n                            * 0.25,\n                            grid_step\n                        )\n\n                        if len(xs) == 0:\n                            xs = np.asarray(\n                                [0.0]\n                            )\n                        if len(ys) == 0:\n                            ys = np.asarray(\n                                [0.0]\n                            )\n\n                        positions = [\n                            (\n                                float(x),\n                                float(y)\n                            )\n                            for y in ys\n                            for x in xs\n                        ]\n\n                        sparse = visual.Rect(\n                            win=win,\n                            width=square_size,\n                            height=square_size,\n                            units="deg",\n                            fillColor=WHITE,\n                            lineColor=WHITE,\n                            colorSpace="rgb",\n                            pos=(0, 0)\n                        )\n\n                        prepared[\n                            "positions"\n                        ] = positions\n                        prepared[\n                            "sparse"\n                        ] = sparse\n                        warm_draw([\n                            sparse.draw\n                        ])\n\n                elif stype == "chirp_intensity":\n                    prepared.update(\n                        build_temporal_resources(\n                            step_cfg\n                        )\n                    )\n\n                # ON/OFF uses the already-existing full-field object.\n                return prepared\n\n            def run_current_stimulus():\n                global details\n                # ==========================================================\n                # ON / OFF\n                # ==========================================================\n                if stimulus_type == "on_off":\n                    baseline_f = seconds_to_frames(\n                        cfg["baseline_s"], hz\n                    )\n                    on_f = seconds_to_frames(\n                        cfg["on_s"], hz\n                    )\n                    gray_f = seconds_to_frames(\n                        cfg["gray_interval_s"], hz\n                    )\n                    off_f = seconds_to_frames(\n                        cfg["off_s"], hz\n                    )\n                    final_f = seconds_to_frames(\n                        cfg["final_gray_s"], hz\n                    )\n                    reps = int(\n                        cfg["repetitions"]\n                    )\n\n                    on_color = rgb255_to_psychopy(\n                        cfg["on_color_rgb"]\n                    )\n                    off_color = rgb255_to_psychopy(\n                        cfg["off_color_rgb"]\n                    )\n\n                    details = {\n                        "baseline_frames":\n                            baseline_f,\n                        "on_frames":\n                            on_f,\n                        "gray_interval_frames":\n                            gray_f,\n                        "off_frames":\n                            off_f,\n                        "final_gray_frames":\n                            final_f,\n                        "on_color_rgb_255":\n                            cfg["on_color_rgb"],\n                        "off_color_rgb_255":\n                            cfg["off_color_rgb"],\n                    }\n\n                    for rep in range(\n                        1, reps + 1\n                    ):\n                        solid_frames(\n                            GRAY,\n                            baseline_f,\n                            "on_off",\n                            rep,\n                            "ON/OFF",\n                            "baseline_gray"\n                        )\n\n                        if on_f > 0:\n                            queue_ttl_event(\n                                "ON_START",\n                                rep,\n                                "ON"\n                            )\n\n                        solid_frames(\n                            on_color,\n                            on_f,\n                            "on_off",\n                            rep,\n                            "ON",\n                            "on"\n                        )\n\n                        solid_frames(\n                            GRAY,\n                            gray_f,\n                            "on_off",\n                            rep,\n                            "ON/OFF",\n                            "gray_interval"\n                        )\n\n                        if off_f > 0:\n                            queue_ttl_event(\n                                "OFF_START",\n                                rep,\n                                "OFF"\n                            )\n\n                        solid_frames(\n                            off_color,\n                            off_f,\n                            "on_off",\n                            rep,\n                            "OFF",\n                            "off"\n                        )\n\n                        solid_frames(\n                            GRAY,\n                            final_f,\n                            "on_off",\n                            rep,\n                            "ON/OFF",\n                            "final_gray"\n                        )\n\n                        write_json_atomic(\n                            status_path,\n                            {\n                                "state":\n                                    "running",\n                                "message":\n                                    f"ON/OFF · repetición "\n                                    f"{rep}/{reps}",\n                                "hz": hz,\n                                "command_id":\n                                    command_id,\n                                "details":\n                                    details\n                            }\n                        )\n\n                elif stimulus_type == "motion":\n                    mode = str(\n                        cfg["motion_mode"]\n                    )\n                    baseline_f = seconds_to_frames(\n                        cfg["motion_baseline_s"],\n                        hz\n                    )\n                    stim_f = seconds_to_frames(\n                        cfg["motion_duration_s"],\n                        hz\n                    )\n                    isi_f = seconds_to_frames(\n                        cfg["motion_isi_s"],\n                        hz\n                    )\n                    reps = int(\n                        cfg["motion_repetitions"]\n                    )\n\n                    background_rgb = rgb255_to_psychopy(\n                        cfg[\n                            "motion_background_rgb"\n                        ]\n                    )\n                    stimulus_rgb = rgb255_to_psychopy(\n                        cfg[\n                            "motion_stimulus_rgb"\n                        ]\n                    )\n\n                    if stim_f <= 0:\n                        raise ValueError(\n                            "Motion: la duración debe producir "\n                            "al menos 1 frame."\n                        )\n                    if reps < 1:\n                        raise ValueError(\n                            "Motion: repeticiones debe ser >= 1."\n                        )\n\n                    if (\n                        prepared_step is not None\n                        and prepared_step.get(\n                            "stimulus_type"\n                        ) == "motion"\n                        and "motion_stim"\n                        in prepared_step\n                    ):\n                        motion_stim = prepared_step[\n                            "motion_stim"\n                        ]\n                        motion_inner_stim = (\n                            prepared_step.get(\n                                "motion_inner_stim"\n                            )\n                        )\n                    else:\n                        motion_stim = None\n                        motion_inner_stim = None\n\n                    details = {\n                        "mode": mode,\n                        "baseline_frames":\n                            baseline_f,\n                        "stimulus_frames":\n                            stim_f,\n                        "isi_frames":\n                            isi_f,\n                        "repetitions":\n                            reps,\n                        "background_rgb_255":\n                            cfg[\n                                "motion_background_rgb"\n                            ],\n                        "stimulus_rgb_255":\n                            cfg[\n                                "motion_stimulus_rgb"\n                            ],\n                    }\n\n                    fullfield.fillColor = (\n                        background_rgb\n                    )\n                    fullfield.lineColor = (\n                        background_rgb\n                    )\n\n                    trial = 0\n\n                    if mode in (\n                        "Moving Bar",\n                        "Moving Spot",\n                    ):\n                        directions = parse_float_list(\n                            cfg[\n                                "motion_directions"\n                            ]\n                        )\n                        if not directions:\n                            raise ValueError(\n                                "Motion: no hay direcciones."\n                            )\n\n                        details[\n                            "directions"\n                        ] = directions\n\n                        if mode == "Moving Bar":\n                            bar_width = float(\n                                cfg[\n                                    "motion_bar_width_deg"\n                                ]\n                            )\n                            bar_length = float(\n                                cfg[\n                                    "motion_bar_length_deg"\n                                ]\n                            )\n\n                            if (\n                                bar_width <= 0\n                                or bar_length <= 0\n                            ):\n                                raise ValueError(\n                                    "Motion: dimensiones de barra "\n                                    "deben ser > 0."\n                                )\n\n                            details[\n                                "bar_width_deg"\n                            ] = bar_width\n                            details[\n                                "bar_length_deg"\n                            ] = bar_length\n\n                            if motion_stim is None:\n                                motion_stim = visual.Rect(\n                                    win=win,\n                                    width=bar_length,\n                                    height=bar_width,\n                                    units="deg",\n                                    fillColor=stimulus_rgb,\n                                    lineColor=stimulus_rgb,\n                                    colorSpace="rgb",\n                                    pos=(0, 0)\n                                )\n\n                        else:\n                            spot_diameter = float(\n                                cfg[\n                                    "motion_spot_diameter_deg"\n                                ]\n                            )\n                            if spot_diameter <= 0:\n                                raise ValueError(\n                                    "Motion: diámetro del spot "\n                                    "debe ser > 0."\n                                )\n\n                            details[\n                                "spot_diameter_deg"\n                            ] = spot_diameter\n\n                            if motion_stim is None:\n                                motion_stim = visual.Circle(\n                                    win=win,\n                                    radius=(\n                                        spot_diameter\n                                        / 2.0\n                                    ),\n                                    units="deg",\n                                    pos=(0, 0),\n                                    fillColor=stimulus_rgb,\n                                    lineColor=stimulus_rgb,\n                                    colorSpace="rgb"\n                                )\n\n                        for rep in range(\n                            1, reps + 1\n                        ):\n                            for direction in directions:\n                                trial += 1\n\n                                condition = (\n                                    f"{mode} "\n                                    f"{direction:g}°"\n                                )\n\n                                solid_frames(\n                                    GRAY,\n                                    baseline_f,\n                                    "motion",\n                                    trial,\n                                    condition,\n                                    "baseline_gray"\n                                )\n\n                                theta = math.radians(\n                                    direction\n                                )\n                                dx = math.cos(theta)\n                                dy = math.sin(theta)\n\n                                screen_half_extent = (\n                                    abs(dx)\n                                    * width_deg\n                                    / 2.0\n                                    + abs(dy)\n                                    * height_deg\n                                    / 2.0\n                                )\n\n                                if mode == "Moving Bar":\n                                    radius_along_motion = (\n                                        bar_width\n                                        / 2.0\n                                    )\n                                    motion_stim.ori = (\n                                        90.0\n                                        - direction\n                                    )\n                                    ttl_name = (\n                                        "MOTION_BAR_START"\n                                    )\n                                    stage_name = (\n                                        "moving_bar"\n                                    )\n                                else:\n                                    radius_along_motion = (\n                                        spot_diameter\n                                        / 2.0\n                                    )\n                                    ttl_name = (\n                                        "MOTION_SPOT_START"\n                                    )\n                                    stage_name = (\n                                        "moving_spot"\n                                    )\n\n                                travel_half = (\n                                    screen_half_extent\n                                    + radius_along_motion\n                                )\n\n                                queue_ttl_event(\n                                    ttl_name,\n                                    trial,\n                                    condition\n                                )\n\n                                for f in range(\n                                    1, stim_f + 1\n                                ):\n                                    check_abort()\n\n                                    u = (\n                                        0.0\n                                        if stim_f <= 1\n                                        else (\n                                            (f - 1)\n                                            / (stim_f - 1)\n                                        )\n                                    )\n                                    scalar = (\n                                        -travel_half\n                                        + 2.0\n                                        * travel_half\n                                        * u\n                                    )\n\n                                    motion_stim.pos = (\n                                        dx * scalar,\n                                        dy * scalar\n                                    )\n\n                                    # Baseline/ISI are neutral gray, but the\n                                    # stimulus background is user-defined.\n                                    # Reapply it explicitly on every frame.\n                                    fullfield.fillColor = background_rgb\n                                    fullfield.lineColor = background_rgb\n                                    fullfield.draw()\n                                    motion_stim.draw()\n                                    flip_and_log(\n                                        "motion",\n                                        trial,\n                                        condition,\n                                        stage_name,\n                                        f\n                                    )\n\n                                solid_frames(\n                                    GRAY,\n                                    isi_f,\n                                    "motion",\n                                    trial,\n                                    condition,\n                                    "inter_trial_gray"\n                                )\n\n                                write_json_atomic(\n                                    status_path,\n                                    {\n                                        "state":\n                                            "running",\n                                        "message":\n                                            f"{mode} · "\n                                            f"rep {rep}/{reps} · "\n                                            f"{direction:g}°",\n                                        "hz": hz,\n                                        "command_id":\n                                            command_id,\n                                        "details":\n                                            details\n                                    }\n                                )\n\n                    elif mode in (\n                        "Looming",\n                        "Receding",\n                    ):\n                        small_diameter = float(\n                            cfg[\n                                "motion_small_diameter_deg"\n                            ]\n                        )\n                        large_diameter = float(\n                            cfg[\n                                "motion_large_diameter_deg"\n                            ]\n                        )\n                        center_x = float(\n                            cfg[\n                                "motion_center_x_deg"\n                            ]\n                        )\n                        center_y = float(\n                            cfg[\n                                "motion_center_y_deg"\n                            ]\n                        )\n\n                        if (\n                            small_diameter <= 0\n                            or large_diameter\n                            <= small_diameter\n                        ):\n                            raise ValueError(\n                                "Motion: diámetro grande > "\n                                "diámetro pequeño > 0."\n                            )\n\n                        details.update({\n                            "small_diameter_deg":\n                                small_diameter,\n                            "large_diameter_deg":\n                                large_diameter,\n                            "center_x_deg":\n                                center_x,\n                            "center_y_deg":\n                                center_y,\n                        })\n\n                        if motion_stim is None:\n                            motion_stim = visual.Circle(\n                                win=win,\n                                radius=(\n                                    large_diameter\n                                    / 2.0\n                                ),\n                                units="deg",\n                                pos=(\n                                    center_x,\n                                    center_y\n                                ),\n                                fillColor=stimulus_rgb,\n                                lineColor=stimulus_rgb,\n                                colorSpace="rgb"\n                            )\n\n                        motion_stim.pos = (\n                            center_x,\n                            center_y\n                        )\n\n                        for rep in range(\n                            1, reps + 1\n                        ):\n                            trial += 1\n                            condition = mode\n\n                            solid_frames(\n                                GRAY,\n                                baseline_f,\n                                "motion",\n                                trial,\n                                condition,\n                                "baseline_gray"\n                            )\n\n                            if mode == "Looming":\n                                ttl_name = (\n                                    "LOOMING_START"\n                                )\n                                stage_name = (\n                                    "looming"\n                                )\n                            else:\n                                ttl_name = (\n                                    "RECEDING_START"\n                                )\n                                stage_name = (\n                                    "receding"\n                                )\n\n                            queue_ttl_event(\n                                ttl_name,\n                                trial,\n                                condition\n                            )\n\n                            for f in range(\n                                1, stim_f + 1\n                            ):\n                                check_abort()\n\n                                u = (\n                                    0.0\n                                    if stim_f <= 1\n                                    else (\n                                        (f - 1)\n                                        / (stim_f - 1)\n                                    )\n                                )\n\n                                if mode == "Looming":\n                                    diameter = (\n                                        small_diameter\n                                        + (\n                                            large_diameter\n                                            - small_diameter\n                                        )\n                                        * u\n                                    )\n                                else:\n                                    diameter = (\n                                        large_diameter\n                                        - (\n                                            large_diameter\n                                            - small_diameter\n                                        )\n                                        * u\n                                    )\n\n                                motion_stim.radius = (\n                                    diameter\n                                    / 2.0\n                                )\n\n                                fullfield.fillColor = background_rgb\n                                fullfield.lineColor = background_rgb\n                                fullfield.draw()\n                                motion_stim.draw()\n                                flip_and_log(\n                                    "motion",\n                                    trial,\n                                    condition,\n                                    stage_name,\n                                    f\n                                )\n\n                            solid_frames(\n                                GRAY,\n                                isi_f,\n                                "motion",\n                                trial,\n                                condition,\n                                "inter_trial_gray"\n                            )\n\n                            write_json_atomic(\n                                status_path,\n                                {\n                                    "state":\n                                        "running",\n                                    "message":\n                                        f"{mode} · "\n                                        f"rep {rep}/{reps}",\n                                    "hz": hz,\n                                    "command_id":\n                                        command_id,\n                                    "details":\n                                        details\n                                }\n                            )\n\n                    elif mode in (\n                        "Expanding Annulus",\n                        "Receding Annulus",\n                    ):\n                        small_diameter = float(\n                            cfg.get(\n                                "motion_annulus_small_diameter_deg",\n                                8.0\n                            )\n                        )\n                        large_diameter = float(\n                            cfg.get(\n                                "motion_annulus_large_diameter_deg",\n                                40.0\n                            )\n                        )\n                        thickness = float(\n                            cfg.get(\n                                "motion_annulus_thickness_deg",\n                                2.0\n                            )\n                        )\n                        center_x = float(\n                            cfg[\n                                "motion_center_x_deg"\n                            ]\n                        )\n                        center_y = float(\n                            cfg[\n                                "motion_center_y_deg"\n                            ]\n                        )\n\n                        if (\n                            small_diameter <= 0\n                            or large_diameter\n                            <= small_diameter\n                        ):\n                            raise ValueError(\n                                "Annulus: diámetro exterior grande > "\n                                "diámetro exterior pequeño > 0."\n                            )\n\n                        if thickness <= 0:\n                            raise ValueError(\n                                "Annulus: el grosor radial debe ser > 0."\n                            )\n\n                        if (\n                            2.0 * thickness\n                            >= small_diameter\n                        ):\n                            raise ValueError(\n                                "Annulus: el grosor radial debe ser menor "\n                                "que la mitad del diámetro exterior pequeño."\n                            )\n\n                        actual_duration_s = (\n                            stim_f / hz\n                        )\n\n                        details.update({\n                            "small_outer_diameter_deg":\n                                small_diameter,\n                            "large_outer_diameter_deg":\n                                large_diameter,\n                            "annulus_thickness_deg":\n                                thickness,\n                            "small_inner_diameter_deg":\n                                (\n                                    small_diameter\n                                    - 2.0 * thickness\n                                ),\n                            "large_inner_diameter_deg":\n                                (\n                                    large_diameter\n                                    - 2.0 * thickness\n                                ),\n                            "center_x_deg":\n                                center_x,\n                            "center_y_deg":\n                                center_y,\n                            "diameter_interpolation":\n                                "linear_outer_diameter",\n                            "thickness_definition":\n                                "constant_radial_thickness",\n                            "outer_diameter_speed_deg_s":\n                                (\n                                    (\n                                        large_diameter\n                                        - small_diameter\n                                    )\n                                    / actual_duration_s\n                                ),\n                            "radial_edge_speed_deg_s":\n                                (\n                                    (\n                                        large_diameter\n                                        - small_diameter\n                                    )\n                                    / 2.0\n                                    / actual_duration_s\n                                ),\n                        })\n\n                        if motion_stim is None:\n                            motion_stim = visual.Circle(\n                                win=win,\n                                radius=large_diameter / 2.0,\n                                units="deg",\n                                pos=(\n                                    center_x,\n                                    center_y\n                                ),\n                                fillColor=stimulus_rgb,\n                                lineColor=stimulus_rgb,\n                                colorSpace="rgb"\n                            )\n\n                        if motion_inner_stim is None:\n                            motion_inner_stim = visual.Circle(\n                                win=win,\n                                radius=(\n                                    (\n                                        large_diameter\n                                        - 2.0 * thickness\n                                    )\n                                    / 2.0\n                                ),\n                                units="deg",\n                                pos=(\n                                    center_x,\n                                    center_y\n                                ),\n                                fillColor=background_rgb,\n                                lineColor=background_rgb,\n                                colorSpace="rgb"\n                            )\n\n                        motion_stim.pos = (\n                            center_x,\n                            center_y\n                        )\n                        motion_inner_stim.pos = (\n                            center_x,\n                            center_y\n                        )\n\n                        for rep in range(\n                            1, reps + 1\n                        ):\n                            trial += 1\n                            condition = mode\n\n                            solid_frames(\n                                GRAY,\n                                baseline_f,\n                                "motion",\n                                trial,\n                                condition,\n                                "baseline_gray"\n                            )\n\n                            if mode == "Expanding Annulus":\n                                ttl_name = (\n                                    "EXPANDING_ANNULUS_START"\n                                )\n                                stage_name = (\n                                    "expanding_annulus"\n                                )\n                            else:\n                                ttl_name = (\n                                    "RECEDING_ANNULUS_START"\n                                )\n                                stage_name = (\n                                    "receding_annulus"\n                                )\n\n                            queue_ttl_event(\n                                ttl_name,\n                                trial,\n                                condition\n                            )\n\n                            for f in range(\n                                1, stim_f + 1\n                            ):\n                                check_abort()\n\n                                u = (\n                                    0.0\n                                    if stim_f <= 1\n                                    else (\n                                        (f - 1)\n                                        / (stim_f - 1)\n                                    )\n                                )\n\n                                if mode == "Expanding Annulus":\n                                    outer_diameter = (\n                                        small_diameter\n                                        + (\n                                            large_diameter\n                                            - small_diameter\n                                        )\n                                        * u\n                                    )\n                                else:\n                                    outer_diameter = (\n                                        large_diameter\n                                        - (\n                                            large_diameter\n                                            - small_diameter\n                                        )\n                                        * u\n                                    )\n\n                                inner_diameter = (\n                                    outer_diameter\n                                    - 2.0 * thickness\n                                )\n\n                                motion_stim.radius = (\n                                    outer_diameter / 2.0\n                                )\n                                motion_inner_stim.radius = (\n                                    inner_diameter / 2.0\n                                )\n\n                                fullfield.fillColor = background_rgb\n                                fullfield.lineColor = background_rgb\n                                fullfield.draw()\n                                motion_stim.draw()\n                                motion_inner_stim.draw()\n\n                                flip_and_log(\n                                    "motion",\n                                    trial,\n                                    condition,\n                                    stage_name,\n                                    f\n                                )\n\n                            solid_frames(\n                                GRAY,\n                                isi_f,\n                                "motion",\n                                trial,\n                                condition,\n                                "inter_trial_gray"\n                            )\n\n                            write_json_atomic(\n                                status_path,\n                                {\n                                    "state":\n                                        "running",\n                                    "message":\n                                        f"{mode} · "\n                                        f"rep {rep}/{reps}",\n                                    "hz": hz,\n                                    "command_id":\n                                        command_id,\n                                    "details":\n                                        details\n                                }\n                            )\n\n                    else:\n                        raise ValueError(\n                            f"Modo Motion desconocido: "\n                            f"{mode}"\n                        )\n\n                # ==========================================================\n                # GRATING\n                # ==========================================================\n                elif stimulus_type == "grating":\n                    mode = str(\n                        cfg.get(\n                            "grating_mode",\n                            "Drifting"\n                        )\n                    )\n                    if mode == "Gabor":\n                        mode = "Static Gabor"\n\n                    baseline_f = seconds_to_frames(\n                        cfg[\n                            "grating_baseline_s"\n                        ],\n                        hz\n                    )\n                    stim_f = seconds_to_frames(\n                        cfg[\n                            "grating_duration_s"\n                        ],\n                        hz\n                    )\n                    isi_f = seconds_to_frames(\n                        cfg[\n                            "grating_isi_s"\n                        ],\n                        hz\n                    )\n                    reps = int(\n                        cfg[\n                            "grating_repetitions"\n                        ]\n                    )\n\n                    angles_text = cfg.get(\n                        "grating_angles",\n                        cfg.get(\n                            "grating_directions",\n                            ""\n                        )\n                    )\n                    angles = parse_float_list(\n                        angles_text\n                    )\n\n                    sf = float(\n                        cfg[\n                            "grating_sf_cpd"\n                        ]\n                    )\n                    tf = float(\n                        cfg.get(\n                            "grating_tf_hz",\n                            1.0\n                        )\n                    )\n                    reversal_hz = float(\n                        cfg.get(\n                            "grating_reversal_hz",\n                            2.0\n                        )\n                    )\n                    phase_initial = float(\n                        cfg.get(\n                            "grating_phase_cycles",\n                            0.0\n                        )\n                    )\n\n                    adjustable_contrast = float(\n                        cfg[\n                            "grating_contrast"\n                        ]\n                    )\n                    absolute = bool(\n                        cfg[\n                            "grating_absolute_contrast"\n                        ]\n                    )\n                    waveform = str(\n                        cfg[\n                            "grating_waveform"\n                        ]\n                    )\n                    asymmetric_polarity = str(\n                        cfg.get(\n                            "grating_asymmetric_polarity",\n                            "A_TO_B"\n                        )\n                    ).strip().upper()\n                    if asymmetric_polarity not in (\n                        "A_TO_B",\n                        "B_TO_A",\n                    ):\n                        raise ValueError(\n                            "Asymmetric Drift: polaridad desconocida."\n                        )\n\n                    if not angles:\n                        raise ValueError(\n                            "Grating: no hay direcciones/orientaciones."\n                        )\n                    if sf <= 0:\n                        raise ValueError(\n                            "Grating: frecuencia espacial debe ser > 0."\n                        )\n                    if reps < 1:\n                        raise ValueError(\n                            "Grating: repeticiones debe ser >= 1."\n                        )\n                    if not (\n                        0.0\n                        <= adjustable_contrast\n                        <= 1.0\n                    ):\n                        raise ValueError(\n                            "Grating: contraste debe estar entre 0 y 1."\n                        )\n\n                    if (\n                        mode in (\n                            "Drifting",\n                            "Drifting Gabor",\n                            "Asymmetric Drift",\n                        )\n                        and tf <= 0\n                    ):\n                        raise ValueError(\n                            f"{mode}: frecuencia temporal debe ser > 0."\n                        )\n\n                    if (\n                        mode == "Phase Reversal"\n                        and reversal_hz <= 0\n                    ):\n                        raise ValueError(\n                            "Phase Reversal: frecuencia de inversión "\n                            "debe ser > 0."\n                        )\n\n                    contrast_used = (\n                        1.0\n                        if absolute\n                        else adjustable_contrast\n                    )\n\n                    gabor_size = float(\n                        cfg.get(\n                            "grating_gabor_size_deg",\n                            30.0\n                        )\n                    )\n                    gabor_sigma = float(\n                        cfg.get(\n                            "grating_gabor_sigma_deg",\n                            6.0\n                        )\n                    )\n                    gabor_x = float(\n                        cfg.get(\n                            "grating_gabor_x_deg",\n                            0.0\n                        )\n                    )\n                    gabor_y = float(\n                        cfg.get(\n                            "grating_gabor_y_deg",\n                            0.0\n                        )\n                    )\n\n                    if mode in (\n                        "Static Gabor",\n                        "Drifting Gabor",\n                    ):\n                        if (\n                            gabor_size <= 0\n                            or gabor_sigma <= 0\n                            or gabor_sigma\n                            > gabor_size / 2.0\n                        ):\n                            raise ValueError(\n                                f"{mode}: tamaño > 0 y "\n                                "0 < sigma <= tamaño/2."\n                            )\n\n                    if (\n                        prepared_step is not None\n                        and prepared_step.get(\n                            "stimulus_type"\n                        ) == "grating"\n                        and "grating"\n                        in prepared_step\n                    ):\n                        texture = prepared_step[\n                            "texture"\n                        ]\n                        grating = prepared_step[\n                            "grating"\n                        ]\n                    else:\n                        if mode == "Asymmetric Drift":\n                            texture = asymmetric_grating_texture(\n                                cfg[\n                                    "grating_color_a_rgb"\n                                ],\n                                cfg[\n                                    "grating_color_b_rgb"\n                                ],\n                                contrast_used,\n                                asymmetric_polarity\n                            )\n                        else:\n                            texture = colored_grating_texture(\n                                cfg[\n                                    "grating_color_a_rgb"\n                                ],\n                                cfg[\n                                    "grating_color_b_rgb"\n                                ],\n                                waveform,\n                                contrast_used\n                            )\n\n                        if mode in (\n                            "Static Gabor",\n                            "Drifting Gabor",\n                        ):\n                            grat_size = gabor_size\n                            mask = gaussian_mask_deg(\n                                gabor_size,\n                                gabor_sigma\n                            )\n                            pos = (\n                                gabor_x,\n                                gabor_y\n                            )\n                        else:\n                            grat_size = (\n                                2.2\n                                * math.hypot(\n                                    width_deg,\n                                    height_deg\n                                )\n                            )\n                            mask = None\n                            pos = (0, 0)\n\n                        grating = visual.GratingStim(\n                            win=win,\n                            tex=texture,\n                            mask=mask,\n                            units="deg",\n                            size=(\n                                grat_size,\n                                grat_size\n                            ),\n                            sf=sf,\n                            contrast=1.0,\n                            pos=pos,\n                            color=(1, 1, 1),\n                            colorSpace="rgb",\n                            interpolate=True\n                        )\n\n                    if mode in (\n                        "Static Gabor",\n                        "Drifting Gabor",\n                    ):\n                        gabor_bg_rgb_255 = cfg.get(\n                            "grating_gabor_background_rgb",\n                            [128, 128, 128]\n                        )\n                        gabor_bg = rgb255_to_psychopy(\n                            gabor_bg_rgb_255\n                        )\n                        grating.pos = (\n                            gabor_x,\n                            gabor_y\n                        )\n                    else:\n                        gabor_bg_rgb_255 = None\n                        gabor_bg = None\n                        grating.pos = (0, 0)\n\n                    effective_s = (\n                        stim_f / hz\n                        if stim_f\n                        else 0.0\n                    )\n\n                    details = {\n                        "mode": mode,\n                        "stimulus_frames":\n                            stim_f,\n                        "requested_duration_s":\n                            float(\n                                cfg[\n                                    "grating_duration_s"\n                                ]\n                            ),\n                        "effective_duration_s":\n                            effective_s,\n                        "angles":\n                            angles,\n                        "sf_cpd":\n                            sf,\n                        "tf_hz":\n                            (\n                                tf\n                                if mode in (\n                                    "Drifting",\n                                    "Drifting Gabor",\n                                    "Asymmetric Drift",\n                                )\n                                else None\n                            ),\n                        "reversal_hz":\n                            (\n                                reversal_hz\n                                if mode\n                                == "Phase Reversal"\n                                else None\n                            ),\n                        "phase_cycles":\n                            phase_initial,\n                        "contrast_used":\n                            contrast_used,\n                        "absolute_contrast":\n                            absolute,\n                        "waveform":\n                            waveform,\n                        "color_a_rgb_255":\n                            cfg[\n                                "grating_color_a_rgb"\n                            ],\n                        "color_b_rgb_255":\n                            cfg[\n                                "grating_color_b_rgb"\n                            ],\n                        "asymmetric_polarity":\n                            (\n                                asymmetric_polarity\n                                if mode == "Asymmetric Drift"\n                                else None\n                            ),\n                        "asymmetric_profile":\n                            (\n                                "half_cosine_smooth_transition_plus_abrupt_reset"\n                                if mode == "Asymmetric Drift"\n                                else None\n                            ),\n                    }\n\n                    if mode in (\n                        "Drifting",\n                        "Drifting Gabor",\n                        "Asymmetric Drift",\n                    ):\n                        details[\n                            "angle_semantics"\n                        ] = (\n                            "movement_direction_deg; "\n                            "0=izquierda→derecha; "\n                            "positivo antihorario"\n                        )\n                        details[\n                            "phase_cycles_per_frame"\n                        ] = (\n                            tf / hz\n                        )\n                    else:\n                        details[\n                            "angle_semantics"\n                        ] = (\n                            "bar_orientation_deg; "\n                            "0=barras verticales; "\n                            "positivo antihorario"\n                        )\n\n                    if mode == "Phase Reversal":\n                        details[\n                            "expected_frames_per_reversal"\n                        ] = (\n                            hz\n                            / reversal_hz\n                        )\n\n                    if mode in (\n                        "Static Gabor",\n                        "Drifting Gabor",\n                    ):\n                        details.update({\n                            "gabor_size_deg":\n                                gabor_size,\n                            "gabor_sigma_deg":\n                                gabor_sigma,\n                            "gabor_x_deg":\n                                gabor_x,\n                            "gabor_y_deg":\n                                gabor_y,\n                            "gabor_background_rgb_255":\n                                gabor_bg_rgb_255,\n                        })\n\n                    trial = 0\n\n                    for rep in range(\n                        1,\n                        reps + 1\n                    ):\n                        for angle in angles:\n                            trial += 1\n\n                            if mode == "Asymmetric Drift":\n                                condition = (\n                                    f"{mode} {angle:g}° "\n                                    f"{asymmetric_polarity}"\n                                )\n                            else:\n                                condition = (\n                                    f"{mode} "\n                                    f"{angle:g}°"\n                                )\n\n                            solid_frames(\n                                GRAY,\n                                baseline_f,\n                                "grating",\n                                trial,\n                                condition,\n                                "baseline_gray"\n                            )\n\n                            # PsychoPy positive ori is clockwise.\n                            # UI convention is CCW.\n                            grating.ori = -angle\n\n                            if mode == "Drifting":\n                                phase_value = 0.0\n                                phase_per_frame = (\n                                    tf / hz\n                                )\n                                ttl_name = (\n                                    "GRATING_DRIFT_START"\n                                )\n                                stage_name = (\n                                    "drifting_grating"\n                                )\n\n                            elif mode == "Asymmetric Drift":\n                                phase_value = 0.0\n                                phase_per_frame = (\n                                    tf / hz\n                                )\n                                grating.phase = (\n                                    phase_value,\n                                    0.0\n                                )\n                                ttl_name = (\n                                    "GRATING_ASYMMETRIC_START"\n                                )\n                                stage_name = (\n                                    "asymmetric_grating"\n                                )\n\n                            elif mode == "Static":\n                                grating.phase = (\n                                    phase_initial,\n                                    0.0\n                                )\n                                ttl_name = (\n                                    "GRATING_STATIC_START"\n                                )\n                                stage_name = (\n                                    "static_grating"\n                                )\n\n                            elif mode == "Phase Reversal":\n                                ttl_name = (\n                                    "GRATING_PHASE_REVERSAL_START"\n                                )\n                                stage_name = (\n                                    "phase_reversal_grating"\n                                )\n\n                            elif mode == "Static Gabor":\n                                grating.phase = (\n                                    phase_initial,\n                                    0.0\n                                )\n                                ttl_name = (\n                                    "GABOR_STATIC_START"\n                                )\n                                stage_name = (\n                                    "static_gabor"\n                                )\n\n                            elif mode == "Drifting Gabor":\n                                phase_value = (\n                                    phase_initial\n                                )\n                                phase_per_frame = (\n                                    tf / hz\n                                )\n                                grating.phase = (\n                                    phase_value,\n                                    0.0\n                                )\n                                ttl_name = (\n                                    "GABOR_DRIFT_START"\n                                )\n                                stage_name = (\n                                    "drifting_gabor"\n                                )\n\n                            else:\n                                raise ValueError(\n                                    f"Modo Grating desconocido: {mode}"\n                                )\n\n                            if stim_f > 0:\n                                queue_ttl_event(\n                                    ttl_name,\n                                    trial,\n                                    condition\n                                )\n\n                            for f in range(\n                                1,\n                                stim_f + 1\n                            ):\n                                check_abort()\n\n                                if mode in (\n                                    "Drifting",\n                                    "Drifting Gabor",\n                                    "Asymmetric Drift",\n                                ):\n                                    phase_value += (\n                                        phase_per_frame\n                                    )\n                                    grating.phase = (\n                                        phase_value,\n                                        0.0\n                                    )\n\n                                elif mode == "Phase Reversal":\n                                    reversal_index = int(\n                                        math.floor(\n                                            (\n                                                (f - 1)\n                                                * reversal_hz\n                                            )\n                                            / hz\n                                        )\n                                    )\n                                    phase_value = (\n                                        phase_initial\n                                        + (\n                                            0.5\n                                            if (\n                                                reversal_index\n                                                % 2\n                                            )\n                                            else 0.0\n                                        )\n                                    )\n                                    grating.phase = (\n                                        phase_value,\n                                        0.0\n                                    )\n\n                                if mode in (\n                                    "Static Gabor",\n                                    "Drifting Gabor",\n                                ):\n                                    fullfield.fillColor = (\n                                        gabor_bg\n                                    )\n                                    fullfield.lineColor = (\n                                        gabor_bg\n                                    )\n                                    fullfield.draw()\n\n                                grating.draw()\n\n                                flip_and_log(\n                                    "grating",\n                                    trial,\n                                    condition,\n                                    stage_name,\n                                    f\n                                )\n\n                            solid_frames(\n                                GRAY,\n                                isi_f,\n                                "grating",\n                                trial,\n                                condition,\n                                "inter_trial_gray"\n                            )\n\n                            write_json_atomic(\n                                status_path,\n                                {\n                                    "state":\n                                        "running",\n                                    "message":\n                                        f"{mode} · "\n                                        f"rep {rep}/{reps} · "\n                                        f"{angle:g}°",\n                                    "hz": hz,\n                                    "command_id":\n                                        command_id,\n                                    "details":\n                                        details\n                                }\n                            )\n\n                # ==========================================================\n                # NOISE\n                # ==========================================================\n                elif stimulus_type == "dense_noise":\n                    baseline_f = seconds_to_frames(\n                        cfg[\n                            "dense_baseline_s"\n                        ],\n                        hz\n                    )\n                    stim_total_f = seconds_to_frames(\n                        cfg[\n                            "dense_duration_s"\n                        ],\n                        hz\n                    )\n                    isi_f = seconds_to_frames(\n                        cfg[\n                            "dense_isi_s"\n                        ],\n                        hz\n                    )\n                    reps = int(\n                        cfg[\n                            "dense_repetitions"\n                        ]\n                    )\n                    update_hz_req = float(\n                        cfg[\n                            "dense_update_hz"\n                        ]\n                    )\n                    check_size_deg = float(\n                        cfg[\n                            "dense_check_size_deg"\n                        ]\n                    )\n                    contrast = float(\n                        cfg[\n                            "dense_contrast"\n                        ]\n                    )\n                    seed_base = int(\n                        cfg[\n                            "dense_seed"\n                        ]\n                    )\n\n                    if (\n                        prepared_step is not None\n                        and prepared_step.get(\n                            "stimulus_type"\n                        ) == "dense_noise"\n                        and "noise_stim"\n                        in prepared_step\n                    ):\n                        resources = (\n                            prepared_step\n                        )\n                    else:\n                        resources = (\n                            build_noise_resources(\n                                cfg\n                            )\n                        )\n\n                    family_mode = resources[\n                        "family_mode"\n                    ]\n                    distribution = resources[\n                        "distribution"\n                    ]\n                    frozen = bool(\n                        resources[\n                            "frozen"\n                        ]\n                    )\n                    frames_per_update = resources[\n                        "frames_per_update"\n                    ]\n                    actual_update_hz = resources[\n                        "actual_update_hz"\n                    ]\n                    n_updates = resources[\n                        "n_updates"\n                    ]\n                    n_cols = resources[\n                        "n_cols"\n                    ]\n                    n_rows = resources[\n                        "n_rows"\n                    ]\n                    noise_size_deg = resources[\n                        "noise_size_deg"\n                    ]\n                    state_sequences = resources[\n                        "state_sequences"\n                    ]\n                    image_sequences = resources[\n                        "image_sequences"\n                    ]\n                    used_seeds = resources[\n                        "used_seeds"\n                    ]\n                    noise_stim = resources[\n                        "noise_stim"\n                    ]\n\n                    sequence_file = resources.get(\n                        "sequence_file"\n                    )\n                    if not sequence_file:\n                        sequence_file = (\n                            save_noise_sequence_before_timing(\n                                resources,\n                                cfg\n                            )\n                        )\n\n                    distribution_label = (\n                        distribution.capitalize()\n                    )\n\n                    if (\n                        family_mode\n                        == "Dense White Noise"\n                    ):\n                        condition_name = (\n                            f"Dense White Noise "\n                            f"({distribution_label})"\n                        )\n                        ttl_name = (\n                            "DENSE_WHITE_NOISE_START"\n                        )\n\n                    elif (\n                        family_mode\n                        == "Gaussian White Noise"\n                    ):\n                        condition_name = (\n                            "Gaussian White Noise"\n                        )\n                        ttl_name = (\n                            "GAUSSIAN_WHITE_NOISE_START"\n                        )\n\n                    else:\n                        condition_name = (\n                            f"Frozen Noise "\n                            f"({distribution_label})"\n                        )\n                        ttl_name = (\n                            "FROZEN_NOISE_START"\n                        )\n\n                    details = {\n                        "family_mode":\n                            family_mode,\n                        "distribution":\n                            distribution_label,\n                        "frozen":\n                            frozen,\n                        "baseline_frames":\n                            baseline_f,\n                        "stimulus_total_frames":\n                            stim_total_f,\n                        "isi_frames":\n                            isi_f,\n                        "requested_update_hz":\n                            update_hz_req,\n                        "actual_update_hz":\n                            actual_update_hz,\n                        "update_limited_by_refresh":\n                            bool(\n                                update_hz_req\n                                > hz\n                            ),\n                        "frames_per_update":\n                            frames_per_update,\n                        "n_updates":\n                            n_updates,\n                        "check_size_deg":\n                            check_size_deg,\n                        "grid_rows":\n                            n_rows,\n                        "grid_cols":\n                            n_cols,\n                        "contrast":\n                            contrast,\n                        "seed_base":\n                            seed_base,\n                        "used_seeds":\n                            used_seeds,\n                        "color_low_rgb_255":\n                            cfg[\n                                "dense_color_low_rgb"\n                            ],\n                        "color_mid_rgb_255":\n                            cfg[\n                                "dense_color_mid_rgb"\n                            ],\n                        "color_high_rgb_255":\n                            cfg[\n                                "dense_color_high_rgb"\n                            ],\n                        "sequence_file":\n                            str(\n                                sequence_file\n                            ),\n                        "sequence_saved_before_timing":\n                            bool(\n                                resources.get(\n                                    "sequence_saved_before_timing",\n                                    True\n                                )\n                            ),\n                    }\n\n                    if distribution == "gaussian":\n                        details[\n                            "gaussian_mean"\n                        ] = (\n                            "color_mid"\n                        )\n                        details[\n                            "gaussian_sigma_normalized"\n                        ] = (\n                            contrast\n                            / 3.0\n                        )\n                        details[\n                            "gaussian_clip_normalized"\n                        ] = [\n                            -contrast,\n                            contrast\n                        ]\n                        details[\n                            "gaussian_color_mapping"\n                        ] = (\n                            "low≈-3σ; mid=mean; high≈+3σ "\n                            "when contrast=1"\n                        )\n\n                    for rep in range(\n                        1,\n                        reps + 1\n                    ):\n                        solid_frames(\n                            GRAY,\n                            baseline_f,\n                            "dense_noise",\n                            rep,\n                            condition_name,\n                            "baseline_gray"\n                        )\n\n                        rep_images = (\n                            image_sequences[\n                                rep - 1\n                            ]\n                        )\n\n                        frames_remaining = (\n                            stim_total_f\n                        )\n\n                        if stim_total_f > 0:\n                            queue_ttl_event(\n                                ttl_name,\n                                rep,\n                                condition_name\n                            )\n\n                        for upd_idx in range(\n                            n_updates\n                        ):\n                            noise_stim.image = (\n                                rep_images[\n                                    upd_idx\n                                ]\n                            )\n\n                            hold = min(\n                                frames_per_update,\n                                frames_remaining\n                            )\n                            frames_remaining -= (\n                                hold\n                            )\n\n                            for f in range(\n                                1,\n                                hold + 1\n                            ):\n                                check_abort()\n                                noise_stim.draw()\n                                flip_and_log(\n                                    "dense_noise",\n                                    rep,\n                                    condition_name,\n                                    (\n                                        f"noise_update_"\n                                        f"{upd_idx + 1}"\n                                    ),\n                                    f\n                                )\n\n                        solid_frames(\n                            GRAY,\n                            isi_f,\n                            "dense_noise",\n                            rep,\n                            condition_name,\n                            "inter_trial_gray"\n                        )\n\n                        write_json_atomic(\n                            status_path,\n                            {\n                                "state":\n                                    "running",\n                                "message":\n                                    f"{condition_name} · "\n                                    f"rep {rep}/{reps}",\n                                "hz":\n                                    hz,\n                                "command_id":\n                                    command_id,\n                                "details":\n                                    details\n                            }\n                        )\n\n                # ==========================================================\n                # RF MAPPING\n                # ==========================================================\n                elif stimulus_type == "rf_mapping":\n                    mode_raw = str(\n                        cfg["rf_mode"]\n                    ).strip()\n                    mode = mode_raw.lower()\n\n                    baseline_f = seconds_to_frames(\n                        cfg["rf_baseline_s"], hz\n                    )\n                    stim_f = seconds_to_frames(\n                        cfg["rf_duration_s"], hz\n                    )\n                    isi_f = seconds_to_frames(\n                        cfg["rf_isi_s"], hz\n                    )\n                    reps = int(\n                        cfg["rf_repetitions"]\n                    )\n\n                    x_deg = float(\n                        cfg["rf_x_deg"]\n                    )\n                    y_deg = float(\n                        cfg["rf_y_deg"]\n                    )\n\n                    stim_rgb = rgb255_to_psychopy(\n                        cfg["rf_stimulus_rgb"]\n                    )\n                    bg_rgb = rgb255_to_psychopy(\n                        cfg["rf_background_rgb"]\n                    )\n\n                    if stim_f <= 0:\n                        raise ValueError(\n                            "RF Mapping: la duración debe producir al menos 1 frame."\n                        )\n                    if reps < 1:\n                        raise ValueError(\n                            "RF Mapping: repeticiones debe ser >= 1."\n                        )\n\n                    trial = 0\n                    details = {\n                        "mode": mode_raw,\n                        "baseline_frames": baseline_f,\n                        "stimulus_frames": stim_f,\n                        "isi_frames": isi_f,\n                        "repetitions": reps,\n                        "x_deg": x_deg,\n                        "y_deg": y_deg,\n                        "stimulus_rgb_255":\n                            cfg["rf_stimulus_rgb"],\n                        "background_rgb_255":\n                            cfg["rf_background_rgb"],\n                    }\n\n                    def present_rf_shape(\n                        draw_callable,\n                        condition_name,\n                        stage_name,\n                        ttl_name\n                    ):\n                        nonlocal trial\n                        trial += 1\n\n                        queue_ttl_event(\n                            ttl_name,\n                            trial,\n                            condition_name\n                        )\n\n                        for frame_i in range(\n                            1, stim_f + 1\n                        ):\n                            check_abort()\n                            fullfield.fillColor = bg_rgb\n                            fullfield.lineColor = bg_rgb\n                            fullfield.draw()\n                            draw_callable()\n                            flip_and_log(\n                                "rf_mapping",\n                                trial,\n                                condition_name,\n                                stage_name,\n                                frame_i\n                            )\n\n                        solid_frames(\n                            GRAY,\n                            isi_f,\n                            "rf_mapping",\n                            trial,\n                            condition_name,\n                            "inter_condition_gray"\n                        )\n\n                    if mode == "spot":\n                        diameter = float(\n                            cfg["rf_spot_diameter_deg"]\n                        )\n                        if diameter <= 0:\n                            raise ValueError(\n                                "RF Spot: diámetro debe ser > 0."\n                            )\n\n                        details[\n                            "spot_diameter_deg"\n                        ] = diameter\n\n                        if (\n                            prepared_step is not None\n                            and prepared_step.get(\n                                "stimulus_type"\n                            ) == "rf_mapping"\n                            and prepared_step.get(\n                                "rf_mode"\n                            ) == "spot"\n                            and "spot"\n                            in prepared_step\n                        ):\n                            spot = prepared_step[\n                                "spot"\n                            ]\n                        else:\n                            spot = visual.Circle(\n                                win=win,\n                                radius=diameter / 2.0,\n                                units="deg",\n                                pos=(x_deg, y_deg),\n                                fillColor=stim_rgb,\n                                lineColor=stim_rgb,\n                                colorSpace="rgb"\n                            )\n\n                        for rep in range(\n                            1, reps + 1\n                        ):\n                            solid_frames(\n                                GRAY,\n                                baseline_f,\n                                "rf_mapping",\n                                trial + 1,\n                                "Spot",\n                                "baseline_gray"\n                            )\n\n                            present_rf_shape(\n                                spot.draw,\n                                (\n                                    f"Spot {diameter:g}° "\n                                    f"X{x_deg:g} Y{y_deg:g}"\n                                ),\n                                "rf_spot",\n                                "RF_SPOT_START"\n                            )\n\n                            write_json_atomic(\n                                status_path,\n                                {\n                                    "state": "running",\n                                    "message":\n                                        f"RF Spot · rep {rep}/{reps}",\n                                    "hz": hz,\n                                    "command_id": command_id,\n                                    "details": details\n                                }\n                            )\n\n                    elif mode == "annulus":\n                        inner = float(\n                            cfg["rf_annulus_inner_deg"]\n                        )\n                        outer = float(\n                            cfg["rf_annulus_outer_deg"]\n                        )\n                        if (\n                            inner <= 0\n                            or outer <= inner\n                        ):\n                            raise ValueError(\n                                "RF Annulus: externo > interno > 0."\n                            )\n\n                        details[\n                            "annulus_inner_deg"\n                        ] = inner\n                        details[\n                            "annulus_outer_deg"\n                        ] = outer\n\n                        if (\n                            prepared_step is not None\n                            and prepared_step.get(\n                                "stimulus_type"\n                            ) == "rf_mapping"\n                            and prepared_step.get(\n                                "rf_mode"\n                            ) == "annulus"\n                            and "outer_circle"\n                            in prepared_step\n                        ):\n                            outer_circle = prepared_step[\n                                "outer_circle"\n                            ]\n                            inner_circle = prepared_step[\n                                "inner_circle"\n                            ]\n                        else:\n                            outer_circle = visual.Circle(\n                                win=win,\n                                radius=outer / 2.0,\n                                units="deg",\n                                pos=(x_deg, y_deg),\n                                fillColor=stim_rgb,\n                                lineColor=stim_rgb,\n                                colorSpace="rgb"\n                            )\n                            inner_circle = visual.Circle(\n                                win=win,\n                                radius=inner / 2.0,\n                                units="deg",\n                                pos=(x_deg, y_deg),\n                                fillColor=bg_rgb,\n                                lineColor=bg_rgb,\n                                colorSpace="rgb"\n                            )\n\n                        def draw_annulus():\n                            outer_circle.draw()\n                            inner_circle.draw()\n\n                        for rep in range(\n                            1, reps + 1\n                        ):\n                            solid_frames(\n                                GRAY,\n                                baseline_f,\n                                "rf_mapping",\n                                trial + 1,\n                                "Annulus",\n                                "baseline_gray"\n                            )\n\n                            present_rf_shape(\n                                draw_annulus,\n                                (\n                                    f"Annulus {inner:g}-{outer:g}° "\n                                    f"X{x_deg:g} Y{y_deg:g}"\n                                ),\n                                "rf_annulus",\n                                "RF_ANNULUS_START"\n                            )\n\n                            write_json_atomic(\n                                status_path,\n                                {\n                                    "state": "running",\n                                    "message":\n                                        f"RF Annulus · rep {rep}/{reps}",\n                                    "hz": hz,\n                                    "command_id": command_id,\n                                    "details": details\n                                }\n                            )\n\n                    elif mode == "size series":\n                        sizes = parse_float_list(\n                            cfg["rf_size_series_deg"]\n                        )\n                        if not sizes:\n                            raise ValueError(\n                                "RF Size Series: no hay diámetros."\n                            )\n                        if any(v <= 0 for v in sizes):\n                            raise ValueError(\n                                "RF Size Series: todos los diámetros deben ser > 0."\n                            )\n\n                        details[\n                            "size_series_deg"\n                        ] = sizes\n\n                        for rep in range(\n                            1, reps + 1\n                        ):\n                            solid_frames(\n                                GRAY,\n                                baseline_f,\n                                "rf_mapping",\n                                trial + 1,\n                                "Size series",\n                                "baseline_gray"\n                            )\n\n                            prepared_size_spots = None\n                            if (\n                                prepared_step is not None\n                                and prepared_step.get(\n                                    "stimulus_type"\n                                ) == "rf_mapping"\n                                and prepared_step.get(\n                                    "rf_mode"\n                                ) == "size series"\n                            ):\n                                prepared_size_spots = (\n                                    prepared_step.get(\n                                        "size_spots"\n                                    )\n                                )\n\n                            for size_idx, diameter in enumerate(\n                                sizes\n                            ):\n                                if (\n                                    prepared_size_spots is not None\n                                    and size_idx\n                                    < len(\n                                        prepared_size_spots\n                                    )\n                                ):\n                                    spot = (\n                                        prepared_size_spots[\n                                            size_idx\n                                        ]\n                                    )\n                                else:\n                                    spot = visual.Circle(\n                                        win=win,\n                                        radius=diameter / 2.0,\n                                        units="deg",\n                                        pos=(x_deg, y_deg),\n                                        fillColor=stim_rgb,\n                                        lineColor=stim_rgb,\n                                        colorSpace="rgb"\n                                    )\n\n                                present_rf_shape(\n                                    spot.draw,\n                                    (\n                                        f"Size {diameter:g}° "\n                                        f"X{x_deg:g} Y{y_deg:g}"\n                                    ),\n                                    "rf_size_series",\n                                    "RF_SIZE_START"\n                                )\n\n                            write_json_atomic(\n                                status_path,\n                                {\n                                    "state": "running",\n                                    "message":\n                                        f"RF Size Series · rep {rep}/{reps}",\n                                    "hz": hz,\n                                    "command_id": command_id,\n                                    "details": details\n                                }\n                            )\n\n                    elif mode == "annulus size series":\n                        sizes = parse_float_list(\n                            cfg[\n                                "rf_annulus_size_series_deg"\n                            ]\n                        )\n                        thickness = float(\n                            cfg[\n                                "rf_annulus_series_thickness_deg"\n                            ]\n                        )\n\n                        if not sizes:\n                            raise ValueError(\n                                "RF Annulus Size Series: no hay diámetros."\n                            )\n                        if any(\n                            diameter <= 0\n                            for diameter in sizes\n                        ):\n                            raise ValueError(\n                                "RF Annulus Size Series: todos los diámetros deben ser > 0."\n                            )\n                        if thickness <= 0:\n                            raise ValueError(\n                                "RF Annulus Size Series: el grosor radial debe ser > 0."\n                            )\n                        if any(\n                            2.0 * thickness >= diameter\n                            for diameter in sizes\n                        ):\n                            raise ValueError(\n                                "RF Annulus Size Series: grosor incompatible "\n                                "con al menos un diámetro."\n                            )\n\n                        details.update({\n                            "annulus_size_series_deg":\n                                sizes,\n                            "annulus_series_thickness_deg":\n                                thickness,\n                            "annulus_inner_diameters_deg":\n                                [\n                                    diameter\n                                    - 2.0 * thickness\n                                    for diameter in sizes\n                                ],\n                            "size_conditions":\n                                len(sizes),\n                            "thickness_definition":\n                                "constant_radial_thickness",\n                        })\n\n                        prepared_pairs = None\n\n                        if (\n                            prepared_step is not None\n                            and prepared_step.get(\n                                "stimulus_type"\n                            ) == "rf_mapping"\n                            and prepared_step.get(\n                                "rf_mode"\n                            ) == "annulus size series"\n                            and "annulus_size_pairs"\n                            in prepared_step\n                        ):\n                            prepared_pairs = prepared_step[\n                                "annulus_size_pairs"\n                            ]\n\n                        # Standalone fallback if, for any reason, this\n                        # stimulus was executed without the normal preload.\n                        # The standard GUI/protocol path should use the\n                        # already-prewarmed pairs above.\n                        if prepared_pairs is None:\n                            prepared_pairs = []\n\n                            for diameter in sizes:\n                                inner_diameter = (\n                                    diameter\n                                    - 2.0 * thickness\n                                )\n\n                                fallback_outer = visual.Circle(\n                                    win=win,\n                                    radius=diameter / 2.0,\n                                    units="deg",\n                                    pos=(x_deg, y_deg),\n                                    fillColor=stim_rgb,\n                                    lineColor=stim_rgb,\n                                    colorSpace="rgb"\n                                )\n                                fallback_inner = visual.Circle(\n                                    win=win,\n                                    radius=inner_diameter / 2.0,\n                                    units="deg",\n                                    pos=(x_deg, y_deg),\n                                    fillColor=bg_rgb,\n                                    lineColor=bg_rgb,\n                                    colorSpace="rgb"\n                                )\n\n                                prepared_pairs.append(\n                                    (\n                                        float(diameter),\n                                        fallback_outer,\n                                        fallback_inner,\n                                    )\n                                )\n\n                        details[\n                            "all_sizes_prewarmed_before_timing"\n                        ] = (\n                            prepared_step is not None\n                            and "annulus_size_pairs"\n                            in prepared_step\n                        )\n                        details[\n                            "preloaded_annulus_objects"\n                        ] = len(\n                            prepared_pairs\n                        )\n\n                        for rep in range(\n                            1, reps + 1\n                        ):\n                            solid_frames(\n                                GRAY,\n                                baseline_f,\n                                "rf_mapping",\n                                trial + 1,\n                                "Annulus Size series",\n                                "baseline_gray"\n                            )\n\n                            for (\n                                diameter,\n                                outer_circle,\n                                inner_circle,\n                            ) in prepared_pairs:\n\n                                def draw_annulus_size(\n                                    outer=outer_circle,\n                                    inner=inner_circle,\n                                ):\n                                    outer.draw()\n                                    inner.draw()\n\n                                present_rf_shape(\n                                    draw_annulus_size,\n                                    (\n                                        f"Annulus Size {diameter:g}° "\n                                        f"X{x_deg:g} Y{y_deg:g}"\n                                    ),\n                                    "rf_annulus_size_series",\n                                    "RF_ANNULUS_SIZE_START"\n                                )\n\n                            write_json_atomic(\n                                status_path,\n                                {\n                                    "state": "running",\n                                    "message":\n                                        f"RF Annulus Size Series · "\n                                        f"rep {rep}/{reps}",\n                                    "hz": hz,\n                                    "command_id": command_id,\n                                    "details": details\n                                }\n                            )\n\n                    elif mode == "sparse noise":\n                        square_size = float(\n                            cfg["rf_sparse_square_deg"]\n                        )\n                        grid_step = float(\n                            cfg["rf_sparse_grid_step_deg"]\n                        )\n                        n_presentations = int(\n                            cfg["rf_sparse_presentations"]\n                        )\n                        polarity = str(\n                            cfg["rf_sparse_polarity"]\n                        ).strip().lower()\n                        seed_base = int(\n                            cfg["rf_sparse_seed"]\n                        )\n\n                        if (\n                            square_size <= 0\n                            or grid_step <= 0\n                        ):\n                            raise ValueError(\n                                "Sparse Noise: tamaño y paso deben ser > 0."\n                            )\n                        if n_presentations < 1:\n                            raise ValueError(\n                                "Sparse Noise: presentaciones debe ser >= 1."\n                            )\n                        if polarity not in (\n                            "bright", "dark", "both"\n                        ):\n                            raise ValueError(\n                                "Sparse Noise: polaridad no válida."\n                            )\n\n                        if (\n                            prepared_step is not None\n                            and prepared_step.get(\n                                "stimulus_type"\n                            ) == "rf_mapping"\n                            and prepared_step.get(\n                                "rf_mode"\n                            ) == "sparse noise"\n                            and "sparse"\n                            in prepared_step\n                        ):\n                            positions = prepared_step[\n                                "positions"\n                            ]\n                            sparse = prepared_step[\n                                "sparse"\n                            ]\n                        else:\n                            half = square_size / 2.0\n                            x_min = -width_deg / 2.0 + half\n                            x_max = width_deg / 2.0 - half\n                            y_min = -height_deg / 2.0 + half\n                            y_max = height_deg / 2.0 - half\n\n                            xs = np.arange(\n                                x_min,\n                                x_max + grid_step * 0.25,\n                                grid_step\n                            )\n                            ys = np.arange(\n                                y_min,\n                                y_max + grid_step * 0.25,\n                                grid_step\n                            )\n\n                            if len(xs) == 0:\n                                xs = np.asarray([0.0])\n                            if len(ys) == 0:\n                                ys = np.asarray([0.0])\n\n                            positions = [\n                                (float(x), float(y))\n                                for y in ys\n                                for x in xs\n                            ]\n\n                            sparse = visual.Rect(\n                                win=win,\n                                width=square_size,\n                                height=square_size,\n                                units="deg",\n                                fillColor=WHITE,\n                                lineColor=WHITE,\n                                colorSpace="rgb",\n                                pos=(0, 0)\n                            )\n\n                        details.update({\n                            "sparse_square_deg":\n                                square_size,\n                            "sparse_grid_step_deg":\n                                grid_step,\n                            "sparse_presentations_per_rep":\n                                n_presentations,\n                            "sparse_polarity":\n                                polarity,\n                            "sparse_seed":\n                                seed_base,\n                            "sparse_grid_positions":\n                                len(positions),\n                            "sparse_background":\n                                "gray_50_percent",\n                        })\n\n                        for rep in range(\n                            1, reps + 1\n                        ):\n                            solid_frames(\n                                GRAY,\n                                baseline_f,\n                                "rf_mapping",\n                                trial + 1,\n                                "Sparse Noise",\n                                "baseline_gray"\n                            )\n\n                            rng = np.random.default_rng(\n                                seed_base + rep - 1\n                            )\n\n                            for presentation in range(\n                                1,\n                                n_presentations + 1\n                            ):\n                                pos_index = int(\n                                    rng.integers(\n                                        0,\n                                        len(positions)\n                                    )\n                                )\n                                px, py = positions[\n                                    pos_index\n                                ]\n\n                                if polarity == "bright":\n                                    level = "Bright"\n                                    square_color = WHITE\n                                elif polarity == "dark":\n                                    level = "Dark"\n                                    square_color = BLACK\n                                else:\n                                    if int(\n                                        rng.integers(0, 2)\n                                    ) == 1:\n                                        level = "Bright"\n                                        square_color = WHITE\n                                    else:\n                                        level = "Dark"\n                                        square_color = BLACK\n\n                                sparse.pos = (px, py)\n                                sparse.fillColor = square_color\n                                sparse.lineColor = square_color\n\n                                condition = (\n                                    f"{level} "\n                                    f"X{px:.3f} Y{py:.3f}"\n                                )\n\n                                trial += 1\n                                queue_ttl_event(\n                                    "RF_SPARSE_START",\n                                    trial,\n                                    condition\n                                )\n\n                                for frame_i in range(\n                                    1, stim_f + 1\n                                ):\n                                    check_abort()\n                                    fullfield.fillColor = GRAY\n                                    fullfield.lineColor = GRAY\n                                    fullfield.draw()\n                                    sparse.draw()\n                                    flip_and_log(\n                                        "rf_mapping",\n                                        trial,\n                                        condition,\n                                        "rf_sparse_noise",\n                                        frame_i\n                                    )\n\n                                solid_frames(\n                                    GRAY,\n                                    isi_f,\n                                    "rf_mapping",\n                                    trial,\n                                    condition,\n                                    "inter_condition_gray"\n                                )\n\n                            write_json_atomic(\n                                status_path,\n                                {\n                                    "state": "running",\n                                    "message":\n                                        f"RF Sparse Noise · rep {rep}/{reps}",\n                                    "hz": hz,\n                                    "command_id": command_id,\n                                    "details": details\n                                }\n                            )\n\n                    else:\n                        raise ValueError(\n                            f"Modo RF Mapping desconocido: {mode_raw}"\n                        )\n\n                # ==========================================================\n                # CHIRP + INTENSITY\n                # ==========================================================\n                elif stimulus_type == "chirp_intensity":\n                    mode = str(\n                        cfg.get(\n                            "temporal_mode",\n                            "Chirp + Intensity"\n                        )\n                    )\n\n                    basal_f = seconds_to_frames(\n                        cfg[\n                            "chirp_basal_s"\n                        ],\n                        hz\n                    )\n                    final_gray_f = seconds_to_frames(\n                        cfg[\n                            "chirp_final_gray_s"\n                        ],\n                        hz\n                    )\n                    reps = int(\n                        cfg[\n                            "chirp_repetitions"\n                        ]\n                    )\n\n                    if reps < 1:\n                        raise ValueError(\n                            "Temporal: repeticiones debe ser >= 1."\n                        )\n\n                    high_color = np.asarray(\n                        rgb255_to_psychopy(\n                            cfg[\n                                "chirp_on_color_rgb"\n                            ]\n                        ),\n                        dtype=np.float32\n                    )\n                    low_color = np.asarray(\n                        rgb255_to_psychopy(\n                            cfg[\n                                "chirp_off_color_rgb"\n                            ]\n                        ),\n                        dtype=np.float32\n                    )\n                    midpoint_color = (\n                        (\n                            high_color\n                            + low_color\n                        )\n                        / 2.0\n                    )\n                    half_range = (\n                        (\n                            high_color\n                            - low_color\n                        )\n                        / 2.0\n                    )\n\n                    if (\n                        prepared_step is not None\n                        and prepared_step.get(\n                            "stimulus_type"\n                        ) == "chirp_intensity"\n                        and prepared_step.get(\n                            "temporal_mode"\n                        ) == mode\n                    ):\n                        temporal_resources = (\n                            prepared_step\n                        )\n                    else:\n                        temporal_resources = (\n                            build_temporal_resources(\n                                cfg\n                            )\n                        )\n\n                    if mode == "Chirp + Intensity":\n                        step_on_f = seconds_to_frames(\n                            cfg[\n                                "chirp_step_on_s"\n                            ],\n                            hz\n                        )\n                        step_off_f = seconds_to_frames(\n                            cfg[\n                                "chirp_step_off_s"\n                            ],\n                            hz\n                        )\n                        pause1_f = seconds_to_frames(\n                            cfg[\n                                "chirp_pause1_s"\n                            ],\n                            hz\n                        )\n                        chirp_duration_f = seconds_to_frames(\n                            cfg[\n                                "chirp_frequency_duration_s"\n                            ],\n                            hz\n                        )\n                        pause2_f = seconds_to_frames(\n                            cfg[\n                                "chirp_pause2_s"\n                            ],\n                            hz\n                        )\n                        intensity_duration_f = seconds_to_frames(\n                            cfg[\n                                "chirp_intensity_duration_s"\n                            ],\n                            hz\n                        )\n\n                        f_min = float(\n                            cfg[\n                                "chirp_frequency_min_hz"\n                            ]\n                        )\n                        f_max = float(\n                            cfg[\n                                "chirp_frequency_max_hz"\n                            ]\n                        )\n                        intensity_freq = float(\n                            cfg[\n                                "chirp_intensity_frequency_hz"\n                            ]\n                        )\n                        intensity_start_pct = float(\n                            cfg[\n                                "chirp_intensity_start_pct"\n                            ]\n                        )\n\n                        chirp_states = temporal_resources[\n                            "chirp_states"\n                        ]\n                        chirp_inst_freq = temporal_resources[\n                            "chirp_inst_freq"\n                        ]\n                        intensity_states = temporal_resources[\n                            "intensity_states"\n                        ]\n                        intensity_levels_pct = temporal_resources[\n                            "intensity_levels_pct"\n                        ]\n\n                        details = {\n                            "temporal_mode":\n                                mode,\n                            "protocol":\n                                "A basal → B ON/OFF → C frequency chirp → "\n                                "D intensity ramp",\n                            "basal_frames":\n                                basal_f,\n                            "step_on_frames":\n                                step_on_f,\n                            "step_off_frames":\n                                step_off_f,\n                            "pause1_frames":\n                                pause1_f,\n                            "chirp_frames":\n                                chirp_duration_f,\n                            "frequency_min_hz":\n                                f_min,\n                            "frequency_max_hz":\n                                f_max,\n                            "frequency_progression":\n                                "linear",\n                            "frequency_duty_cycle":\n                                0.5,\n                            "pause2_frames":\n                                pause2_f,\n                            "intensity_frames":\n                                intensity_duration_f,\n                            "intensity_frequency_hz":\n                                intensity_freq,\n                            "intensity_start_pct":\n                                intensity_start_pct,\n                            "intensity_end_pct":\n                                100.0,\n                            "intensity_effective_cycles":\n                                (\n                                    max(\n                                        1,\n                                        int(\n                                            math.floor(\n                                                (\n                                                    max(\n                                                        0,\n                                                        intensity_duration_f\n                                                        - 1\n                                                    )\n                                                    / hz\n                                                )\n                                                * intensity_freq\n                                            )\n                                        )\n                                        + 1\n                                    )\n                                    if intensity_duration_f > 0\n                                    else 0\n                                ),\n                            "intensity_last_on_level_pct":\n                                (\n                                    max(\n                                        intensity_levels_pct\n                                    )\n                                    if intensity_levels_pct\n                                    else None\n                                ),\n                            "intensity_duty_cycle":\n                                0.5,\n                            "on_color_rgb_255":\n                                cfg[\n                                    "chirp_on_color_rgb"\n                                ],\n                            "off_color_rgb_255":\n                                cfg[\n                                    "chirp_off_color_rgb"\n                                ],\n                            "final_gray_frames":\n                                final_gray_f,\n                            "repetitions":\n                                reps,\n                            "ttl_principal_conditions":\n                                [\n                                    "B",\n                                    "C",\n                                    "D"\n                                ],\n                        }\n\n                        for rep in range(\n                            1,\n                            reps + 1\n                        ):\n                            solid_frames(\n                                GRAY,\n                                basal_f,\n                                "chirp_intensity",\n                                rep,\n                                "A_BASAL",\n                                "A_gray"\n                            )\n\n                            if (\n                                step_on_f\n                                + step_off_f\n                            ) > 0:\n                                queue_ttl_event(\n                                    "B_START",\n                                    rep,\n                                    "B_ON_OFF"\n                                )\n\n                            solid_frames(\n                                tuple(\n                                    high_color.tolist()\n                                ),\n                                step_on_f,\n                                "chirp_intensity",\n                                rep,\n                                "B_ON",\n                                "B_step_on"\n                            )\n                            solid_frames(\n                                tuple(\n                                    low_color.tolist()\n                                ),\n                                step_off_f,\n                                "chirp_intensity",\n                                rep,\n                                "B_OFF",\n                                "B_step_off"\n                            )\n\n                            solid_frames(\n                                GRAY,\n                                pause1_f,\n                                "chirp_intensity",\n                                rep,\n                                "PAUSE_B_C",\n                                "gray_pause_1"\n                            )\n\n                            if chirp_duration_f > 0:\n                                queue_ttl_event(\n                                    "C_START",\n                                    rep,\n                                    "FREQUENCY_CHIRP"\n                                )\n\n                            for frame_idx in range(\n                                chirp_duration_f\n                            ):\n                                check_abort()\n\n                                if chirp_states[\n                                    frame_idx\n                                ]:\n                                    color = tuple(\n                                        high_color.tolist()\n                                    )\n                                    state_name = (\n                                        "ON"\n                                    )\n                                else:\n                                    color = tuple(\n                                        low_color.tolist()\n                                    )\n                                    state_name = (\n                                        "OFF"\n                                    )\n\n                                fullfield.fillColor = (\n                                    color\n                                )\n                                fullfield.lineColor = (\n                                    color\n                                )\n                                fullfield.draw()\n\n                                flip_and_log(\n                                    "chirp_intensity",\n                                    rep,\n                                    (\n                                        f"C_CHIRP_{state_name}_"\n                                        f"{chirp_inst_freq[frame_idx]:.4f}Hz"\n                                    ),\n                                    "C_frequency_chirp",\n                                    frame_idx + 1\n                                )\n\n                            solid_frames(\n                                GRAY,\n                                pause2_f,\n                                "chirp_intensity",\n                                rep,\n                                "PAUSE_C_D",\n                                "gray_pause_2"\n                            )\n\n                            if intensity_duration_f > 0:\n                                queue_ttl_event(\n                                    "D_START",\n                                    rep,\n                                    "INTENSITY_RAMP"\n                                )\n\n                            for frame_idx in range(\n                                intensity_duration_f\n                            ):\n                                check_abort()\n\n                                if intensity_states[\n                                    frame_idx\n                                ]:\n                                    pct = (\n                                        intensity_levels_pct[\n                                            frame_idx\n                                        ]\n                                    )\n                                    alpha = (\n                                        pct\n                                        / 100.0\n                                    )\n                                    color_vec = (\n                                        low_color\n                                        + alpha\n                                        * (\n                                            high_color\n                                            - low_color\n                                        )\n                                    )\n                                    condition = (\n                                        f"D_ON_{pct:.3f}%"\n                                    )\n                                else:\n                                    color_vec = (\n                                        low_color\n                                    )\n                                    condition = (\n                                        "D_OFF"\n                                    )\n\n                                color = tuple(\n                                    color_vec.tolist()\n                                )\n                                fullfield.fillColor = (\n                                    color\n                                )\n                                fullfield.lineColor = (\n                                    color\n                                )\n                                fullfield.draw()\n\n                                flip_and_log(\n                                    "chirp_intensity",\n                                    rep,\n                                    condition,\n                                    "D_intensity_ramp",\n                                    frame_idx + 1\n                                )\n\n                            solid_frames(\n                                GRAY,\n                                final_gray_f,\n                                "chirp_intensity",\n                                rep,\n                                "FINAL_GRAY",\n                                "final_gray"\n                            )\n\n                            write_json_atomic(\n                                status_path,\n                                {\n                                    "state":\n                                        "running",\n                                    "message":\n                                        f"Temporal · Chirp + Intensity · "\n                                        f"rep {rep}/{reps}",\n                                    "hz":\n                                        hz,\n                                    "command_id":\n                                        command_id,\n                                    "details":\n                                        details\n                                }\n                            )\n\n                    elif mode == "Sinusoidal Flicker":\n                        duration_f = seconds_to_frames(\n                            cfg[\n                                "temporal_duration_s"\n                            ],\n                            hz\n                        )\n                        contrast_fraction = float(\n                            temporal_resources[\n                                "contrast_fraction"\n                            ]\n                        )\n                        sine_values = temporal_resources[\n                            "sine_values"\n                        ]\n                        frequency = float(\n                            cfg[\n                                "temporal_frequency_hz"\n                            ]\n                        )\n\n                        details = {\n                            "temporal_mode":\n                                mode,\n                            "basal_frames":\n                                basal_f,\n                            "stimulus_frames":\n                                duration_f,\n                            "frequency_hz":\n                                frequency,\n                            "contrast_pct":\n                                float(\n                                    cfg[\n                                        "temporal_contrast_pct"\n                                    ]\n                                ),\n                            "phase_start_cycles":\n                                0.0,\n                            "modulation":\n                                "sinusoidal about color midpoint",\n                            "high_color_rgb_255":\n                                cfg[\n                                    "chirp_on_color_rgb"\n                                ],\n                            "low_color_rgb_255":\n                                cfg[\n                                    "chirp_off_color_rgb"\n                                ],\n                            "final_gray_frames":\n                                final_gray_f,\n                            "repetitions":\n                                reps,\n                            "ttl_principal_conditions":\n                                [\n                                    "SINUSOIDAL_FLICKER"\n                                ],\n                        }\n\n                        for rep in range(\n                            1,\n                            reps + 1\n                        ):\n                            solid_frames(\n                                GRAY,\n                                basal_f,\n                                "chirp_intensity",\n                                rep,\n                                "TEMPORAL_BASAL",\n                                "baseline_gray"\n                            )\n\n                            if duration_f > 0:\n                                queue_ttl_event(\n                                    "SINE_FLICKER_START",\n                                    rep,\n                                    "SINUSOIDAL_FLICKER"\n                                )\n\n                            for frame_idx in range(\n                                duration_f\n                            ):\n                                check_abort()\n\n                                modulation = (\n                                    contrast_fraction\n                                    * float(\n                                        sine_values[\n                                            frame_idx\n                                        ]\n                                    )\n                                )\n                                color_vec = (\n                                    midpoint_color\n                                    + half_range\n                                    * modulation\n                                )\n                                color = tuple(\n                                    color_vec.tolist()\n                                )\n\n                                fullfield.fillColor = (\n                                    color\n                                )\n                                fullfield.lineColor = (\n                                    color\n                                )\n                                fullfield.draw()\n\n                                flip_and_log(\n                                    "chirp_intensity",\n                                    rep,\n                                    "SINUSOIDAL_FLICKER",\n                                    "sinusoidal_flicker",\n                                    frame_idx + 1\n                                )\n\n                            solid_frames(\n                                GRAY,\n                                final_gray_f,\n                                "chirp_intensity",\n                                rep,\n                                "FINAL_GRAY",\n                                "final_gray"\n                            )\n\n                            write_json_atomic(\n                                status_path,\n                                {\n                                    "state":\n                                        "running",\n                                    "message":\n                                        f"Temporal · Sinusoidal · "\n                                        f"rep {rep}/{reps}",\n                                    "hz":\n                                        hz,\n                                    "command_id":\n                                        command_id,\n                                    "details":\n                                        details\n                                }\n                            )\n\n                    elif mode == "Gaussian Flicker":\n                        duration_f = seconds_to_frames(\n                            cfg[\n                                "temporal_duration_s"\n                            ],\n                            hz\n                        )\n                        frames_per_update = int(\n                            temporal_resources[\n                                "frames_per_update"\n                            ]\n                        )\n                        actual_update_hz = float(\n                            temporal_resources[\n                                "actual_update_hz"\n                            ]\n                        )\n                        sequences = temporal_resources[\n                            "gaussian_sequences"\n                        ]\n                        used_seeds = temporal_resources[\n                            "used_seeds"\n                        ]\n                        contrast_fraction = float(\n                            temporal_resources[\n                                "contrast_fraction"\n                            ]\n                        )\n                        requested_update_hz = float(\n                            cfg[\n                                "temporal_update_hz"\n                            ]\n                        )\n\n                        details = {\n                            "temporal_mode":\n                                mode,\n                            "basal_frames":\n                                basal_f,\n                            "stimulus_frames":\n                                duration_f,\n                            "requested_update_hz":\n                                requested_update_hz,\n                            "actual_update_hz":\n                                actual_update_hz,\n                            "update_limited_by_refresh":\n                                bool(\n                                    requested_update_hz\n                                    > hz\n                                ),\n                            "frames_per_update":\n                                frames_per_update,\n                            "n_updates":\n                                int(\n                                    temporal_resources[\n                                        "n_updates"\n                                    ]\n                                ),\n                            "contrast_pct":\n                                float(\n                                    cfg[\n                                        "temporal_contrast_pct"\n                                    ]\n                                ),\n                            "gaussian_mean":\n                                "color_midpoint",\n                            "gaussian_sigma_normalized":\n                                (\n                                    1.0\n                                    / 3.0\n                                ),\n                            "gaussian_clip_normalized":\n                                [\n                                    -1.0,\n                                    1.0\n                                ],\n                            "seed_base":\n                                int(\n                                    cfg[\n                                        "temporal_seed"\n                                    ]\n                                ),\n                            "used_seeds":\n                                used_seeds,\n                            "high_color_rgb_255":\n                                cfg[\n                                    "chirp_on_color_rgb"\n                                ],\n                            "low_color_rgb_255":\n                                cfg[\n                                    "chirp_off_color_rgb"\n                                ],\n                            "final_gray_frames":\n                                final_gray_f,\n                            "repetitions":\n                                reps,\n                            "ttl_principal_conditions":\n                                [\n                                    "GAUSSIAN_FLICKER"\n                                ],\n                        }\n\n                        for rep in range(\n                            1,\n                            reps + 1\n                        ):\n                            solid_frames(\n                                GRAY,\n                                basal_f,\n                                "chirp_intensity",\n                                rep,\n                                "TEMPORAL_BASAL",\n                                "baseline_gray"\n                            )\n\n                            if duration_f > 0:\n                                queue_ttl_event(\n                                    "GAUSSIAN_FLICKER_START",\n                                    rep,\n                                    "GAUSSIAN_FLICKER"\n                                )\n\n                            seq = sequences[\n                                rep - 1\n                            ]\n\n                            for frame_idx in range(\n                                duration_f\n                            ):\n                                check_abort()\n\n                                update_idx = min(\n                                    len(seq) - 1,\n                                    frame_idx\n                                    // frames_per_update\n                                )\n                                normalized = (\n                                    contrast_fraction\n                                    * float(\n                                        seq[\n                                            update_idx\n                                        ]\n                                    )\n                                )\n                                color_vec = (\n                                    midpoint_color\n                                    + half_range\n                                    * normalized\n                                )\n                                color = tuple(\n                                    color_vec.tolist()\n                                )\n\n                                fullfield.fillColor = (\n                                    color\n                                )\n                                fullfield.lineColor = (\n                                    color\n                                )\n                                fullfield.draw()\n\n                                flip_and_log(\n                                    "chirp_intensity",\n                                    rep,\n                                    (\n                                        f"GAUSSIAN_"\n                                        f"{normalized:+.6f}"\n                                    ),\n                                    "gaussian_flicker",\n                                    frame_idx + 1\n                                )\n\n                            solid_frames(\n                                GRAY,\n                                final_gray_f,\n                                "chirp_intensity",\n                                rep,\n                                "FINAL_GRAY",\n                                "final_gray"\n                            )\n\n                            write_json_atomic(\n                                status_path,\n                                {\n                                    "state":\n                                        "running",\n                                    "message":\n                                        f"Temporal · Gaussian · "\n                                        f"rep {rep}/{reps}",\n                                    "hz":\n                                        hz,\n                                    "command_id":\n                                        command_id,\n                                    "details":\n                                        details\n                                }\n                            )\n\n                    elif mode == "Contrast Series":\n                        duration_f = seconds_to_frames(\n                            cfg[\n                                "temporal_duration_s"\n                            ],\n                            hz\n                        )\n                        isi_f = seconds_to_frames(\n                            cfg[\n                                "temporal_contrast_isi_s"\n                            ],\n                            hz\n                        )\n                        levels = temporal_resources[\n                            "contrast_levels_pct"\n                        ]\n                        sine_values = temporal_resources[\n                            "sine_values"\n                        ]\n                        frequency = float(\n                            cfg[\n                                "temporal_frequency_hz"\n                            ]\n                        )\n\n                        details = {\n                            "temporal_mode":\n                                mode,\n                            "basal_frames":\n                                basal_f,\n                            "stimulus_frames_per_contrast":\n                                duration_f,\n                            "gray_between_contrasts_frames":\n                                isi_f,\n                            "frequency_hz":\n                                frequency,\n                            "contrast_levels_pct":\n                                levels,\n                            "phase_start_cycles":\n                                0.0,\n                            "modulation":\n                                "sinusoidal about color midpoint",\n                            "high_color_rgb_255":\n                                cfg[\n                                    "chirp_on_color_rgb"\n                                ],\n                            "low_color_rgb_255":\n                                cfg[\n                                    "chirp_off_color_rgb"\n                                ],\n                            "final_gray_frames":\n                                final_gray_f,\n                            "repetitions":\n                                reps,\n                            "ttl_principal_conditions":\n                                [\n                                    "CONTRAST_LEVEL_ONSET"\n                                ],\n                        }\n\n                        trial = 0\n\n                        for rep in range(\n                            1,\n                            reps + 1\n                        ):\n                            solid_frames(\n                                GRAY,\n                                basal_f,\n                                "chirp_intensity",\n                                rep,\n                                "TEMPORAL_BASAL",\n                                "baseline_gray"\n                            )\n\n                            for level in levels:\n                                trial += 1\n                                condition = (\n                                    f"CONTRAST_{level:g}%"\n                                )\n                                contrast_fraction = (\n                                    float(level)\n                                    / 100.0\n                                )\n\n                                if duration_f > 0:\n                                    queue_ttl_event(\n                                        "CONTRAST_LEVEL_START",\n                                        trial,\n                                        condition\n                                    )\n\n                                for frame_idx in range(\n                                    duration_f\n                                ):\n                                    check_abort()\n\n                                    modulation = (\n                                        contrast_fraction\n                                        * float(\n                                            sine_values[\n                                                frame_idx\n                                            ]\n                                        )\n                                    )\n                                    color_vec = (\n                                        midpoint_color\n                                        + half_range\n                                        * modulation\n                                    )\n                                    color = tuple(\n                                        color_vec.tolist()\n                                    )\n\n                                    fullfield.fillColor = (\n                                        color\n                                    )\n                                    fullfield.lineColor = (\n                                        color\n                                    )\n                                    fullfield.draw()\n\n                                    flip_and_log(\n                                        "chirp_intensity",\n                                        trial,\n                                        condition,\n                                        "contrast_series",\n                                        frame_idx + 1\n                                    )\n\n                                solid_frames(\n                                    GRAY,\n                                    isi_f,\n                                    "chirp_intensity",\n                                    trial,\n                                    condition,\n                                    "contrast_inter_trial_gray"\n                                )\n\n                            solid_frames(\n                                GRAY,\n                                final_gray_f,\n                                "chirp_intensity",\n                                rep,\n                                "FINAL_GRAY",\n                                "final_gray"\n                            )\n\n                            write_json_atomic(\n                                status_path,\n                                {\n                                    "state":\n                                        "running",\n                                    "message":\n                                        f"Temporal · Contrast Series · "\n                                        f"rep {rep}/{reps}",\n                                    "hz":\n                                        hz,\n                                    "command_id":\n                                        command_id,\n                                    "details":\n                                        details\n                                }\n                            )\n\n                    else:\n                        raise ValueError(\n                            f"Modo Temporal desconocido: {mode}"\n                        )\n\n                else:\n                    raise ValueError(\n                        f"Tipo de estímulo desconocido: "\n                        f"{stimulus_type}"\n                    )\n\n\n            if stimulus_type == "protocol":\n                protocol_cfg = cfg\n                protocol_name = str(\n                    protocol_cfg.get(\n                        "protocol_name",\n                        "Protocolo"\n                    )\n                )\n                protocol_version = int(\n                    protocol_cfg.get(\n                        "protocol_version", 1\n                    )\n                )\n                protocol_repetitions = int(\n                    protocol_cfg.get(\n                        "protocol_repetitions", 1\n                    )\n                )\n                protocol_steps = list(\n                    protocol_cfg.get(\n                        "protocol_steps", []\n                    )\n                )\n\n                if protocol_repetitions < 1:\n                    raise ValueError(\n                        "El protocolo debe repetirse al menos una vez."\n                    )\n                if not protocol_steps:\n                    raise ValueError(\n                        "El protocolo no contiene pasos."\n                    )\n\n                protocol_step_results = []\n                protocol_pause_events = []\n                protocol_name_context = protocol_name\n                protocol_version_context = protocol_version\n\n                # ------------------------------------------------------\n                # PRELOAD / PREWARM PHASE\n                # ------------------------------------------------------\n                # Nothing is logged yet. Every warm-up flip is covered by\n                # neutral gray, so timing starts only after all heavy visual\n                # resources have already been created.\n                write_json_atomic(\n                    status_path,\n                    {\n                        "state": "running",\n                        "message": (\n                            f"Preparando protocolo "\n                            f"{protocol_name}..."\n                        ),\n                        "hz": hz,\n                        "command_id": command_id,\n                        "protocol_name":\n                            protocol_name,\n                        "protocol_version":\n                            protocol_version,\n                        "preparing": True,\n                    }\n                )\n\n                protocol_prepared_steps = {}\n\n                for prep_position, prep_step in enumerate(\n                    protocol_steps,\n                    start=1\n                ):\n                    check_abort()\n\n                    if prep_step.get(\n                        "kind"\n                    ) == "stimulus":\n                        write_json_atomic(\n                            status_path,\n                            {\n                                "state": "running",\n                                "message": (\n                                    f"Preparando protocolo · "\n                                    f"paso {prep_position}/"\n                                    f"{len(protocol_steps)} · "\n                                    f"{prep_step.get(\'label\', \'Estímulo\')}"\n                                ),\n                                "hz": hz,\n                                "command_id":\n                                    command_id,\n                                "protocol_name":\n                                    protocol_name,\n                                "protocol_version":\n                                    protocol_version,\n                                "preparing": True,\n                                "preparing_step":\n                                    prep_position,\n                            }\n                        )\n\n                        protocol_prepared_steps[\n                            prep_position\n                        ] = prepare_protocol_step(\n                            prep_step\n                        )\n\n                # Two final gray flips settle the back/front buffers after\n                # any shader/texture warm-up. These are deliberately outside\n                # the experimental log.\n                show_solid(GRAY)\n                show_solid(GRAY)\n\n                # The first logged protocol frame establishes its own timing\n                # reference; no preparation flip can be counted as a drop.\n                previous_flip = None\n\n                for protocol_rep in range(\n                    1,\n                    protocol_repetitions + 1\n                ):\n                    protocol_rep_context = (\n                        protocol_rep\n                    )\n\n                    for step_position, step in enumerate(\n                        protocol_steps,\n                        start=1\n                    ):\n                        check_abort()\n\n                        next_step_name = str(\n                            step.get(\n                                "label",\n                                step.get(\n                                    "stimulus_type",\n                                    step.get(\n                                        "kind",\n                                        "Paso"\n                                    )\n                                )\n                            )\n                        )\n\n                        wait_for_protocol_resume(\n                            protocol_name,\n                            protocol_version,\n                            protocol_rep,\n                            step_position,\n                            len(protocol_steps),\n                            next_step_name,\n                            protocol_pause_events\n                        )\n\n                        check_abort()\n\n                        protocol_step_index_context = int(\n                            step.get(\n                                "step_index",\n                                step_position\n                            )\n                        )\n                        protocol_step_name_context = str(\n                            step.get(\n                                "label",\n                                step.get(\n                                    "stimulus_type",\n                                    step.get(\n                                        "kind",\n                                        "Paso"\n                                    )\n                                )\n                            )\n                        )\n                        protocol_step_kind_context = str(\n                            step.get(\n                                "kind",\n                                "stimulus"\n                            )\n                        )\n\n                        write_json_atomic(\n                            status_path,\n                            {\n                                "state": "running",\n                                "message": (\n                                    f"Protocolo {protocol_name} · "\n                                    f"rep {protocol_rep}/"\n                                    f"{protocol_repetitions} · "\n                                    f"paso {step_position}/"\n                                    f"{len(protocol_steps)} · "\n                                    f"{protocol_step_name_context}"\n                                ),\n                                "hz": hz,\n                                "command_id": command_id,\n                                "protocol_name":\n                                    protocol_name,\n                                "protocol_version":\n                                    protocol_version,\n                                "protocol_rep":\n                                    protocol_rep,\n                                "protocol_step":\n                                    step_position,\n                                "protocol_total_steps":\n                                    len(protocol_steps),\n                            }\n                        )\n\n                        if (\n                            protocol_step_kind_context\n                            == "pause"\n                        ):\n                            prepared_step = None\n                            pause_s = float(\n                                step.get(\n                                    "duration_s", 0.0\n                                )\n                            )\n                            if pause_s <= 0:\n                                raise ValueError(\n                                    "Una pausa del protocolo "\n                                    "tiene duración <= 0."\n                                )\n\n                            pause_f = seconds_to_frames(\n                                pause_s, hz\n                            )\n\n                            pause_color_name = str(step.get("color", "adaptation")).strip().lower()\n                            if pause_color_name == "black":\n                                pause_color = BLACK\n                                pause_label = "negro"\n                            elif pause_color_name == "gray50":\n                                pause_color = (0.0, 0.0, 0.0)\n                                pause_label = "gris medio"\n                            else:\n                                pause_color = GRAY\n                                pause_label = "fondo de adaptación"\n\n                            solid_frames(\n                                pause_color,\n                                pause_f,\n                                "protocol_pause",\n                                protocol_step_index_context,\n                                f"Pausa {pause_label} {pause_s:g} s",\n                                "protocol_pause",\n                                after_first_frame=(\n                                    emit_generic_pause_marker\n                                    if ttl_mode == "generic_ftdi" else None\n                                )\n                            )\n\n                            protocol_step_results.append({\n                                "protocol_rep":\n                                    protocol_rep,\n                                "step_index":\n                                    protocol_step_index_context,\n                                "step_name":\n                                    protocol_step_name_context,\n                                "kind": "pause",\n                                "duration_s":\n                                    pause_s,\n                                "frames":\n                                    pause_f,\n                                "aux_pause_marker":\n                                    bool(ttl_mode == "generic_ftdi"),\n                                "aux_pause_marker_width_ms":\n                                    (\n                                        1000.0 * pause_marker_width_s\n                                        if ttl_mode == "generic_ftdi" else 0.0\n                                    ),\n                            })\n\n                        elif (\n                            protocol_step_kind_context\n                            == "stimulus"\n                        ):\n                            step_cfg = dict(\n                                step.get(\n                                    "config", {}\n                                )\n                            )\n\n                            step_stimulus_type = str(\n                                step_cfg.get(\n                                    "stimulus_type",\n                                    step.get(\n                                        "stimulus_type",\n                                        ""\n                                    )\n                                )\n                            )\n\n                            if not step_stimulus_type:\n                                raise ValueError(\n                                    "Un paso de estímulo no "\n                                    "contiene stimulus_type."\n                                )\n\n                            cfg = step_cfg\n                            stimulus_type = (\n                                step_stimulus_type\n                            )\n                            prepared_step = (\n                                protocol_prepared_steps.get(\n                                    step_position\n                                )\n                            )\n                            details = {}\n\n                            frame_before = global_frame\n                            ttl_before = len(\n                                ttl_events\n                            )\n\n                            run_current_stimulus()\n                            # Each protocol stimulus ends with a clean AUX baseline.\n                            # UNIVERSAL receives no extra EVENT pulse for this reset.\n                            force_generic_aux_low(\n                                f"PROTOCOL_STEP_{protocol_step_index_context}_END_RESET"\n                            )\n\n                            protocol_step_results.append({\n                                "protocol_rep":\n                                    protocol_rep,\n                                "step_index":\n                                    protocol_step_index_context,\n                                "step_name":\n                                    protocol_step_name_context,\n                                "kind": "stimulus",\n                                "stimulus_type":\n                                    step_stimulus_type,\n                                "frames":\n                                    global_frame\n                                    - frame_before,\n                                "ttl_transitions":\n                                    len(ttl_events)\n                                    - ttl_before,\n                                "details":\n                                    json.loads(\n                                        json.dumps(\n                                            details\n                                        )\n                                    ),\n                            })\n\n                        else:\n                            raise ValueError(\n                                "Tipo de paso de protocolo "\n                                f"desconocido: "\n                                f"{protocol_step_kind_context}"\n                            )\n\n                # Release protocol-only references before final result/log.\n                prepared_step = None\n                protocol_prepared_steps.clear()\n\n                # Restore the command-level identity for the single result/log.\n                cfg = protocol_cfg\n                stimulus_type = "protocol"\n\n                details = {\n                    "protocol_id":\n                        protocol_cfg.get(\n                            "protocol_id"\n                        ),\n                    "protocol_name":\n                        protocol_name,\n                    "protocol_version":\n                        protocol_version,\n                    "protocol_repetitions":\n                        protocol_repetitions,\n                    "protocol_steps":\n                        len(protocol_steps),\n                    "preload_before_timing":\n                        True,\n                    "safe_pause_mode":\n                        "between_protocol_steps",\n                    "manual_pause_count":\n                        len(\n                            protocol_pause_events\n                        ),\n                    "manual_pause_total_s":\n                        sum(\n                            float(\n                                pause_item.get(\n                                    "duration_s",\n                                    0.0\n                                )\n                            )\n                            for pause_item\n                            in protocol_pause_events\n                        ),\n                    "manual_pause_events":\n                        protocol_pause_events,\n                    "step_results":\n                        protocol_step_results,\n                }\n\n            else:\n                run_current_stimulus()\n\n            # Always finish in neutral gray. Event pulses normally\n            # returned LOW one frame after onset; if a falling edge is\n            # pending on the final safety flip it is executed and stamped.\n            finish_trigger_run("STIMULUS_END_SAFETY")\n\n            stamp = run_stamp\n            log_prefix = safe_run_prefix\n\n            csv_path = run_dir / (\n                f"{run_base_name}.csv"\n            )\n\n            config_snapshot_path = run_dir / (\n                f"{run_base_name}_config.json"\n            )\n\n            fieldnames = [\n                "stimulus_type", "trial", "condition",\n                "stage", "frame_in_stage", "global_frame",\n                "flip_time_s",\n                "interval_from_previous_flip_s",\n                "expected_interval_s", "interval_error_ms",\n                "possible_dropped_frame", "ttl_state",\n                "protocol_name", "protocol_version",\n                "protocol_rep", "protocol_step_index",\n                "protocol_step_name", "protocol_step_kind"\n            ]\n\n            with csv_path.open(\n                "w", newline="", encoding="utf-8"\n            ) as fh:\n                writer = csv.DictWriter(\n                    fh, fieldnames=fieldnames\n                )\n                writer.writeheader()\n                writer.writerows(rows)\n\n            ttl_csv_path = run_dir / (\n                f"{run_base_name}_ttl.csv"\n            )\n            ttl_fieldnames = [\n                "event_index", "state", "level",\n                "event_name", "trial", "condition",\n                "scheduled_frame", "callback_time_s",\n                "flip_time_s", "mode", "pin",\n                "port", "signal", "maxone_bit",\n                "event_role", "pulse_width_frames",\n                "event_offset_requested_ms",\n                "pulse_width_requested_ms",\n                "frame_anchor_perf_s", "transition_perf_s",\n                "event_offset_measured_ms",\n                "pulse_width_measured_ms", "universal_code",\n                "aux_event_state", "aux_event_level",\n                "aux_event_policy", "universal_pulse_phase",\n                "protocol_name", "protocol_version",\n                "protocol_rep", "protocol_step_index",\n                "protocol_step_name", "protocol_step_kind"\n            ]\n            with ttl_csv_path.open(\n                "w", newline="", encoding="utf-8"\n            ) as ttl_fh:\n                ttl_writer = csv.DictWriter(\n                    ttl_fh, fieldnames=ttl_fieldnames\n                )\n                ttl_writer.writeheader()\n                ttl_writer.writerows(ttl_events)\n\n\n            # ------------------------------------------------------\n            # EXPERIMENT CONFIGURATION SNAPSHOT\n            # ------------------------------------------------------\n            # Primary CSV logs are already safely written at this point.\n            expected_interval_s = (\n                1.0 / hz\n                if hz and hz > 0\n                else None\n            )\n\n            snapshot = {\n                "snapshot_schema_version": 1,\n                "created_at":\n                    datetime.now().isoformat(\n                        timespec="milliseconds"\n                    ),\n                "app_version": cfg.get(\n                    "app_version", ""\n                ),\n                "command_id": command_id,\n                "stimulus_type": stimulus_type,\n                "measured_refresh_hz": hz,\n                "expected_frame_interval_s":\n                    expected_interval_s,\n                "refresh_measurement_method":\n                    "perf_counter_median_of_real_win_flip_intervals",\n                "refresh_measurement_samples_raw":\n                    refresh_measurement.get("n_raw", ""),\n                "refresh_measurement_samples_used":\n                    refresh_measurement.get("n_used", ""),\n                "refresh_measurement_p05_p95_spread_ms":\n                    refresh_measurement.get("p05_p95_spread_ms", ""),\n                "timing_strategy":\n                    "monotonic_real_flip_calibration_plus_absolute_pacing",\n                "screen_index": cfg.get(\n                    "screen_index"\n                ),\n                "screen_width_px": cfg.get(\n                    "screen_width_px"\n                ),\n                "screen_height_px": cfg.get(\n                    "screen_height_px"\n                ),\n                "screen_x": cfg.get(\n                    "screen_x"\n                ),\n                "screen_y": cfg.get(\n                    "screen_y"\n                ),\n                "physical_width_cm": cfg.get(\n                    "physical_width_cm"\n                ),\n                "physical_height_cm": cfg.get(\n                    "physical_height_cm"\n                ),\n                "distance_cm": cfg.get(\n                    "distance_cm"\n                ),\n                "visual_width_deg": width_deg,\n                "visual_height_deg": height_deg,\n                "ttl_mode": cfg.get(\n                    "ttl_mode"\n                ),\n                "ttl_pin": cfg.get(\n                    "ttl_pin"\n                ),\n                "ttl_strategy":\n                    (\n                        "maxone_ftdi_frame_plus_event_pulses"\n                        if ttl_mode == "maxone_ftdi"\n                        else (\n                            "generic_ftdi_universal_pulse_plus_aux_state"\n                            if ttl_mode == "generic_ftdi"\n                            else "principal_condition_event_pulses"\n                        )\n                    ),\n                "trigger_port_requested":\n                    ttl_port_requested,\n                "trigger_port_resolved":\n                    ttl_port_resolved,\n                "frame_sync_signal":\n                    (\n                        "RTS / MaxOne bit 3"\n                        if ttl_mode == "maxone_ftdi"\n                        else ("RTS# -> FRAME_SYNC + EVENT on UNIVERSAL" if ttl_mode == "generic_ftdi" else "")\n                    ),\n                "event_signal":\n                    (\n                        "TX/BREAK / MaxOne bit 0"\n                        if ttl_mode == "maxone_ftdi"\n                        else ("TX/BREAK -> AUX condition state; UNIVERSAL EVENT pulse remains on RTS" if ttl_mode == "generic_ftdi" else "")\n                    ),\n                "frame_sync_pulses":\n                    frame_sync_pulses,\n                "frame_sync_min_width_ms": 1000.0 * (\n                    GenericTTLFTDI.FRAME_PULSE_MIN_S\n                    if ttl_mode == "generic_ftdi"\n                    else MaxOneFTDI.FRAME_PULSE_MIN_S\n                ),\n                "event_offset_requested_ms": (\n                    1000.0 * event_offset_requested_s\n                    if ttl_mode == "generic_ftdi" else ""\n                ),\n                "event_pulse_min_width_ms": (\n                    1000.0 * event_pulse_width_s\n                    if ttl_mode == "generic_ftdi" else ""\n                ),\n                "pause_marker_aux_width_ms": (\n                    1000.0 * pause_marker_width_s\n                    if ttl_mode == "generic_ftdi" else ""\n                ),\n                "universal_encoding": (\n                    "FRAME pulse + EVENT pulse ~3 ms later"\n                    if ttl_mode == "generic_ftdi" else ""\n                ),\n                "aux_event_encoding": (\n                    "ON/OFF: ON=HIGH, OFF=LOW; other principal conditions alternate HIGH/LOW; protocol pauses: short AUX HIGH marker then LOW; forced LOW at stimulus end"\n                    if ttl_mode == "generic_ftdi" else ""\n                ),\n                "adaptation_rgb": cfg.get("adaptation_rgb", [128, 128, 128]),\n                "idle_rgb": cfg.get("idle_rgb", [0, 0, 0]),\n                "session_metadata":\n                    cfg.get(\n                        "session_metadata",\n                        {}\n                    ),\n                "executed_config": cfg,\n                "result_details": details,\n            }\n\n            snapshot_ok = True\n            snapshot_error = ""\n\n            try:\n                write_json_atomic(\n                    config_snapshot_path,\n                    snapshot\n                )\n            except Exception as snapshot_exc:\n                snapshot_ok = False\n                snapshot_error = (\n                    f"{type(snapshot_exc).__name__}: "\n                    f"{snapshot_exc}"\n                )\n\n            result = {\n                "ok": True,\n                "command_id": command_id,\n                "stimulus_type": stimulus_type,\n                "measured_hz": hz,\n                "refresh_measurement_method":\n                    "perf_counter_median_of_real_win_flip_intervals",\n                "refresh_measurement_samples_raw":\n                    refresh_measurement.get("n_raw", ""),\n                "refresh_measurement_samples_used":\n                    refresh_measurement.get("n_used", ""),\n                "refresh_measurement_p05_p95_spread_ms":\n                    refresh_measurement.get("p05_p95_spread_ms", ""),\n                "resolution": [width_px, height_px],\n                "visual_field_deg":\n                    [width_deg, height_deg],\n                "details": details,\n                "timing_strategy":\n                    "monotonic_real_flip_calibration_plus_absolute_pacing",\n                "timing_target_hz": hz,\n                "pacing_wait_total_s": pacing_wait_total_s,\n                "pacing_late_frame_count": pacing_late_frame_count,\n                "pacing_max_lateness_ms":\n                    1000.0 * pacing_max_lateness_s,\n                "possible_dropped_frames":\n                    possible_drops,\n                "total_frames_logged": global_frame,\n                "run_folder": str(run_dir),\n                "log_file": str(csv_path),\n                "ttl_mode": ttl_mode,\n                "ttl_pin": ttl_pin,\n                "ttl_port_requested": ttl_port_requested,\n                "ttl_port_resolved": ttl_port_resolved,\n                "ttl_strategy":\n                    (\n                        "maxone_ftdi_frame_plus_event_pulses"\n                        if ttl_mode == "maxone_ftdi"\n                        else (\n                            "generic_ftdi_universal_pulse_plus_aux_state"\n                            if ttl_mode == "generic_ftdi"\n                            else "principal_condition_event_pulses"\n                        )\n                    ),\n                "frame_sync_pulses": frame_sync_pulses,\n                "frame_sync_min_width_ms": 1000.0 * (\n                    GenericTTLFTDI.FRAME_PULSE_MIN_S\n                    if ttl_mode == "generic_ftdi"\n                    else MaxOneFTDI.FRAME_PULSE_MIN_S\n                ),\n                "event_offset_requested_ms": (\n                    1000.0 * event_offset_requested_s\n                    if ttl_mode == "generic_ftdi" else ""\n                ),\n                "event_pulse_min_width_ms": (\n                    1000.0 * event_pulse_width_s\n                    if ttl_mode == "generic_ftdi" else ""\n                ),\n                "pause_marker_aux_width_ms": (\n                    1000.0 * pause_marker_width_s\n                    if ttl_mode == "generic_ftdi" else ""\n                ),\n                "universal_encoding": (\n                    "FRAME pulse + EVENT pulse ~3 ms later"\n                    if ttl_mode == "generic_ftdi" else ""\n                ),\n                "aux_event_encoding": (\n                    "ON/OFF: ON=HIGH, OFF=LOW; other principal conditions alternate HIGH/LOW; protocol pauses: short AUX HIGH marker then LOW; forced LOW at stimulus end"\n                    if ttl_mode == "generic_ftdi" else ""\n                ),\n                "frame_sync_signal":\n                    (\n                        "RTS / MaxOne bit 3"\n                        if ttl_mode == "maxone_ftdi"\n                        else ("RTS# -> FRAME_SYNC + EVENT on UNIVERSAL" if ttl_mode == "generic_ftdi" else "")\n                    ),\n                "event_signal":\n                    (\n                        "TX/BREAK / MaxOne bit 0"\n                        if ttl_mode == "maxone_ftdi"\n                        else ("TX/BREAK -> AUX condition state; UNIVERSAL EVENT pulse remains on RTS" if ttl_mode == "generic_ftdi" else "")\n                    ),\n                "ttl_pulse_width_frames": ttl_pulse_width_frames,\n                "ttl_pulse_width_ms_approx":\n                    (\n                        1000.0 * event_pulse_width_s\n                        if ttl_mode == "generic_ftdi"\n                        else (1000.0 / hz) * ttl_pulse_width_frames\n                    ),\n                "ttl_pulses":\n                    sum(\n                        1 for ev in ttl_events\n                        if ev.get("state") == 1\n                    ),\n                "ttl_transitions": len(ttl_events),\n                "ttl_log_file": str(ttl_csv_path),\n                "config_snapshot_file":\n                    (\n                        str(config_snapshot_path)\n                        if snapshot_ok\n                        else ""\n                    ),\n                "config_snapshot_ok":\n                    snapshot_ok,\n                "config_snapshot_error":\n                    snapshot_error,\n                "protocol_name":\n                    (\n                        details.get(\n                            "protocol_name", ""\n                        )\n                        if stimulus_type == "protocol"\n                        else ""\n                    ),\n                "protocol_version":\n                    (\n                        details.get(\n                            "protocol_version", ""\n                        )\n                        if stimulus_type == "protocol"\n                        else ""\n                    ),\n            }\n\n            write_json_atomic(result_path, result)\n            show_solid(IDLE_COLOR)\n            write_json_atomic(status_path, {\n                "state": "ready",\n                "message":\n                    "Pantalla experimental estable · "\n                    "espera segura",\n                "hz": hz,\n                "screen_index": screen_index,\n                "resolution":\n                    [width_px, height_px],\n                "last_command_id": command_id,\n                "last_result_ok": True\n            })\n\n        except KeyboardInterrupt:\n            # Safe abort: TTL LOW and immediate return to gray;\n            # the PsychoPy window remains open.\n            finish_trigger_run("ABORT_LOW", suppress_errors=True)\n\n            try:\n                abort_map.seek(0)\n                abort_map.write(b"\\x00")\n                abort_map.flush()\n            except Exception:\n                pass\n\n            show_solid(IDLE_COLOR)\n            write_json_atomic(result_path, {\n                "ok": False,\n                "aborted": True,\n                "trigger_error": trigger_backend_fault,\n                "ttl_port_resolved": ttl_port_resolved,\n                "frame_sync_pulses": frame_sync_pulses,\n                "command_id": command_id,\n                "stimulus_type":\n                    command_stimulus_type,\n                "protocol_name":\n                    (\n                        protocol_name_context\n                        if command_stimulus_type\n                        == "protocol"\n                        else ""\n                    ),\n                "protocol_version":\n                    (\n                        protocol_version_context\n                        if command_stimulus_type\n                        == "protocol"\n                        else ""\n                    ),\n                "error": "Estímulo abortado."\n            })\n            write_json_atomic(status_path, {\n                "state": "ready",\n                "message":\n                    "Estímulo abortado · espera segura",\n                "hz": hz,\n                "screen_index": screen_index,\n                "last_command_id": command_id\n            })\n\n        except Exception as exc:\n            # Stimulus errors do NOT destroy the PsychoPy window.\n            # Return both trigger outputs to rest even after an I/O/flip error.\n            finish_trigger_run("ERROR_LOW", suppress_errors=True)\n\n            show_solid(IDLE_COLOR)\n            write_json_atomic(result_path, {\n                "ok": False,\n                "command_id": command_id,\n                "stimulus_type":\n                    command_stimulus_type,\n                "protocol_name":\n                    (\n                        protocol_name_context\n                        if command_stimulus_type\n                        == "protocol"\n                        else ""\n                    ),\n                "protocol_version":\n                    (\n                        protocol_version_context\n                        if command_stimulus_type\n                        == "protocol"\n                        else ""\n                    ),\n                "error":\n                    f"{type(exc).__name__}: {exc}",\n                "traceback": traceback.format_exc(),\n                "trigger_error": trigger_backend_fault,\n                "ttl_port_resolved": ttl_port_resolved,\n                "frame_sync_pulses": frame_sync_pulses,\n            })\n            write_json_atomic(status_path, {\n                "state": "ready",\n                "message":\n                    "Error de estímulo · espera segura",\n                "hz": hz,\n                "screen_index": screen_index,\n                "last_command_id": command_id,\n                "last_result_ok": False\n            })\n\n        finally:\n            trigger_stopping = True\n            close_trigger_backend()\n\n        core.wait(0.005)\n\nexcept Exception as exc:\n    # If the PsychoPy process itself fails, the independent safety\n    # window behind it remains gray.\n    try:\n        write_json_atomic(status_path, {\n            "state": "fatal",\n            "message":\n                f"Motor PsychoPy detenido: "\n                f"{type(exc).__name__}: {exc}",\n            "traceback": traceback.format_exc()\n        })\n    except Exception:\n        pass\n\nfinally:\n    try:\n        cleanup = globals().get("close_trigger_backend")\n        if callable(cleanup):\n            cleanup()\n    except Exception:\n        pass\n    try:\n        if abort_map is not None:\n            abort_map.close()\n    except Exception:\n        pass\n    try:\n        if abort_file is not None:\n            abort_file.close()\n    except Exception:\n        pass\n    try:\n        if win is not None:\n            show_solid(IDLE_COLOR)\n            win.close()\n    except Exception:\n        pass'
app = App()
try:
    app.mainloop()
finally:
    try:
        app._restore_global_gamma_safely()
    except Exception:
        pass
